# Phase 8 · baseline · B

In [ ]:
import importlib.util
from importlib.metadata import PackageNotFoundError, version
import os
import platform
import subprocess
import sys
from pathlib import Path

print(f"Python {platform.python_version()}", flush=True)
try:
    installed = version("pyspark")
except PackageNotFoundError:
    installed = None
if installed != "4.3.0":
    subprocess.run([sys.executable, "-m", "pip", "install", "pyspark==4.3.0"], check=True)
if importlib.util.find_spec("pyarrow") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "pyarrow"], check=True)
subprocess.run(["java", "-version"], check=True)
os.environ["PYSPARK_PYTHON"] = sys.executable

In [ ]:
project = Path('/kaggle/temp/mmds_phase8/code')
sources = {'src/ingest/dataset_audit.py': '"""Measure the MMDS source without changing or exporting traffic records."""\n\nimport argparse\nfrom collections import Counter\nimport csv\nfrom datetime import datetime\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport time\nimport unicodedata\n\n\nLABEL_NAMES = {"label", "traffic_label", "class", "attack_type"}\nTIME_NAMES = {"timestamp", "time_stamp", "flow_timestamp"}\nIDENTIFIER_NAMES = {"flow_id", "src_ip", "dst_ip", "source_ip", "destination_ip"}\nNONNEGATIVE = re.compile(r"duration|length|(?:^|_)len(?:_|$)|pkts?|packets?|bytes?|byts?|count|(?:^|_)iat(?:_|$)|flags?|active|idle|(?:^|_)rate(?:_|$)|(?:^|_)ps$")\nSENTINELS = {"infinity", "+infinity", "-infinity", "inf", "+inf", "-inf"}\nBENIGN = "BENIGN"\n# The quantized comparison is limited to these non-identity flow measurements.\nNEAR_FIELDS = {\n    "flow_duration", "tot_fwd_pkts", "tot_bwd_pkts",\n    "totlen_fwd_pkts", "totlen_bwd_pkts", "flow_iat_mean",\n}\n\n\ndef normalize_column(value):\n    value = unicodedata.normalize("NFKC", value).lstrip("\\ufeff").strip().casefold()\n    return re.sub(r"[^\\w]+", "_", value, flags=re.UNICODE).strip("_")\n\n\ndef normalize_label(value):\n    return " ".join(unicodedata.normalize("NFKC", value).split()).casefold() if value is not None else None\n\n\ndef proposed_binary(value):\n    """Only BENIGN has a default mapping; observed attack families need review."""\n    return "benign" if normalize_label(value) == BENIGN.casefold() else "unmapped"\n\n\ndef capture_day(name):\n    match = re.search(r"-(\\d{2}-\\d{2}-\\d{4})_", name)\n    return datetime.strptime(match.group(1), "%d-%m-%Y").date().isoformat() if match else None\n\n\ndef inspect_schema(header):\n    names = [normalize_column(col) for col in header]\n    original_counts = Counter(header)\n    normalized_counts = Counter(names)\n    return {\n        "original": header,\n        "normalized": names,\n        "exact_duplicates": sorted(k for k, n in original_counts.items() if n > 1),\n        "normalized_duplicates": sorted(k for k, n in normalized_counts.items() if n > 1),\n        "whitespace_columns": [i for i, raw in enumerate(header) if raw != raw.strip()],\n        "encoding_columns": [i for i, raw in enumerate(header) if any(ord(c) > 127 for c in raw)],\n        "unsupported_indices": [i for i, name in enumerate(names) if not name or normalized_counts[name] > 1],\n        "label_indices": [i for i, name in enumerate(names) if name in LABEL_NAMES],\n        "timestamp_indices": [i for i, name in enumerate(names) if name in TIME_NAMES],\n    }\n\n\ndef compare_schema(reference, candidate):\n    left, right = reference["normalized"], candidate["normalized"]\n    return {\n        "same_order": left == right,\n        "same_original_header": reference["original"] == candidate["original"],\n        "missing": list((Counter(left) - Counter(right)).elements()),\n        "extra": list((Counter(right) - Counter(left)).elements()),\n        "compatible": Counter(left) == Counter(right)\n        and not reference["unsupported_indices"] and not candidate["unsupported_indices"],\n    }\n\n\ndef read_inventory(root, config):\n    root = Path(root)\n    expected = {f\'{config["directory"]}/{name}\': size for name, size in config["files"].items()}\n    available = {p.relative_to(root).as_posix(): p for p in root.rglob("*") if p.is_file()}\n    names = Counter(path.rsplit("/", 1)[-1] for path in available)\n    items = []\n    for rel in sorted(expected.keys() | available.keys()):\n        path = available.get(rel)\n        name = rel.rsplit("/", 1)[-1]\n        size = None\n        access = "missing"\n        if path is not None:\n            try:\n                size = path.stat().st_size\n                with path.open("rb") as source:\n                    source.read(1)\n                access = "readable"\n            except OSError:\n                access = "unreadable"\n        items.append({\n            "path": rel, "name": name, "capture_day": capture_day(name) if rel in expected else None,\n            "bytes": size, "expected_bytes": expected.get(rel), "access": access,\n            "unexpected": rel not in expected, "duplicate_basename": names[name] > 1,\n            "size_matches": size == expected[rel] if rel in expected else None,\n        })\n    return items\n\n\ndef counts_reconcile(counts, labels=None):\n    return (\n        counts["physical_lines"] == sum(counts[key] for key in ("blank", "headers", "malformed", "parsed"))\n        and counts["malformed"] == sum(counts[key] for key in ("bad_width_or_quotes", "spark_corrupt"))\n        and counts["data_records"] == counts["parsed"] + counts["malformed"]\n        and (labels is None or sum(row["count"] for row in labels) == counts["parsed"])\n    )\n\n\ndef stable_sample(raw, seed, modulus):\n    from pyspark.sql import functions as F\n\n    return F.pmod(F.xxhash64(F.lit(seed), raw), F.lit(modulus)) == 0\n\n\ndef timestamp_columns(raw, formats):\n    from pyspark.sql import functions as F\n\n    value = F.trim(raw)\n    primary_options = [F.try_to_timestamp(value, F.lit(fmt)) for fmt in formats]\n    shadow_options = [\n        F.try_to_timestamp(value, F.lit(fmt.replace("d/M/yyyy", "M/d/yyyy")))\n        for fmt in formats if fmt.startswith("d/M/yyyy")\n    ]\n    primary = F.coalesce(*primary_options)\n    shadow = F.coalesce(*shadow_options)\n    matched = F.coalesce(*[F.when(expr.isNotNull(), F.lit(fmt)) for fmt, expr in zip(formats, primary_options)])\n    return primary, shadow, matched\n\n\ndef numeric_fields(value):\n    from pyspark.sql import functions as F\n\n    stripped = F.trim(value)\n    present = value.isNotNull() & (stripped != "")\n    number = stripped.try_cast("double")\n    nan = present & F.isnan(number)\n    inf = present & (\n        F.lower(stripped).isin(*sorted(SENTINELS)) | F.coalesce(F.abs(number) == float("inf"), F.lit(False))\n    )\n    finite = present & number.isNotNull() & ~nan & ~inf\n    failed = present & number.isNull() & ~inf\n    return present, number, nan, inf, finite, failed\n\n\ndef atomic_json(path, value):\n    stage = path.with_suffix(".tmp")\n    stage.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str) + "\\n", encoding="utf-8")\n    stage.replace(path)\n\n\ndef spark_session(config, scratch):\n    from pyspark.sql import SparkSession\n\n    spark_config = config["spark"]\n    os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")\n    return (\n        SparkSession.builder.appName("mmds-phase-1-dataset-audit")\n        .master(spark_config["master"])\n        .config("spark.driver.memory", spark_config["driver_memory"])\n        .config("spark.local.dir", str(scratch))\n        .config("spark.sql.shuffle.partitions", spark_config["shuffle_partitions"])\n        .config("spark.sql.files.maxPartitionBytes", spark_config["input_partition_bytes"])\n        .config("spark.sql.files.openCostInBytes", 4194304)\n        .config("spark.sql.adaptive.enabled", "true")\n        .config("spark.sql.session.timeZone", "UTC")\n        .config("spark.sql.timestampType", "TIMESTAMP_NTZ")\n        .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")\n        .config("spark.sql.csv.parser.columnPruning.enabled", "false")\n        .config("spark.ui.enabled", "false")\n        .getOrCreate()\n    )\n\n\ndef record_frame(spark, path, header_line, width):\n    from pyspark.sql import functions as F\n\n    raw = F.col("raw")\n    blank = raw.rlike(r"^\\s*$")\n    header = raw == header_line\n    comma_width = F.length(raw) - F.length(F.replace(raw, F.lit(","), F.lit(""))) + 1\n    quoted = F.instr(raw, \'"\') > 0\n    field = r\'(?:"(?:[^"]|"")*"|[^",]*)\'\n    quote_grammar = "^" + field + "(?:," + field + ")*$"\n    masked = F.regexp_replace(raw, r\'"(?:[^"]|"")*"\', \'""\')\n    quoted_width = F.length(masked) - F.length(F.replace(masked, F.lit(","), F.lit(""))) + 1\n    valid = F.when(quoted, raw.rlike(quote_grammar) & (quoted_width == width)).otherwise(comma_width == width)\n    status = F.when(blank, "blank").when(header, "headers").when(~valid, "bad_width_or_quotes")\n    ddl = ", ".join([f"c{i} STRING" for i in range(width)] + ["_corrupt STRING"])\n    options = {\n        "mode": "PERMISSIVE", "columnNameOfCorruptRecord": "_corrupt",\n        "quote": \'"\', "escape": \'"\', "nullValue": "\\u0000", "ignoreLeadingWhiteSpace": "false",\n        "ignoreTrailingWhiteSpace": "false", "unescapedQuoteHandling": "RAISE_ERROR",\n    }\n    physical = spark.read.text(str(path)).toDF("raw")\n    parsed = physical.withColumn("csv", F.when(status.isNull(), F.from_csv(raw, ddl, options)))\n    kind = F.coalesce(status, F.when(F.col("csv._corrupt").isNotNull(), "spark_corrupt"), F.lit("parsed"))\n    return parsed.select("raw", kind.alias("kind"), *[F.col(f"csv.c{i}").alias(f"c{i}") for i in range(width)])\n\n\ndef count_records(frame):\n    counts = {row["kind"]: row["count"] for row in frame.groupBy("kind").count().collect()}\n    summary = {k: counts.get(k, 0) for k in ("blank", "headers", "bad_width_or_quotes", "spark_corrupt", "parsed")}\n    summary["malformed"] = summary["bad_width_or_quotes"] + summary["spark_corrupt"]\n    summary["data_records"] = summary["parsed"] + summary["malformed"]\n    summary["physical_lines"] = summary["data_records"] + summary["headers"] + summary["blank"]\n    summary["repeated_header_lines"] = max(0, summary["headers"] - 1)\n    return summary\n\n\ndef count_if(condition):\n    from pyspark.sql import functions as F\n\n    return F.count(F.when(condition, 1))\n\n\ndef profile_columns(frame, schema, config):\n    from pyspark.sql import functions as F\n\n    sampled = frame.where(stable_sample(F.col("raw"), config["sample_seed"], config["sample_modulus"]))\n    sample_count = sampled.count()\n    # Exact value counts within the small sample; collect only a row per column.\n    elements = [\n        F.struct(F.lit(i).alias("index"), F.col(f"c{i}").alias("value"))\n        for i in range(len(schema["original"]))\n    ]\n    sample_summary = (\n        sampled.select(F.explode(F.array(*elements)).alias("cell")).select("cell.*")\n        .where(F.col("value").isNotNull() & (F.trim("value") != ""))\n        .groupBy("index", "value").count()\n        .groupBy("index").agg(F.sum("count").alias("present"), F.max("count").alias("dominant"))\n    )\n    sample_counts = {row["index"]: (row["present"], row["dominant"]) for row in sample_summary.collect()}\n    profiles = []\n    # Limit the size of each aggregate projection and reuse the on-disk parse.\n    for start in range(0, len(schema["original"]), 24):\n        full = []\n        stop = min(start + 24, len(schema["original"]))\n        for i in range(start, stop):\n            col = F.col(f"c{i}")\n            present, number, nan, inf, finite, failed = numeric_fields(col)\n            measures = {\n                "null": count_if(col.isNull()), "empty_or_whitespace": count_if(col.isNotNull() & (F.trim(col) == "")),\n                "present": count_if(present), "finite_numeric": count_if(finite),\n                "nan": count_if(nan), "infinite": count_if(inf), "numeric_conversion_failures": count_if(failed),\n                "negative": count_if(finite & (number < 0)),\n                "lexical_min": F.min(F.when(present, col)), "lexical_max": F.max(F.when(present, col)),\n            }\n            full.extend(expr.alias(f"c{i}_{key}") for key, expr in measures.items())\n        full_result = frame.agg(*full).first().asDict()\n        for i in range(start, stop):\n            prefix = f"c{i}_"\n            row = {key[len(prefix):]: value for key, value in full_result.items() if key.startswith(prefix)}\n            low, high = row.pop("lexical_min"), row.pop("lexical_max")\n            row["constant_nonempty"] = low == high if row["present"] else None\n            row["sample_present"], top_count = sample_counts.get(i, (0, 0))\n            row["sample_dominant_count"] = top_count\n            row["sample_dominant_fraction"] = top_count / row["sample_present"] if row["sample_present"] else None\n            row["near_constant_sample"] = row["sample_dominant_fraction"] >= 0.99 if row["sample_present"] else None\n            name = schema["normalized"][i]\n            support = row["finite_numeric"] + row["nan"] + row["infinite"]\n            identifier = name in IDENTIFIER_NAMES\n            numeric = support > 0 and support / row["present"] >= 0.95\n            row["type_candidate"] = (\n                "label" if i in schema["label_indices"] else "timestamp" if i in schema["timestamp_indices"]\n                else "unsupported" if i in schema["unsupported_indices"]\n                else "identifier" if identifier else "categorical" if name in {"protocol", "protocol_number"}\n                else "numeric" if numeric else "categorical" if row["present"] else "unobserved"\n            )\n            row["invalid_negative"] = (\n                row["negative"] if numeric and NONNEGATIVE.search(name) and not name.startswith("init_") else None\n            )\n            if row["type_candidate"] != "numeric":\n                row["numeric_conversion_failures"] = None\n            profiles.append(dict(row, index=i, original=schema["original"][i], normalized=name, sample_rows=sample_count))\n    return profiles\n\n\ndef summarize_time(frame, schema, formats, day):\n    from pyspark.sql import functions as F\n\n    results = []\n    for i in schema["timestamp_indices"]:\n        raw = F.col(f"c{i}")\n        primary, shadow, matched = timestamp_columns(raw, formats)\n        stamped = frame.select(raw.alias("source"), primary.alias("parsed"), shadow.alias("shadow"), matched.alias("format"))\n        parsed, shadow, source = F.col("parsed"), F.col("shadow"), F.col("source")\n        present = source.isNotNull() & (F.trim(source) != "")\n        date = F.to_date(parsed)\n        aggregates = stamped.agg(\n            F.count("*").alias("rows"), count_if(~present).alias("missing"), count_if(present & parsed.isNull()).alias("invalid"),\n            count_if(parsed.isNotNull()).alias("parsed"), F.min(parsed).alias("earliest"), F.max(parsed).alias("latest"),\n            count_if(shadow.isNotNull() & parsed.isNull()).alias("mdy_only"),\n            count_if(shadow.isNotNull() & parsed.isNotNull() & (F.to_date(shadow) != date)).alias("ambiguous"),\n            count_if(date == F.lit(day)).alias("date_matches"),\n            count_if(date.isNotNull() & (date != F.lit(day))).alias("date_mismatches"),\n            count_if(F.to_date(shadow) == F.lit(day)).alias("mdy_date_matches"),\n        ).first().asDict()\n        dates = [row.asDict() for row in stamped.where(parsed.isNotNull()).groupBy(date.alias("date"), "format").count().limit(501).collect()]\n        if len(dates) <= 500:\n            aggregates["date_formats"] = sorted(dates, key=lambda row: (row["date"], row["format"]))\n        results.append(dict(aggregates, index=i, original=schema["original"][i]))\n    return results\n\n\ndef fingerprint_frame(frame, schema, path, config):\n    from pyspark.sql import functions as F\n\n    label = schema["label_indices"]\n    if len(label) != 1:\n        return None\n    columns = sorted(\n        (name, i) for i, name in enumerate(schema["normalized"])\n        if name not in LABEL_NAMES | TIME_NAMES | IDENTIFIER_NAMES and i not in schema["unsupported_indices"]\n    )\n    schema_key = hashlib.sha256(json.dumps([name for name, _ in columns]).encode()).hexdigest()[:16]\n    selected = [(name, i) for name, i in columns if name in NEAR_FIELDS]\n    raw = F.col("raw")\n    near = F.lit(None).cast("string")\n    if len(selected) >= 3:\n        buckets = [F.floor(F.col(f"c{i}").try_cast("double") / (1000 if "duration" in name or "iat" in name else 10))\n                   for name, i in selected]\n        complete = F.lit(True)\n        for bucket in buckets:\n            complete = complete & bucket.isNotNull()\n        near = F.when(stable_sample(raw, config["sample_seed"], config["sample_modulus"]) & complete,\n                      F.sha2(F.to_json(F.array(*buckets)), 256))\n    return frame.select(\n        F.sha2(raw, 256).alias("row_hash"),\n        F.sha2(F.to_json(F.array(*[F.col(f"c{i}") for _, i in columns])), 256).alias("feature_hash"),\n        F.col(f"c{label[0]}").alias("label"), near.alias("near_hash"),\n        F.lit(schema_key).alias("schema_key"), F.lit(path).alias("source"),\n    )\n\n\ndef summarize_fingerprints(spark, paths):\n    from pyspark.sql import functions as F\n\n    rows = spark.read.parquet(*[str(path) for path in paths])\n    duplicate_groups = rows.groupBy("row_hash").count().where(F.col("count") > 1)\n    duplicates = duplicate_groups.agg(F.count("*").alias("groups"), F.sum(F.col("count") - 1).alias("excess")).first().asDict()\n    known = rows.where(F.col("label").isNotNull() & (F.trim(F.col("label")) != ""))\n    conflicts = (known.groupBy("schema_key", "feature_hash")\n                 .agg(F.min("label").alias("first"), F.max("label").alias("last"), F.count("*").alias("rows"))\n                 .where(F.col("first") != F.col("last"))\n                 .agg(F.count("*").alias("groups"), F.sum("rows").alias("rows")).first().asDict())\n    near_sample = rows.where(F.col("near_hash").isNotNull())\n    near = (near_sample.groupBy("schema_key", "near_hash")\n            .agg(F.min("feature_hash").alias("first"), F.max("feature_hash").alias("last"), F.count("*").alias("rows"))\n            .where(F.col("first") != F.col("last"))\n            .agg(F.count("*").alias("groups"), F.sum("rows").alias("rows")).first().asDict())\n    near["eligible_sample_rows"] = near_sample.count()\n    return {\n        "raw_line_duplicates_sha256": {key: value or 0 for key, value in duplicates.items()},\n        "label_conflicting_features_sha256": {key: value or 0 for key, value in conflicts.items()},\n        "near_sample_buckets": {key: value or 0 for key, value in near.items()},\n    }\n\n\ndef exact_duplicate_summary(spark, root, inventory):\n    """Group full parsed string arrays within identical original headers."""\n    from pyspark.sql import functions as F\n\n    schemas = {}\n    complete = []\n    for item in inventory:\n        if item["unexpected"] or item["access"] != "readable" or not item["size_matches"]:\n            raise ValueError("Exact duplicate measurement requires the verified inventory.")\n        path = root / item["path"]\n        with path.open("rb") as source:\n            header_line = source.readline(65536).decode("utf-8").rstrip("\\r\\n").lstrip("\\ufeff")\n        header = tuple(next(csv.reader([header_line], strict=True)))\n        schema_key = schemas.setdefault(header, len(schemas))\n        frame = record_frame(spark, path, header_line, len(header))\n        complete.append(frame.where(F.col("kind") == "parsed").select(\n            F.lit(schema_key).alias("schema"),\n            F.array(*[F.col(f"c{i}") for i in range(len(header))]).alias("values"),\n            F.lit(item["capture_day"]).alias("day"),\n        ))\n    combined = complete[0]\n    for frame in complete[1:]:\n        combined = combined.unionByName(frame)\n    groups = combined.groupBy("schema", "values").agg(\n        F.count("*").alias("count"), F.min("day").alias("first_day"), F.max("day").alias("last_day"),\n    )\n    result = groups.agg(\n        F.sum("count").alias("parsed_rows"),\n        F.count("*").alias("distinct_rows"),\n        count_if(F.col("count") > 1).alias("duplicate_groups"),\n        F.sum(F.col("count") - 1).alias("excess_copies"),\n        count_if(F.col("first_day") != F.col("last_day")).alias("cross_day_groups"),\n    ).first().asDict()\n    return result | {"files": len(complete), "original_schemas": len(schemas)}\n\n\ndef audit_exact_duplicates(config, root, output, scratch):\n    root, output, scratch = map(Path, (root, output, scratch))\n    if not root.is_dir() or any(path.resolve().is_relative_to(root.resolve()) for path in (output, scratch)):\n        raise ValueError("Use a mounted dataset root and separate writable output and scratch directories")\n    output.mkdir(parents=True, exist_ok=True)\n    scratch.mkdir(parents=True, exist_ok=True)\n    inventory = read_inventory(root, config)\n    atomic_json(output / "inventory.json", {\n        "dataset": {key: config[key] for key in ("owner", "slug", "version")},\n        "expected_bytes": sum(config["files"].values()),\n        "actual_total_bytes": sum(item["bytes"] or 0 for item in inventory),\n        "files": inventory,\n    })\n    spark = spark_session(config, scratch)\n    spark.sparkContext.setLogLevel("WARN")\n    try:\n        started = time.monotonic()\n        print("Measuring exact duplicate field arrays across the mounted captures.", flush=True)\n        result = exact_duplicate_summary(spark, root, inventory)\n        result["elapsed_seconds"] = round(time.monotonic() - started, 3)\n        result["spark_version"] = spark.version\n        result["spark"] = config["spark"]\n        atomic_json(output / "exact_duplicates.json", result)\n        print(f"Exact duplicate aggregation completed in {result[\'elapsed_seconds\']} s", flush=True)\n    finally:\n        spark.stop()\n\n\ndef measure_file(spark, path, item, output, cache_dir, config, result):\n    from pyspark import StorageLevel\n    from pyspark.sql import functions as F\n\n    with path.open("rb") as source:\n        header_bytes = source.readline(65536)\n    if not header_bytes.endswith(b"\\n"):\n        raise ValueError("Missing header newline or header exceeds 64 KiB")\n    header_line = header_bytes.decode("utf-8").rstrip("\\r\\n")\n    schema = inspect_schema(next(csv.reader([header_line.lstrip("\\ufeff")], strict=True)))\n    schema["utf8_bom"] = header_bytes.startswith(b"\\xef\\xbb\\xbf")\n    result["schema"] = schema\n    cache_path = cache_dir / (item["name"] + ".parquet")\n    stage = output / (item["name"] + ".json")\n    frame = record_frame(spark, path, header_line.lstrip("\\ufeff"), len(schema["original"])).persist(StorageLevel.DISK_ONLY)\n    try:\n        started = time.monotonic()\n        counts = count_records(frame)\n        if not counts_reconcile(counts) or not counts["headers"]:\n            raise ValueError("Header or row counts failed reconciliation")\n        result["counts"] = counts\n        result["stage_seconds"]["parse_and_count"] = round(time.monotonic() - started, 3)\n        atomic_json(stage, result)\n        valid = frame.where(F.col("kind") == "parsed")\n        started = time.monotonic()\n        if len(schema["label_indices"]) == 1:\n            label_col = F.col(f\'c{schema["label_indices"][0]}\')\n            labels = valid.groupBy(label_col.alias("label")).count().orderBy("label").limit(501).collect()\n            if len(labels) > 500:\n                raise ValueError("Label cardinality exceeds compact audit limit")\n            result["labels"] = [row.asDict() for row in labels]\n            if not counts_reconcile(counts, result["labels"]):\n                raise ValueError("Label counts do not reconcile")\n        result["timestamps"] = summarize_time(valid, schema, config["timestamp_formats"], item["capture_day"])\n        result["stage_seconds"]["labels_and_time"] = round(time.monotonic() - started, 3)\n        atomic_json(stage, result)\n        started = time.monotonic()\n        result["quality"] = profile_columns(valid, schema, config)\n        result["stage_seconds"]["quality"] = round(time.monotonic() - started, 3)\n        atomic_json(stage, result)\n        started = time.monotonic()\n        fingerprints = fingerprint_frame(valid, schema, item["path"], config)\n        if fingerprints is not None:\n            fingerprints.write.mode("overwrite").parquet(str(cache_path))\n            result["fingerprints"] = True\n        result["stage_seconds"]["fingerprints"] = round(time.monotonic() - started, 3)\n        return cache_path if fingerprints is not None else None\n    finally:\n        frame.unpersist(blocking=True)\n\n\ndef audit_dataset(config, root, output, scratch, only_days=None):\n    root, output, scratch = map(Path, (root, output, scratch))\n    if not root.is_dir() or any(path.resolve().is_relative_to(root.resolve()) for path in (output, scratch)):\n        raise ValueError("Use a mounted dataset root and separate writable output and scratch directories")\n    output.mkdir(parents=True, exist_ok=True)\n    scratch.mkdir(parents=True, exist_ok=True)\n    fingerprints_dir = scratch / "fingerprints"\n    fingerprints_dir.mkdir(exist_ok=True)\n    (scratch / "spark").mkdir(exist_ok=True)\n    inventory = read_inventory(root, config)\n    atomic_json(output / "inventory.json", {\n        "dataset": {key: config[key] for key in ("owner", "slug", "version")},\n        "expected_bytes": sum(config["files"].values()),\n        "actual_total_bytes": sum(item["bytes"] or 0 for item in inventory),\n        "files": inventory,\n    })\n    spark = spark_session(config, scratch / "spark")\n    spark.sparkContext.setLogLevel("WARN")\n    run = {\n        "spark": config["spark"] | {\n            "version": spark.version, "timezone": "UTC", "timestamp_type": "TIMESTAMP_NTZ",\n            "executor_cores": spark.sparkContext.defaultParallelism, "spark_application": spark.sparkContext.appName,\n        },\n        "timestamp_formats": config["timestamp_formats"], "files": {},\n    }\n    started = time.monotonic()\n    atomic_json(output / "run_summary.json", run)\n    fingerprints = []\n    try:\n        # Complete the smaller files first so expensive captures do not prevent partial findings.\n        for item in sorted(inventory, key=lambda entry: entry["bytes"] or 0):\n            if item["unexpected"] or (only_days is not None and item["capture_day"] not in only_days):\n                continue\n            name = item["name"]\n            result = {"name": name, "stage_seconds": {}, "status": "partial"}\n            stage = output / (name + ".json")\n            print(f"Audit start {name}", flush=True)\n            try:\n                if item["access"] != "readable" or not item["size_matches"]:\n                    raise ValueError("Missing, inaccessible or changed source file")\n                path = measure_file(spark, root / item["path"], item, output, fingerprints_dir, config, result)\n                if path is not None:\n                    fingerprints.append(path)\n                result["status"] = "complete"\n            except Exception as error:\n                result["error"] = f"{type(error).__name__}: {str(error)[:900]}"\n                print(result["error"], flush=True)\n            atomic_json(stage, result)\n            run["files"][name] = {"status": result["status"], "stage_seconds": result["stage_seconds"]}\n            atomic_json(output / "run_summary.json", run)\n            print(f"Audit {result[\'status\']} {name}: {result.get(\'counts\', {}).get(\'parsed\')} parsed", flush=True)\n\n        if fingerprints:\n            fingerprint_started = time.monotonic()\n            try:\n                run["fingerprint_summary"] = summarize_fingerprints(spark, fingerprints)\n                run["fingerprint_files"] = len(fingerprints)\n            except Exception as error:\n                run["fingerprint_error"] = f"{type(error).__name__}: {str(error)[:900]}"\n                print(run["fingerprint_error"], flush=True)\n            run["fingerprint_seconds"] = round(time.monotonic() - fingerprint_started, 3)\n    finally:\n        run["elapsed_seconds"] = round(time.monotonic() - started, 3)\n        atomic_json(output / "run_summary.json", run)\n        spark.stop()\n    return run\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--config", type=Path, default=Path("configs/dataset.yaml"))\n    parser.add_argument("--root", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--scratch", type=Path, required=True)\n    parser.add_argument("--day", action="append", help="Audit only this ISO capture day; repeat for more days.")\n    parser.add_argument("--exact-duplicates", action="store_true", help="Run only the exact duplicate aggregation.")\n    args = parser.parse_args()\n    config = json.loads(args.config.read_text())\n    if args.exact_duplicates:\n        audit_exact_duplicates(config, args.root, args.output, args.scratch)\n    else:\n        audit_dataset(config, args.root, args.output, args.scratch, args.day)\n\n\nif __name__ == "__main__":\n    main()\n', 'src/ingest/schema_contract.py': '"""Versioned schema, record identity and strict Phase 1 source reconciliation."""\n\nimport csv\nfrom dataclasses import dataclass\nimport json\nfrom pathlib import Path\n\nfrom pyspark.sql import functions as F\n\nfrom src.ingest.dataset_audit import normalize_column, record_frame\n\n\nEXTRA_FIELDS = ("flow_id", "src_ip", "src_port", "dst_ip")\nTEXT_METADATA = ("flow_id", "src_ip", "dst_ip")\nSOURCE_COUNTS = ("parsed", "headers", "blank", "bad_width_or_quotes", "spark_corrupt")\nFEATURE_CONTEXT = ("record_fingerprint", "split_feature_key", "source_day", "temporal_eligibility")\nTARGETS = ("original_label", "normalized_label", "binary_label", "attack_family")\n\n\n@dataclass\nclass DataContract:\n    policy: dict\n\n    @classmethod\n    def load(cls, path):\n        return cls(json.loads(Path(path).read_text(encoding="utf-8")))\n\n    @property\n    def common_values(self):\n        return tuple(name for name in map(normalize_column, self.policy["headers"]["common_80"])\n                     if name not in {"label", "timestamp"})\n\n    @property\n    def feature_names(self):\n        return tuple(name for name in self.common_values if name not in self.policy["excluded_shared_features"])\n\n    def source_variant(self, source_day):\n        return "extended_84" if source_day == self.policy["extended_day"] else "common_80"\n\n    def verified_header(self, path, source_day):\n        variant = self.source_variant(source_day)\n        with path.open("rb") as source:\n            line = source.readline(65536)\n        if not line.endswith(b"\\n"):\n            raise ValueError(f"Missing or oversized header in {path.name}")\n        decoded = line.decode("utf-8-sig").rstrip("\\r\\n")\n        header = next(csv.reader([decoded], strict=True))\n        if header != self.policy["headers"][variant]:\n            raise ValueError(f"Unexpected original header/order in {path.name} (expected {variant})")\n        return variant, decoded, tuple(map(normalize_column, header))\n\n    def label_lookup(self):\n        # Match the exact audited originals. Normalization is for display only.\n        pairs = []\n        for original, values in self.policy["label_mapping"].items():\n            pairs.extend((F.lit(original), F.struct(\n                F.lit(values["normalized"]).alias("normalized"),\n                F.lit(values["binary"]).cast("byte").alias("binary"),\n                F.lit(values["family"]).alias("family"),\n            )))\n        return F.create_map(*pairs)\n\n\ndef source_records(spark, contract, path, source_day):\n    variant, line, names = contract.verified_header(path, source_day)\n    frame = record_frame(spark, path, line, len(names))\n    return variant, frame.select("raw", "kind", *[F.col(f"c{i}").alias(name) for i, name in enumerate(names)])\n\n\ndef record_key(raw, variant):\n    # Use exact UTF-8 source-line text (excluding line break), scoped by header variant.\n    return F.unhex(F.sha2(F.concat(F.lit(variant + "\\u0000"), raw), 256))\n\n\ndef reconcile_source(observed, expected):\n    for name in SOURCE_COUNTS:\n        if observed[name] != expected[name]:\n            raise ValueError(f"Source count mismatch for {name}: {observed[name]} vs {expected[name]}")\n    if observed["physical_lines"] != expected["physical_lines"]:\n        raise ValueError("Source physical lines do not reconcile")\n    return observed | {"repeated_headers": max(0, observed["headers"] - 1)}\n\n\ndef safe_union(frames):\n    """Align only frames that already have matching canonical names and column types."""\n    result = frames[0]\n    expected = {field.name: field.dataType for field in result.schema}\n    for frame in frames[1:]:\n        actual = {field.name: field.dataType for field in frame.schema}\n        if actual != expected:\n            raise ValueError("Canonical frames have different columns or types")\n        result = result.unionByName(frame)\n    return result\n', 'src/ingest/cleaning.py': '"""Phase 2 value-level diagnostics and policy transformations."""\n\nfrom pyspark.sql import Window, functions as F\n\nfrom src.ingest.dataset_audit import NONNEGATIVE, SENTINELS, timestamp_columns\nfrom src.ingest.schema_contract import EXTRA_FIELDS, FEATURE_CONTEXT, TARGETS, TEXT_METADATA, record_key\n\n\ndef issue_rule(item):\n    text, value = item["text"], item["value"]\n    missing = text.isNull() | (F.trim(text) == "")\n    inf = F.lower(F.trim(text)).isin(*sorted(SENTINELS)) | F.coalesce(F.abs(value) == float("inf"), F.lit(False))\n    return (\n        F.when(missing, "source_null")\n        .when(F.isnan(value), "nan")\n        .when(inf & (F.trim(text).startswith("-") | (value < 0)), "negative_infinity")\n        .when(inf, "positive_infinity")\n        .when(value.isNull(), "cast_failure")\n        .when(F.array_contains(item["sentinels"], value), "sentinel")\n        .when(item["nonnegative"] & (value < 0), "invalid_negative")\n    )\n\n\ndef canonical_frame(frame, contract, variant, source_name, source_day, timestamp_formats):\n    numeric_names = tuple(name for name in (*contract.common_values, "src_port") if name in frame.columns)\n    raw_values = F.array(*[F.col(name) for name in numeric_names])\n    config = contract.policy\n    policies = config["numeric_actions"]\n    policy_pairs = [F.lit(v) for pair in policies.items() for v in pair]\n    actions = F.create_map(*policy_pairs)\n    descriptors = F.array(*[\n        F.struct(F.lit(name).alias("field"), F.lit(bool(NONNEGATIVE.search(name) and not name.startswith("init_"))).alias("nonnegative"),\n                 F.array(*[F.lit(value).cast("double") for value in config["sentinel_values"].get(name, [])])\n                 .cast("array<double>").alias("sentinels"))\n        for name in numeric_names\n    ])\n    parsed, _, _ = timestamp_columns(F.col("timestamp"), timestamp_formats)\n    base = (\n        frame.where(F.col("kind") == "parsed")\n        .withColumn("_text", raw_values)\n        .withColumn("_numbers", F.transform("_text", lambda value: F.trim(value).try_cast("double")))\n        .withColumn("_rules", descriptors)\n        .withColumn("_tokens", F.zip_with(F.arrays_zip("_text", "_numbers"), "_rules",\n                                         lambda x, r: F.struct(r["field"].alias("field"), x["_text"].alias("text"),\n                                                              x["_numbers"].alias("value"), r["nonnegative"],\n                                                              r["sentinels"])))\n        .withColumn("_issues", F.transform("_tokens", lambda token: F.struct(\n            token["field"], token["text"].alias("source_text"), issue_rule(token).alias("reason"),\n        )))\n        .withColumn("quality_issues", F.transform(F.filter("_issues", lambda item: item["reason"].isNotNull()),\n                                                 lambda item: F.struct(item["field"], item["source_text"], item["reason"],\n                                                                       actions[item["reason"]].alias("action"))))\n        .withColumn("record_fingerprint", record_key(F.col("raw"), variant))\n        .withColumn("_label", contract.label_lookup()[F.col("label")])\n        .withColumn("parsed_event_time", parsed)\n    )\n    match = F.to_date("parsed_event_time") == F.lit(source_day).cast("date")\n    status = (F.when(F.col("timestamp").isNull() | (F.trim("timestamp") == ""), "missing")\n              .when(F.col("parsed_event_time").isNull(), "unparseable")\n              .when(match, "matched")\n              .otherwise("outside_capture_day"))\n    fields = {name: F.col("_numbers")[i].alias(name) for i, name in enumerate(numeric_names)}\n    for name in EXTRA_FIELDS:\n        if name in TEXT_METADATA:\n            fields[name] = (F.col(name) if name in frame.columns else F.lit(None).cast("string")).alias(name)\n        elif name not in fields:\n            fields[name] = F.lit(None).cast("double").alias(name)\n    missing_fields = [name for name in EXTRA_FIELDS if name not in frame.columns]\n    return base.select(\n        *fields.values(),\n        F.col("label").alias("original_label"), F.col("_label.normalized").alias("normalized_label"),\n        F.col("_label.binary").alias("binary_label"), F.col("_label.family").alias("attack_family"),\n        F.col("timestamp").alias("original_timestamp"), "parsed_event_time",\n        F.lit(source_name).alias("source_file"), F.lit(source_day).alias("source_day"),\n        "record_fingerprint", F.lit(variant).alias("schema_variant"),\n        F.lit(config["version"]).alias("cleaning_policy_version"),\n        F.lit(config["dataset_version"]).cast("int").alias("dataset_version"),\n        status.alias("timestamp_parse_status"), match.alias("timestamp_capture_day_match"),\n        F.coalesce(match, F.lit(False)).alias("temporal_eligibility"),\n        F.array(*[F.lit(name) for name in missing_fields]).cast("array<string>").alias("structural_missing_fields"),\n        "quality_issues",\n    )\n\n\ndef clean_frame(canonical, contract):\n    null_fields = F.transform(F.filter("quality_issues", lambda issue: issue["action"] == "nullify"),\n                              lambda issue: issue["field"])\n    base = canonical.withColumn("_null_fields", null_fields)\n    numeric = {*contract.common_values, "src_port"}\n    return base.select(*[\n        F.when(F.array_contains("_null_fields", name), F.lit(None).cast("double"))\n        .otherwise(F.col(name)).alias(name) if name in numeric else F.col(name)\n        for name in canonical.columns\n    ])\n\n\ndef feature_frame(clean, contract):\n    # Hash the shared cleaned feature vector in fixed contract order, with a policy version prefix.\n    values = F.array(*[F.col(name) for name in contract.feature_names])\n    split_key = F.unhex(F.sha2(F.concat(F.lit(contract.policy["version"] + "\\u0000"), F.to_json(values)), 256))\n    with_key = clean.withColumn("split_feature_key", split_key)\n    return with_key.select(*contract.feature_names, *TARGETS, *FEATURE_CONTEXT)\n\n\ndef exact_groups(frame):\n    return (frame.groupBy("record_fingerprint")\n            .agg(F.count("*").alias("multiplicity"), F.min("source_file").alias("representative_source"))\n            .where(F.col("multiplicity") > 1))\n\n\ndef conflict_groups(frame):\n    return (frame.groupBy("split_feature_key")\n            .agg(F.count("*").alias("rows"), F.count_distinct("original_label").alias("label_count"))\n            .where(F.col("label_count") > 1))\n\n\ndef apply_duplicate_policy(frame, mode):\n    if mode in {"preserve_all", "diagnostic"}:\n        return frame\n    if mode != "exact_representative":\n        raise ValueError(f"Unsupported duplicate policy: {mode}")\n    # Rows tied on source_file and digest are identical visible occurrences in that source.\n    window = Window.partitionBy("record_fingerprint").orderBy("source_file")\n    return frame.withColumn("_representative", F.row_number().over(window)).where("_representative = 1").drop("_representative")\n', 'src/ingest/parquet_pipeline.py': '"""Execute, reconcile and checkpoint the Phase 2 data layers."""\n\nimport argparse\nfrom collections import Counter\nimport csv\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport time\n\nfrom pyspark import StorageLevel\nfrom pyspark.sql import Observation, functions as F\n\nfrom src.ingest.cleaning import apply_duplicate_policy, canonical_frame, clean_frame, conflict_groups, exact_groups, feature_frame\nfrom src.ingest.dataset_audit import atomic_json, count_if, read_inventory, spark_session\nfrom src.ingest.schema_contract import DataContract, SOURCE_COUNTS, reconcile_source, source_records\nfrom src.sql.aggregation_demo import run_aggregation_demo\n\n\nLAYERS = ("raw_aligned", "clean", "feature_ready")\nKINDS = ("nan", "positive_infinity", "negative_infinity", "invalid_negative", "sentinel", "cast_failure", "source_null")\n\n\ndef canonical_metrics():\n    issues = F.col("quality_issues")\n    metrics = [\n        F.count("*").alias("canonical_rows"),\n        count_if(F.col("binary_label") == 0).alias("benign"),\n        count_if(F.col("binary_label") == 1).alias("attack"),\n        count_if(F.col("binary_label").isNull()).alias("unknown_labels"),\n        count_if(F.col("timestamp_parse_status") == "outside_capture_day").alias("timestamp_outlier"),\n        count_if(F.col("timestamp_parse_status") == "unparseable").alias("timestamp_unparseable"),\n        count_if(F.col("timestamp_parse_status") == "missing").alias("timestamp_missing"),\n        count_if(F.col("temporal_eligibility")).alias("temporal_eligible"),\n        count_if(F.size(issues) > 0).alias("quality_affected_rows"),\n        F.coalesce(F.sum(F.size(issues)), F.lit(0)).alias("quality_affected_cells"),\n        count_if(F.size("structural_missing_fields") > 0).alias("structural_missing_rows"),\n        F.coalesce(F.sum(F.size("structural_missing_fields")), F.lit(0)).alias("structural_missing_cells"),\n    ]\n    for reason in KINDS:\n        selected = F.size(F.filter(issues, lambda issue: issue["reason"] == reason))\n        metrics.extend([\n            count_if(selected > 0).alias(reason + "_rows"),\n            F.coalesce(F.sum(selected), F.lit(0)).alias(reason + "_cells"),\n        ])\n    return metrics\n\n\ndef parquet_footers(path):\n    """Verify files and row counts via footer metadata only."""\n    import pyarrow.parquet as parquet\n\n    files = sorted(path.rglob("part-*.parquet"))\n    if not files:\n        raise ValueError(f"No Parquet data files under {path.name}")\n    rows, size = 0, 0\n    codecs = set()\n    for file in files:\n        metadata = parquet.read_metadata(file)\n        rows += metadata.num_rows\n        size += file.stat().st_size\n        for group in range(metadata.num_row_groups):\n            row_group = metadata.row_group(group)\n            codecs.update(row_group.column(i).compression for i in range(row_group.num_columns))\n    if codecs != {"SNAPPY"}:\n        raise ValueError(f"Parquet compression mismatch in {path.name}: {sorted(codecs)}")\n    return {"records": rows, "bytes": size, "files": len(files), "compression": "SNAPPY"}\n\n\ndef write_day(frame, path, number_of_files, expected_rows):\n    started = time.monotonic()\n    (frame.drop("source_day").coalesce(number_of_files).write.mode("overwrite")\n     .option("compression", "snappy").parquet(str(path)))\n    write_s = round(time.monotonic() - started, 3)\n    observed = parquet_footers(path)\n    if observed["records"] != expected_rows:\n        raise ValueError(f"Parquet row-count mismatch at {path.name}")\n    return observed | {"write_seconds": write_s}\n\n\ndef driver_peak_bytes(spark):\n    pid = spark._jvm.java.lang.ProcessHandle.current().pid()\n    status = Path(f"/proc/{pid}/status")\n    if status.is_file():\n        line = next((line for line in status.read_text().splitlines() if line.startswith("VmHWM:")), None)\n        if line is not None:\n            return int(line.split()[1]) * 1024\n    return None\n\n\ndef process_capture(spark, root, stores, contract, source, formats):\n    source_name, day = source["name"], source["capture_day"]\n    path = root / source["path"]\n    expected = source["counts"]\n    start = time.monotonic()\n    variant, frame = source_records(spark, contract, path, day)\n    physical = Observation()\n    measured = frame.observe(\n        physical, F.count("*").alias("physical_lines"),\n        *[count_if(F.col("kind") == name).alias(name) for name in SOURCE_COUNTS],\n    )\n    normalized = Observation()\n    canonical = (canonical_frame(measured, contract, variant, source_name, day, formats)\n                 .observe(normalized, *canonical_metrics()).persist(StorageLevel.DISK_ONLY))\n    target = contract.policy["target_source_bytes_per_part"]\n    parts = max(1, (source["bytes"] + target - 1) // target)\n    try:\n        materialized = canonical.count()\n        parse_seconds = round(time.monotonic() - start, 3)\n        reconciliation = reconcile_source(physical.get, expected)\n        stats = normalized.get\n        if stats["canonical_rows"] != expected["parsed"] or materialized != expected["parsed"]:\n            raise ValueError("Canonical rows failed reconciliation")\n        if stats["unknown_labels"] or stats["benign"] + stats["attack"] != materialized:\n            raise ValueError(f"Unmapped labels in {source_name}: {stats[\'unknown_labels\']}")\n        if stats["timestamp_outlier"] + stats["timestamp_missing"] + stats["timestamp_unparseable"] + stats["temporal_eligible"] != materialized:\n            raise ValueError("Canonical timestamp statuses failed reconciliation")\n        raw_output = write_day(canonical, stores["raw_aligned"] / f"source_day={day}", parts, materialized)\n        clean_output = write_day(clean_frame(canonical, contract), stores["clean"] / f"source_day={day}", parts, materialized)\n    finally:\n        canonical.unpersist(blocking=True)\n    # The third layer is built from the persisted clean data.\n    clean_path = stores["clean"] / f"source_day={day}"\n    feature_source = spark.read.parquet(str(clean_path)).withColumn("source_day", F.lit(day))\n    feature_output = write_day(feature_frame(feature_source, contract), stores["feature_ready"] / f"source_day={day}", parts, materialized)\n    return {\n        "source_file": source_name, "source_day": day, "schema_variant": variant,\n        "source_bytes": source["bytes"], "expected_records": expected["parsed"],\n        "expected_headers": expected["headers"], "read_transform_seconds": parse_seconds,\n        "effective_read_mib_s": round(source["bytes"] / (1024 ** 2) / max(parse_seconds, 0.001), 3),\n        "driver_peak_rss_bytes": driver_peak_bytes(spark),\n        "counts": reconciliation | stats,\n        "layers": dict(zip(LAYERS, (raw_output, clean_output, feature_output))),\n    }\n\n\ndef reconcile_full(summary, expected):\n    counts = summary["totals"]\n    observed = {key: counts[key] for key in ("benign", "attack", "timestamp_outlier")}\n    observed.update(nan=counts["nan_cells"], infinity=counts["positive_infinity_cells"] + counts["negative_infinity_cells"],\n                    invalid_negative=counts["invalid_negative_cells"])\n    if any(observed[key] != expected[key] for key in observed):\n        raise ValueError(f"Phase 1 totals disagree: {observed}")\n    if counts["canonical_rows"] != counts["parsed"] or any(counts["canonical_rows"] != summary["layers"][name]["records"] for name in LAYERS):\n        raise ValueError("Layer totals do not reconcile")\n\n\ndef layer_summary(files, data_root):\n    output = {}\n    for name in LAYERS:\n        metrics = [row["layers"][name] for row in files]\n        output[name] = {\n            "path": (data_root / name).as_posix(), "partition_column": "source_day",\n            "partitions": len(metrics), "format": "parquet", "compression": "SNAPPY",\n            **{key: sum(item[key] for item in metrics) for key in ("records", "bytes", "files")},\n            "write_seconds": round(sum(item["write_seconds"] for item in metrics), 3),\n        }\n    return output\n\n\ndef write_csv(path, rows, fields):\n    with path.open("w", encoding="utf-8", newline="") as stream:\n        writer = csv.DictWriter(stream, fields, lineterminator="\\n")\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef write_evidence(evidence, summary, policy):\n    atomic_json(evidence / "validation_summary.json", summary)\n    atomic_json(evidence / "parquet_layers.json", {\n        key: summary[key] for key in ("status", "dataset", "policy_version", "layers", "duplicate_policy", "duplicate_diagnostics")\n        if key in summary\n    })\n    captured = summary["completed_files"]\n    counts = [{\n        "source_file": item["source_file"], "source_day": item["source_day"],\n        "expected_records": item["expected_records"], "observed_records": item["counts"]["parsed"],\n        "physical_lines": item["counts"]["physical_lines"], "expected_headers": item["expected_headers"],\n        "headers": item["counts"]["headers"], "repeated_headers": item["counts"]["repeated_headers"],\n        "canonical": item["counts"]["canonical_rows"], "clean": item["layers"]["clean"]["records"],\n        "feature_ready": item["layers"]["feature_ready"]["records"], "benign": item["counts"]["benign"],\n        "attack": item["counts"]["attack"], "timestamp_outlier": item["counts"]["timestamp_outlier"],\n    } for item in captured]\n    timing = [{\n        "source_file": item["source_file"], "source_bytes": item["source_bytes"],\n        "read_transform_seconds": item["read_transform_seconds"], "effective_read_mib_s": item["effective_read_mib_s"],\n        "driver_peak_rss_bytes": item["driver_peak_rss_bytes"],\n        **{name + "_write_seconds": item["layers"][name]["write_seconds"] for name in LAYERS},\n    } for item in captured]\n    quality = [{\n        "source_file": item["source_file"], "reason": reason,\n        "affected_rows": item["counts"][reason + "_rows"], "affected_cells": item["counts"][reason + "_cells"],\n        "action": policy["numeric_actions"].get(reason, "structural_null" if reason == "structural_missing" else "all_quality_issues"),\n    } for item in captured for reason in (*KINDS, "structural_missing", "quality_affected")]\n    if counts:\n        write_csv(evidence / "phase2_count_reconciliation.csv", counts, list(counts[0]))\n        write_csv(evidence / "phase2_performance.csv", timing, list(timing[0]))\n        write_csv(evidence / "phase2_quality_actions.csv", quality, list(quality[0]))\n\n\ndef run_duplicate_mode(spark, stores, policy, data_root, expected_excess):\n    mode = policy["duplicate_policy"]\n    if mode == "preserve_all":\n        return {"records_removed_primary": 0}\n    if mode not in {"diagnostic", "exact_representative"}:\n        raise ValueError(f"Unknown duplicate policy {mode}")\n    started = time.monotonic()\n    folder = data_root / "diagnostics"\n    folder.mkdir(exist_ok=True)\n    aligned = spark.read.parquet(str(stores["raw_aligned"])).select("record_fingerprint", "source_file")\n    keys = folder / "exact_groups"\n    exact_groups(aligned).write.mode("overwrite").option("compression", "snappy").parquet(str(keys))\n    exact = spark.read.parquet(str(keys)).agg(F.count("*").alias("groups"),\n                                              F.coalesce(F.sum(F.col("multiplicity") - 1), F.lit(0)).alias("excess")).first().asDict()\n    if expected_excess is not None and exact["excess"] != expected_excess:\n        raise ValueError("Exact raw-line duplicate excess differs from Phase 1")\n    features = spark.read.parquet(str(stores["feature_ready"])).select("split_feature_key", "original_label")\n    conflicts_path = folder / "label_conflicts"\n    conflict_groups(features).write.mode("overwrite").option("compression", "snappy").parquet(str(conflicts_path))\n    conflicts = spark.read.parquet(str(conflicts_path)).agg(F.count("*").alias("groups"),\n        F.coalesce(F.sum("rows"), F.lit(0)).alias("participating_rows")).first().asDict()\n    result = {"records_removed_primary": 0, "exact": exact, "cleaned_feature_conflicts": conflicts,\n              "diagnostics_path": folder.as_posix()}\n    if mode == "exact_representative":\n        clean = spark.read.parquet(str(stores["clean"]))\n        target = data_root / "exact_representatives"\n        (apply_duplicate_policy(clean, mode).write.mode("overwrite").option("compression", "snappy")\n         .partitionBy("source_day").parquet(str(target)))\n        measured = parquet_footers(target)\n        if measured["records"] != sum(parquet_footers(p)["records"] for p in stores["clean"].glob("source_day=*")) - exact["excess"]:\n            raise ValueError("Representative output failed multiplicity reconciliation")\n        result["representatives"] = measured | {"path": target.as_posix()}\n    return result | {"seconds": round(time.monotonic() - started, 3)}\n\n\ndef resource_limits(spark, policy, path):\n    metrics = {"spark_version": spark.version, **policy["spark"], "default_parallelism": spark.sparkContext.defaultParallelism,\n               "cpu_affinity": len(os.sched_getaffinity(0)), "scratch_output_free_bytes_start": shutil.disk_usage(path).free}\n    for name in ("memory.max", "cpu.max"):\n        resource = Path("/sys/fs/cgroup") / name\n        if resource.exists():\n            metrics[name] = resource.read_text().strip()\n    return metrics\n\n\ndef run_pipeline(contract, source_config, phase1, root, output, scratch, only_days=None):\n    root, output, scratch = map(Path, (root, output, scratch))\n    if not root.is_dir() or any(p.resolve().is_relative_to(root.resolve()) for p in (output, scratch)):\n        raise ValueError("The verified mount and writable outputs must be separate")\n    output.mkdir(parents=True, exist_ok=True)\n    scratch.mkdir(parents=True, exist_ok=True)\n    actual = read_inventory(root, source_config)\n    audited = {row["path"]: row for row in phase1["files"]}\n    if len(actual) != len(audited) or any(\n        row["path"] not in audited or row["bytes"] != audited[row["path"]]["bytes"]\n        or row["access"] != "readable" or row["unexpected"] or row["duplicate_basename"] for row in actual\n    ):\n        raise ValueError("Mounted inventory differs from Phase 1")\n    policy = contract.policy\n    data_root = output / policy["version"]\n    stores = {name: data_root / name for name in LAYERS}\n    evidence = output / "evidence"\n    evidence.mkdir(exist_ok=True)\n    spark = spark_session({"spark": policy["spark"]}, scratch)\n    spark.conf.set("spark.sql.sources.partitionColumnTypeInference.enabled", "false")\n    spark.sparkContext.setLogLevel("WARN")\n    summary = {\n        "status": "running", "dataset": phase1["dataset"], "policy_version": policy["version"],\n        "duplicate_policy": policy["duplicate_policy"], "expected_files": len(audited),\n        "resources": resource_limits(spark, policy, output), "completed_files": [], "failures": [],\n        "phase1_conflict_groups": 22129,\n    }\n    started = time.monotonic()\n    totals = Counter()\n    try:\n        sources = sorted(audited.values(), key=lambda file: file["bytes"])\n        for source in sources:\n            day = source["capture_day"]\n            if only_days is not None and day not in only_days:\n                continue\n            try:\n                marker = evidence / f"capture-{day}.json"\n                if marker.is_file():\n                    result = json.loads(marker.read_text())\n                    for name in LAYERS:\n                        metrics = parquet_footers(stores[name] / f"source_day={day}")\n                        if any(result["layers"][name][key] != value for key, value in metrics.items()):\n                            raise ValueError("Existing checkpoint no longer matches its data files")\n                else:\n                    print(f"PHASE2 start {day} {source[\'bytes\']} bytes", flush=True)\n                    result = process_capture(spark, root, stores, contract, source, source_config["timestamp_formats"])\n                    atomic_json(marker, result)\n                summary["completed_files"].append(result)\n                totals.update(result["counts"])\n                print(f"PHASE2 completed {day} {result[\'counts\'][\'canonical_rows\']} rows "\n                      f"in {result[\'read_transform_seconds\']} s read and transform", flush=True)\n            except Exception as error:\n                summary["failures"].append({"source_day": day, "stage": "capture", "type": type(error).__name__})\n                print(f"PHASE2 failed {day}: {type(error).__name__}: {str(error)[:1000]}", flush=True)\n            summary.update(totals=dict(totals), layers=layer_summary(summary["completed_files"], data_root))\n            write_evidence(evidence, summary, policy)\n        full = len(summary["completed_files"]) == len(sources)\n        if full:\n            reconcile_full(summary, policy["expected_full"])\n            stored = stores["feature_ready"] / f"source_day={sources[0][\'capture_day\']}"\n            summary["aggregation_demo"] = run_aggregation_demo(spark.read.parquet(str(stored)))\n            summary["duplicate_diagnostics"] = run_duplicate_mode(\n                spark, stores, policy, data_root, policy["expected_full"]["duplicate_excess"])\n        summary["status"] = "validated" if full else "partial"\n    except Exception as error:\n        summary["status"] = "partial"\n        summary["failures"].append({"stage": "global_validation", "type": type(error).__name__})\n        print(f"PHASE2 validation failed: {type(error).__name__}: {str(error)[:1000]}", flush=True)\n    finally:\n        summary["elapsed_seconds"] = round(time.monotonic() - started, 3)\n        summary["resources"]["driver_peak_rss_bytes"] = driver_peak_bytes(spark)\n        write_evidence(evidence, summary, policy)\n        print(f"PHASE2 final {summary[\'status\']} in {summary[\'elapsed_seconds\']} seconds", flush=True)\n        # Persist compact evidence in logs as well as the notebook output.\n        print("PHASE2_EVIDENCE " + json.dumps(summary, separators=(",", ":")), flush=True)\n        spark.stop()\n    return summary\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--root", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--scratch", type=Path, required=True)\n    parser.add_argument("--day", action="append")\n    args = parser.parse_args()\n    contract = DataContract.load("configs/cleaning.yaml")\n    load = lambda path: json.loads(Path(path).read_text())\n    summary = run_pipeline(contract, load("configs/dataset.yaml"), load("manifests/dataset_inventory.json"),\n                           args.root, args.output, args.scratch, args.day)\n    raise SystemExit(0 if summary["status"] == "validated" else 1)\n\n\nif __name__ == "__main__":\n    main()\n', 'src/sql/aggregation_demo.py': '"""Compare equivalent SQL, DataFrame and RDD aggregations on a bounded sample."""\n\nfrom pyspark.sql import functions as F\n\n\ndef run_aggregation_demo(features):\n    sample = (features.select("protocol", "binary_label", "record_fingerprint")\n              .where(F.pmod(F.xxhash64("record_fingerprint"), F.lit(1024)) == 0)\n              .select("protocol", "binary_label").cache())\n    sample.createOrReplaceTempView("phase2_demo_sample")\n    try:\n        sql = features.sparkSession.sql(\n            "SELECT protocol, count(*) AS records, sum(binary_label) AS attacks "\n            "FROM phase2_demo_sample GROUP BY protocol")\n        dataframe = sample.groupBy("protocol").agg(F.count("*").alias("records"), F.sum("binary_label").alias("attacks"))\n        rdd = (sample.rdd.map(lambda row: (row.protocol, (1, row.binary_label)))\n               .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])))\n        result = {row.protocol: (row.records, row.attacks) for row in sql.collect()}\n        if result != {row.protocol: (row.records, row.attacks) for row in dataframe.collect()} or result != dict(rdd.collect()):\n            raise ValueError("SQL, DataFrame and RDD aggregation disagree")\n        if not result:\n            raise ValueError("The demonstration sample is empty")\n        return {\n            "sample_rule": "smallest capture by source bytes; pmod(xxhash64(record_fingerprint), 1024) == 0",\n            "rows": sum(count for count, _ in result.values()),\n            "groups": [{"protocol": key, "records": value[0], "attacks": value[1]}\n                       for key, value in sorted(result.items())],\n            "sql_plan": sql._jdf.queryExecution().executedPlan().toString(),\n            "dataframe_plan": dataframe._jdf.queryExecution().executedPlan().toString(),\n            "rdd_dag": rdd.toDebugString().decode("utf-8"),\n        }\n    finally:\n        features.sparkSession.catalog.dropTempView("phase2_demo_sample")\n        sample.unpersist()\n', 'src/sql/traffic_analytics.py': '"""Validated input and reusable SQL aggregations for Phase 3."""\n\nfrom collections import Counter\nimport csv\nfrom hashlib import sha256\nimport json\nimport math\nfrom pathlib import Path\nimport re\nimport time\n\nfrom pyspark.sql import functions as F\n\nfrom src.ingest.dataset_audit import atomic_json\nfrom src.ingest.parquet_pipeline import parquet_footers\nfrom src.ingest.schema_contract import FEATURE_CONTEXT, TARGETS\n\n\ndef identifier(name):\n    if not re.fullmatch(r"[a-z][a-z0-9_]*", name):\n        raise ValueError(f"Invalid configured SQL identifier {name!r}")\n    return name\n\n\ndef locate_producer(input_root, expected_sha):\n    relative = "mmds_phase2/evidence/validation_summary.json"\n    root = Path(input_root)\n    candidates = set(root.rglob(relative))\n    for depth in (1, 2, 3, 4):\n        candidates.update(root.glob("*/" * depth + relative))\n    matched = {path.resolve(): path for path in sorted(candidates)\n               if sha256(path.read_bytes()).hexdigest() == expected_sha}\n    if len(matched) != 1:\n        raise ValueError("Expected one mounted input matching the Phase 2 receipt")\n    return next(iter(matched.values()))\n\n\ndef require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef label_grouping(frame):\n    return frame.groupBy("source_day", *TARGETS, "temporal_eligibility").count()\n\n\ndef compact(frame, maximum):\n    rows = frame.limit(maximum + 1).collect()\n    require(len(rows) <= maximum, "Aggregate exceeds compact output limit")\n    def scalar(value):\n        if isinstance(value, float):\n            return value if math.isfinite(value) else None\n        return value\n\n    return [{k: scalar(v) for k, v in row.asDict().items()} for row in rows]\n\n\ndef validate_input(spark, receipt, manifest, policy):\n    summary = json.loads(receipt.read_text())\n    expected_sha = manifest["independent_mount"]["summary_sha256"]\n    require(sha256(receipt.read_bytes()).hexdigest() == expected_sha, "Phase 2 receipt checksum differs")\n    require(summary["status"] == "validated" and summary["dataset"] == manifest["dataset"], "Wrong producer dataset")\n    require(summary["policy_version"] == manifest["policy_version"] == policy["version"], "Wrong cleaning policy")\n    require(summary["totals"] == manifest["totals"], "Wrong producer totals")\n    root = receipt.parent.parent / summary["policy_version"]\n    frames = {}\n    evidence = {}\n    for name in ("feature_ready", "clean", "raw_aligned"):\n        directory = root / name\n        actual = parquet_footers(directory)\n        expected = manifest["layers"][name]\n        days = sorted(path.name.split("=", 1)[1] for path in directory.glob("source_day=*") if path.is_dir())\n        require(all(expected[key] == value for key, value in actual.items()), f"{name} is unreadable or differs from manifest")\n        require(len(days) == expected["partitions"] == 10, f"{name} partition count differs")\n        require(days == sorted(row["source_day"] for row in summary["completed_files"]), f"{name} day list differs")\n        frames[name] = spark.read.parquet(str(directory))\n        evidence[name] = actual | {"path": str(directory), "partitions": len(days)}\n    features, clean = frames["feature_ready"], frames["clean"]\n    require(set(features.columns) == set(manifest["feature_columns"]) | set(FEATURE_CONTEXT) | set(TARGETS), "Feature schema differs")\n    for frame in (features, clean):\n        types = dict(frame.dtypes)\n        require(all(types.get(name) == "double" for name in manifest["feature_columns"]), "Feature types differ")\n        require(types.get("source_day") == "string" and types.get("binary_label") == "tinyint", "Target or partition type differs")\n        require(types.get("record_fingerprint") == "binary" and types.get("temporal_eligibility") == "boolean", "Provenance types differ")\n    require(dict(clean.dtypes).get("parsed_event_time") == "timestamp_ntz", "Wall-clock timestamp is unavailable")\n    require({row[0] for row in clean.select("cleaning_policy_version").distinct().collect()} == {policy["version"]}, "Stored cleaning policy differs")\n    clean_groups = compact(label_grouping(clean).orderBy("source_day", "original_label", "temporal_eligibility"), 500)\n    feature_groups = compact(label_grouping(features).orderBy("source_day", "original_label", "temporal_eligibility"), 500)\n    require(clean_groups == feature_groups, "Layer targets or temporal flags differ")\n    expected_totals = summary["totals"]\n    labels, day_counts = Counter(), Counter()\n    ineligible = 0\n    for row in clean_groups:\n        label = policy["label_mapping"].get(row["original_label"])\n        require(label is not None and (row["normalized_label"], row["binary_label"], row["attack_family"]) ==\n                (label["normalized"], label["binary"], label["family"]), "Observed label mapping differs")\n        require(row["temporal_eligibility"] in (True, False), "Null temporal flag")\n        labels[row["binary_label"]] += row["count"]\n        day_counts[row["source_day"]] += row["count"]\n        if not row["temporal_eligibility"]:\n            ineligible += row["count"]\n    require(dict(labels) == {0: expected_totals["benign"], 1: expected_totals["attack"]}, "Binary counts differ")\n    require(sum(labels.values()) == expected_totals["canonical_rows"] == 16232943 and ineligible == 14, "Row or temporal counts differ")\n    require(dict(day_counts) == {row["source_day"]: row["counts"]["canonical_rows"] for row in summary["completed_files"]}, "Per-day counts differ")\n    bad_time = clean.where(\n        ~F.col("temporal_eligibility").eqNullSafe(\n            F.col("parsed_event_time").isNotNull() & (F.to_date("parsed_event_time") == F.col("source_day").cast("date"))))\n    require(bad_time.limit(1).count() == 0, "Temporal eligibility disagrees with stored event time")\n    return frames, root, {\n        "status": "validated", "dataset": summary["dataset"], "phase2_receipt_sha256": expected_sha,\n        "policy_version": summary["policy_version"], "producer": manifest["producer"], "layers": evidence,\n        "records": sum(labels.values()), "benign": labels[0], "attack": labels[1], "temporal_ineligible": ineligible,\n        "source_days": sorted(day_counts), "features": manifest["feature_columns"],\n    }\n\n\ndef finite(column):\n    name = identifier(column)\n    return f"CASE WHEN {name} IS NOT NULL AND NOT isnan({name}) AND abs({name}) < CAST(\'Infinity\' AS DOUBLE) THEN {name} END"\n\n\ndef register_traffic(clean, metrics):\n    """Project only the columns needed by exploratory queries."""\n    selected = set(metrics) - {"total_packets", "total_bytes"}\n    columns = ["source_day", *TARGETS, "temporal_eligibility", "parsed_event_time",\n               *sorted(selected | {"protocol", "dst_port", "fin_flag_cnt"})]\n    projected = (clean.select(*columns, "quality_issues")\n                 .withColumn("quality_cells", F.size("quality_issues"))\n                 .withColumn("invalid_cells", F.size(F.filter("quality_issues", lambda issue: issue["reason"] != "sentinel")))\n                 .drop("quality_issues"))\n    return (\n        projected.withColumn("population", F.when(F.col("binary_label") == 0, "benign").otherwise("attack"))\n        .withColumn("total_packets", F.col("tot_fwd_pkts") + F.col("tot_bwd_pkts"))\n        .withColumn("total_bytes", F.col("totlen_fwd_pkts") + F.col("totlen_bwd_pkts"))\n    )\n\n\nDAILY_SQL = """SELECT source_day, count(*) AS records, count_if(binary_label = 0) AS benign,\n count_if(binary_label = 1) AS attacks, avg(CAST(binary_label AS DOUBLE)) AS attack_share,\n sum(total_packets) AS packet_count, sum(total_bytes) AS byte_count\n FROM traffic GROUP BY source_day ORDER BY source_day"""\n\nTOTAL_SQL = """SELECT population, count(*) AS records, sum(total_packets) AS packet_count,\n sum(total_bytes) AS byte_count, count(total_packets) AS packet_observations,\n count(total_bytes) AS byte_observations FROM traffic GROUP BY population ORDER BY population"""\n\nFAMILY_SQL = """SELECT attack_family, count(*) AS records, count(*) / sum(count(*)) OVER () AS share_of_attacks,\n count(DISTINCT source_day) AS capture_days,\n count_if(quality_cells > 0) AS quality_affected_rows, avg(CAST(quality_cells > 0 AS DOUBLE)) AS quality_affected_share,\n count_if(invalid_cells > 0) AS invalid_affected_rows, avg(CAST(invalid_cells > 0 AS DOUBLE)) AS invalid_affected_share\n FROM traffic WHERE binary_label = 1 GROUP BY attack_family ORDER BY records DESC, attack_family"""\n\nLABEL_SQL = """SELECT attack_family, original_label, count(*) AS records, count(DISTINCT source_day) AS capture_days\n FROM traffic GROUP BY attack_family, original_label ORDER BY attack_family, original_label"""\n\nPROTOCOL_SQL = """SELECT population, attack_family, protocol, count(*) AS records\n FROM traffic GROUP BY population, attack_family, protocol ORDER BY population, attack_family, protocol"""\n\nPORT_SQL = """WITH grouped AS (\n SELECT population, coalesce(attack_family, \'All families\') AS attack_family,\n CASE WHEN grouping(attack_family) = 1 THEN \'population\' ELSE \'family\' END AS scope,\n dst_port, count(*) AS records FROM traffic\n GROUP BY GROUPING SETS ((population, attack_family, dst_port), (population, dst_port))),\n ranked AS (SELECT *, row_number() OVER (PARTITION BY scope, population, attack_family ORDER BY records DESC, dst_port) AS rank,\n sum(records) OVER (PARTITION BY scope, population, attack_family) AS population_records FROM grouped)\n SELECT * FROM ranked WHERE rank <= {limit} ORDER BY scope, population, attack_family, rank"""\n\ndef feature_statistics_sql(features, group, accuracy):\n    group = identifier(group)\n    terms = []\n    for feature in features:\n        x = finite(feature)\n        terms.append(\n            f"named_struct(\'feature\', \'{feature}\', \'valid\', count({x}), \'population_count\', count(*), "\n            f"\'mean\', avg({x}), \'stddev\', stddev_samp({x}), \'min\', min({x}), \'max\', max({x}), "\n            f"\'quantiles\', percentile_approx({x}, array(0.1D, 0.5D, 0.9D, 0.99D), {int(accuracy)}))"\n        )\n    return (f"SELECT {group}, m.* FROM (SELECT {group}, explode(array({\', \'.join(terms)})) AS m "\n            f"FROM traffic GROUP BY {group}) ORDER BY {group}, m.feature")\n\n\ndef missing_sql(features):\n    terms = [\n        f"named_struct(\'feature\', \'{name}\', \'missing\', count(*) - count({finite(name)}), \'records\', count(*))"\n        for name in features\n    ]\n    return f"SELECT m.* FROM (SELECT explode(array({\', \'.join(terms)})) AS m FROM features) ORDER BY m.feature"\n\n\ndef correlations_sql(pairs):\n    terms = []\n    for x, y in pairs:\n        a, b = finite(x), finite(y)\n        terms.append(\n            f"named_struct(\'x\', \'{x}\', \'y\', \'{y}\', \'pairs\', count_if(({a}) IS NOT NULL AND ({b}) IS NOT NULL), "\n            f"\'pearson\', corr({a}, {b}))"\n        )\n    return (f"SELECT population, m.* FROM (SELECT population, explode(array({\', \'.join(terms)})) AS m "\n            "FROM traffic GROUP BY population) ORDER BY population, m.x, m.y")\n\n\ndef write_rows(destination, rows):\n    if not rows:\n        raise ValueError(f"Unexpected empty result {destination.name}")\n    stage = destination.with_suffix(".tmp")\n    with stage.open("w", encoding="utf-8", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]), lineterminator="\\n")\n        writer.writeheader()\n        writer.writerows({key: json.dumps(value, separators=(",", ":")) if isinstance(value, (list, dict)) else value\n                          for key, value in row.items()} for row in rows)\n    stage.replace(destination)\n\n\ndef timed_sql(spark, query, limit, name, output, performance):\n    started = time.monotonic()\n    rows = compact(spark.sql(query), limit)\n    seconds = round(time.monotonic() - started, 6)\n    write_rows(output / f"phase3_{name}.csv", rows)\n    performance.append({"workload": name, "api": "SQL", "iteration": 1, "seconds": seconds, "result_rows": len(rows)})\n    print(f"PHASE3 query {name}: {len(rows)} result rows in {seconds:.3f} s", flush=True)\n    return rows\n\n\ndef write_checkpoint(output, validation, completed, performance):\n    atomic_json(output / "phase3_validation.json", validation | {"completed_queries": list(completed)})\n    if performance:\n        write_rows(output / "phase3_query_performance.csv", performance)\n', 'src/ml/feature_registry.py': '"""Phase 4 predictor contract and the predictor-equivalence key."""\n\nfrom pyspark.sql import functions as F\n\n\nMETA = ("source_day", "binary_label", "attack_family", "original_label", "temporal_eligibility")\nFORBIDDEN = {\n    *META, "normalized_label", "dst_port", "src_port", "src_ip", "dst_ip", "flow_id",\n    "source_file", "parsed_event_time", "original_timestamp", "record_fingerprint",\n    "split_feature_key", "predictor_key", "schema_variant", "quality_issues", "protocol",\n    "init_fwd_win_byts", "init_bwd_win_byts",\n}\n\n\ndef registry(config, upstream):\n    names = config["numerical_features"]\n    if len(names) != len(set(names)) or not names or set(names) & FORBIDDEN or not set(names) <= set(upstream):\n        raise ValueError("Invalid Phase 4 predictor registry")\n    return list(names)\n\n\ndef finite(column):\n    value = F.col(column).cast("double")\n    return F.when(value.isNotNull() & ~F.isnan(value) & (F.abs(value) < float("inf")),\n                  F.when(value == 0.0, F.lit(0.0)).otherwise(value))\n\n\ndef predictor_input(frame, names, version):\n    """Hash the actual primary scalar vector before training-only transformations."""\n    data = frame.select(*META, "record_fingerprint", "split_feature_key", *map(lambda name: finite(name).alias(name), names))\n    return data.withColumn("predictor_key", F.unhex(F.sha2(\n        F.concat(F.lit(version + "\\u0000"), F.to_json(F.array(*names))), 256,\n    )))\n', 'src/ml/split_protocols.py': '"""Deterministic disjoint predictor cohorts and ordered capture-day policies."""\n\nfrom pyspark.sql import functions as F\n\n\nSPLITS = ("train", "validation", "test")\n\n\ndef check_days(protocol, design, days):\n    selected = [day for name in SPLITS for day in design[name]]\n    if len(set(selected)) != len(selected) or set(selected) != set(days) or any(not design[name] for name in SPLITS):\n        raise ValueError("Capture days must map exactly once to a nonempty partition")\n    if protocol == "B" and not (max(design["train"]) < min(design["validation"])\n                                and max(design["validation"]) < min(design["test"])):\n        raise ValueError("Chronological holdout must be strictly ordered")\n\n\ndef assign_population(frame, protocol, config, days):\n    if protocol == "A":\n        ratios = config["random_permyriad"]\n        if sum(ratios.values()) != 10000 or any(ratios[name] <= 0 for name in SPLITS):\n            raise ValueError("Invalid random split proportions")\n        index = F.pmod(F.xxhash64(F.lit(config["seed"]), "predictor_key"), F.lit(10000))\n        return frame.withColumn("split_rank", F.when(index < ratios["train"], 0)\n                                .when(index < ratios["train"] + ratios["validation"], 1).otherwise(2))\n    if protocol not in ("B", "C"):\n        raise ValueError(f"Unknown protocol {protocol}")\n    design = config["protocols"][protocol]\n    check_days(protocol, design, days)\n    pairs = [value for rank, name in enumerate(SPLITS) for day in design[name] for value in (F.lit(day), F.lit(rank))]\n    return frame.withColumn("split_rank", F.create_map(*pairs)[F.col("source_day")])\n\n\ndef crossing_groups(frame, key):\n    return (frame.groupBy(key).agg(F.min("split_rank").alias("first_rank"), F.max("split_rank").alias("last_rank"),\n                                   F.count("*").alias("occurrences"))\n            .where("first_rank <> last_rank"))\n\n\ndef mark_retention(assigned, crossing, count):\n    groups = crossing.select("predictor_key", "first_rank")\n    # Bound executor replication and use Spark\'s normal join above the cap.\n    groups = F.broadcast(groups) if count <= 400000 else groups\n    return (assigned.join(groups, "predictor_key", "left")\n            .withColumn("retained", F.col("first_rank").isNull() | (F.col("first_rank") == F.col("split_rank")))\n            .drop("first_rank"))\n\n\ndef group_sample(frame, config):\n    fraction = config["group_sample_permyriad"]\n    if not 0 < fraction <= 10000:\n        raise ValueError("Invalid grouped sampling fraction")\n    return frame.withColumn("sampled", F.pmod(F.xxhash64(F.lit(config["seed"]), F.lit("training-sample"), "predictor_key"),\n                                             F.lit(10000)) < fraction)\n\n\ndef group_summary(groups):\n    row = groups.agg(F.count("*").alias("groups"), F.coalesce(F.sum("occurrences"), F.lit(0)).alias("rows")).first()\n    return row.asDict()\n\n\ndef label_conflicts(frame):\n    return (frame.groupBy("predictor_key")\n            .agg(F.min("binary_label").alias("low"), F.max("binary_label").alias("high"),\n                 F.count("*").alias("occurrences"))\n            .where("low <> high"))\n', 'src/ml/preprocessing.py': '"""Spark MLlib preprocessing with an explicit training-only fit boundary."""\n\nfrom pyspark.ml import Pipeline, PipelineModel\nfrom pyspark.ml.feature import Imputer, StandardScaler, VectorAssembler\nfrom pyspark.sql import functions as F\n\n\ndef fit_preprocessing(train, names):\n    if train.where("split_rank <> 0 OR split_rank IS NULL").limit(1).count():\n        raise ValueError("Preprocessing requires train rows only")\n    metrics = train.agg(*[\n        expr.alias(f"{name}_{key}")\n        for name in names for key, expr in (("min", F.min(name)), ("max", F.max(name)))\n    ]).first().asDict()\n    active = [name for name in names if metrics[f"{name}_min"] is not None and metrics[f"{name}_min"] != metrics[f"{name}_max"]]\n    if not active:\n        raise ValueError("Training population has no varying finite predictors")\n    imputed = [name + "_imputed" for name in active]\n    model = Pipeline(stages=[\n        Imputer(strategy="median", relativeError=0.001, inputCols=active, outputCols=imputed),\n        VectorAssembler(inputCols=imputed, outputCol="features", handleInvalid="error"),\n    ]).fit(train)\n    scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withStd=True,\n                            withMean=False).fit(model.transform(train))\n    scaled = PipelineModel(stages=[*model.stages, scaler])\n    medians = model.stages[0].surrogateDF.first().asDict()\n    return model, scaled, {"active": active, "dropped_train_constant_or_empty": [name for name in names if name not in active],\n                           "training_medians": medians}\n\n\ndef verify_vectors(assembled, active_count):\n    from pyspark.ml.functions import vector_to_array\n\n    for name in ("features", "scaled_features"):\n        values = vector_to_array(name)\n        invalid = (F.size(values) != active_count) | F.exists(\n            values, lambda item: item.isNull() | F.isnan(item) | (F.abs(item) == float("inf")))\n        if assembled.where(invalid).limit(1).count():\n            raise ValueError(f"Nonfinite or invalid {name} supplied to estimators")\n', 'src/ml/baselines.py': '"""Two prespecified binary baselines using training data only."""\n\nfrom pyspark.sql import functions as F\n\n\ndef fit_baselines(train, benign, attacks):\n    if train.where("split_rank <> 0 OR split_rank IS NULL").limit(1).count():\n        raise ValueError("Baseline calibration requires train rows only")\n    threshold = train.where("binary_label = 0").agg(F.percentile_approx("flow_duration", 0.5, 4096)).first()[0]\n    if threshold is None:\n        raise ValueError("Benign reference is unavailable for duration baseline")\n    return {"majority": {"attack_prediction": int(attacks > benign)}, "duration_rule": {"benign_training_median_us": threshold}}\n\n\ndef baseline_score(frame, name, parameters):\n    if name == "majority":\n        score = F.lit(float(parameters["attack_prediction"]))\n    elif name == "duration_rule":\n        score = F.when(F.col("flow_duration") <= parameters["benign_training_median_us"], 1.0).otherwise(0.0)\n    else:\n        raise ValueError(f"Unknown baseline {name}")\n    return frame.withColumn("score", score)\n', 'src/ml/evaluate_supervised.py': '"""Distributed exact confusion counts and bounded score-bin AUC estimates."""\n\nfrom collections import Counter\nimport time\n\nfrom pyspark.ml.functions import vector_to_array\nfrom pyspark.sql import functions as F\n\n\ndef rate(numerator, denominator):\n    return numerator / denominator if denominator else None\n\n\ndef confusion_metrics(tp, tn, fp, fn):\n    recall, specificity = rate(tp, tp + fn), rate(tn, tn + fp)\n    return {"tp": tp, "tn": tn, "fp": fp, "fn": fn, "rows": tp + tn + fp + fn,\n            "accuracy": rate(tp + tn, tp + tn + fp + fn), "precision": rate(tp, tp + fp), "recall": recall,\n            "specificity": specificity, "f1": rate(2 * tp, 2 * tp + fp + fn),\n            "balanced_accuracy": (recall + specificity) / 2 if recall is not None and specificity is not None else None}\n\n\ndef binned_auc(histogram):\n    positives = sum(counts[1] for counts in histogram.values())\n    negatives = sum(counts[0] for counts in histogram.values())\n    if not positives or not negatives:\n        return {"roc_auc": None, "pr_auc": None}\n    grown_pos = grown_neg = 0\n    last_tpr = last_fpr = 0.0\n    roc = pr = 0.0\n    for score_bin in sorted(histogram, reverse=True):\n        counts = histogram[score_bin]\n        grown_neg += counts[0]\n        grown_pos += counts[1]\n        tpr, fpr = grown_pos / positives, grown_neg / negatives\n        precision = grown_pos / (grown_pos + grown_neg)\n        roc += (fpr - last_fpr) * (tpr + last_tpr) / 2\n        pr += (tpr - last_tpr) * precision\n        last_tpr, last_fpr = tpr, fpr\n    return {"roc_auc": roc, "pr_auc": pr}\n\n\ndef estimator_score(model, frame):\n    return model.transform(frame).withColumn("score", vector_to_array("probability")[1])\n\n\ndef evaluate(scored, config):\n    """One Spark action scores rows and gathers only small aggregated histograms."""\n    started = time.monotonic()\n    value = F.col("score")\n    bins = config["score_bins"]\n    invalid = value.isNull() | F.isnan(value) | (value < 0.0) | (value > 1.0)\n    rows = (scored.select("binary_label", "attack_family", "original_label",\n                         F.when(invalid, -1).otherwise(F.least(F.floor(value * bins), F.lit(bins - 1))).alias("score_bin"),\n                         F.when(value >= config["threshold"], 1).otherwise(0).alias("pred"))\n            .groupBy("binary_label", "attack_family", "original_label", "score_bin", "pred").count()\n            .limit(65001).collect())\n    seconds = round(time.monotonic() - started, 3)\n    if len(rows) >= 65001 or any(row.score_bin < 0 or row.binary_label not in (0, 1) for row in rows):\n        raise ValueError("Invalid model scores or aggregate exceeded the compact limit")\n    histogram, conf, groups = {}, Counter(), {}\n    for row in rows:\n        histogram.setdefault(row.score_bin, Counter())[row.binary_label] += row["count"]\n        conf[row.binary_label, row.pred] += row["count"]\n        groups.setdefault((row.binary_label, row.attack_family, row.original_label), Counter())[row.pred] += row["count"]\n    metrics = confusion_metrics(conf[1, 1], conf[0, 0], conf[0, 1], conf[1, 0])\n    return metrics | binned_auc(histogram) | {"prediction_seconds": seconds}, groups\n\n\ndef family_recall(groups, train_families, train_labels, families, original_labels, minimum):\n    observed = {}\n    for (binary, family, label), pred in groups.items():\n        observed.setdefault(("family", family, binary), Counter()).update(pred)\n        observed.setdefault(("original_label", label, binary), Counter()).update(pred)\n    rows = []\n    for level, catalog, train in (("family", families, train_families), ("original_label", original_labels, train_labels)):\n        for name, binary in sorted(catalog.items()):\n            counts = observed.get((level, name, binary), Counter())\n            support = counts[0] + counts[1]\n            rows.append({"level": level, "category": name, "binary_label": binary,\n                         "evaluated_rows": support, "tp": counts[1] if binary else None,\n                         "fn": counts[0] if binary else None, "fp": counts[1] if not binary else None,\n                         "recall": rate(counts[1], support) if binary else None,\n                         "benign_false_positive_rate": rate(counts[1], support) if not binary else None,\n                         "seen_in_training": train.get(name, 0) > 0, "training_rows": train.get(name, 0),\n                         "support_status": "absent" if not support else "low" if support < minimum else "reported"})\n    return rows\n', 'src/ml/model_explanations.py': '"""Map fitted coefficients or split importances to the exact vector order."""\n\n\ndef explain(estimator, model_name, active):\n    if model_name == "logistic_regression":\n        method, values = "scaled_lr_coefficient", model_values(estimator.coefficients)\n    else:\n        method, values = "impurity_split_importance", model_values(estimator.featureImportances)\n    if len(values) != len(active):\n        raise ValueError("Fitted vector and registry dimensions disagree")\n    return [{"feature": feature, "method": method, "value": value}\n            for feature, value in sorted(zip(active, values), key=lambda item: (-abs(item[1]), item[0]))]\n\n\ndef model_values(vector):\n    return [float(value) for value in vector.toArray()]\n', 'src/ml/train_supervised.py': '"""Execute one predeclared Phase 4 protocol over the verified Kaggle input."""\n\nfrom collections import Counter\nimport json\nfrom hashlib import sha256\nfrom pathlib import Path\nimport time\n\nfrom pyspark import StorageLevel\nfrom pyspark.ml import PipelineModel\nfrom pyspark.ml.classification import GBTClassifier, LogisticRegression, RandomForestClassifier\nfrom pyspark.ml.functions import vector_to_array\nfrom pyspark.sql import functions as F\n\nfrom src.ingest.dataset_audit import atomic_json, spark_session\nfrom src.ingest.parquet_pipeline import driver_peak_bytes, parquet_footers, resource_limits\nfrom src.ml.baselines import baseline_score, fit_baselines\nfrom src.ml.evaluate_supervised import estimator_score, evaluate, family_recall\nfrom src.ml.feature_registry import predictor_input, registry\nfrom src.ml.model_explanations import explain\nfrom src.ml.preprocessing import fit_preprocessing, verify_vectors\nfrom src.ml.split_protocols import (\n    SPLITS, assign_population, crossing_groups, group_sample, group_summary, label_conflicts, mark_retention,\n)\nfrom src.sql.traffic_analytics import compact, label_grouping, locate_producer, require, write_rows\n\n\nAUDIT_KEYS = ("predictor_key", "record_fingerprint", "split_feature_key")\n\n\ndef read_json(path):\n    return json.loads(path.read_text())\n\n\ndef verified_input(spark, receipt, manifest, clean_policy):\n    root = receipt.parent.parent\n    evidence = read_json(receipt)\n    require(evidence["status"] == "validated" and evidence["dataset"] == manifest["dataset"], "Producer dataset differs")\n    require(evidence["policy_version"] == clean_policy["version"] == manifest["policy_version"], "Cleaning policy differs")\n    path = root / manifest["policy_version"] / "feature_ready"\n    footer = parquet_footers(path)\n    expected = manifest["layers"]["feature_ready"]\n    require(all(footer[key] == expected[key] for key in footer), "Saved feature-ready Parquet differs")\n    days = sorted(x.name.split("=", 1)[1] for x in path.glob("source_day=*") if x.is_dir())\n    require(len(days) == expected["partitions"] == 10, "Verified input must contain ten days")\n    frame = spark.read.parquet(str(path))\n    extras = {"source_day", "binary_label", "original_label", "normalized_label", "attack_family",\n              "temporal_eligibility", "record_fingerprint", "split_feature_key"}\n    require(set(frame.columns) == set(manifest["feature_columns"]) | extras, "Input schema differs")\n    types = dict(frame.dtypes)\n    require(all(types.get(name) == "double" for name in manifest["feature_columns"]), "Input feature types differ")\n    require(types["binary_label"] == "tinyint" and types["source_day"] == "string", "Label or day type differs")\n    label_totals, day_totals = Counter(), Counter()\n    ineligible = 0\n    for row in compact(label_grouping(frame), 500):\n        mapping = clean_policy["label_mapping"].get(row["original_label"])\n        require(mapping and (mapping["binary"], mapping["family"], mapping["normalized"]) ==\n                (row["binary_label"], row["attack_family"], row["normalized_label"]), "Label integrity check failed")\n        require(row["temporal_eligibility"] in (True, False), "Temporal flag is unavailable")\n        label_totals[row["binary_label"]] += row["count"]\n        day_totals[row["source_day"]] += row["count"]\n        ineligible += 0 if row["temporal_eligibility"] else row["count"]\n    require(dict(label_totals) == {0: 13484708, 1: 2748235} and sum(label_totals.values()) == 16232943,\n            "Input row or label totals differ")\n    require(ineligible == 14, "Timestamp eligibility count differs")\n    require(dict(day_totals) == {row["source_day"]: row["counts"]["canonical_rows"] for row in evidence["completed_files"]},\n            "Capture-day totals differ")\n    return frame, {"path": str(path), "days": days, "totals": dict(label_totals), "rows": footer["records"],\n                   "temporally_flagged": ineligible, "policy_version": manifest["policy_version"],\n                   "receipt_sha256": sha256(receipt.read_bytes()).hexdigest(), "producer": manifest["producer"]}\n\n\ndef split_rows(frame, config):\n    return compact(frame.groupBy("split_rank", "retained", "sampled", "source_day", "binary_label",\n                                 "attack_family", "original_label").count()\n                   .orderBy("split_rank", "retained", "sampled", "source_day", "original_label"), 1500)\n\n\ndef supports(rows, rank, sample_only=False):\n    labels, families, originals = Counter(), Counter(), Counter()\n    for row in rows:\n        if row["split_rank"] == rank and row["retained"] and (not sample_only or row["sampled"]):\n            labels[row["binary_label"]] += row["count"]\n            families[row["attack_family"]] += row["count"]\n            originals[row["original_label"]] += row["count"]\n    return {"labels": dict(labels), "families": dict(families), "original_labels": dict(originals), "rows": sum(labels.values())}\n\n\ndef create_estimator(name, params, config):\n    common = {"labelCol": "binary_label"}\n    if name == "logistic_regression":\n        return LogisticRegression(**common, **params, featuresCol="scaled_features", standardization=False, tol=1e-5)\n    trees = {"featuresCol": "features", "seed": config["seed"], "maxBins": 24, "maxMemoryInMB": 192, **common}\n    if name == "random_forest":\n        return RandomForestClassifier(**trees, **params, featureSubsetStrategy="sqrt", subsamplingRate=1.0)\n    if name == "gradient_boosted_trees":\n        return GBTClassifier(**trees, **params, subsamplingRate=1.0, minInstancesPerNode=30)\n    raise ValueError(f"Unknown estimator {name}")\n\n\ndef select_trial(trials):\n    candidates = [row for row in trials if row["pr_auc"] is not None]\n    if not candidates:\n        raise ValueError("No trial has a defined validation PR-AUC")\n    return max(candidates, key=lambda row: (row["pr_auc"], -row["trial"]))\n\n\ndef save_and_reload(estimator, preprocessor, model_name, dataset, directory):\n    pipeline = PipelineModel(stages=[*preprocessor.stages, estimator])\n    directory.parent.mkdir(parents=True, exist_ok=True)\n    pipeline.write().save(str(directory))\n    reopened = PipelineModel.load(str(directory))\n    require(len(reopened.stages) == len(pipeline.stages), "Reloaded pipeline has a different stage count")\n    def scores(model):\n        return sorted(row[0] for row in model.transform(dataset).select(vector_to_array("probability")[1]).collect())\n\n    probe = dataset.orderBy("predictor_key", "binary_label").limit(24).persist()\n    try:\n        before, after = scores(pipeline), scores(reopened)\n        require(bool(before) and before == after, "Reloaded probabilities differ")\n    finally:\n        probe.unpersist()\n    files = sorted(x for x in directory.rglob("*") if x.is_file() and not x.name.endswith(".crc"))\n    return {"path": str(directory), "model": model_name, "reload_probe_rows": len(after),\n            "model_bytes": sum(x.stat().st_size for x in files), "model_files": len(files),\n            "sha256": {x.relative_to(directory).as_posix(): sha256(x.read_bytes()).hexdigest() for x in files}}\n\n\ndef run(project, input_root, output, scratch, protocol):\n    output.mkdir(parents=True, exist_ok=True)\n    scratch.mkdir(parents=True, exist_ok=True)\n    config = read_json(project / "configs/ml_experiments.yaml")\n    manifest = read_json(project / "manifests/parquet_layers.json")\n    policy = read_json(project / "configs/cleaning.yaml")\n    atomic_json(output / "phase4_config.json", config)\n    spark = spark_session(config, scratch)\n    spark.conf.set("spark.sql.sources.partitionColumnTypeInference.enabled", "false")\n    spark.sparkContext.setLogLevel("WARN")\n    start = time.monotonic()\n    cached = []\n\n    def disk(frame):\n        result = frame.persist(StorageLevel.DISK_ONLY)\n        cached.append(result)\n        return result\n\n    metrics, family_rows, performance = [], [], []\n    summary = {"protocol": protocol, "version": config["version"], "status": "running",\n               "design": config["protocols"][protocol], "resources": resource_limits(spark, config, output),\n               "models": [], "trial_selections": {}}\n    atomic_json(output / "phase4_summary.json", summary)\n\n    def checkpoint():\n        for name, rows in (("metrics", metrics), ("family_recall", family_rows), ("training_performance", performance)):\n            if rows:\n                write_rows(output / f"phase4_{name}.csv", rows)\n        atomic_json(output / "phase4_summary.json", summary)\n\n    try:\n        receipt = locate_producer(input_root, manifest["independent_mount"]["summary_sha256"])\n        frame, summary["input"] = verified_input(spark, receipt, manifest, policy)\n        names = registry(config, manifest["feature_columns"])\n        summary["registered_predictors"] = names\n        print(f"PHASE4 {protocol} verified {summary[\'input\'][\'rows\']} input rows", flush=True)\n        assigned = disk(assign_population(predictor_input(frame, names, config["version"]), protocol, config, summary["input"]["days"]))\n        require(assigned.count() == summary["input"]["rows"], "Assignment lost input rows")\n        groups = disk(crossing_groups(assigned, "predictor_key"))\n        before = group_summary(groups)\n        if protocol == "A":\n            require(before["groups"] == 0, "Grouped random assignment crossed an equivalence boundary")\n        retained = disk(group_sample(mark_retention(assigned, groups, before["groups"]), config))\n        counts = split_rows(retained, config)\n        require(sum(row["count"] for row in counts) == summary["input"]["rows"], "Split partition rows do not reconcile")\n        write_rows(output / "phase4_split_diagnostics.csv", counts)\n        active_source = retained.where("retained")\n        after = {key: group_summary(crossing_groups(active_source, key)) for key in AUDIT_KEYS}\n        require(all(count["groups"] == 0 for count in after.values()), "Retained partition keys overlap")\n        summary["overlap"] = {"predictor_before": before, "after": after,\n                              "purged_rows": sum(row["count"] for row in counts if not row["retained"])}\n        summary["retained_predictor_conflicts"] = group_summary(label_conflicts(active_source))\n        summary["split_support"] = {name: supports(counts, rank) for rank, name in enumerate(SPLITS)}\n        sample_support = supports(counts, 0, True)\n        summary["train_sample_support"] = sample_support\n        require(all(summary["split_support"][name]["rows"] > 0 for name in SPLITS), "A required partition is empty")\n        require(set(sample_support["labels"]) == {0, 1}, "Training sample lacks a binary class")\n        print(f"PHASE4 {protocol} retained split sizes {[(n, summary[\'split_support\'][n][\'rows\']) for n in SPLITS]}", flush=True)\n        assigned.unpersist(blocking=True)\n        groups.unpersist(blocking=True)\n        training = active_source.where("split_rank = 0")\n        sample = disk(training.where("sampled"))\n        calibration = fit_baselines(training, summary["split_support"]["train"]["labels"].get(0, 0),\n                                    summary["split_support"]["train"]["labels"].get(1, 0))\n        summary["baselines"] = calibration\n        eval_sources = {name: active_source.where(F.col("split_rank") == rank) for rank, name in enumerate(SPLITS) if rank}\n        catalogs = {name: values["binary"] for name, values in policy["label_mapping"].items()}\n        families = {values["family"]: values["binary"] for values in policy["label_mapping"].values()}\n\n        def record(scored, model, cohort, split, training_seconds):\n            result, groups = evaluate(scored, config)\n            require(result["rows"] == summary["split_support"][split]["rows"], f"{model} prediction population differs")\n            support = summary["train_sample_support"] if cohort == "group_sample" else summary["split_support"]["train"]\n            stamp = {"protocol": protocol, "model": model, "training_cohort": cohort, "split": split}\n            metrics.append(stamp | {"training_rows": support["rows"], "training_seconds": training_seconds} | result)\n            family_rows.extend(stamp | row for row in family_recall(\n                groups, support["families"], support["original_labels"], families, catalogs, config["rare_family_minimum"]))\n            checkpoint()\n            print(f"PHASE4 {protocol} {model} {cohort} {split}: {result[\'rows\']} scored", flush=True)\n            return result\n\n        # Baseline validation scores are established before fitting complex estimators.\n        for model, params in calibration.items():\n            record(baseline_score(eval_sources["validation"], model, params), model, "full_train_baseline", "validation", 0.0)\n\n        prep_started = time.monotonic()\n        common, scaled, summary["preprocessing"] = fit_preprocessing(sample, names)\n        summary["preprocess_seconds"] = round(time.monotonic() - prep_started, 3)\n        columns = ["split_rank", "sampled", "binary_label", "attack_family", "original_label", "features", "scaled_features"]\n        design = disk(scaled.transform(active_source).select(*columns))\n        require(design.count() == sum(v["rows"] for v in summary["split_support"].values()), "Transform lost retained rows")\n        verify_vectors(design, len(summary["preprocessing"]["active"]))\n        vector_keys = design.select("split_rank", F.unhex(F.sha2(F.to_json(vector_to_array("features")), 256)).alias("vector_key"))\n        summary["post_imputation_overlap"] = group_summary(crossing_groups(vector_keys, "vector_key"))\n        checkpoint()\n        sampled_vector = disk(design.where("split_rank = 0 AND sampled"))\n        sampled_vector.count()\n        vectors = {name: design.where(F.col("split_rank") == rank) for rank, name in enumerate(SPLITS)}\n        winners = []\n        for name, choices in config["trials"].items():\n            runs, fitted = [], {}\n            for i, params in enumerate(choices, 1):\n                fit_start = time.monotonic()\n                model = create_estimator(name, params, config).fit(sampled_vector)\n                fit_s = round(time.monotonic() - fit_start, 3)\n                measured, _ = evaluate(estimator_score(model, vectors["validation"]), config)\n                require(measured["rows"] == summary["split_support"]["validation"]["rows"], "Trial validation rows differ")\n                row = {"protocol": protocol, "model": name, "training_cohort": "group_sample", "trial": i,\n                       "parameters": params, "training_rows": sample_support["rows"], "training_seconds": fit_s,\n                       "validation_inference_seconds": measured["prediction_seconds"], "pr_auc": measured["pr_auc"],\n                       "roc_auc": measured["roc_auc"], "validation_f1": measured["f1"], "selected": False}\n                runs.append(row)\n                performance.append(row)\n                fitted[i] = model\n                checkpoint()\n                print(f"PHASE4 {protocol} {name} trial {i}: fit {fit_s:.3f}s, validation PR-AUC {measured[\'pr_auc\']}", flush=True)\n            selected = select_trial(runs)\n            selected["selected"] = True\n            summary["trial_selections"][name] = {"trial": selected["trial"], "parameters": selected["parameters"]}\n            winners.append((name, "group_sample", fitted[selected["trial"]], selected["training_seconds"]))\n\n        # Additional full-training LR uses fixed sample-selected settings and the same train-only transform.\n        name = "logistic_regression"\n        params = summary["trial_selections"][name]["parameters"]\n        fit_start = time.monotonic()\n        full_model = create_estimator(name, params, config).fit(vectors["train"])\n        full_time = round(time.monotonic() - fit_start, 3)\n        full_result, _ = evaluate(estimator_score(full_model, vectors["validation"]), config)\n        performance.append({"protocol": protocol, "model": name, "training_cohort": "full_train_refit", "trial": 1,\n                            "parameters": params, "training_rows": summary["split_support"]["train"]["rows"],\n                            "training_seconds": full_time, "validation_inference_seconds": full_result["prediction_seconds"],\n                            "pr_auc": full_result["pr_auc"], "roc_auc": full_result["roc_auc"],\n                            "validation_f1": full_result["f1"], "selected": True})\n        winners.append((name, "full_train_refit", full_model, full_time))\n        summary["selection_frozen"] = True\n        checkpoint()\n        print(f"PHASE4 {protocol} selections frozen before testing", flush=True)\n\n        for baseline, params in calibration.items():\n            record(baseline_score(eval_sources["test"], baseline, params), baseline, "full_train_baseline", "test", 0.0)\n        for name, cohort, model, fit_s in winners:\n            for split in ("validation", "test"):\n                record(estimator_score(model, vectors[split]), name, cohort, split, fit_s)\n            stored = save_and_reload(model, scaled if name == "logistic_regression" else common, name,\n                                     sample, output / "models" / f"{name}_{cohort}")\n            stored.update(training_cohort=cohort, training_seconds=fit_s,\n                          parameters=summary["trial_selections"][name]["parameters"],\n                          explanation=explain(model, name, summary["preprocessing"]["active"]))\n            summary["models"].append(stored)\n            checkpoint()\n        summary.update(status="validated", elapsed_seconds=round(time.monotonic() - start, 3))\n        summary["resources"]["driver_peak_rss_bytes"] = driver_peak_bytes(spark)\n        checkpoint()\n        print(f"PHASE4 {protocol} complete: {len(metrics)} evaluation rows, {len(summary[\'models\'])} reloaded pipelines "\n              f"in {summary[\'elapsed_seconds\']}s", flush=True)\n    except Exception as exc:\n        summary.update(status="partial", error=f"{type(exc).__name__}: {str(exc)[:500]}",\n                       elapsed_seconds=round(time.monotonic() - start, 3))\n        checkpoint()\n        raise\n    finally:\n        for item in reversed(cached):\n            item.unpersist()\n        spark.stop()\n', 'configs/cleaning.yaml': '{\n  "version": "phase2-v1",\n  "dataset_version": 1,\n  "headers": {\n    "common_80": [\n      "Dst Port",\n      "Protocol",\n      "Timestamp",\n      "Flow Duration",\n      "Tot Fwd Pkts",\n      "Tot Bwd Pkts",\n      "TotLen Fwd Pkts",\n      "TotLen Bwd Pkts",\n      "Fwd Pkt Len Max",\n      "Fwd Pkt Len Min",\n      "Fwd Pkt Len Mean",\n      "Fwd Pkt Len Std",\n      "Bwd Pkt Len Max",\n      "Bwd Pkt Len Min",\n      "Bwd Pkt Len Mean",\n      "Bwd Pkt Len Std",\n      "Flow Byts/s",\n      "Flow Pkts/s",\n      "Flow IAT Mean",\n      "Flow IAT Std",\n      "Flow IAT Max",\n      "Flow IAT Min",\n      "Fwd IAT Tot",\n      "Fwd IAT Mean",\n      "Fwd IAT Std",\n      "Fwd IAT Max",\n      "Fwd IAT Min",\n      "Bwd IAT Tot",\n      "Bwd IAT Mean",\n      "Bwd IAT Std",\n      "Bwd IAT Max",\n      "Bwd IAT Min",\n      "Fwd PSH Flags",\n      "Bwd PSH Flags",\n      "Fwd URG Flags",\n      "Bwd URG Flags",\n      "Fwd Header Len",\n      "Bwd Header Len",\n      "Fwd Pkts/s",\n      "Bwd Pkts/s",\n      "Pkt Len Min",\n      "Pkt Len Max",\n      "Pkt Len Mean",\n      "Pkt Len Std",\n      "Pkt Len Var",\n      "FIN Flag Cnt",\n      "SYN Flag Cnt",\n      "RST Flag Cnt",\n      "PSH Flag Cnt",\n      "ACK Flag Cnt",\n      "URG Flag Cnt",\n      "CWE Flag Count",\n      "ECE Flag Cnt",\n      "Down/Up Ratio",\n      "Pkt Size Avg",\n      "Fwd Seg Size Avg",\n      "Bwd Seg Size Avg",\n      "Fwd Byts/b Avg",\n      "Fwd Pkts/b Avg",\n      "Fwd Blk Rate Avg",\n      "Bwd Byts/b Avg",\n      "Bwd Pkts/b Avg",\n      "Bwd Blk Rate Avg",\n      "Subflow Fwd Pkts",\n      "Subflow Fwd Byts",\n      "Subflow Bwd Pkts",\n      "Subflow Bwd Byts",\n      "Init Fwd Win Byts",\n      "Init Bwd Win Byts",\n      "Fwd Act Data Pkts",\n      "Fwd Seg Size Min",\n      "Active Mean",\n      "Active Std",\n      "Active Max",\n      "Active Min",\n      "Idle Mean",\n      "Idle Std",\n      "Idle Max",\n      "Idle Min",\n      "Label"\n    ],\n    "extended_84": [\n      "Flow ID",\n      "Src IP",\n      "Src Port",\n      "Dst IP",\n      "Dst Port",\n      "Protocol",\n      "Timestamp",\n      "Flow Duration",\n      "Tot Fwd Pkts",\n      "Tot Bwd Pkts",\n      "TotLen Fwd Pkts",\n      "TotLen Bwd Pkts",\n      "Fwd Pkt Len Max",\n      "Fwd Pkt Len Min",\n      "Fwd Pkt Len Mean",\n      "Fwd Pkt Len Std",\n      "Bwd Pkt Len Max",\n      "Bwd Pkt Len Min",\n      "Bwd Pkt Len Mean",\n      "Bwd Pkt Len Std",\n      "Flow Byts/s",\n      "Flow Pkts/s",\n      "Flow IAT Mean",\n      "Flow IAT Std",\n      "Flow IAT Max",\n      "Flow IAT Min",\n      "Fwd IAT Tot",\n      "Fwd IAT Mean",\n      "Fwd IAT Std",\n      "Fwd IAT Max",\n      "Fwd IAT Min",\n      "Bwd IAT Tot",\n      "Bwd IAT Mean",\n      "Bwd IAT Std",\n      "Bwd IAT Max",\n      "Bwd IAT Min",\n      "Fwd PSH Flags",\n      "Bwd PSH Flags",\n      "Fwd URG Flags",\n      "Bwd URG Flags",\n      "Fwd Header Len",\n      "Bwd Header Len",\n      "Fwd Pkts/s",\n      "Bwd Pkts/s",\n      "Pkt Len Min",\n      "Pkt Len Max",\n      "Pkt Len Mean",\n      "Pkt Len Std",\n      "Pkt Len Var",\n      "FIN Flag Cnt",\n      "SYN Flag Cnt",\n      "RST Flag Cnt",\n      "PSH Flag Cnt",\n      "ACK Flag Cnt",\n      "URG Flag Cnt",\n      "CWE Flag Count",\n      "ECE Flag Cnt",\n      "Down/Up Ratio",\n      "Pkt Size Avg",\n      "Fwd Seg Size Avg",\n      "Bwd Seg Size Avg",\n      "Fwd Byts/b Avg",\n      "Fwd Pkts/b Avg",\n      "Fwd Blk Rate Avg",\n      "Bwd Byts/b Avg",\n      "Bwd Pkts/b Avg",\n      "Bwd Blk Rate Avg",\n      "Subflow Fwd Pkts",\n      "Subflow Fwd Byts",\n      "Subflow Bwd Pkts",\n      "Subflow Bwd Byts",\n      "Init Fwd Win Byts",\n      "Init Bwd Win Byts",\n      "Fwd Act Data Pkts",\n      "Fwd Seg Size Min",\n      "Active Mean",\n      "Active Std",\n      "Active Max",\n      "Active Min",\n      "Idle Mean",\n      "Idle Std",\n      "Idle Max",\n      "Idle Min",\n      "Label"\n    ]\n  },\n  "extended_day": "2018-02-20",\n  "label_mapping": {\n    "Benign": {\n      "normalized": "benign",\n      "binary": 0,\n      "family": "Benign"\n    },\n    "Bot": {\n      "normalized": "bot",\n      "binary": 1,\n      "family": "Bot"\n    },\n    "Brute Force -Web": {\n      "normalized": "brute force -web",\n      "binary": 1,\n      "family": "Web attack"\n    },\n    "Brute Force -XSS": {\n      "normalized": "brute force -xss",\n      "binary": 1,\n      "family": "Web attack"\n    },\n    "DDOS attack-HOIC": {\n      "normalized": "ddos attack-hoic",\n      "binary": 1,\n      "family": "DDoS"\n    },\n    "DDOS attack-LOIC-UDP": {\n      "normalized": "ddos attack-loic-udp",\n      "binary": 1,\n      "family": "DDoS"\n    },\n    "DDoS attacks-LOIC-HTTP": {\n      "normalized": "ddos attacks-loic-http",\n      "binary": 1,\n      "family": "DDoS"\n    },\n    "DoS attacks-GoldenEye": {\n      "normalized": "dos attacks-goldeneye",\n      "binary": 1,\n      "family": "DoS"\n    },\n    "DoS attacks-Hulk": {\n      "normalized": "dos attacks-hulk",\n      "binary": 1,\n      "family": "DoS"\n    },\n    "DoS attacks-SlowHTTPTest": {\n      "normalized": "dos attacks-slowhttptest",\n      "binary": 1,\n      "family": "DoS"\n    },\n    "DoS attacks-Slowloris": {\n      "normalized": "dos attacks-slowloris",\n      "binary": 1,\n      "family": "DoS"\n    },\n    "FTP-BruteForce": {\n      "normalized": "ftp-bruteforce",\n      "binary": 1,\n      "family": "Credential brute force"\n    },\n    "Infilteration": {\n      "normalized": "infilteration",\n      "binary": 1,\n      "family": "Infiltration"\n    },\n    "SQL Injection": {\n      "normalized": "sql injection",\n      "binary": 1,\n      "family": "Web attack"\n    },\n    "SSH-Bruteforce": {\n      "normalized": "ssh-bruteforce",\n      "binary": 1,\n      "family": "Credential brute force"\n    }\n  },\n  "sentinel_values": {\n    "init_fwd_win_byts": [\n      -1\n    ],\n    "init_bwd_win_byts": [\n      -1\n    ]\n  },\n  "numeric_actions": {\n    "nan": "nullify",\n    "positive_infinity": "nullify",\n    "negative_infinity": "nullify",\n    "invalid_negative": "nullify",\n    "sentinel": "nullify",\n    "cast_failure": "typed_null",\n    "source_null": "retain_null"\n  },\n  "excluded_shared_features": [\n    "dst_port"\n  ],\n  "duplicate_policy": "diagnostic",\n  "expected_full": {\n    "benign": 13484708,\n    "attack": 2748235,\n    "nan": 59721,\n    "infinity": 131799,\n    "invalid_negative": 106,\n    "timestamp_outlier": 14,\n    "duplicate_excess": 410707\n  },\n  "spark": {\n    "master": "local[4]",\n    "driver_memory": "8g",\n    "input_partition_bytes": 67108864,\n    "shuffle_partitions": 48\n  },\n  "target_source_bytes_per_part": 134217728\n}\n', 'configs/ml_experiments.yaml': '{\n  "version": "phase4-v1",\n  "seed": 23013,\n  "group_sample_permyriad": 625,\n  "score_bins": 2048,\n  "threshold": 0.5,\n  "rare_family_minimum": 100,\n  "numerical_features": [\n    "flow_duration", "tot_fwd_pkts", "tot_bwd_pkts", "totlen_fwd_pkts", "totlen_bwd_pkts",\n    "pkt_len_mean", "pkt_len_std", "flow_iat_mean", "flow_iat_std", "fwd_iat_mean",\n    "bwd_iat_mean", "flow_pkts_s", "flow_byts_s", "down_up_ratio",\n    "syn_flag_cnt", "ack_flag_cnt", "rst_flag_cnt", "fin_flag_cnt"\n  ],\n  "high_missingness_excluded": ["init_fwd_win_byts", "init_bwd_win_byts"],\n  "optional_categorical": ["protocol"],\n  "random_permyriad": {"train": 7000, "validation": 1500, "test": 1500},\n  "protocols": {\n    "A": {"name": "predictor-group random"},\n    "B": {\n      "name": "chronological capture-day holdout",\n      "train": ["2018-02-14", "2018-02-15", "2018-02-16", "2018-02-20", "2018-02-21", "2018-02-22"],\n      "validation": ["2018-02-23", "2018-02-28"],\n      "test": ["2018-03-01", "2018-03-02"]\n    },\n    "C": {\n      "name": "web-attack capture holdout",\n      "train": ["2018-02-14", "2018-02-15", "2018-02-16", "2018-02-20", "2018-02-21", "2018-03-01", "2018-03-02"],\n      "validation": ["2018-02-28"],\n      "test": ["2018-02-22", "2018-02-23"]\n    }\n  },\n  "trials": {\n    "logistic_regression": [\n      {"maxIter": 12, "regParam": 0.005, "elasticNetParam": 0.0},\n      {"maxIter": 12, "regParam": 0.05, "elasticNetParam": 0.0}\n    ],\n    "random_forest": [\n      {"numTrees": 12, "maxDepth": 5, "minInstancesPerNode": 30},\n      {"numTrees": 20, "maxDepth": 7, "minInstancesPerNode": 30}\n    ],\n    "gradient_boosted_trees": [\n      {"maxIter": 6, "maxDepth": 3, "stepSize": 0.1},\n      {"maxIter": 10, "maxDepth": 4, "stepSize": 0.15}\n    ]\n  },\n  "spark": {"master": "local[4]", "driver_memory": "8g", "shuffle_partitions": 48, "input_partition_bytes": 67108864}\n}\n', 'manifests/parquet_layers.json': '{\n  "status": "validated",\n  "dataset": {\n    "owner": "saisathwikregula",\n    "slug": "mmds-dataset-23013",\n    "version": 1\n  },\n  "policy_version": "phase2-v1",\n  "layers": {\n    "raw_aligned": {\n      "path": "/kaggle/working/mmds_phase2/phase2-v1/raw_aligned",\n      "partition_column": "source_day",\n      "partitions": 10,\n      "format": "parquet",\n      "compression": "SNAPPY",\n      "records": 16232943,\n      "bytes": 2712371594,\n      "files": 55,\n      "write_seconds": 240.044\n    },\n    "clean": {\n      "path": "/kaggle/working/mmds_phase2/phase2-v1/clean",\n      "partition_column": "source_day",\n      "partitions": 10,\n      "format": "parquet",\n      "compression": "SNAPPY",\n      "records": 16232943,\n      "bytes": 2708993526,\n      "files": 55,\n      "write_seconds": 297.806\n    },\n    "feature_ready": {\n      "path": "/kaggle/working/mmds_phase2/phase2-v1/feature_ready",\n      "partition_column": "source_day",\n      "partitions": 10,\n      "format": "parquet",\n      "compression": "SNAPPY",\n      "records": 16232943,\n      "bytes": 3003003600,\n      "files": 53,\n      "write_seconds": 314.859\n    }\n  },\n  "duplicate_policy": "diagnostic",\n  "duplicate_diagnostics": {\n    "records_removed_primary": 0,\n    "exact": {\n      "groups": 107670,\n      "excess": 410707\n    },\n    "cleaned_feature_conflicts": {\n      "groups": 21763,\n      "participating_rows": 710014\n    },\n    "diagnostics_path": "/kaggle/working/mmds_phase2/phase2-v1/diagnostics",\n    "seconds": 72.76\n  },\n  "producer": {\n    "url": "https://www.kaggle.com/code/ramcharan2345/mmds-phase-2-spark-data-engineering",\n    "kernel_version": 1,\n    "status": "COMPLETE"\n  },\n  "resources": {\n    "spark_version": "4.3.0",\n    "master": "local[4]",\n    "driver_memory": "8g",\n    "input_partition_bytes": 67108864,\n    "shuffle_partitions": 48,\n    "default_parallelism": 4,\n    "cpu_affinity": 4,\n    "scratch_output_free_bytes_start": 20940369920,\n    "memory.max": "32212254720",\n    "cpu.max": "400000 100000",\n    "driver_peak_rss_bytes": 5784305664\n  },\n  "totals": {\n    "physical_lines": 16233012,\n    "parsed": 16232943,\n    "headers": 69,\n    "blank": 0,\n    "bad_width_or_quotes": 0,\n    "spark_corrupt": 0,\n    "repeated_headers": 59,\n    "canonical_rows": 16232943,\n    "benign": 13484708,\n    "attack": 2748235,\n    "unknown_labels": 0,\n    "timestamp_outlier": 14,\n    "timestamp_unparseable": 0,\n    "timestamp_missing": 0,\n    "temporal_eligible": 16232929,\n    "quality_affected_rows": 8257114,\n    "quality_affected_cells": 12879783,\n    "structural_missing_rows": 8284195,\n    "structural_missing_cells": 33136780,\n    "nan_rows": 59721,\n    "nan_cells": 59721,\n    "positive_infinity_rows": 95760,\n    "positive_infinity_cells": 131799,\n    "negative_infinity_rows": 0,\n    "negative_infinity_cells": 0,\n    "invalid_negative_rows": 15,\n    "invalid_negative_cells": 106,\n    "sentinel_rows": 8255549,\n    "sentinel_cells": 12688157,\n    "cast_failure_rows": 0,\n    "cast_failure_cells": 0,\n    "source_null_rows": 0,\n    "source_null_cells": 0\n  },\n  "independent_mount": {\n    "layers": {\n      "raw_aligned": {\n        "records": 16232943,\n        "bytes": 2712371594,\n        "benign": 13484708,\n        "attack": 2748235,\n        "files": 55,\n        "partitions": 10\n      },\n      "clean": {\n        "records": 16232943,\n        "bytes": 2708993526,\n        "benign": 13484708,\n        "attack": 2748235,\n        "files": 55,\n        "partitions": 10\n      },\n      "feature_ready": {\n        "records": 16232943,\n        "bytes": 3003003600,\n        "benign": 13484708,\n        "attack": 2748235,\n        "files": 53,\n        "partitions": 10\n      }\n    },\n    "clean_cells_verified_null": 12879783,\n    "clean_quality_affected_rows": 8257114,\n    "clean_quality_reasons": {\n      "sentinel": 12688157,\n      "nan": 59721,\n      "positive_infinity": 131799,\n      "invalid_negative": 106\n    },\n    "clean_timestamp_outliers": 14,\n    "summary_sha256": "1fa00cd2b9c853ee33eaacee8ca7632dbb41f739573326280d6ffbfb2ccff25a",\n    "producer_mount": "/kaggle/input/notebooks/ramcharan2345/mmds-phase-2-spark-data-engineering/mmds_phase2",\n    "elapsed_seconds": 101.019,\n    "url": "https://www.kaggle.com/code/ramcharan2345/mmds-phase-2-parquet-reuse-check",\n    "kernel_version": 1,\n    "status": "COMPLETE"\n  },\n  "feature_columns": [\n    "protocol",\n    "flow_duration",\n    "tot_fwd_pkts",\n    "tot_bwd_pkts",\n    "totlen_fwd_pkts",\n    "totlen_bwd_pkts",\n    "fwd_pkt_len_max",\n    "fwd_pkt_len_min",\n    "fwd_pkt_len_mean",\n    "fwd_pkt_len_std",\n    "bwd_pkt_len_max",\n    "bwd_pkt_len_min",\n    "bwd_pkt_len_mean",\n    "bwd_pkt_len_std",\n    "flow_byts_s",\n    "flow_pkts_s",\n    "flow_iat_mean",\n    "flow_iat_std",\n    "flow_iat_max",\n    "flow_iat_min",\n    "fwd_iat_tot",\n    "fwd_iat_mean",\n    "fwd_iat_std",\n    "fwd_iat_max",\n    "fwd_iat_min",\n    "bwd_iat_tot",\n    "bwd_iat_mean",\n    "bwd_iat_std",\n    "bwd_iat_max",\n    "bwd_iat_min",\n    "fwd_psh_flags",\n    "bwd_psh_flags",\n    "fwd_urg_flags",\n    "bwd_urg_flags",\n    "fwd_header_len",\n    "bwd_header_len",\n    "fwd_pkts_s",\n    "bwd_pkts_s",\n    "pkt_len_min",\n    "pkt_len_max",\n    "pkt_len_mean",\n    "pkt_len_std",\n    "pkt_len_var",\n    "fin_flag_cnt",\n    "syn_flag_cnt",\n    "rst_flag_cnt",\n    "psh_flag_cnt",\n    "ack_flag_cnt",\n    "urg_flag_cnt",\n    "cwe_flag_count",\n    "ece_flag_cnt",\n    "down_up_ratio",\n    "pkt_size_avg",\n    "fwd_seg_size_avg",\n    "bwd_seg_size_avg",\n    "fwd_byts_b_avg",\n    "fwd_pkts_b_avg",\n    "fwd_blk_rate_avg",\n    "bwd_byts_b_avg",\n    "bwd_pkts_b_avg",\n    "bwd_blk_rate_avg",\n    "subflow_fwd_pkts",\n    "subflow_fwd_byts",\n    "subflow_bwd_pkts",\n    "subflow_bwd_byts",\n    "init_fwd_win_byts",\n    "init_bwd_win_byts",\n    "fwd_act_data_pkts",\n    "fwd_seg_size_min",\n    "active_mean",\n    "active_std",\n    "active_max",\n    "active_min",\n    "idle_mean",\n    "idle_std",\n    "idle_max",\n    "idle_min"\n  ]\n}\n', 'src/ml/train_clustering.py': '"""Run a preregistered clustering version against the verified mounted Phase 2 data."""\n\nfrom collections import Counter\nfrom hashlib import sha256\nimport json\nfrom pathlib import Path\nimport time\n\nfrom pyspark import StorageLevel\nfrom pyspark.ml import PipelineModel\nfrom pyspark.ml.functions import vector_to_array\nfrom pyspark.sql import functions as F\n\nfrom src.ingest.dataset_audit import atomic_json, spark_session\nfrom src.ingest.parquet_pipeline import driver_peak_bytes, parquet_footers, resource_limits\nfrom src.ml.cluster_diagnostics import centroid_dominance, choose_k, compare_assignments, fit_kmeans, silhouette\nfrom src.ml.cluster_evolution import occupancy\nfrom src.ml.clustering_features import clustering_registry, fit_transform, verify_finite_vectors\nfrom src.ml.feature_registry import META, finite\nfrom src.ml.novelty_scoring import calibrate, novelty_metrics, squared_distance\nfrom src.ml.split_protocols import assign_population, crossing_groups, group_summary, mark_retention\nfrom src.ml.train_supervised import verified_input\nfrom src.sql.traffic_analytics import compact, locate_producer, require, write_rows\nfrom src.visualization.clustering_charts import draw_phase5\n\n\ndef read(path):\n    return json.loads(path.read_text())\n\n\ndef cohorts(clean, names, phase4, experiment, days, extra_metadata=()):\n    # Exactly the Phase 4 predictor digest, with clean timestamps carried only as metadata.\n    projected = clean.select(*META, "parsed_event_time", "record_fingerprint", *extra_metadata,\n                             *[finite(x).alias(x) for x in names])\n    projected = projected.withColumn("predictor_key", F.unhex(F.sha2(\n        F.concat(F.lit(phase4["version"] + "\\u0000"), F.to_json(F.array(*names))), 256)))\n    if experiment == "G":\n        return projected.withColumn("split_rank", F.lit(0))\n    return assign_population(projected, experiment, phase4, days)\n\n\ndef pick_reference(frame, config):\n    value = F.pmod(F.xxhash64("predictor_key", F.lit(config["seed"]), F.lit("phase5-reference")), F.lit(10000))\n    ranked = frame.where(F.col("split_rank") == 0).withColumn("sample_bucket", value)\n    boundary = config["training_permyriad"]\n    training = ranked.where(F.col("sample_bucket") < boundary)\n    probe = ranked.where((F.col("sample_bucket") >= boundary) & (F.col("sample_bucket") < boundary + config["internal_permyriad"]))\n    return training, probe\n\n\ndef split_counts(data):\n    return compact(data.groupBy("split_rank", "source_day", "binary_label", "attack_family")\n                   .count().orderBy("split_rank", "source_day", "binary_label", "attack_family"), 1000)\n\n\ndef verify_phase4_match(rows, experiment, phase4_receipt):\n    if experiment == "G":\n        return None\n    expected = phase4_receipt["protocols"][experiment]\n    ranks = ("train", "validation", "test")\n    for i, split in enumerate(ranks):\n        families, labels = Counter(), Counter()\n        for row in rows:\n            if row["split_rank"] == i:\n                families[row["attack_family"]] += row["count"]\n                labels[str(row["binary_label"])] += row["count"]\n        support = expected["split_support"][split]\n        require(dict(families) == support["families"] and dict(labels) == support["labels"], "Phase 4 retained cohort differs")\n    return {"protocol": experiment, "kaggle_version": expected["version"], "checked": "matched split rules and family/binary totals",\n            "purged_rows": expected["overlap"]["purged_rows"]}\n\n\ndef profiles(scores, centers, config):\n    return compact(scores.groupBy("cluster").agg(\n        F.count("*").alias("records"), F.count(F.when(F.col("binary_label") == 1, 1)).alias("attacks"),\n        F.avg("score").alias("mean_score"),\n        F.percentile_approx("score", 0.95, config["quantile_accuracy"]).alias("score_p95"),\n        F.avg("flow_duration").alias("mean_duration_us"), F.avg("tot_fwd_pkts").alias("mean_forward_packets"),\n        F.avg("tot_bwd_pkts").alias("mean_backward_packets"), F.avg("totlen_fwd_pkts").alias("mean_forward_bytes"),\n        F.avg("totlen_bwd_pkts").alias("mean_backward_bytes"), F.avg("flow_pkts_s").alias("mean_rate")\n    ).orderBy("cluster"), centers)\n\n\ndef saved_pipeline(preprocessing, model, probe, target):\n    pipeline = PipelineModel(stages=[*preprocessing.stages, model])\n    pipeline.write().save(str(target))\n    recovered = PipelineModel.load(str(target))\n    small = probe.limit(32).cache()\n    try:\n        expected = sorted(row[0] for row in pipeline.transform(small).select("cluster").collect())\n        actual = sorted(row[0] for row in recovered.transform(small).select("cluster").collect())\n        require(expected and actual == expected, "Saved pipeline reload did not reproduce assignments")\n    finally:\n        small.unpersist()\n    files = [p for p in target.rglob("*") if p.is_file() and not p.name.endswith(".crc")]\n    return {"path": str(target), "reload_rows": len(actual),\n            "sha256": {p.relative_to(target).as_posix(): sha256(p.read_bytes()).hexdigest() for p in sorted(files)},\n            "bytes": sum(p.stat().st_size for p in files)}\n\n\ndef run(project, input_root, output, scratch, experiment):\n    config = read(project / "configs/clustering_experiments.yaml")\n    phase4 = read(project / "configs/ml_experiments.yaml")\n    phase4_receipt = read(project / "manifests/phase4_models.json")\n    manifest = read(project / "manifests/parquet_layers.json")\n    policy = read(project / "configs/cleaning.yaml")\n    names = clustering_registry(phase4, manifest["feature_columns"])\n    output.mkdir(parents=True, exist_ok=True)\n    scratch.mkdir(parents=True, exist_ok=True)\n    spark = spark_session(config, scratch)\n    spark.conf.set("spark.sql.sources.partitionColumnTypeInference.enabled", "false")\n    spark.sparkContext.setLogLevel("WARN")\n    started = time.monotonic()\n    cached = []\n\n    def disk(data):\n        result = data.persist(StorageLevel.DISK_ONLY)\n        cached.append(result)\n        return result\n\n    summary = {"experiment": experiment, "config": config, "status": "running",\n               "resources": resource_limits(spark, config, output), "features": names}\n    path = output / "phase5_summary.json"\n    atomic_json(path, summary)\n\n    def emit(name, rows):\n        write_rows(output / f"phase5_{name}.csv", rows)\n        print(f"PHASE5 {experiment} {name}: {len(rows)} rows", flush=True)\n\n    try:\n        receipt = locate_producer(input_root, manifest["independent_mount"]["summary_sha256"])\n        _, summary["input"] = verified_input(spark, receipt, manifest, policy)\n        clean_path = receipt.parent.parent / manifest["policy_version"] / "clean"\n        require(all(manifest["layers"]["clean"][key] == value for key, value in parquet_footers(clean_path).items()),\n                "Clean Parquet metadata differs")\n        clean = spark.read.parquet(str(clean_path))\n        require(dict(clean.dtypes).get("parsed_event_time") == "timestamp_ntz" and set(names) <= set(clean.columns), "Clean schema changed")\n        base = disk(cohorts(clean, names, phase4, experiment, summary["input"]["days"]))\n        require(base.count() == summary["input"]["rows"], "Input split lost records")\n        observed = split_counts(base)\n        label_total = Counter()\n        for row in observed:\n            label_total[row["binary_label"]] += row["count"]\n        require(dict(label_total) == {0: 13484708, 1: 2748235}, "Clean layer binary counts changed")\n        expected_families = read(receipt)["completed_files"]\n        # The feature-ready layer verifies exact original-label mappings before this check.\n        require(len(expected_families) == len({r["source_day"] for r in observed}), "Clean capture-day inventory changed")\n        require(base.where("record_fingerprint IS NULL OR temporal_eligibility IS NULL").limit(1).count() == 0,\n                "Required record provenance is missing")\n        if experiment == "G":\n            retained = base\n            summary["purge"] = {"groups_before": 0, "groups_after": 0, "rows": 0}\n        else:\n            crossings = disk(crossing_groups(base, "predictor_key"))\n            counts = group_summary(crossings)\n            retained = disk(mark_retention(base, crossings, counts["groups"]).where("retained").drop("retained"))\n            retained_count = retained.count()\n            remaining = group_summary(crossing_groups(retained, "predictor_key"))\n            require(remaining["groups"] == 0, "Overlap remains between reference and later cohorts")\n            summary["purge"] = {"groups_before": counts["groups"], "groups_after": remaining["groups"],\n                                "rows": summary["input"]["rows"] - retained_count}\n            base.unpersist(blocking=True)\n            crossings.unpersist(blocking=True)\n        split = split_counts(retained)\n        summary["matched_phase4"] = verify_phase4_match(split, experiment, phase4_receipt)\n        if summary["matched_phase4"]:\n            require(summary["matched_phase4"]["purged_rows"] == summary["purge"]["rows"], "Phase 4 purge size changed")\n        else:\n            classified = Counter()\n            for row in split:\n                classified[row["attack_family"]] += row["count"]\n            canonical = phase4_receipt["protocols"]["A"]["split_support"]\n            aggregate = Counter()\n            for entry in canonical.values():\n                aggregate.update(entry["families"])\n            require(classified == aggregate, "Clean family frequencies differ")\n        emit("cohorts", split)\n        reference_rows = sum(row["count"] for row in split if row["split_rank"] == 0)\n        train, probe = pick_reference(retained, config)\n        train_features, probe_features = disk(train.select(*names)), disk(probe.select(*names))\n        train_rows, probe_rows = train_features.count(), probe_features.count()\n        require(1 < probe_rows <= config["max_internal_rows"] and train_rows > probe_rows, "Bounded reference cohort differs")\n        prep, diagnostics = fit_transform(train_features, names)\n        summary["preprocessing"] = diagnostics\n        summary["training"] = {"fit_rows": train_rows, "internal_rows": probe_rows, "reference_rows": reference_rows,\n                               "share_of_reference": train_rows / reference_rows}\n        fit = disk(prep.transform(train_features).select("features"))\n        internal = disk(prep.transform(probe_features).select("features"))\n        verify_finite_vectors(fit, len(diagnostics["active"]))\n        verify_finite_vectors(internal, len(diagnostics["active"]))\n        trials, models = [], {}\n        for k in config["candidate_k"]:\n            model, info = fit_kmeans(fit, k, config, config["seed"])\n            require(info["training_rows"] == train_rows, "K-Means training sizes do not reconcile")\n            info["silhouette"] = silhouette(model, internal)\n            info["silhouette_rows"] = probe_rows\n            trials.append(info)\n            models[k] = model\n            atomic_json(path, summary | {"trials": trials})\n            print(f"PHASE5 {experiment} k={k} trained {train_rows} rows, silhouette {info[\'silhouette\']}", flush=True)\n        selected = choose_k(trials)\n        best = models[selected["k"]]\n        summary["selected_k"] = selected["k"]\n        summary["trials"] = trials\n        emit("model_selection", [row | {"selected": row["k"] == selected["k"]} for row in trials])\n\n        # Evolve both sensitivity models exclusively on the same fit and internal reference rows.\n        alternate, alt_seed_info = fit_kmeans(fit, selected["k"], config, config["seed"] + 1)\n        seed_ari, _ = compare_assignments(best, alternate, internal.withColumn("alternate_features", F.col("features")))\n        linear, linear_prep = fit_transform(train_features, names, transform="linear")\n        linear_fit = disk(linear.transform(train_features).select("features"))\n        verify_finite_vectors(linear_fit, len(linear_prep["active"]))\n        linear_model, linear_info = fit_kmeans(linear_fit, selected["k"], config, config["seed"])\n        paired = linear.transform(prep.transform(probe_features).withColumnRenamed("features", "primary_features")\n                                  .select(*names, "primary_features"))             .select(F.col("primary_features").alias("features"), F.col("features").alias("alternate_features"))\n        paired = disk(paired)\n        scale_ari, count = compare_assignments(best, linear_model, paired)\n        summary["robustness"] = {"paired_rows": count, "seed_ari": seed_ari, "scale_ari": scale_ari,\n                                 "alternate_seed": alt_seed_info, "linear": linear_info,\n                                 "linear_silhouette": silhouette(linear_model, paired.select(F.col("alternate_features").alias("features"))),\n                                 "dominance": centroid_dominance(best.clusterCenters(), selected["cluster_sizes"], diagnostics["active"])}\n        # Distances for threshold calibration come from the disjoint internal training-day sample.\n        calibration = calibrate(squared_distance(best, internal).select(F.lit(0).alias("split_rank"), "score"),\n                                config["reference_percentiles"], config["quantile_accuracy"])\n        summary["calibration"] = calibration\n        require(calibration["rows"] == probe_rows, "Thresholds were calibrated on an unexpected cohort")\n        for frame in (fit, internal, linear_fit, paired, train_features):\n            frame.unpersist(blocking=True)\n        assembled = disk(prep.transform(retained).select(\n            "source_day", "split_rank", "temporal_eligibility", "parsed_event_time", "binary_label", "attack_family",\n            "original_label", "record_fingerprint", "features", "flow_duration", "tot_fwd_pkts", "tot_bwd_pkts",\n            "totlen_fwd_pkts", "totlen_bwd_pkts", "flow_pkts_s",\n        ))\n        verify_finite_vectors(assembled, len(diagnostics["active"]))\n        total = sum(row["count"] for row in split)\n        scored = disk(squared_distance(best, assembled).drop("features"))\n        require(scored.count() == total, "Scoring changed retained count")\n        assembled.unpersist(blocking=True)\n        require(scored.where("score IS NULL OR isnan(score) OR score < 0 OR abs(score) = double(\'Infinity\')").limit(1).count() == 0,\n                "Centroid distance is invalid")\n        # Diagnose invalid and missing inputs in the retained train/evaluation populations.\n        null_counts = compact(retained.groupBy("split_rank").agg(F.count("*").alias("rows"), *[\n            F.count(F.when(F.col(name).isNull(), 1)).alias(name) for name in names\n        ]).orderBy("split_rank"), 3)\n        emit("transform_diagnostics", [{"split_rank": row["split_rank"], "rows": row["rows"], "feature": name,\n                                         "nullified_or_missing": row[name]} for row in null_counts for name in names])\n        selected_profiles = profiles(scored, selected["k"], config)\n        require(sum(row["records"] for row in selected_profiles) == total, "Cluster profiles lost rows")\n        emit("cluster_profiles", [row | {"attack_share": row["attacks"] / row["records"],\n                                        "binary_purity": max(row["attacks"], row["records"] - row["attacks"]) / row["records"]}\n                                 for row in selected_profiles])\n        emit("centroids", [{"cluster": i, "feature": name, "center": float(center[j])}\n                          for i, center in enumerate(best.clusterCenters()) for j, name in enumerate(diagnostics["active"])])\n        composition = compact(scored.groupBy("source_day", "split_rank", "cluster", "binary_label", "attack_family", "original_label")\n                              .count().orderBy("source_day", "cluster", "original_label"), 3500)\n        emit("composition", composition)\n        require(sum(row["count"] for row in composition) == total, "Composition changed retained count")\n        windows = compact(scored.where("temporal_eligibility").withColumn(\n            "window_hour", (F.floor(F.hour("parsed_event_time") / config["window_hours"]) * config["window_hours"]).cast("int"))\n            .groupBy("source_day", "split_rank", "window_hour", "cluster", "attack_family").count()\n            .orderBy("source_day", "window_hour", "cluster", "attack_family"), 6500)\n        emit("window_family", windows)\n        primary = calibration["cutoffs"][str(config["primary_percentile"])]\n        populated, shifts, reference = occupancy(scored, selected["k"], config, primary)\n        require(sum(row["records"] for row in populated if row["window_hour"] == -1) == total, "Daily occupancy changed retained count")\n        require(sum(row["records"] for row in populated if row["window_hour"] >= 0) == sum(row["count"] for row in windows),\n                "Eligible event-time tables differ")\n        emit("cluster_occupancy", populated)\n        emit("temporal_shift", shifts)\n        summary["reference_occupancy"] = reference\n        catalog = sorted({entry["family"] for entry in policy["label_mapping"].values() if entry["binary"]})\n        emit("novelty_metrics", novelty_metrics(scored, calibration["cutoffs"], catalog, config["rare_support"]))\n        histogram = compact(scored.withColumn("score_bin", F.floor(F.log1p("score") * 4).cast("int"))\n                            .groupBy("split_rank", "binary_label", "score_bin").count()\n                            .orderBy("split_rank", "binary_label", "score_bin"), 1000)\n        emit("score_histogram", histogram)\n        require(sum(row["count"] for row in histogram) == total, "Distance histogram changed retained count")\n        if experiment == "G":\n            duplicates = compact(scored.groupBy("record_fingerprint", "cluster").count().groupBy("cluster").agg(\n                F.sum("count").alias("records"), F.count("*").alias("exact_digest_representatives"),\n                F.sum(F.col("count") - 1).alias("excess")\n            ).orderBy("cluster"), selected["k"])\n            require(sum(row["excess"] for row in duplicates) == 410707, "Exact duplicate sensitivity differs from Phase 2")\n            emit("duplicate_sensitivity", duplicates)\n        summary["retained_rows"] = total\n        summary["model"] = saved_pipeline(prep, best, probe_features, output / "model")\n        summary.update(status="validated", elapsed_seconds=round(time.monotonic() - started, 3))\n        summary["resources"]["driver_peak_rss_bytes"] = driver_peak_bytes(spark)\n        atomic_json(path, summary)\n        summary["figures"] = draw_phase5(output, experiment, project / "reports/phase4_metrics.csv")\n        atomic_json(path, summary)\n        print(f"PHASE5 {experiment} complete k={selected[\'k\']} trained {train_rows}, scored {total} in {summary[\'elapsed_seconds\']}s", flush=True)\n    except Exception as exc:\n        summary.update(status="partial", error=f"{type(exc).__name__}: {str(exc)[:500]}")\n        atomic_json(path, summary)\n        raise\n    finally:\n        for frame in reversed(cached):\n            frame.unpersist()\n        spark.stop()\n', 'src/ml/clustering_features.py': '"""Label-free Spark feature preparation for distance-based experiments."""\n\nfrom pyspark.ml import Pipeline, PipelineModel\nfrom pyspark.ml.feature import Imputer, SQLTransformer, StandardScaler, VectorAssembler\nfrom pyspark.ml.functions import vector_to_array\nfrom pyspark.sql import functions as F\n\nfrom src.ml.feature_registry import FORBIDDEN, registry\nfrom src.sql.traffic_analytics import identifier, require\n\n\ndef clustering_registry(phase4, upstream):\n    return registry(phase4, upstream)\n\n\ndef fit_transform(training_values, names, transform="signed_log1p"):\n    if set(training_values.columns) != set(names) or set(names) & FORBIDDEN:\n        raise ValueError("Reference fit takes approved predictor columns only")\n    if transform not in {"signed_log1p", "linear"}:\n        raise ValueError("Unknown clustering transform")\n    clean = []\n    for name in names:\n        x = identifier(name)\n        value = (f"CASE WHEN {x} IS NOT NULL AND NOT isnan({x}) "\n                 f"AND abs({x}) < CAST(\'Infinity\' AS DOUBLE) THEN {x} END")\n        numerical = f"signum({value}) * log1p(abs({value}))" if transform == "signed_log1p" else value\n        clean.append(f"{numerical} AS {x}_valid")\n    sanitizer = SQLTransformer(statement="SELECT *, " + ", ".join(clean) + " FROM __THIS__")\n    finite = sanitizer.transform(training_values)\n    extrema = finite.agg(*[\n        expr.alias(f"{name}_{part}") for name in names\n        for part, expr in (("min", F.min(f"{name}_valid")), ("max", F.max(f"{name}_valid")),\n                           ("null", F.count(F.when(F.col(f"{name}_valid").isNull(), 1))))\n    ]).first().asDict()\n    active = [x for x in names if extrema[x + "_min"] is not None and extrema[x + "_min"] != extrema[x + "_max"]]\n    if not active:\n        raise ValueError("Reference population has no varying predictors")\n    assembled = Pipeline(stages=[\n        sanitizer, Imputer(inputCols=[x + "_valid" for x in active], outputCols=[x + "_filled" for x in active],\n                           strategy="median", relativeError=0.001),\n        VectorAssembler(inputCols=[x + "_filled" for x in active], outputCol="unscaled", handleInvalid="error"),\n    ]).fit(training_values)\n    scaler = StandardScaler(inputCol="unscaled", outputCol="features", withStd=True, withMean=True).fit(\n        assembled.transform(training_values))\n    stats = {name: {"invalid_or_null": extrema[name + "_null"], "available": name in active} for name in names}\n    for i, name in enumerate(active):\n        stats[name]["reference_std"] = float(scaler.std[i])\n        stats[name]["near_constant"] = float(scaler.std[i]) < 1e-8\n    return PipelineModel(stages=[*assembled.stages, scaler]), {\n        "transform": transform, "fit_columns": list(names), "active": active,\n        "dropped": [name for name in names if name not in active], "reference_stats": stats,\n        "imputation_surrogates": assembled.stages[1].surrogateDF.first().asDict(),\n        "reference_means": [float(x) for x in scaler.mean], "reference_scales": [float(x) for x in scaler.std],\n    }\n\n\ndef verify_finite_vectors(data, size):\n    values = vector_to_array("features")\n    bad = (F.size(values) != size) | F.exists(values, lambda x: F.isnan(x) | (F.abs(x) == float("inf")) | x.isNull())\n    require(data.where(bad).limit(1).count() == 0, "Nonfinite clustering vector")\n', 'src/ml/novelty_scoring.py': '"""Nearest-centroid squared distances and reference-only score calibration."""\n\nfrom pyspark.ml.functions import vector_to_array\nfrom pyspark.sql import functions as F\n\n\ndef squared_distance(model, data):\n    centers = F.array(*[F.array(*[F.lit(float(x)) for x in center]) for center in model.clusterCenters()])\n    prediction = model.transform(data)\n    assigned = F.element_at(centers, (F.col("cluster") + 1).cast("int"))\n    deltas = F.zip_with(vector_to_array("features"), assigned, lambda x, y: (x - y) * (x - y))\n    return prediction.withColumn("score", F.aggregate(deltas, F.lit(0.0), lambda acc, value: acc + value))\n\n\ndef calibrate(reference_scores, percentiles, accuracy):\n    if set(reference_scores.columns) != {"split_rank", "score"} or reference_scores.where("split_rank <> 0 OR split_rank IS NULL").limit(1).count():\n        raise ValueError("Threshold calibration requires reference score rows only")\n    row = reference_scores.agg(F.count("*").alias("rows"),\n                               F.percentile_approx("score", percentiles, accuracy).alias("cutoffs")).first()\n    if not row.rows or row.cutoffs is None:\n        raise ValueError("Empty novelty calibration population")\n    cutoffs = dict(zip(map(str, percentiles), row.cutoffs))\n    exceedance = reference_scores.agg(*[\n        F.count(F.when(F.col("score") > value, 1)).alias(p) for p, value in cutoffs.items()\n    ]).first().asDict()\n    return {"rows": row.rows, "cutoffs": cutoffs, "accuracy": accuracy, "reference_flags": exceedance}\n\n\ndef safe_div(a, b):\n    return a / b if b else None\n\n\ndef novelty_row(positive, negative, tp, fp):\n    count = positive + negative\n    return {"rows": count, "attacks": positive, "benign": negative, "tp": tp, "fp": fp,\n            "flags": tp + fp, "flag_rate": safe_div(tp + fp, count),\n            "benign_fpr": safe_div(fp, negative), "attack_recall": safe_div(tp, positive),\n            "precision": safe_div(tp, tp + fp)}\n\n\ndef novelty_metrics(data, cutoffs, catalog, minimum):\n    rows = []\n    for p, cutoff in cutoffs.items():\n        flag = F.col("score") > cutoff\n        projection = data.withColumn("flag", flag)\n        measured = projection.groupBy("split_rank", "binary_label", "attack_family").agg(\n            F.count("*").alias("records"), F.count(F.when(F.col("flag"), 1)).alias("flagged")).collect()\n        totals, support = {}, {}\n        for row in measured:\n            totals.setdefault(row.split_rank, [0, 0, 0, 0])\n            totals[row.split_rank][0 if row.binary_label else 1] += row.records\n            totals[row.split_rank][2 if row.binary_label else 3] += row.flagged\n            support[row.split_rank, row.attack_family] = (row.records, row.flagged)\n        for rank, counts in sorted(totals.items()):\n            rows.append({"split_rank": rank, "percentile": p, "threshold": cutoff, "scope": "binary", "family": "All",\n                         **novelty_row(*counts), "support_status": "observed"})\n            for family in catalog:\n                s, hit = support.get((rank, family), (0, 0))\n                rows.append({"split_rank": rank, "percentile": p, "threshold": cutoff, "scope": "family", "family": family,\n                             **(novelty_row(s, 0, hit, 0) | {"precision": None}),\n                             "support_status": "absent" if not s else "low" if s < minimum else "observed"})\n    return rows\n', 'src/ml/cluster_diagnostics.py': '"""Training-only model selection and permutation-invariant robustness metrics."""\n\nimport math\nimport time\n\nfrom pyspark.ml.clustering import KMeans\nfrom pyspark.ml.evaluation import ClusteringEvaluator\nfrom pyspark.sql import functions as F\n\nfrom src.ingest.parquet_pipeline import driver_peak_bytes\n\n\ndef fit_kmeans(vectors, k, config, seed):\n    if vectors.columns != ["features"]:\n        raise ValueError("K-Means receives only the fitted feature vector")\n    start = time.monotonic()\n    model = KMeans(k=k, seed=seed, initMode="k-means||", initSteps=config["init_steps"],\n                   maxIter=config["max_iter"], tol=config["tolerance"], featuresCol="features",\n                   predictionCol="cluster", distanceMeasure="euclidean").fit(vectors)\n    elapsed = round(time.monotonic() - start, 3)\n    summary = model.summary\n    sizes = [int(v) for v in summary.clusterSizes]\n    fraction = min(sizes) / sum(sizes) if sizes and sum(sizes) else 0.0\n    return model, {\n        "k": k, "seed": seed, "initialization": "k-means||", "budget": config["max_iter"], "iterations": summary.numIter,\n        "wssse": summary.trainingCost, "cluster_sizes": sizes, "training_rows": sum(sizes),\n        "empty_clusters": k - sum(size > 0 for size in sizes), "smallest_share": fraction, "fit_seconds": elapsed,\n        "jvm_peak_rss_bytes": driver_peak_bytes(vectors.sparkSession),\n    }\n\n\ndef silhouette(model, probe):\n    predictions = model.transform(probe)\n    if predictions.select("cluster").distinct().count() < 2:\n        return None\n    value = ClusteringEvaluator(featuresCol="features", predictionCol="cluster",\n                                metricName="silhouette", distanceMeasure="squaredEuclidean").evaluate(predictions)\n    return value if math.isfinite(value) else None\n\n\ndef choose_k(rows):\n    eligible = [row for row in rows if not row["empty_clusters"] and row["smallest_share"] >= 0.001\n                and row["silhouette"] is not None]\n    if not eligible:\n        raise ValueError("All candidates are degenerate or lack silhouette support")\n    return max(eligible, key=lambda row: (row["silhouette"], -row["k"]))\n\n\ndef adjusted_rand(contingency):\n    n = sum(n for _, _, n in contingency)\n    if n < 2:\n        return None\n    left, right = {}, {}\n    for a, b, count in contingency:\n        left[a] = left.get(a, 0) + count\n        right[b] = right.get(b, 0) + count\n    observed = sum(math.comb(count, 2) for _, _, count in contingency)\n    a, b = (sum(math.comb(count, 2) for count in side.values()) for side in (left, right))\n    expected = a * b / math.comb(n, 2)\n    denominator = (a + b) / 2 - expected\n    return (observed - expected) / denominator if denominator else 1.0\n\n\ndef compare_assignments(standard, alternative, probe):\n    pairs = (alternative.transform(probe.withColumnRenamed("features", "primary_features")\n                                   .withColumnRenamed("alternate_features", "features"))\n             .withColumnRenamed("cluster", "other_cluster").drop("features")\n             .withColumnRenamed("primary_features", "features"))\n    counts = standard.transform(pairs).groupBy("cluster", "other_cluster").count().collect()\n    return adjusted_rand([(r.cluster, r.other_cluster, r["count"]) for r in counts]), sum(row["count"] for row in counts)\n\n\ndef centroid_dominance(centers, sizes, features):\n    normalized = [n / sum(sizes) for n in sizes]\n    sums = []\n    for j, feature in enumerate(features):\n        mean = sum(weight * float(c[j]) for weight, c in zip(normalized, centers))\n        sums.append((feature, sum(weight * (float(c[j]) - mean) ** 2 for weight, c in zip(normalized, centers))))\n    total = sum(v for _, v in sums)\n    return [{"feature": name, "share_between_centroids": share / total if total else 0.0}\n            for name, share in sorted(sums, key=lambda item: (-item[1], item[0]))]\n', 'src/ml/cluster_evolution.py': '"""Capture-day and three-hour occupancy comparisons on a frozen model."""\n\nimport math\n\nfrom pyspark.sql import functions as F\n\nfrom src.sql.traffic_analytics import compact\n\n\ndef divergence(left, right):\n    if sum(left) == 0 or sum(right) == 0 or len(left) != len(right):\n        return None\n    p = [value / sum(left) for value in left]\n    q = [value / sum(right) for value in right]\n    distance = sum(\n        (a * math.log2(2 * a / (a + b)) if a else 0.0)\n        + (b * math.log2(2 * b / (a + b)) if b else 0.0) for a, b in zip(p, q)\n    ) / 2\n    return max(0.0, distance)\n\n\ndef occupancy(data, k, config, cutoff):\n    hours = config["window_hours"]\n    bins = F.floor(F.hour("parsed_event_time") / hours) * hours\n    day = data.withColumn("window_hour", F.lit(-1))\n    window = data.where("temporal_eligibility").withColumn("window_hour", bins.cast("int"))\n    fields = ("source_day", "split_rank", "window_hour", "cluster", "score", "binary_label", "flow_duration", "tot_fwd_pkts", "totlen_fwd_pkts")\n    scoped = day.select(*fields).unionByName(window.select(*fields))\n    raw = compact(scoped.groupBy("source_day", "split_rank", "window_hour", "cluster").agg(\n        F.count("*").alias("records"), F.count(F.when(F.col("binary_label") == 1, 1)).alias("attacks"),\n        F.count(F.when(F.col("score") > cutoff, 1)).alias("flags"),\n        F.avg("score").alias("mean_score"), F.percentile_approx("score", 0.95, config["quantile_accuracy"]).alias("score_p95"),\n        F.avg("flow_duration").alias("mean_duration_us"), F.avg("tot_fwd_pkts").alias("mean_forward_packets"),\n        F.avg("totlen_fwd_pkts").alias("mean_forward_bytes")\n    ).orderBy("source_day", "window_hour", "cluster"), 2000)\n    reference = [0] * k\n    for row in raw:\n        if row["split_rank"] == 0 and row["window_hour"] == -1:\n            reference[row["cluster"]] += row["records"]\n    cohorts = {}\n    for row in raw:\n        key = row["source_day"], row["split_rank"], row["window_hour"]\n        cohorts.setdefault(key, [0] * k)[row["cluster"]] += row["records"]\n    rows = [row | {"occupancy": row["records"] / sum(cohorts[row["source_day"], row["split_rank"], row["window_hour"]])} for row in raw]\n    shifts = [{"source_day": day, "split_rank": rank, "window_hour": hour, "rows": sum(counts),\n               "occupied_clusters": sum(n > 0 for n in counts), "js_divergence_bits": divergence(reference, counts),\n               "reference_rows": sum(reference)} for (day, rank, hour), counts in sorted(cohorts.items())]\n    return rows, shifts, reference\n', 'src/visualization/clustering_charts.py': '"""Generate compact Phase 5 figures from reconciled aggregate tables."""\n\nfrom collections import Counter\nimport json\nfrom pathlib import Path\n\nimport matplotlib\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\n\n\n\ndef draw_phase5(root, experiment, supervised_csv):\n    from csv import DictReader\n\n    def read(name):\n        with (root / f"phase5_{name}.csv").open(newline="", encoding="utf-8") as stream:\n            return list(DictReader(stream))\n\n    summary = json.loads((root / "phase5_summary.json").read_text())\n    profiles, composition = read("cluster_profiles"), read("composition")\n    occupancy, shifts, centroids = read("cluster_occupancy"), read("temporal_shift"), read("centroids")\n    novelty, histogram = read("novelty_metrics"), read("score_histogram")\n    total = sum(int(item["records"]) for item in profiles)\n    if total != summary["retained_rows"] or sum(int(item["count"]) for item in composition) != total:\n        raise ValueError("Chart totals do not match the retained population")\n    folder = root / "figures"\n    folder.mkdir(exist_ok=True)\n    prefix = f"phase5_{experiment.lower()}"\n    k = summary["selected_k"]\n    ids = list(range(k))\n    figures = []\n    colors = plt.get_cmap("tab20")(np.linspace(0, 1, k))\n\n    def show(fig, name):\n        target = folder / f"{prefix}_{name}.png"\n        fig.savefig(target, dpi=120, bbox_inches="tight", metadata={"Software": "MMDS Phase 5"})\n        plt.close(fig)\n        figures.append(target.name)\n\n    def bar(name, items, label, title):\n        fig, ax = plt.subplots(figsize=(6.8, 3.4), layout="constrained")\n        ax.bar([str(a) for a, _ in items], [b for _, b in items], color="#087e8b")\n        ax.set_ylabel(label)\n        ax.set_title(title, loc="left")\n        ax.grid(axis="y", alpha=.2)\n        show(fig, name)\n\n    bar("cluster_sizes", [(r["cluster"], int(r["records"])) for r in profiles], "Records",\n        f"{experiment} · cluster size across retained captures")\n    bar("cluster_attack_share", [(r["cluster"], 100 * float(r["attack_share"])) for r in profiles],\n        "Observed attack share (%)", f"{experiment} · external label composition")\n    families = sorted({r["attack_family"] for r in composition})\n    matrix = Counter()\n    for row in composition:\n        matrix[int(row["cluster"]), row["attack_family"]] += int(row["count"])\n    counts = [[matrix[i, family] for family in families] for i in ids]\n    fig, ax = plt.subplots(figsize=(7.8, 3.9), layout="constrained")\n    view = ax.imshow(np.log10(1 + np.array(counts)), aspect="auto", cmap="cividis")\n    ax.set_xticks(range(len(families)), families, rotation=25, ha="right")\n    ax.set_yticks(ids, ids)\n    ax.set_ylabel("Cluster")\n    ax.set_title(f"{experiment} · cluster by observed family")\n    fig.colorbar(view, ax=ax, label="log10(1 + records)")\n    show(fig, "family_heatmap")\n\n    daily = [r for r in occupancy if r["window_hour"] == "-1"]\n    days = sorted({r["source_day"] for r in daily})\n    posts = {(r["source_day"], int(r["cluster"])): float(r["occupancy"]) for r in daily}\n    if sum(int(r["records"]) for r in daily) != total or any(\n        abs(sum(posts.get((day, i), 0) for i in ids) - 1) > 1e-8 for day in days\n    ):\n        raise ValueError("Chart occupancy partitions do not reconcile")\n    fig, ax = plt.subplots(figsize=(8.1, 3.8), layout="constrained")\n    bottom = np.zeros(len(days))\n    for i in ids:\n        y = [100 * posts.get((day, i), 0) for day in days]\n        ax.bar([d[5:] for d in days], y, bottom=bottom, color=colors[i], label=str(i))\n        bottom += y\n    ax.set_ylabel("Cluster occupancy (%)")\n    ax.set_title(f"{experiment} · occupancy by capture day")\n    ax.legend(ncols=min(k, 6), title="Cluster", fontsize=7)\n    show(fig, "daily_occupancy")\n    bar("temporal_shift", [(r["source_day"][5:], float(r["js_divergence_bits"]))\n                           for r in shifts if r["window_hour"] == "-1"], "Jensen–Shannon divergence (bits)",\n        f"{experiment} · daily occupancy versus reference")\n\n    names = summary["preprocessing"]["active"]\n    positions = {(int(r["cluster"]), r["feature"]): float(r["center"]) for r in centroids}\n    fig, ax = plt.subplots(figsize=(9.5, 4.6), layout="constrained")\n    view = ax.imshow([[positions[i, key] for key in names] for i in ids], aspect="auto", cmap="coolwarm")\n    ax.set_xticks(range(len(names)), names, rotation=60, ha="right", fontsize=7)\n    ax.set_yticks(ids, ids)\n    ax.set_ylabel("Cluster")\n    ax.set_title(f"{experiment} · centered and scaled signed-log centroids")\n    fig.colorbar(view, ax=ax, label="Reference standard deviations")\n    show(fig, "centroid_profiles")\n\n    rank = "0" if experiment == "G" else "2"\n    fig, ax = plt.subplots(figsize=(7, 3.6), layout="constrained")\n    for label, title, color in (("0", "Benign", "#087e8b"), ("1", "Attack", "#b44a31")):\n        rows = sorted((r for r in histogram if r["split_rank"] == rank and r["binary_label"] == label),\n                      key=lambda row: int(row["score_bin"]))\n        if rows:\n            denominator = sum(int(r["count"]) for r in rows)\n            ax.plot([int(r["score_bin"]) / 4 for r in rows], [int(r["count"]) / denominator for r in rows],\n                    marker=".", label=title, color=color)\n    ax.set_xlabel("log1p(squared distance), 0.25-width bins")\n    ax.set_ylabel("Within-label fraction per bin")\n    ax.set_title(f"{experiment} · {\'all retained\' if rank == \'0\' else \'test\'} distance distribution")\n    ax.legend()\n    show(fig, "distance_histogram")\n\n    day_counts = Counter()\n    day_flags = Counter()\n    for row in daily:\n        day_counts[row["source_day"]] += int(row["records"])\n        day_flags[row["source_day"]] += int(row["flags"])\n    bar("daily_novelty", [(day[5:], 100 * day_flags[day] / day_counts[day]) for day in days],\n        "Rows above reference P99 (%)", f"{experiment} · novelty flags by capture day")\n\n    primary = next(r for r in novelty if r["split_rank"] == rank and r["percentile"] == "0.99" and r["scope"] == "binary")\n    bar("binary_novelty", [("Benign", 100 * float(primary["benign_fpr"])), ("Attack", 100 * float(primary["attack_recall"]))],\n        "Within-label flag rate (%)", f"{experiment} · {\'all retained\' if rank == \'0\' else \'test\'} at reference P99")\n    if experiment != "G":\n        with Path(supervised_csv).open(newline="", encoding="utf-8") as stream:\n            rf = next(row for row in DictReader(stream) if row["protocol"] == experiment and row["split"] == "test"\n                      and row["model"] == "random_forest" and row["training_cohort"] == "group_sample")\n        if int(rf["tp"]) + int(rf["fn"]) != int(primary["attacks"]) or int(rf["tn"]) + int(rf["fp"]) != int(primary["benign"]):\n            raise ValueError("Comparison chart test denominators differ")\n        fig, ax = plt.subplots(figsize=(7.0, 3.8), layout="constrained")\n        x = np.arange(2)\n        ax.bar(x - .16, [100 * (1 - float(rf["specificity"])), 100 * float(rf["recall"])], width=.32, label="Phase 4 RF, 0.5")\n        ax.bar(x + .16, [100 * float(primary["benign_fpr"]), 100 * float(primary["attack_recall"])], width=.32,\n               label="Centroid distance, reference P99")\n        ax.set_xticks(x, ["Benign false positives", "Attack recall"])\n        ax.set_ylabel("Rate (%)")\n        ax.set_title(f"{experiment} · cohort-matched aggregate signals")\n        ax.legend()\n        show(fig, "supervised_comparison")\n    return figures\n', 'configs/clustering_experiments.yaml': '{\n  "version": "phase5-v1",\n  "seed": 23013,\n  "candidate_k": [3, 5, 8, 12],\n  "max_iter": 16,\n  "init_steps": 2,\n  "tolerance": 0.0001,\n  "training_permyriad": 250,\n  "internal_permyriad": 12,\n  "reference_percentiles": [0.95, 0.99, 0.999],\n  "primary_percentile": 0.99,\n  "quantile_accuracy": 8192,\n  "max_internal_rows": 40000,\n  "rare_support": 100,\n  "window_hours": 3,\n  "lag_reference": "full retained phase4 train-day population",\n  "spark": {"master": "local[4]", "driver_memory": "8g", "shuffle_partitions": 48, "input_partition_bytes": 67108864}\n}\n', 'manifests/phase4_models.json': '{\n  "status": "validated",\n  "kernel_url": "https://www.kaggle.com/code/ramcharan2345/mmds-phase-4-supervised-attack-detection",\n  "config_version": "phase4-v1",\n  "seed": 23013,\n  "metric_policy": {\n    "positive_label": 1,\n    "threshold": 0.5,\n    "score_bins": 2048,\n    "pr_auc": "step-integrated binned average precision",\n    "roc_auc": "trapezoidal binned ROC"\n  },\n  "protocols": {\n    "A": {\n      "version": "phase4-v1",\n      "status": "validated",\n      "log_events": 952,\n      "protocol": "A",\n      "design": {\n        "name": "predictor-group random"\n      },\n      "resources": {\n        "spark_version": "4.3.0",\n        "master": "local[4]",\n        "driver_memory": "8g",\n        "shuffle_partitions": 48,\n        "input_partition_bytes": 67108864,\n        "default_parallelism": 4,\n        "cpu_affinity": 4,\n        "scratch_output_free_bytes_start": 20940279808,\n        "memory.max": "32212254720",\n        "cpu.max": "400000 100000",\n        "driver_peak_rss_bytes": 7474003968\n      },\n      "models": [\n        {\n          "path": "/kaggle/working/mmds_phase4/models/logistic_regression_group_sample",\n          "model": "logistic_regression",\n          "reload_probe_rows": 695307,\n          "model_bytes": 16239,\n          "model_files": 16,\n          "sha256": {\n            "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "metadata/part-00000-d22084df-5d33-404a-9fda-26a9776312cd-c000.txt": "f005026679ac611c02e066dd3303ee750f90f51995f41ff60b7e0eb1fba64375",\n            "stages/0_Imputer_abfd4cd929fc/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_abfd4cd929fc/data/part-00000-a26e794d-22f5-4b92-8881-621f6128fc27-c000.snappy.parquet": "6782881f601f55774f6a3a328f36b8e67f1a4c597e029385bf75967133da6c93",\n            "stages/0_Imputer_abfd4cd929fc/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_abfd4cd929fc/metadata/part-00000-63fcca9b-f1ab-4f0b-a432-1d2b90bbd47a-c000.txt": "7c47f4e28af6ded54377e01998e0a20f73361071b79171107cdea893e3ee1a5a",\n            "stages/1_VectorAssembler_74fb923cd0b6/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/1_VectorAssembler_74fb923cd0b6/metadata/part-00000-cdec3ca8-3b2e-44f5-8df1-f4d37510420d-c000.txt": "93504030dc87b4b4f7024d52d575af05938a50e7b0cc192b2f0175f8b654b899",\n            "stages/2_StandardScaler_7ec078203426/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_StandardScaler_7ec078203426/data/part-00000-f00ed6b1-97a9-4b2f-abc6-ae310b7bb5f8-c000.snappy.parquet": "9c258408f8b723d8defb95ac34ec00ba12ac69d56608731095db3bba6f26fc56",\n            "stages/2_StandardScaler_7ec078203426/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_StandardScaler_7ec078203426/metadata/part-00000-34893155-4a3a-42c0-896c-30c3482ed6b9-c000.txt": "0ca8ef31bd4cae35781d3a71af933f5e35073b8c4eaadfd5b1f278517f76e529",\n            "stages/3_LogisticRegression_adbfc1c28bd8/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/3_LogisticRegression_adbfc1c28bd8/data/part-00000-cb77979f-85c3-40d3-9084-547f9be98e01-c000.snappy.parquet": "64026013080a9f13f3ed3bec051db9d6acac0bd6f0706a7a17c8ea3975e49f23",\n            "stages/3_LogisticRegression_adbfc1c28bd8/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/3_LogisticRegression_adbfc1c28bd8/metadata/part-00000-14a1eb4f-b7b0-4932-bfec-8795703e1cc2-c000.txt": "1acb85d84306f1006da5833a4bcf9dd02a1ee659c5975642a01bf5611c791447"\n          },\n          "training_cohort": "group_sample",\n          "training_seconds": 6.295,\n          "parameters": {\n            "maxIter": 12,\n            "regParam": 0.05,\n            "elasticNetParam": 0.0\n          },\n          "explanation": [\n            {\n              "feature": "ack_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": 0.6055004815071301\n            },\n            {\n              "feature": "rst_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": 0.41801635461493525\n            },\n            {\n              "feature": "pkt_len_mean",\n              "method": "scaled_lr_coefficient",\n              "value": -0.33783932660100696\n            },\n            {\n              "feature": "syn_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": -0.28812792814204863\n            },\n            {\n              "feature": "flow_duration",\n              "method": "scaled_lr_coefficient",\n              "value": -0.21893573945720465\n            },\n            {\n              "feature": "flow_pkts_s",\n              "method": "scaled_lr_coefficient",\n              "value": -0.1911204322503146\n            },\n            {\n              "feature": "flow_byts_s",\n              "method": "scaled_lr_coefficient",\n              "value": -0.10934142934216814\n            },\n            {\n              "feature": "flow_iat_std",\n              "method": "scaled_lr_coefficient",\n              "value": -0.09230271255991071\n            },\n            {\n              "feature": "down_up_ratio",\n              "method": "scaled_lr_coefficient",\n              "value": -0.06399728305368631\n            },\n            {\n              "feature": "fin_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": -0.05260695834107492\n            },\n            {\n              "feature": "fwd_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": -0.04171377005068665\n            },\n            {\n              "feature": "flow_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": -0.03784969621292684\n            },\n            {\n              "feature": "tot_fwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.033926155607644295\n            },\n            {\n              "feature": "totlen_fwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.028382859854910233\n            },\n            {\n              "feature": "bwd_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": 0.016827809341042027\n            },\n            {\n              "feature": "pkt_len_std",\n              "method": "scaled_lr_coefficient",\n              "value": -0.006682537282281606\n            },\n            {\n              "feature": "tot_bwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": -0.002908947666697675\n            },\n            {\n              "feature": "totlen_bwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": -0.0010682323226017172\n            }\n          ]\n        },\n        {\n          "path": "/kaggle/working/mmds_phase4/models/random_forest_group_sample",\n          "model": "random_forest",\n          "reload_probe_rows": 695307,\n          "model_bytes": 64114,\n          "model_files": 14,\n          "sha256": {\n            "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "metadata/part-00000-03f8af52-37ff-4e51-b578-9c9a31cb044a-c000.txt": "c472c97b7d4f76325a5c2bff4801f373f84771ce67a5754dd62adc5556344050",\n            "stages/0_Imputer_abfd4cd929fc/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_abfd4cd929fc/data/part-00000-1afea790-1b40-4f38-88b9-4900cc34bcdc-c000.snappy.parquet": "6782881f601f55774f6a3a328f36b8e67f1a4c597e029385bf75967133da6c93",\n            "stages/0_Imputer_abfd4cd929fc/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_abfd4cd929fc/metadata/part-00000-46157776-e1d4-41bc-9c67-47ef8c332784-c000.txt": "dfb6194e70d6199e21d5264bf1b8ed6b623287d2d889fde1e04ffabab2d9630c",\n            "stages/1_VectorAssembler_74fb923cd0b6/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/1_VectorAssembler_74fb923cd0b6/metadata/part-00000-f92d70a1-c40f-4703-b90f-45ad52828272-c000.txt": "87be86c491ff69af50fcde183eeca14f496880cf07c4e51c58f54d526bf41e0d",\n            "stages/2_RandomForestClassifier_98377622b0a4/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_RandomForestClassifier_98377622b0a4/data/part-00000-54695e2b-40a2-4d19-8359-87a4f76599e5-c000.snappy.parquet": "49f34e3c812c007226aeabf69eca1c4c8ea94eb49ecbcdeac5030cc2ba50365d",\n            "stages/2_RandomForestClassifier_98377622b0a4/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_RandomForestClassifier_98377622b0a4/metadata/part-00000-4f7eabc6-63f5-4786-94bf-0de646ae4dac-c000.txt": "89a32402b2207e53d6aa590d23e5f6cfae9a78254603f5788f7380dc1d603e5f",\n            "stages/2_RandomForestClassifier_98377622b0a4/treesMetadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_RandomForestClassifier_98377622b0a4/treesMetadata/part-00000-e6ba7265-32a3-4c98-a108-f38ba02146c2-c000.snappy.parquet": "f059e6874d66606dbfb53557ade88e8d088050bde1890daf40a73204ead7c197"\n          },\n          "training_cohort": "group_sample",\n          "training_seconds": 11.737,\n          "parameters": {\n            "numTrees": 20,\n            "maxDepth": 7,\n            "minInstancesPerNode": 30\n          },\n          "explanation": [\n            {\n              "feature": "fwd_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.1485828303115913\n            },\n            {\n              "feature": "flow_pkts_s",\n              "method": "impurity_split_importance",\n              "value": 0.12844340395588738\n            },\n            {\n              "feature": "totlen_fwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.1280578502643492\n            },\n            {\n              "feature": "flow_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.0847239780596514\n            },\n            {\n              "feature": "flow_duration",\n              "method": "impurity_split_importance",\n              "value": 0.08105967665769179\n            },\n            {\n              "feature": "tot_bwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.07936652024408121\n            },\n            {\n              "feature": "pkt_len_mean",\n              "method": "impurity_split_importance",\n              "value": 0.0696051952123036\n            },\n            {\n              "feature": "flow_byts_s",\n              "method": "impurity_split_importance",\n              "value": 0.05286345774391453\n            },\n            {\n              "feature": "tot_fwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.05121029536467747\n            },\n            {\n              "feature": "totlen_bwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.0465217396021467\n            },\n            {\n              "feature": "pkt_len_std",\n              "method": "impurity_split_importance",\n              "value": 0.03721990062869542\n            },\n            {\n              "feature": "ack_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.035757636894877015\n            },\n            {\n              "feature": "flow_iat_std",\n              "method": "impurity_split_importance",\n              "value": 0.02376530816764866\n            },\n            {\n              "feature": "rst_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.0162936756054958\n            },\n            {\n              "feature": "down_up_ratio",\n              "method": "impurity_split_importance",\n              "value": 0.007973356728062384\n            },\n            {\n              "feature": "bwd_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.007940022192018519\n            },\n            {\n              "feature": "syn_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.0003883854687734745\n            },\n            {\n              "feature": "fin_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.0002267668981342781\n            }\n          ]\n        },\n        {\n          "path": "/kaggle/working/mmds_phase4/models/gradient_boosted_trees_group_sample",\n          "model": "gradient_boosted_trees",\n          "reload_probe_rows": 695307,\n          "model_bytes": 31043,\n          "model_files": 14,\n          "sha256": {\n            "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "metadata/part-00000-9997ea4b-620c-4725-ab56-fb60547bb9ca-c000.txt": "3a8db28bac890f120b68ac7c817836f4e782d8d0ea0fa29d475b76e13fc0b340",\n            "stages/0_Imputer_abfd4cd929fc/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_abfd4cd929fc/data/part-00000-9f0438c1-df56-44c6-80be-c7c3e8053633-c000.snappy.parquet": "6782881f601f55774f6a3a328f36b8e67f1a4c597e029385bf75967133da6c93",\n            "stages/0_Imputer_abfd4cd929fc/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_abfd4cd929fc/metadata/part-00000-230b1995-5c8d-46e5-be65-f2d1c5a6ad5f-c000.txt": "a74ab818a4d5303c5ac0f500ea9e51e7953458a8a8bb5f660892f7fc63718798",\n            "stages/1_VectorAssembler_74fb923cd0b6/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/1_VectorAssembler_74fb923cd0b6/metadata/part-00000-e9eb1da3-7ab7-4f5e-a707-9986e33ad612-c000.txt": "20fe83ecfa8949ef2e674ee7234f02eddf8b15c57a5165c400fffcef8fc82e03",\n            "stages/2_GBTClassifier_a572097ba766/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_GBTClassifier_a572097ba766/data/part-00000-b233c444-18d1-49f3-962e-2ea467d359cb-c000.snappy.parquet": "ecb29f883aa85e5cb7840f65d423e6052074e6e2bd9654b4e713601239fedd69",\n            "stages/2_GBTClassifier_a572097ba766/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_GBTClassifier_a572097ba766/metadata/part-00000-7d715cac-4f06-4492-8119-14dc1ecd8a25-c000.txt": "d5585381f22369d61863594ec2961d7fb2a2ec9ffda17c8386a848f8858138f7",\n            "stages/2_GBTClassifier_a572097ba766/treesMetadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_GBTClassifier_a572097ba766/treesMetadata/part-00000-1ab94296-2336-49c8-987f-c633b0b25e06-c000.snappy.parquet": "f4b6fd6dc04571ec9886d8999a5f355a09938bede8c9acff28490820d4a51bc5"\n          },\n          "training_cohort": "group_sample",\n          "training_seconds": 19.557,\n          "parameters": {\n            "maxIter": 10,\n            "maxDepth": 4,\n            "stepSize": 0.15\n          },\n          "explanation": [\n            {\n              "feature": "fwd_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.2943983652590758\n            },\n            {\n              "feature": "totlen_fwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.19264049041299314\n            },\n            {\n              "feature": "tot_fwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.09968025403414996\n            },\n            {\n              "feature": "flow_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.08468494650972742\n            },\n            {\n              "feature": "flow_iat_std",\n              "method": "impurity_split_importance",\n              "value": 0.06280931085634042\n            },\n            {\n              "feature": "tot_bwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.05522076067527951\n            },\n            {\n              "feature": "rst_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.04857612370895901\n            },\n            {\n              "feature": "flow_pkts_s",\n              "method": "impurity_split_importance",\n              "value": 0.029906156494087812\n            },\n            {\n              "feature": "ack_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.02867864603479734\n            },\n            {\n              "feature": "flow_duration",\n              "method": "impurity_split_importance",\n              "value": 0.02703419735576855\n            },\n            {\n              "feature": "flow_byts_s",\n              "method": "impurity_split_importance",\n              "value": 0.02153572869042957\n            },\n            {\n              "feature": "pkt_len_mean",\n              "method": "impurity_split_importance",\n              "value": 0.019431320174945952\n            },\n            {\n              "feature": "totlen_bwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.014504644201957304\n            },\n            {\n              "feature": "pkt_len_std",\n              "method": "impurity_split_importance",\n              "value": 0.011030620287699642\n            },\n            {\n              "feature": "bwd_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.0036449596053326314\n            },\n            {\n              "feature": "down_up_ratio",\n              "method": "impurity_split_importance",\n              "value": 0.003488000014053678\n            },\n            {\n              "feature": "syn_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.0023378444989426003\n            },\n            {\n              "feature": "fin_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.00039763118545956694\n            }\n          ]\n        },\n        {\n          "path": "/kaggle/working/mmds_phase4/models/logistic_regression_full_train_refit",\n          "model": "logistic_regression",\n          "reload_probe_rows": 695307,\n          "model_bytes": 16239,\n          "model_files": 16,\n          "sha256": {\n            "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "metadata/part-00000-af34b176-c9ed-4a14-bc17-c4528a9db97c-c000.txt": "5f9c7fcfe76d6793e40b30320a2c6965cad9fbaf6947cfc7b4376502e2e1cbbd",\n            "stages/0_Imputer_abfd4cd929fc/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_abfd4cd929fc/data/part-00000-6ece0894-f8b7-447e-b38f-0a3a6f7be9e3-c000.snappy.parquet": "6782881f601f55774f6a3a328f36b8e67f1a4c597e029385bf75967133da6c93",\n            "stages/0_Imputer_abfd4cd929fc/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_abfd4cd929fc/metadata/part-00000-f5cb6293-72c8-4eda-a90b-7772f83cd552-c000.txt": "0dfe3449b12edc0e7e2265ed506be64328550439fa9f3ff05a748dfa63a8e13e",\n            "stages/1_VectorAssembler_74fb923cd0b6/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/1_VectorAssembler_74fb923cd0b6/metadata/part-00000-55025e42-4898-4e6c-84cd-cbf88d885849-c000.txt": "5f8c9591510f6db8ba2d716013d0abc429e08d51ee782afa0a23d03d198648f9",\n            "stages/2_StandardScaler_7ec078203426/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_StandardScaler_7ec078203426/data/part-00000-11ef9831-1439-48d8-a72d-5c655aa5981a-c000.snappy.parquet": "9c258408f8b723d8defb95ac34ec00ba12ac69d56608731095db3bba6f26fc56",\n            "stages/2_StandardScaler_7ec078203426/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_StandardScaler_7ec078203426/metadata/part-00000-11f18883-902c-4946-bd20-6f4256506f0f-c000.txt": "a08f37a7cf9420b3b9c0ab7eab3199adcd4adce78d809339286cd543c741de1e",\n            "stages/3_LogisticRegression_1d57b334e6b3/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/3_LogisticRegression_1d57b334e6b3/data/part-00000-4b28ecbb-85e7-4326-9f97-2e17c9903b37-c000.snappy.parquet": "a00af7e5250ccffc4308c60bf19f7ff14e6b7bdc51700c556b80ca919a2187a8",\n            "stages/3_LogisticRegression_1d57b334e6b3/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/3_LogisticRegression_1d57b334e6b3/metadata/part-00000-d5ee61ca-ea00-4da9-b0e0-16a68b4a113b-c000.txt": "96d230188f28f3dc821df7a0551e4c67015f676638a31104f8501b9626f2f008"\n          },\n          "training_cohort": "full_train_refit",\n          "training_seconds": 25.26,\n          "parameters": {\n            "maxIter": 12,\n            "regParam": 0.05,\n            "elasticNetParam": 0.0\n          },\n          "explanation": [\n            {\n              "feature": "ack_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": 0.47266925598336645\n            },\n            {\n              "feature": "flow_pkts_s",\n              "method": "scaled_lr_coefficient",\n              "value": 0.46701478324029866\n            },\n            {\n              "feature": "pkt_len_mean",\n              "method": "scaled_lr_coefficient",\n              "value": -0.40680489960034694\n            },\n            {\n              "feature": "rst_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": 0.3730107928819338\n            },\n            {\n              "feature": "syn_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": -0.29921406286165514\n            },\n            {\n              "feature": "flow_duration",\n              "method": "scaled_lr_coefficient",\n              "value": -0.24159368747030682\n            },\n            {\n              "feature": "flow_byts_s",\n              "method": "scaled_lr_coefficient",\n              "value": -0.23808594939360386\n            },\n            {\n              "feature": "fin_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": -0.10447304907465696\n            },\n            {\n              "feature": "fwd_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": -0.040445744628794016\n            },\n            {\n              "feature": "pkt_len_std",\n              "method": "scaled_lr_coefficient",\n              "value": -0.03633762510402606\n            },\n            {\n              "feature": "tot_fwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.03439898174550569\n            },\n            {\n              "feature": "flow_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": -0.032870148850709296\n            },\n            {\n              "feature": "down_up_ratio",\n              "method": "scaled_lr_coefficient",\n              "value": 0.027685986407391772\n            },\n            {\n              "feature": "totlen_fwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.023698087067449074\n            },\n            {\n              "feature": "bwd_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": -0.018089822328427284\n            },\n            {\n              "feature": "tot_bwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": -0.007690750197213877\n            },\n            {\n              "feature": "totlen_bwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.0006219430260135988\n            },\n            {\n              "feature": "flow_iat_std",\n              "method": "scaled_lr_coefficient",\n              "value": -0.0002691260185619522\n            }\n          ]\n        }\n      ],\n      "trial_selections": {\n        "logistic_regression": {\n          "trial": 2,\n          "parameters": {\n            "maxIter": 12,\n            "regParam": 0.05,\n            "elasticNetParam": 0.0\n          }\n        },\n        "random_forest": {\n          "trial": 2,\n          "parameters": {\n            "numTrees": 20,\n            "maxDepth": 7,\n            "minInstancesPerNode": 30\n          }\n        },\n        "gradient_boosted_trees": {\n          "trial": 2,\n          "parameters": {\n            "maxIter": 10,\n            "maxDepth": 4,\n            "stepSize": 0.15\n          }\n        }\n      },\n      "input": {\n        "path": "/kaggle/input/notebooks/ramcharan2345/mmds-phase-2-spark-data-engineering/mmds_phase2/phase2-v1/feature_ready",\n        "days": [\n          "2018-02-14",\n          "2018-02-15",\n          "2018-02-16",\n          "2018-02-20",\n          "2018-02-21",\n          "2018-02-22",\n          "2018-02-23",\n          "2018-02-28",\n          "2018-03-01",\n          "2018-03-02"\n        ],\n        "totals": {\n          "0": 13484708,\n          "1": 2748235\n        },\n        "rows": 16232943,\n        "temporally_flagged": 14,\n        "policy_version": "phase2-v1",\n        "receipt_sha256": "1fa00cd2b9c853ee33eaacee8ca7632dbb41f739573326280d6ffbfb2ccff25a",\n        "producer": {\n          "url": "https://www.kaggle.com/code/ramcharan2345/mmds-phase-2-spark-data-engineering",\n          "kernel_version": 1,\n          "status": "COMPLETE"\n        }\n      },\n      "registered_predictors": [\n        "flow_duration",\n        "tot_fwd_pkts",\n        "tot_bwd_pkts",\n        "totlen_fwd_pkts",\n        "totlen_bwd_pkts",\n        "pkt_len_mean",\n        "pkt_len_std",\n        "flow_iat_mean",\n        "flow_iat_std",\n        "fwd_iat_mean",\n        "bwd_iat_mean",\n        "flow_pkts_s",\n        "flow_byts_s",\n        "down_up_ratio",\n        "syn_flag_cnt",\n        "ack_flag_cnt",\n        "rst_flag_cnt",\n        "fin_flag_cnt"\n      ],\n      "overlap": {\n        "predictor_before": {\n          "groups": 0,\n          "rows": 0\n        },\n        "after": {\n          "predictor_key": {\n            "groups": 0,\n            "rows": 0\n          },\n          "record_fingerprint": {\n            "groups": 0,\n            "rows": 0\n          },\n          "split_feature_key": {\n            "groups": 0,\n            "rows": 0\n          }\n        },\n        "purged_rows": 0\n      },\n      "retained_predictor_conflicts": {\n        "groups": 25011,\n        "rows": 1792767\n      },\n      "split_support": {\n        "train": {\n          "labels": {\n            "0": 9426379,\n            "1": 2029546\n          },\n          "families": {\n            "Benign": 9426379,\n            "Credential brute force": 341040,\n            "DoS": 479941,\n            "DDoS": 885772,\n            "Web attack": 634,\n            "Infiltration": 116242,\n            "Bot": 205917\n          },\n          "original_labels": {\n            "Benign": 9426379,\n            "FTP-BruteForce": 183538,\n            "SSH-Bruteforce": 157502,\n            "DoS attacks-GoldenEye": 29251,\n            "DoS attacks-Slowloris": 7475,\n            "DoS attacks-Hulk": 323298,\n            "DoS attacks-SlowHTTPTest": 119917,\n            "DDoS attacks-LOIC-HTTP": 403700,\n            "DDOS attack-HOIC": 480857,\n            "DDOS attack-LOIC-UDP": 1215,\n            "Brute Force -Web": 411,\n            "Brute Force -XSS": 163,\n            "SQL Injection": 60,\n            "Infilteration": 116242,\n            "Bot": 205917\n          },\n          "rows": 11455925\n        },\n        "validation": {\n          "labels": {\n            "0": 1990157,\n            "1": 354569\n          },\n          "families": {\n            "Benign": 1990157,\n            "Credential brute force": 17801,\n            "DoS": 85648,\n            "DDoS": 189228,\n            "Web attack": 140,\n            "Infiltration": 21285,\n            "Bot": 40467\n          },\n          "original_labels": {\n            "Benign": 1990157,\n            "FTP-BruteForce": 2697,\n            "SSH-Bruteforce": 15104,\n            "DoS attacks-GoldenEye": 6145,\n            "DoS attacks-Slowloris": 1933,\n            "DoS attacks-Hulk": 70165,\n            "DoS attacks-SlowHTTPTest": 7405,\n            "DDoS attacks-LOIC-HTTP": 86510,\n            "DDOS attack-HOIC": 102441,\n            "DDOS attack-LOIC-UDP": 277,\n            "Brute Force -Web": 107,\n            "Brute Force -XSS": 25,\n            "SQL Injection": 8,\n            "Infilteration": 21285,\n            "Bot": 40467\n          },\n          "rows": 2344726\n        },\n        "test": {\n          "labels": {\n            "0": 2068172,\n            "1": 364120\n          },\n          "families": {\n            "Benign": 2068172,\n            "Credential brute force": 22108,\n            "DoS": 88711,\n            "DDoS": 188933,\n            "Web attack": 154,\n            "Infiltration": 24407,\n            "Bot": 39807\n          },\n          "original_labels": {\n            "Benign": 2068172,\n            "FTP-BruteForce": 7125,\n            "SSH-Bruteforce": 14983,\n            "DoS attacks-GoldenEye": 6112,\n            "DoS attacks-Slowloris": 1582,\n            "DoS attacks-Hulk": 68449,\n            "DoS attacks-SlowHTTPTest": 12568,\n            "DDoS attacks-LOIC-HTTP": 85981,\n            "DDOS attack-HOIC": 102714,\n            "DDOS attack-LOIC-UDP": 238,\n            "Brute Force -Web": 93,\n            "Brute Force -XSS": 42,\n            "SQL Injection": 19,\n            "Infilteration": 24407,\n            "Bot": 39807\n          },\n          "rows": 2432292\n        }\n      },\n      "train_sample_support": {\n        "labels": {\n          "0": 591111,\n          "1": 104196\n        },\n        "families": {\n          "Benign": 591111,\n          "Credential brute force": 4989,\n          "DoS": 22380,\n          "DDoS": 56636,\n          "Web attack": 44,\n          "Infiltration": 6207,\n          "Bot": 13940\n        },\n        "original_labels": {\n          "Benign": 591111,\n          "FTP-BruteForce": 806,\n          "SSH-Bruteforce": 4183,\n          "DoS attacks-GoldenEye": 1878,\n          "DoS attacks-Slowloris": 472,\n          "DoS attacks-Hulk": 19884,\n          "DoS attacks-SlowHTTPTest": 146,\n          "DDoS attacks-LOIC-HTTP": 25296,\n          "DDOS attack-HOIC": 31263,\n          "DDOS attack-LOIC-UDP": 77,\n          "Brute Force -Web": 27,\n          "Brute Force -XSS": 12,\n          "SQL Injection": 5,\n          "Infilteration": 6207,\n          "Bot": 13940\n        },\n        "rows": 695307\n      },\n      "baselines": {\n        "majority": {\n          "attack_prediction": 0\n        },\n        "duration_rule": {\n          "benign_training_median_us": 34807.0\n        }\n      },\n      "preprocessing": {\n        "active": [\n          "flow_duration",\n          "tot_fwd_pkts",\n          "tot_bwd_pkts",\n          "totlen_fwd_pkts",\n          "totlen_bwd_pkts",\n          "pkt_len_mean",\n          "pkt_len_std",\n          "flow_iat_mean",\n          "flow_iat_std",\n          "fwd_iat_mean",\n          "bwd_iat_mean",\n          "flow_pkts_s",\n          "flow_byts_s",\n          "down_up_ratio",\n          "syn_flag_cnt",\n          "ack_flag_cnt",\n          "rst_flag_cnt",\n          "fin_flag_cnt"\n        ],\n        "dropped_train_constant_or_empty": [],\n        "training_medians": {\n          "flow_duration": 24873.0,\n          "tot_fwd_pkts": 2.0,\n          "tot_bwd_pkts": 1.0,\n          "totlen_fwd_pkts": 44.0,\n          "totlen_bwd_pkts": 104.0,\n          "pkt_len_mean": 53.6666666667,\n          "pkt_len_std": 33.9587985653203,\n          "flow_iat_mean": 12681.0,\n          "flow_iat_std": 90.86785276799858,\n          "fwd_iat_mean": 4816.0,\n          "bwd_iat_mean": 0.0,\n          "flow_pkts_s": 126.1438957822,\n          "flow_byts_s": 979.496885506,\n          "down_up_ratio": 0.0,\n          "syn_flag_cnt": 0.0,\n          "ack_flag_cnt": 0.0,\n          "rst_flag_cnt": 0.0,\n          "fin_flag_cnt": 0.0\n        }\n      },\n      "preprocess_seconds": 14.6,\n      "post_imputation_overlap": {\n        "groups": 0,\n        "rows": 0\n      },\n      "selection_frozen": true,\n      "elapsed_seconds": 517.592,\n      "downloaded_reload": [\n        {\n          "artifact_directory": "logistic_regression_group_sample",\n          "finite_probe_predictions": 3\n        },\n        {\n          "artifact_directory": "random_forest_group_sample",\n          "finite_probe_predictions": 3\n        },\n        {\n          "artifact_directory": "gradient_boosted_trees_group_sample",\n          "finite_probe_predictions": 3\n        },\n        {\n          "artifact_directory": "logistic_regression_full_train_refit",\n          "finite_probe_predictions": 3\n        }\n      ]\n    },\n    "B": {\n      "version": "phase4-v1",\n      "status": "validated",\n      "log_events": 1091,\n      "protocol": "B",\n      "design": {\n        "name": "chronological capture-day holdout",\n        "train": [\n          "2018-02-14",\n          "2018-02-15",\n          "2018-02-16",\n          "2018-02-20",\n          "2018-02-21",\n          "2018-02-22"\n        ],\n        "validation": [\n          "2018-02-23",\n          "2018-02-28"\n        ],\n        "test": [\n          "2018-03-01",\n          "2018-03-02"\n        ]\n      },\n      "resources": {\n        "spark_version": "4.3.0",\n        "master": "local[4]",\n        "driver_memory": "8g",\n        "shuffle_partitions": 48,\n        "input_partition_bytes": 67108864,\n        "default_parallelism": 4,\n        "cpu_affinity": 4,\n        "scratch_output_free_bytes_start": 20940279808,\n        "memory.max": "32212254720",\n        "cpu.max": "400000 100000",\n        "driver_peak_rss_bytes": 6735474688\n      },\n      "models": [\n        {\n          "path": "/kaggle/working/mmds_phase4/models/logistic_regression_group_sample",\n          "model": "logistic_regression",\n          "reload_probe_rows": 795270,\n          "model_bytes": 16241,\n          "model_files": 16,\n          "sha256": {\n            "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "metadata/part-00000-d09704e3-8b08-46c6-a97c-9d2b4c0bd829-c000.txt": "b222cf21e85042226cad2bab56a4e300e57ec59719dee464ed21214056369120",\n            "stages/0_Imputer_7052b64974ed/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_7052b64974ed/data/part-00000-36cc507f-5d3c-46f9-8d2f-5363bc28c598-c000.snappy.parquet": "22f8f563ed7937c121a2add4a56ab773e344bff06b77e54a2a1c916067da552e",\n            "stages/0_Imputer_7052b64974ed/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_7052b64974ed/metadata/part-00000-381bd52c-4e68-47b0-9d90-698c937d24d6-c000.txt": "8920fa2bcbe8905239fce1612d2adc7edd9350ba3b5b175cf5f3808938a16c56",\n            "stages/1_VectorAssembler_1d71e7b472ad/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/1_VectorAssembler_1d71e7b472ad/metadata/part-00000-d045abaa-13f9-420c-b4ed-be77d4ebaec3-c000.txt": "315c19d9becbaa11fd72a95a79320cf57a5d7b17a42117f2aa6847388d741e3a",\n            "stages/2_StandardScaler_aa30bbdb44b9/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_StandardScaler_aa30bbdb44b9/data/part-00000-2a7c52a9-2af2-43a3-bb68-26ffa3b26989-c000.snappy.parquet": "1e06bec0086ac57a353dca45e6703337e8d0c598cce0aa0d505a2846821723b2",\n            "stages/2_StandardScaler_aa30bbdb44b9/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_StandardScaler_aa30bbdb44b9/metadata/part-00000-31c7267e-71ab-4d7d-ab32-c5618d2dbac5-c000.txt": "cbd9564b9c94ddc4f2e0452c84b149d59ece9b84805a33780d0b9b1cfa8e6f22",\n            "stages/3_LogisticRegression_93ba6b170c64/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/3_LogisticRegression_93ba6b170c64/data/part-00000-eca9e110-62c5-469b-a24c-54d7e36f72fb-c000.snappy.parquet": "b2cb257a862d1dc1c25a1cc6a117f63f3501cae8bc3d20d6a6978b5e706f1940",\n            "stages/3_LogisticRegression_93ba6b170c64/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/3_LogisticRegression_93ba6b170c64/metadata/part-00000-ad94d93b-661f-418a-9b58-e7e85b634947-c000.txt": "6e46ad499a82c57e9c52715d65c9c14379d68068a77dcfe5187bd8445d95daed"\n          },\n          "training_cohort": "group_sample",\n          "training_seconds": 13.5,\n          "parameters": {\n            "maxIter": 12,\n            "regParam": 0.005,\n            "elasticNetParam": 0.0\n          },\n          "explanation": [\n            {\n              "feature": "pkt_len_mean",\n              "method": "scaled_lr_coefficient",\n              "value": -1.780591224500976\n            },\n            {\n              "feature": "flow_byts_s",\n              "method": "scaled_lr_coefficient",\n              "value": -1.112350263625203\n            },\n            {\n              "feature": "pkt_len_std",\n              "method": "scaled_lr_coefficient",\n              "value": 0.923384841756616\n            },\n            {\n              "feature": "flow_duration",\n              "method": "scaled_lr_coefficient",\n              "value": -0.8852168829523376\n            },\n            {\n              "feature": "ack_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": 0.7615032677887672\n            },\n            {\n              "feature": "syn_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": -0.7260335062130642\n            },\n            {\n              "feature": "flow_pkts_s",\n              "method": "scaled_lr_coefficient",\n              "value": 0.6303179128463633\n            },\n            {\n              "feature": "rst_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": 0.5938143723566249\n            },\n            {\n              "feature": "bwd_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": 0.42173284043500003\n            },\n            {\n              "feature": "flow_iat_std",\n              "method": "scaled_lr_coefficient",\n              "value": -0.38031640864136457\n            },\n            {\n              "feature": "fin_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": -0.11848661645584345\n            },\n            {\n              "feature": "tot_fwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.10991900173640909\n            },\n            {\n              "feature": "fwd_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": 0.09485773269568075\n            },\n            {\n              "feature": "flow_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": 0.09328298989681266\n            },\n            {\n              "feature": "totlen_fwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.06828791267348731\n            },\n            {\n              "feature": "down_up_ratio",\n              "method": "scaled_lr_coefficient",\n              "value": 0.035025609080342794\n            },\n            {\n              "feature": "tot_bwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": -0.01209622751847114\n            },\n            {\n              "feature": "totlen_bwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.0020208612240548065\n            }\n          ]\n        },\n        {\n          "path": "/kaggle/working/mmds_phase4/models/random_forest_group_sample",\n          "model": "random_forest",\n          "reload_probe_rows": 795270,\n          "model_bytes": 62923,\n          "model_files": 14,\n          "sha256": {\n            "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "metadata/part-00000-e921fc7c-54c9-4406-b75c-5897681c5ac6-c000.txt": "b781e9c5b3ddcc480bfd4bf9375ded0c7a68f2f62be6752befc81a1c60b93f14",\n            "stages/0_Imputer_7052b64974ed/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_7052b64974ed/data/part-00000-58e83166-5de1-41a4-91fb-df039444c268-c000.snappy.parquet": "22f8f563ed7937c121a2add4a56ab773e344bff06b77e54a2a1c916067da552e",\n            "stages/0_Imputer_7052b64974ed/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_7052b64974ed/metadata/part-00000-5ef38d51-4df2-4030-a68a-7e20ea6dfebc-c000.txt": "3dcf615dc6ead2f20284250a3d292977eae18d866bacaf1aa147293fe3302494",\n            "stages/1_VectorAssembler_1d71e7b472ad/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/1_VectorAssembler_1d71e7b472ad/metadata/part-00000-a1a7f9ec-5f45-4366-8c33-6e3acb58b507-c000.txt": "08aa06a7f9a6c6a72fabbba50f8dd3d9edf824fde87adc41811e9d533fe03976",\n            "stages/2_RandomForestClassifier_e65672fa318a/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_RandomForestClassifier_e65672fa318a/data/part-00000-8cf4c00d-8eeb-4f4e-8415-17f0daa8614f-c000.snappy.parquet": "f2446f6db6876001a19d9a3030d77d7218d82d9d8c8678fab71feb696ebb2988",\n            "stages/2_RandomForestClassifier_e65672fa318a/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_RandomForestClassifier_e65672fa318a/metadata/part-00000-22903d16-7742-4297-a8b6-f94998acfbd2-c000.txt": "9a9f25c21aae0ba3d2959cd946f89a3c58b3ce88543f0fe7627036498609d7dd",\n            "stages/2_RandomForestClassifier_e65672fa318a/treesMetadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_RandomForestClassifier_e65672fa318a/treesMetadata/part-00000-5d0f9c54-e9e2-4e49-9430-8fee7f1ab794-c000.snappy.parquet": "111a1b821b5a16165f68182cc327be978b75e07b03aaa27ca64f146054838b01"\n          },\n          "training_cohort": "group_sample",\n          "training_seconds": 19.239,\n          "parameters": {\n            "numTrees": 20,\n            "maxDepth": 7,\n            "minInstancesPerNode": 30\n          },\n          "explanation": [\n            {\n              "feature": "totlen_fwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.15005228174630172\n            },\n            {\n              "feature": "ack_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.09894914887463935\n            },\n            {\n              "feature": "fwd_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.09744757969988256\n            },\n            {\n              "feature": "flow_pkts_s",\n              "method": "impurity_split_importance",\n              "value": 0.08211103799790603\n            },\n            {\n              "feature": "tot_bwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.08190782732459835\n            },\n            {\n              "feature": "flow_duration",\n              "method": "impurity_split_importance",\n              "value": 0.07810701388066098\n            },\n            {\n              "feature": "flow_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.0662482749594117\n            },\n            {\n              "feature": "totlen_bwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.06442200035495507\n            },\n            {\n              "feature": "tot_fwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.06201574725367744\n            },\n            {\n              "feature": "flow_byts_s",\n              "method": "impurity_split_importance",\n              "value": 0.061839212527291465\n            },\n            {\n              "feature": "pkt_len_mean",\n              "method": "impurity_split_importance",\n              "value": 0.06049114563830391\n            },\n            {\n              "feature": "pkt_len_std",\n              "method": "impurity_split_importance",\n              "value": 0.059348765425908626\n            },\n            {\n              "feature": "flow_iat_std",\n              "method": "impurity_split_importance",\n              "value": 0.011960422418420463\n            },\n            {\n              "feature": "down_up_ratio",\n              "method": "impurity_split_importance",\n              "value": 0.011442614078587212\n            },\n            {\n              "feature": "rst_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.00955825948925279\n            },\n            {\n              "feature": "bwd_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.0028809813214295205\n            },\n            {\n              "feature": "fin_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.0006229749712901726\n            },\n            {\n              "feature": "syn_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.0005947120374825553\n            }\n          ]\n        },\n        {\n          "path": "/kaggle/working/mmds_phase4/models/gradient_boosted_trees_group_sample",\n          "model": "gradient_boosted_trees",\n          "reload_probe_rows": 795270,\n          "model_bytes": 20054,\n          "model_files": 14,\n          "sha256": {\n            "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "metadata/part-00000-efbdecf1-4803-45db-ae68-abe98da0e199-c000.txt": "4123305ce71538ad80e064db257b9f735fcec77b5eabc50cc023cf1fde3f1b80",\n            "stages/0_Imputer_7052b64974ed/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_7052b64974ed/data/part-00000-8adbc5fc-0c93-42fa-b346-dd9476f83a90-c000.snappy.parquet": "22f8f563ed7937c121a2add4a56ab773e344bff06b77e54a2a1c916067da552e",\n            "stages/0_Imputer_7052b64974ed/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_7052b64974ed/metadata/part-00000-b6cab653-d0fd-4c77-aca7-aab3cfc2b275-c000.txt": "d03fd702a9c4695d1d13ca1b594cb8fa037f5ae5e9ff6f730a4d411bdb2f05f6",\n            "stages/1_VectorAssembler_1d71e7b472ad/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/1_VectorAssembler_1d71e7b472ad/metadata/part-00000-32eec798-d9d3-45b1-9045-7d5ebc915551-c000.txt": "ede986a40e1120df8143417f1d79f5f295da93aef163afaec6eb614392470d48",\n            "stages/2_GBTClassifier_ed0a682fa045/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_GBTClassifier_ed0a682fa045/data/part-00000-d216835b-7635-4173-8e6e-c09c9b700350-c000.snappy.parquet": "481f7cf4fa2ee3a69a170fe2752ecf03f2f2ba20a86f2cde92d0540d2b30e60e",\n            "stages/2_GBTClassifier_ed0a682fa045/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_GBTClassifier_ed0a682fa045/metadata/part-00000-140fbea1-593b-4019-9aeb-0b01b674de20-c000.txt": "3049f6423197b1b3a40e8355ba44fc77c826bc2426ab0c2ede9eeb537c64eb98",\n            "stages/2_GBTClassifier_ed0a682fa045/treesMetadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_GBTClassifier_ed0a682fa045/treesMetadata/part-00000-2fee5052-d4dd-4dbf-9da0-ba29400facaa-c000.snappy.parquet": "7603b2afadd15d52949931d502b65ea74d4637aa058f3bcd4616ade79fc78234"\n          },\n          "training_cohort": "group_sample",\n          "training_seconds": 17.415,\n          "parameters": {\n            "maxIter": 6,\n            "maxDepth": 3,\n            "stepSize": 0.1\n          },\n          "explanation": [\n            {\n              "feature": "tot_bwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.2339302773742976\n            },\n            {\n              "feature": "ack_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.21187176671645613\n            },\n            {\n              "feature": "totlen_fwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.15919712437755826\n            },\n            {\n              "feature": "fwd_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.11143762140075865\n            },\n            {\n              "feature": "flow_duration",\n              "method": "impurity_split_importance",\n              "value": 0.10278018282023525\n            },\n            {\n              "feature": "flow_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.08532863814951952\n            },\n            {\n              "feature": "flow_iat_std",\n              "method": "impurity_split_importance",\n              "value": 0.03942685523415636\n            },\n            {\n              "feature": "totlen_bwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.027452293578463044\n            },\n            {\n              "feature": "pkt_len_mean",\n              "method": "impurity_split_importance",\n              "value": 0.018163734698595824\n            },\n            {\n              "feature": "pkt_len_std",\n              "method": "impurity_split_importance",\n              "value": 0.0053939156650885615\n            },\n            {\n              "feature": "tot_fwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.005017589984870909\n            },\n            {\n              "feature": "bwd_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.0\n            },\n            {\n              "feature": "down_up_ratio",\n              "method": "impurity_split_importance",\n              "value": 0.0\n            },\n            {\n              "feature": "fin_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.0\n            },\n            {\n              "feature": "flow_byts_s",\n              "method": "impurity_split_importance",\n              "value": 0.0\n            },\n            {\n              "feature": "flow_pkts_s",\n              "method": "impurity_split_importance",\n              "value": 0.0\n            },\n            {\n              "feature": "rst_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.0\n            },\n            {\n              "feature": "syn_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.0\n            }\n          ]\n        },\n        {\n          "path": "/kaggle/working/mmds_phase4/models/logistic_regression_full_train_refit",\n          "model": "logistic_regression",\n          "reload_probe_rows": 795270,\n          "model_bytes": 16241,\n          "model_files": 16,\n          "sha256": {\n            "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "metadata/part-00000-0e78daa9-def3-46bd-8d5f-5607608f4ecb-c000.txt": "c82e755e8e0ae81bb51041d109e4101692b6a30bb7821a0463317d0e038e2a3c",\n            "stages/0_Imputer_7052b64974ed/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_7052b64974ed/data/part-00000-4c23c8bb-1bf5-44f5-87d0-b52d0d40765c-c000.snappy.parquet": "22f8f563ed7937c121a2add4a56ab773e344bff06b77e54a2a1c916067da552e",\n            "stages/0_Imputer_7052b64974ed/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_7052b64974ed/metadata/part-00000-8746ef4d-a351-4a4f-a0a0-59d4f4ad63cf-c000.txt": "aa498b839593c79905d79b3cdc9ad123863e275870570fb39e73f4cae40e2cd2",\n            "stages/1_VectorAssembler_1d71e7b472ad/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/1_VectorAssembler_1d71e7b472ad/metadata/part-00000-80ecd58f-b8a5-4270-9518-9b73b5085a67-c000.txt": "e2633b6193a8cc69fb646fde56d7fa6b4e3d5331111230c1953ebd5cadff28d6",\n            "stages/2_StandardScaler_aa30bbdb44b9/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_StandardScaler_aa30bbdb44b9/data/part-00000-e073eb75-12e6-4a47-a5a7-38a1b3e8cd41-c000.snappy.parquet": "1e06bec0086ac57a353dca45e6703337e8d0c598cce0aa0d505a2846821723b2",\n            "stages/2_StandardScaler_aa30bbdb44b9/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_StandardScaler_aa30bbdb44b9/metadata/part-00000-de360e70-a176-4c7a-a035-1f85b59a51d7-c000.txt": "16e55e02ebe10c0612e1ce1db5d2d4084b26f79c0aa6089f930a1cebf5b0af8c",\n            "stages/3_LogisticRegression_80d3fafce516/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/3_LogisticRegression_80d3fafce516/data/part-00000-a186bbfc-7c71-40fe-b226-a0e11ceddac0-c000.snappy.parquet": "3e97852a5efcad0aff181529fb2fad7b578e7d596277a0086b866f3b404b95d9",\n            "stages/3_LogisticRegression_80d3fafce516/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/3_LogisticRegression_80d3fafce516/metadata/part-00000-c7566b37-a7d4-44ba-85a0-d8d8df05e671-c000.txt": "6e3e4a6e7c88ebe2d671ac4f72e452177d73d214fecbbef5a04556c90e9822e5"\n          },\n          "training_cohort": "full_train_refit",\n          "training_seconds": 44.995,\n          "parameters": {\n            "maxIter": 12,\n            "regParam": 0.005,\n            "elasticNetParam": 0.0\n          },\n          "explanation": [\n            {\n              "feature": "pkt_len_mean",\n              "method": "scaled_lr_coefficient",\n              "value": -1.8670591581079088\n            },\n            {\n              "feature": "pkt_len_std",\n              "method": "scaled_lr_coefficient",\n              "value": 0.9680905133125925\n            },\n            {\n              "feature": "flow_duration",\n              "method": "scaled_lr_coefficient",\n              "value": -0.9397318701779404\n            },\n            {\n              "feature": "flow_byts_s",\n              "method": "scaled_lr_coefficient",\n              "value": -0.6402688432634688\n            },\n            {\n              "feature": "ack_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": 0.6284176725086255\n            },\n            {\n              "feature": "syn_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": -0.6224668326042223\n            },\n            {\n              "feature": "rst_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": 0.5455725718029051\n            },\n            {\n              "feature": "flow_pkts_s",\n              "method": "scaled_lr_coefficient",\n              "value": 0.3899168903099706\n            },\n            {\n              "feature": "down_up_ratio",\n              "method": "scaled_lr_coefficient",\n              "value": 0.18137120862326153\n            },\n            {\n              "feature": "bwd_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": 0.177248781830544\n            },\n            {\n              "feature": "fin_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": -0.1533909502840441\n            },\n            {\n              "feature": "flow_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": 0.13777249981586884\n            },\n            {\n              "feature": "fwd_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": 0.10781704655154939\n            },\n            {\n              "feature": "tot_fwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.10341793377036967\n            },\n            {\n              "feature": "totlen_fwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.061376137349828615\n            },\n            {\n              "feature": "tot_bwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": -0.022734416750892123\n            },\n            {\n              "feature": "totlen_bwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.00469054131037264\n            },\n            {\n              "feature": "flow_iat_std",\n              "method": "scaled_lr_coefficient",\n              "value": -0.0001483209333920456\n            }\n          ]\n        }\n      ],\n      "trial_selections": {\n        "logistic_regression": {\n          "trial": 1,\n          "parameters": {\n            "maxIter": 12,\n            "regParam": 0.005,\n            "elasticNetParam": 0.0\n          }\n        },\n        "random_forest": {\n          "trial": 2,\n          "parameters": {\n            "numTrees": 20,\n            "maxDepth": 7,\n            "minInstancesPerNode": 30\n          }\n        },\n        "gradient_boosted_trees": {\n          "trial": 1,\n          "parameters": {\n            "maxIter": 6,\n            "maxDepth": 3,\n            "stepSize": 0.1\n          }\n        }\n      },\n      "input": {\n        "path": "/kaggle/input/notebooks/ramcharan2345/mmds-phase-2-spark-data-engineering/mmds_phase2/phase2-v1/feature_ready",\n        "days": [\n          "2018-02-14",\n          "2018-02-15",\n          "2018-02-16",\n          "2018-02-20",\n          "2018-02-21",\n          "2018-02-22",\n          "2018-02-23",\n          "2018-02-28",\n          "2018-03-01",\n          "2018-03-02"\n        ],\n        "totals": {\n          "0": 13484708,\n          "1": 2748235\n        },\n        "rows": 16232943,\n        "temporally_flagged": 14,\n        "policy_version": "phase2-v1",\n        "receipt_sha256": "1fa00cd2b9c853ee33eaacee8ca7632dbb41f739573326280d6ffbfb2ccff25a",\n        "producer": {\n          "url": "https://www.kaggle.com/code/ramcharan2345/mmds-phase-2-spark-data-engineering",\n          "kernel_version": 1,\n          "status": "COMPLETE"\n        }\n      },\n      "registered_predictors": [\n        "flow_duration",\n        "tot_fwd_pkts",\n        "tot_bwd_pkts",\n        "totlen_fwd_pkts",\n        "totlen_bwd_pkts",\n        "pkt_len_mean",\n        "pkt_len_std",\n        "flow_iat_mean",\n        "flow_iat_std",\n        "fwd_iat_mean",\n        "bwd_iat_mean",\n        "flow_pkts_s",\n        "flow_byts_s",\n        "down_up_ratio",\n        "syn_flag_cnt",\n        "ack_flag_cnt",\n        "rst_flag_cnt",\n        "fin_flag_cnt"\n      ],\n      "overlap": {\n        "predictor_before": {\n          "groups": 136985,\n          "rows": 2773438\n        },\n        "after": {\n          "predictor_key": {\n            "groups": 0,\n            "rows": 0\n          },\n          "record_fingerprint": {\n            "groups": 0,\n            "rows": 0\n          },\n          "split_feature_key": {\n            "groups": 0,\n            "rows": 0\n          }\n        },\n        "purged_rows": 909956\n      },\n      "retained_predictor_conflicts": {\n        "groups": 12312,\n        "rows": 784773\n      },\n      "split_support": {\n        "train": {\n          "labels": {\n            "0": 10892078,\n            "1": 2299544\n          },\n          "families": {\n            "Benign": 10892078,\n            "Credential brute force": 380949,\n            "DoS": 654300,\n            "DDoS": 1263933,\n            "Web attack": 362\n          },\n          "original_labels": {\n            "Benign": 10892078,\n            "FTP-BruteForce": 193360,\n            "SSH-Bruteforce": 187589,\n            "DoS attacks-GoldenEye": 41508,\n            "DoS attacks-Slowloris": 10990,\n            "DoS attacks-Hulk": 461912,\n            "DoS attacks-SlowHTTPTest": 139890,\n            "DDoS attacks-LOIC-HTTP": 576191,\n            "DDOS attack-HOIC": 686012,\n            "DDOS attack-LOIC-UDP": 1730,\n            "Brute Force -Web": 249,\n            "Brute Force -XSS": 79,\n            "SQL Injection": 34\n          },\n          "rows": 13191622\n        },\n        "validation": {\n          "labels": {\n            "0": 1094200,\n            "1": 57056\n          },\n          "families": {\n            "Benign": 1094200,\n            "Web attack": 423,\n            "Infiltration": 56633\n          },\n          "original_labels": {\n            "Benign": 1094200,\n            "Brute Force -Web": 276,\n            "Brute Force -XSS": 111,\n            "SQL Injection": 36,\n            "Infilteration": 56633\n          },\n          "rows": 1151256\n        },\n        "test": {\n          "labels": {\n            "0": 771584,\n            "1": 208525\n          },\n          "families": {\n            "Benign": 771584,\n            "Infiltration": 65251,\n            "Bot": 143274\n          },\n          "original_labels": {\n            "Benign": 771584,\n            "Infilteration": 65251,\n            "Bot": 143274\n          },\n          "rows": 980109\n        }\n      },\n      "train_sample_support": {\n        "labels": {\n          "0": 664936,\n          "1": 130334\n        },\n        "families": {\n          "Benign": 664936,\n          "Credential brute force": 12384,\n          "DoS": 37445,\n          "DDoS": 80477,\n          "Web attack": 28\n        },\n        "original_labels": {\n          "Benign": 664936,\n          "FTP-BruteForce": 6425,\n          "SSH-Bruteforce": 5959,\n          "DoS attacks-GoldenEye": 2663,\n          "DoS attacks-Slowloris": 684,\n          "DoS attacks-Hulk": 28198,\n          "DoS attacks-SlowHTTPTest": 5900,\n          "DDoS attacks-LOIC-HTTP": 36066,\n          "DDOS attack-HOIC": 44289,\n          "DDOS attack-LOIC-UDP": 122,\n          "Brute Force -Web": 20,\n          "Brute Force -XSS": 5,\n          "SQL Injection": 3\n        },\n        "rows": 795270\n      },\n      "baselines": {\n        "majority": {\n          "attack_prediction": 0\n        },\n        "duration_rule": {\n          "benign_training_median_us": 30201.0\n        }\n      },\n      "preprocessing": {\n        "active": [\n          "flow_duration",\n          "tot_fwd_pkts",\n          "tot_bwd_pkts",\n          "totlen_fwd_pkts",\n          "totlen_bwd_pkts",\n          "pkt_len_mean",\n          "pkt_len_std",\n          "flow_iat_mean",\n          "flow_iat_std",\n          "fwd_iat_mean",\n          "bwd_iat_mean",\n          "flow_pkts_s",\n          "flow_byts_s",\n          "down_up_ratio",\n          "syn_flag_cnt",\n          "ack_flag_cnt",\n          "rst_flag_cnt",\n          "fin_flag_cnt"\n        ],\n        "dropped_train_constant_or_empty": [],\n        "training_medians": {\n          "flow_duration": 29185.0,\n          "tot_fwd_pkts": 2.0,\n          "tot_bwd_pkts": 1.0,\n          "totlen_fwd_pkts": 44.0,\n          "totlen_bwd_pkts": 110.0,\n          "pkt_len_mean": 54.5,\n          "pkt_len_std": 35.21836642056717,\n          "flow_iat_mean": 14048.0,\n          "flow_iat_std": 106.4700270812,\n          "fwd_iat_mean": 6398.0,\n          "bwd_iat_mean": 0.0,\n          "flow_pkts_s": 112.1361945053,\n          "flow_byts_s": 948.1289043378442,\n          "down_up_ratio": 0.0,\n          "syn_flag_cnt": 0.0,\n          "ack_flag_cnt": 0.0,\n          "rst_flag_cnt": 0.0,\n          "fin_flag_cnt": 0.0\n        }\n      },\n      "preprocess_seconds": 21.572,\n      "post_imputation_overlap": {\n        "groups": 0,\n        "rows": 0\n      },\n      "selection_frozen": true,\n      "elapsed_seconds": 745.004,\n      "downloaded_reload": [\n        {\n          "artifact_directory": "logistic_regression_group_sample",\n          "finite_probe_predictions": 3\n        },\n        {\n          "artifact_directory": "random_forest_group_sample",\n          "finite_probe_predictions": 3\n        },\n        {\n          "artifact_directory": "gradient_boosted_trees_group_sample",\n          "finite_probe_predictions": 3\n        },\n        {\n          "artifact_directory": "logistic_regression_full_train_refit",\n          "finite_probe_predictions": 3\n        }\n      ]\n    },\n    "C": {\n      "version": "phase4-v1",\n      "status": "validated",\n      "log_events": 836,\n      "protocol": "C",\n      "design": {\n        "name": "web-attack capture holdout",\n        "train": [\n          "2018-02-14",\n          "2018-02-15",\n          "2018-02-16",\n          "2018-02-20",\n          "2018-02-21",\n          "2018-03-01",\n          "2018-03-02"\n        ],\n        "validation": [\n          "2018-02-28"\n        ],\n        "test": [\n          "2018-02-22",\n          "2018-02-23"\n        ]\n      },\n      "resources": {\n        "spark_version": "4.3.0",\n        "master": "local[4]",\n        "driver_memory": "8g",\n        "shuffle_partitions": 48,\n        "input_partition_bytes": 67108864,\n        "default_parallelism": 4,\n        "cpu_affinity": 4,\n        "scratch_output_free_bytes_start": 20940279808,\n        "memory.max": "32212254720",\n        "cpu.max": "400000 100000",\n        "driver_peak_rss_bytes": 7445041152\n      },\n      "models": [\n        {\n          "path": "/kaggle/working/mmds_phase4/models/logistic_regression_group_sample",\n          "model": "logistic_regression",\n          "reload_probe_rows": 812520,\n          "model_bytes": 16241,\n          "model_files": 16,\n          "sha256": {\n            "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "metadata/part-00000-5f3aed8d-e6c0-4c41-8408-d4ebc7759bff-c000.txt": "6f5a888c299e3b95f5e67ac04ceb3c8e09256811c154ece318a45a788d0ac76e",\n            "stages/0_Imputer_75e58e12f8f2/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_75e58e12f8f2/data/part-00000-568211c1-d62a-4813-87d4-721e07961390-c000.snappy.parquet": "b95f973e5c9d08382e52dde2888bfb19b4371d8eb2000e48801a085ca373c01a",\n            "stages/0_Imputer_75e58e12f8f2/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_75e58e12f8f2/metadata/part-00000-a5350c72-13b1-4671-8b21-405b164fcf5f-c000.txt": "7d40b5f74f6e0dc9cafa6a2511a24c70e55e6e8a88d06433463fc5f8f37511eb",\n            "stages/1_VectorAssembler_bc29ebd64ced/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/1_VectorAssembler_bc29ebd64ced/metadata/part-00000-b23307b0-99ea-4929-ac94-98ec16ab3b92-c000.txt": "c6c93ab74095ae50bc62d2157fa3fed7842b6fb4e5b7d4b7ee30e8d6a0e7f2c1",\n            "stages/2_StandardScaler_611ae33371ff/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_StandardScaler_611ae33371ff/data/part-00000-7df6f915-7e7b-4704-b777-ba432f18668b-c000.snappy.parquet": "e5ca48abe1b71ba77b74e5909088ec9af7ba5e9fbe4af16dbd0c3c2b4c22031d",\n            "stages/2_StandardScaler_611ae33371ff/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_StandardScaler_611ae33371ff/metadata/part-00000-82921780-3bde-4b7b-8af1-38cd239d38bb-c000.txt": "8b9d547dfefa311dc6af3b9f6801b9928e7c151326509dce13baeea0ba0ed736",\n            "stages/3_LogisticRegression_acfcb9fe427d/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/3_LogisticRegression_acfcb9fe427d/data/part-00000-9d7f0f44-8be6-4a81-890c-5495f53dd9f2-c000.snappy.parquet": "21faf6cd326a4950c2d07b386fd8a8c96b1bb7f9a9d916394f09d6c03f14019b",\n            "stages/3_LogisticRegression_acfcb9fe427d/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/3_LogisticRegression_acfcb9fe427d/metadata/part-00000-e761b683-a3b3-4662-b732-5d1a3d903a77-c000.txt": "402637179e659a8a028b928e74f9499db623abf40d4256bffb6fbef1566d5f74"\n          },\n          "training_cohort": "group_sample",\n          "training_seconds": 10.976,\n          "parameters": {\n            "maxIter": 12,\n            "regParam": 0.005,\n            "elasticNetParam": 0.0\n          },\n          "explanation": [\n            {\n              "feature": "pkt_len_mean",\n              "method": "scaled_lr_coefficient",\n              "value": -1.7785150131300438\n            },\n            {\n              "feature": "flow_byts_s",\n              "method": "scaled_lr_coefficient",\n              "value": -1.0410201765929772\n            },\n            {\n              "feature": "flow_duration",\n              "method": "scaled_lr_coefficient",\n              "value": -0.8030793330807489\n            },\n            {\n              "feature": "rst_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": 0.7534496577589579\n            },\n            {\n              "feature": "ack_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": 0.7343297701813489\n            },\n            {\n              "feature": "pkt_len_std",\n              "method": "scaled_lr_coefficient",\n              "value": 0.7049722561445563\n            },\n            {\n              "feature": "syn_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": -0.6645100754422375\n            },\n            {\n              "feature": "flow_pkts_s",\n              "method": "scaled_lr_coefficient",\n              "value": 0.5080633310751165\n            },\n            {\n              "feature": "bwd_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": 0.3738130260259236\n            },\n            {\n              "feature": "flow_iat_std",\n              "method": "scaled_lr_coefficient",\n              "value": -0.3134998622713633\n            },\n            {\n              "feature": "fin_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": -0.1173579512487568\n            },\n            {\n              "feature": "tot_fwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.10171587944469686\n            },\n            {\n              "feature": "fwd_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": 0.07664484544466971\n            },\n            {\n              "feature": "totlen_fwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.07118048655568075\n            },\n            {\n              "feature": "flow_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": 0.06730844086707058\n            },\n            {\n              "feature": "down_up_ratio",\n              "method": "scaled_lr_coefficient",\n              "value": 0.01772022458770494\n            },\n            {\n              "feature": "tot_bwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": -0.01632524693025906\n            },\n            {\n              "feature": "totlen_bwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.0025806242947113618\n            }\n          ]\n        },\n        {\n          "path": "/kaggle/working/mmds_phase4/models/random_forest_group_sample",\n          "model": "random_forest",\n          "reload_probe_rows": 812520,\n          "model_bytes": 30720,\n          "model_files": 14,\n          "sha256": {\n            "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "metadata/part-00000-e87e419d-477f-4831-880c-c3b5b4236ba1-c000.txt": "50b7f1b39629d827f55bac65a9ccf7011fe4a7085ac18e7d18978bea619d9aa8",\n            "stages/0_Imputer_75e58e12f8f2/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_75e58e12f8f2/data/part-00000-088020e3-6b6d-40af-9c21-cf635b36a09a-c000.snappy.parquet": "b95f973e5c9d08382e52dde2888bfb19b4371d8eb2000e48801a085ca373c01a",\n            "stages/0_Imputer_75e58e12f8f2/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_75e58e12f8f2/metadata/part-00000-4cf65d3b-775b-440f-9dad-69454a3372b6-c000.txt": "4a967ba5fcd67d0945183b3015ef2f47ccd2eb1b9702a15bed744e9ee4189111",\n            "stages/1_VectorAssembler_bc29ebd64ced/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/1_VectorAssembler_bc29ebd64ced/metadata/part-00000-2acd1782-0a6d-4bcb-a1b9-3d82e1874db5-c000.txt": "52fe7777948866b88f884557ec2800ffd48ae8b714713fed745ceb6829fc63f6",\n            "stages/2_RandomForestClassifier_2155d2c82f7d/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_RandomForestClassifier_2155d2c82f7d/data/part-00000-0b8e5abd-1473-449e-a5ce-3d82b78c1984-c000.snappy.parquet": "ae6f9f9fe8a564716b870ef1ab34055ad594077b76abf554719aca28bf3e8a25",\n            "stages/2_RandomForestClassifier_2155d2c82f7d/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_RandomForestClassifier_2155d2c82f7d/metadata/part-00000-f53cf08d-2505-478a-99b7-3a2dbad69d7c-c000.txt": "9cb36682d6caa92e2c51d874a10abe1132cd4e212856c8f0f6a3e5a95032e3bc",\n            "stages/2_RandomForestClassifier_2155d2c82f7d/treesMetadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_RandomForestClassifier_2155d2c82f7d/treesMetadata/part-00000-0dde3d69-b538-4de2-ab4c-d4164f038eb4-c000.snappy.parquet": "77dae29a6fe030f4e3acc0a91e88a57402111f5f94f40fb35048f6ec7973ab0d"\n          },\n          "training_cohort": "group_sample",\n          "training_seconds": 9.08,\n          "parameters": {\n            "numTrees": 12,\n            "maxDepth": 5,\n            "minInstancesPerNode": 30\n          },\n          "explanation": [\n            {\n              "feature": "totlen_fwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.1834688886651975\n            },\n            {\n              "feature": "fwd_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.17999052420586634\n            },\n            {\n              "feature": "flow_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.09223460132340447\n            },\n            {\n              "feature": "tot_bwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.07188866158158025\n            },\n            {\n              "feature": "flow_duration",\n              "method": "impurity_split_importance",\n              "value": 0.06977032009191769\n            },\n            {\n              "feature": "flow_pkts_s",\n              "method": "impurity_split_importance",\n              "value": 0.06713696862479576\n            },\n            {\n              "feature": "tot_fwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.06233368101232681\n            },\n            {\n              "feature": "pkt_len_mean",\n              "method": "impurity_split_importance",\n              "value": 0.05861413688890729\n            },\n            {\n              "feature": "ack_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.05150918768659484\n            },\n            {\n              "feature": "pkt_len_std",\n              "method": "impurity_split_importance",\n              "value": 0.04881579751315323\n            },\n            {\n              "feature": "flow_byts_s",\n              "method": "impurity_split_importance",\n              "value": 0.0472943636653173\n            },\n            {\n              "feature": "totlen_bwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.03154728010101541\n            },\n            {\n              "feature": "rst_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.012938145915254228\n            },\n            {\n              "feature": "flow_iat_std",\n              "method": "impurity_split_importance",\n              "value": 0.012289366606927317\n            },\n            {\n              "feature": "down_up_ratio",\n              "method": "impurity_split_importance",\n              "value": 0.00648405002904396\n            },\n            {\n              "feature": "bwd_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.002626307110376393\n            },\n            {\n              "feature": "fin_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.0010577189783211242\n            },\n            {\n              "feature": "syn_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.0\n            }\n          ]\n        },\n        {\n          "path": "/kaggle/working/mmds_phase4/models/gradient_boosted_trees_group_sample",\n          "model": "gradient_boosted_trees",\n          "reload_probe_rows": 812520,\n          "model_bytes": 20175,\n          "model_files": 14,\n          "sha256": {\n            "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "metadata/part-00000-28d7f8b4-8ae7-46cf-9ed3-61ab76345462-c000.txt": "9be6d1901d29d8df31aa5f6c5987491a0bbc62f1d026e1227b99be02647e24a6",\n            "stages/0_Imputer_75e58e12f8f2/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_75e58e12f8f2/data/part-00000-5578f0ca-09f6-4d1f-8a1b-63219920519f-c000.snappy.parquet": "b95f973e5c9d08382e52dde2888bfb19b4371d8eb2000e48801a085ca373c01a",\n            "stages/0_Imputer_75e58e12f8f2/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_75e58e12f8f2/metadata/part-00000-017f6684-c8b5-46ad-8526-a51bbf358713-c000.txt": "2a42bfb2b7bbb8602d3fa621a90eeea9d2e92abb20204d98eadfbfdad2c164ab",\n            "stages/1_VectorAssembler_bc29ebd64ced/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/1_VectorAssembler_bc29ebd64ced/metadata/part-00000-d8c453d2-e5e3-45fa-8e5d-8d2d1ea513a0-c000.txt": "4017fe5436b7188ce9285079cdc097d8ed69f5472e6744ce75353c801caba4ad",\n            "stages/2_GBTClassifier_3fc58e0cc2fc/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_GBTClassifier_3fc58e0cc2fc/data/part-00000-b101510e-4864-4f65-86c8-dd9ac3e9792d-c000.snappy.parquet": "e4920fc7039ec7c457169130f20824828a691b620326d2f170a660511c173f18",\n            "stages/2_GBTClassifier_3fc58e0cc2fc/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_GBTClassifier_3fc58e0cc2fc/metadata/part-00000-f7d506b0-6dcc-409c-a756-af0ae33184eb-c000.txt": "1bec11f595f0766317c60ad02e660259f468cb7eeda98a2bae2b894136dd4e23",\n            "stages/2_GBTClassifier_3fc58e0cc2fc/treesMetadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_GBTClassifier_3fc58e0cc2fc/treesMetadata/part-00000-42c3cf4a-3edf-4e98-978e-1d47e8d34860-c000.snappy.parquet": "2bc9334240b0b57bec706ca40e6f46b17cf643d7d16b9ff3393e16dd3ea8f7a9"\n          },\n          "training_cohort": "group_sample",\n          "training_seconds": 11.686,\n          "parameters": {\n            "maxIter": 6,\n            "maxDepth": 3,\n            "stepSize": 0.1\n          },\n          "explanation": [\n            {\n              "feature": "totlen_fwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.2517482769853455\n            },\n            {\n              "feature": "fwd_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.17895648984132703\n            },\n            {\n              "feature": "tot_fwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.14091877663723415\n            },\n            {\n              "feature": "ack_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.11721792049786338\n            },\n            {\n              "feature": "tot_bwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.09355878219713805\n            },\n            {\n              "feature": "flow_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.08489831401785873\n            },\n            {\n              "feature": "flow_iat_std",\n              "method": "impurity_split_importance",\n              "value": 0.05290235565805834\n            },\n            {\n              "feature": "flow_byts_s",\n              "method": "impurity_split_importance",\n              "value": 0.029652021330934645\n            },\n            {\n              "feature": "rst_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.01869923511136819\n            },\n            {\n              "feature": "flow_duration",\n              "method": "impurity_split_importance",\n              "value": 0.015432340067010655\n            },\n            {\n              "feature": "flow_pkts_s",\n              "method": "impurity_split_importance",\n              "value": 0.011706679680500173\n            },\n            {\n              "feature": "syn_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.0026407667398478285\n            },\n            {\n              "feature": "totlen_bwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.00166804123551327\n            },\n            {\n              "feature": "bwd_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.0\n            },\n            {\n              "feature": "down_up_ratio",\n              "method": "impurity_split_importance",\n              "value": 0.0\n            },\n            {\n              "feature": "fin_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.0\n            },\n            {\n              "feature": "pkt_len_mean",\n              "method": "impurity_split_importance",\n              "value": 0.0\n            },\n            {\n              "feature": "pkt_len_std",\n              "method": "impurity_split_importance",\n              "value": 0.0\n            }\n          ]\n        },\n        {\n          "path": "/kaggle/working/mmds_phase4/models/logistic_regression_full_train_refit",\n          "model": "logistic_regression",\n          "reload_probe_rows": 812520,\n          "model_bytes": 16241,\n          "model_files": 16,\n          "sha256": {\n            "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "metadata/part-00000-91526216-e836-468f-a884-814d345baba3-c000.txt": "29bf7ba3ab603bead56262a35a798ff3ec1d6f7697221f490bd80de514f4e5be",\n            "stages/0_Imputer_75e58e12f8f2/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_75e58e12f8f2/data/part-00000-e6012794-1f89-4dcc-8191-05dbe5ff6bbc-c000.snappy.parquet": "b95f973e5c9d08382e52dde2888bfb19b4371d8eb2000e48801a085ca373c01a",\n            "stages/0_Imputer_75e58e12f8f2/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_75e58e12f8f2/metadata/part-00000-f83819d3-79c9-472d-9d42-8a8987472b8b-c000.txt": "901e89faa78cfdea969ceb4c5f8ea24435f1b1764c31236a31275a68b12493e7",\n            "stages/1_VectorAssembler_bc29ebd64ced/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/1_VectorAssembler_bc29ebd64ced/metadata/part-00000-c7f54243-2ca4-4ec3-87ad-ace77130e722-c000.txt": "77459e19b797b9485568cbb4fd5f07a03b7dcaed763ba36b2f4f9ac3e24c7f17",\n            "stages/2_StandardScaler_611ae33371ff/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_StandardScaler_611ae33371ff/data/part-00000-602906a3-3234-4aea-8937-b0fbcfb7f42d-c000.snappy.parquet": "e5ca48abe1b71ba77b74e5909088ec9af7ba5e9fbe4af16dbd0c3c2b4c22031d",\n            "stages/2_StandardScaler_611ae33371ff/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_StandardScaler_611ae33371ff/metadata/part-00000-1b5979f1-b2ea-4d79-97a1-e0aaa667769d-c000.txt": "3905fc148dc2e1f77c392ddb6326e117bd7b2bb980687811cb4971638bdbdc50",\n            "stages/3_LogisticRegression_396b6b8873c9/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/3_LogisticRegression_396b6b8873c9/data/part-00000-555c029a-021a-42cc-8388-93bd79754a90-c000.snappy.parquet": "52054a1a1c413a2129d5edab7413589d0659a49c8d1fbdc5141004d963766df4",\n            "stages/3_LogisticRegression_396b6b8873c9/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/3_LogisticRegression_396b6b8873c9/metadata/part-00000-a3b741ca-c5cf-4d56-97a8-698539fda113-c000.txt": "650d50212587c62d9b4649d679dd9ff3cde5662431a564c1b8794976517b8084"\n          },\n          "training_cohort": "full_train_refit",\n          "training_seconds": 29.211,\n          "parameters": {\n            "maxIter": 12,\n            "regParam": 0.005,\n            "elasticNetParam": 0.0\n          },\n          "explanation": [\n            {\n              "feature": "pkt_len_mean",\n              "method": "scaled_lr_coefficient",\n              "value": -1.9062686969568232\n            },\n            {\n              "feature": "flow_duration",\n              "method": "scaled_lr_coefficient",\n              "value": -0.8674368222976909\n            },\n            {\n              "feature": "pkt_len_std",\n              "method": "scaled_lr_coefficient",\n              "value": 0.8222475553209422\n            },\n            {\n              "feature": "rst_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": 0.6883605432941551\n            },\n            {\n              "feature": "flow_byts_s",\n              "method": "scaled_lr_coefficient",\n              "value": -0.609026279602007\n            },\n            {\n              "feature": "ack_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": 0.6034332888339793\n            },\n            {\n              "feature": "syn_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": -0.5696343352283386\n            },\n            {\n              "feature": "flow_pkts_s",\n              "method": "scaled_lr_coefficient",\n              "value": 0.3474925176329858\n            },\n            {\n              "feature": "down_up_ratio",\n              "method": "scaled_lr_coefficient",\n              "value": 0.23560894032017093\n            },\n            {\n              "feature": "bwd_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": 0.15253831532400683\n            },\n            {\n              "feature": "fin_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": -0.1473693534529459\n            },\n            {\n              "feature": "flow_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": 0.11848347565720055\n            },\n            {\n              "feature": "fwd_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": 0.10038513374101134\n            },\n            {\n              "feature": "tot_fwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.08887497151124718\n            },\n            {\n              "feature": "totlen_fwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.05895817586666829\n            },\n            {\n              "feature": "totlen_bwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.011839444588369944\n            },\n            {\n              "feature": "tot_bwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": -0.0031745524107773537\n            },\n            {\n              "feature": "flow_iat_std",\n              "method": "scaled_lr_coefficient",\n              "value": -0.0004389912442234386\n            }\n          ]\n        }\n      ],\n      "trial_selections": {\n        "logistic_regression": {\n          "trial": 1,\n          "parameters": {\n            "maxIter": 12,\n            "regParam": 0.005,\n            "elasticNetParam": 0.0\n          }\n        },\n        "random_forest": {\n          "trial": 1,\n          "parameters": {\n            "numTrees": 12,\n            "maxDepth": 5,\n            "minInstancesPerNode": 30\n          }\n        },\n        "gradient_boosted_trees": {\n          "trial": 1,\n          "parameters": {\n            "maxIter": 6,\n            "maxDepth": 3,\n            "stepSize": 0.1\n          }\n        }\n      },\n      "input": {\n        "path": "/kaggle/input/notebooks/ramcharan2345/mmds-phase-2-spark-data-engineering/mmds_phase2/phase2-v1/feature_ready",\n        "days": [\n          "2018-02-14",\n          "2018-02-15",\n          "2018-02-16",\n          "2018-02-20",\n          "2018-02-21",\n          "2018-02-22",\n          "2018-02-23",\n          "2018-02-28",\n          "2018-03-01",\n          "2018-03-02"\n        ],\n        "totals": {\n          "0": 13484708,\n          "1": 2748235\n        },\n        "rows": 16232943,\n        "temporally_flagged": 14,\n        "policy_version": "phase2-v1",\n        "receipt_sha256": "1fa00cd2b9c853ee33eaacee8ca7632dbb41f739573326280d6ffbfb2ccff25a",\n        "producer": {\n          "url": "https://www.kaggle.com/code/ramcharan2345/mmds-phase-2-spark-data-engineering",\n          "kernel_version": 1,\n          "status": "COMPLETE"\n        }\n      },\n      "registered_predictors": [\n        "flow_duration",\n        "tot_fwd_pkts",\n        "tot_bwd_pkts",\n        "totlen_fwd_pkts",\n        "totlen_bwd_pkts",\n        "pkt_len_mean",\n        "pkt_len_std",\n        "flow_iat_mean",\n        "flow_iat_std",\n        "fwd_iat_mean",\n        "bwd_iat_mean",\n        "flow_pkts_s",\n        "flow_byts_s",\n        "down_up_ratio",\n        "syn_flag_cnt",\n        "ack_flag_cnt",\n        "rst_flag_cnt",\n        "fin_flag_cnt"\n      ],\n      "overlap": {\n        "predictor_before": {\n          "groups": 142016,\n          "rows": 2547596\n        },\n        "after": {\n          "predictor_key": {\n            "groups": 0,\n            "rows": 0\n          },\n          "record_fingerprint": {\n            "groups": 0,\n            "rows": 0\n          },\n          "split_feature_key": {\n            "groups": 0,\n            "rows": 0\n          }\n        },\n        "purged_rows": 859101\n      },\n      "retained_predictor_conflicts": {\n        "groups": 17434,\n        "rows": 1363306\n      },\n      "split_support": {\n        "train": {\n          "labels": {\n            "0": 10844286,\n            "1": 2678436\n          },\n          "families": {\n            "Benign": 10844286,\n            "Credential brute force": 380949,\n            "DoS": 654300,\n            "DDoS": 1263933,\n            "Infiltration": 93063,\n            "Bot": 286191\n          },\n          "original_labels": {\n            "Benign": 10844286,\n            "FTP-BruteForce": 193360,\n            "SSH-Bruteforce": 187589,\n            "DoS attacks-GoldenEye": 41508,\n            "DoS attacks-Slowloris": 10990,\n            "DoS attacks-Hulk": 461912,\n            "DoS attacks-SlowHTTPTest": 139890,\n            "DDoS attacks-LOIC-HTTP": 576191,\n            "DDOS attack-HOIC": 686012,\n            "DDOS attack-LOIC-UDP": 1730,\n            "Infilteration": 93063,\n            "Bot": 286191\n          },\n          "rows": 13522722\n        },\n        "validation": {\n          "labels": {\n            "0": 351060,\n            "1": 41842\n          },\n          "families": {\n            "Benign": 351060,\n            "Infiltration": 41842\n          },\n          "original_labels": {\n            "Benign": 351060,\n            "Infilteration": 41842\n          },\n          "rows": 392902\n        },\n        "test": {\n          "labels": {\n            "0": 1457560,\n            "1": 658\n          },\n          "families": {\n            "Benign": 1457560,\n            "Web attack": 658\n          },\n          "original_labels": {\n            "Benign": 1457560,\n            "Brute Force -Web": 423,\n            "Brute Force -XSS": 176,\n            "SQL Injection": 59\n          },\n          "rows": 1458218\n        }\n      },\n      "train_sample_support": {\n        "labels": {\n          "0": 659296,\n          "1": 153224\n        },\n        "families": {\n          "Benign": 659296,\n          "Credential brute force": 12384,\n          "DoS": 37445,\n          "DDoS": 80477,\n          "Infiltration": 5516,\n          "Bot": 17402\n        },\n        "original_labels": {\n          "Benign": 659296,\n          "FTP-BruteForce": 6425,\n          "SSH-Bruteforce": 5959,\n          "DoS attacks-GoldenEye": 2663,\n          "DoS attacks-Slowloris": 684,\n          "DoS attacks-Hulk": 28198,\n          "DoS attacks-SlowHTTPTest": 5900,\n          "DDoS attacks-LOIC-HTTP": 36066,\n          "DDOS attack-HOIC": 44289,\n          "DDOS attack-LOIC-UDP": 122,\n          "Infilteration": 5516,\n          "Bot": 17402\n        },\n        "rows": 812520\n      },\n      "baselines": {\n        "majority": {\n          "attack_prediction": 0\n        },\n        "duration_rule": {\n          "benign_training_median_us": 30920.0\n        }\n      },\n      "preprocessing": {\n        "active": [\n          "flow_duration",\n          "tot_fwd_pkts",\n          "tot_bwd_pkts",\n          "totlen_fwd_pkts",\n          "totlen_bwd_pkts",\n          "pkt_len_mean",\n          "pkt_len_std",\n          "flow_iat_mean",\n          "flow_iat_std",\n          "fwd_iat_mean",\n          "bwd_iat_mean",\n          "flow_pkts_s",\n          "flow_byts_s",\n          "down_up_ratio",\n          "syn_flag_cnt",\n          "ack_flag_cnt",\n          "rst_flag_cnt",\n          "fin_flag_cnt"\n        ],\n        "dropped_train_constant_or_empty": [],\n        "training_medians": {\n          "flow_duration": 25413.0,\n          "tot_fwd_pkts": 2.0,\n          "tot_bwd_pkts": 1.0,\n          "totlen_fwd_pkts": 45.0,\n          "totlen_bwd_pkts": 112.0,\n          "pkt_len_mean": 55.0,\n          "pkt_len_std": 35.21836642056717,\n          "flow_iat_mean": 12840.6666666667,\n          "flow_iat_std": 123.03657992645927,\n          "fwd_iat_mean": 5123.0,\n          "bwd_iat_mean": 0.0,\n          "flow_pkts_s": 122.92563,\n          "flow_byts_s": 1002.68197,\n          "down_up_ratio": 0.0,\n          "syn_flag_cnt": 0.0,\n          "ack_flag_cnt": 0.0,\n          "rst_flag_cnt": 0.0,\n          "fin_flag_cnt": 0.0\n        }\n      },\n      "preprocess_seconds": 14.438,\n      "post_imputation_overlap": {\n        "groups": 0,\n        "rows": 0\n      },\n      "selection_frozen": true,\n      "elapsed_seconds": 491.052,\n      "downloaded_reload": [\n        {\n          "artifact_directory": "logistic_regression_group_sample",\n          "finite_probe_predictions": 3\n        },\n        {\n          "artifact_directory": "random_forest_group_sample",\n          "finite_probe_predictions": 3\n        },\n        {\n          "artifact_directory": "gradient_boosted_trees_group_sample",\n          "finite_probe_predictions": 3\n        },\n        {\n          "artifact_directory": "logistic_regression_full_train_refit",\n          "finite_probe_predictions": 3\n        }\n      ]\n    }\n  }\n}\n', 'reports/phase4_metrics.csv': 'protocol,model,training_cohort,split,training_rows,training_seconds,tp,tn,fp,fn,rows,accuracy,precision,recall,specificity,f1,balanced_accuracy,roc_auc,pr_auc,prediction_seconds\nA,majority,full_train_baseline,validation,11455925,0.0,0,1990157,0,354569,2344726,0.8487801986244875,,0.0,1.0,0.0,0.5,0.5,0.15121980137551252,2.885\nA,duration_rule,full_train_baseline,validation,11455925,0.0,215748,1010854,979303,138821,2344726,0.5231323404099242,0.18053455459223078,0.6084795907143602,0.5079267615570028,0.2784527819723545,0.5582031761356815,0.5582031761356815,0.1690572304147137,2.882\nA,majority,full_train_baseline,test,11455925,0.0,0,2068172,0,364120,2432292,0.8502975794024731,,0.0,1.0,0.0,0.5,0.5,0.14970242059752695,2.509\nA,duration_rule,full_train_baseline,test,11455925,0.0,225592,1010484,1057688,138528,2432292,0.5081939175066151,0.17579327972071568,0.6195539931890586,0.48858798978034707,0.27387641131479906,0.5540709914847028,0.5540709914847028,0.1658671165530317,2.586\nA,logistic_regression,group_sample,validation,695307,6.295,364,1990154,3,354205,2344726,0.8489341611770416,0.9918256130790191,0.0010265984899977155,0.9999984925812385,0.002051073996438795,0.5005125455356181,0.8203354549081758,0.6100444118279686,3.753\nA,logistic_regression,group_sample,test,695307,6.295,324,2068167,5,363796,2432292,0.8504287314187606,0.9848024316109423,0.0008898165439964847,0.9999975824061055,0.0017780265551558653,0.500443699475051,0.8063869233854803,0.5930784428992267,4.051\nA,random_forest,group_sample,validation,695307,11.737,302628,1967648,22509,51941,2344726,0.9682478890923716,0.9307707212651898,0.8535094720632656,0.9886898370329577,0.8904673491185895,0.9210996545481116,0.9676168582168041,0.9220911936832712,5.861\nA,random_forest,group_sample,test,695307,11.737,301553,2045915,22257,62567,2432292,0.9651258977129391,0.931265248139341,0.8281692848511479,0.9892383225379707,0.8766967569374791,0.9087038036945594,0.9656252738284283,0.9165679954807866,5.802\nA,gradient_boosted_trees,group_sample,validation,695307,19.557,304070,1968486,21671,50499,2344726,0.9692202841611344,0.933471684559205,0.8575763814659488,0.9891109093403184,0.8939160088783055,0.9233436454031336,0.9609982675249529,0.8959337445321933,4.47\nA,gradient_boosted_trees,group_sample,test,695307,19.557,309359,2044256,23916,54761,2432292,0.9676531436192694,0.9282394419023329,0.849607272327804,0.9884361648837717,0.8871844507058411,0.9190217186057879,0.9576786382978011,0.889155941145078,4.237\nA,logistic_regression,full_train_refit,validation,11455925,25.26,460,1989107,1050,354109,2344726,0.8485285700759918,0.304635761589404,0.0012973497401070032,0.999472403433498,0.0025836963145818765,0.5003848765868025,0.7910496253085418,0.336277301248326,3.908\nA,logistic_regression,full_train_refit,test,11455925,25.26,1392,1983162,85010,362728,2432292,0.8159193057412515,0.016110738177357007,0.0038229155223552676,0.9588960686055125,0.006179498448466446,0.48135949206393386,0.752074094446787,0.2833215198288908,4.026\nB,majority,full_train_baseline,validation,13191622,0.0,0,1094200,0,57056,1151256,0.9504402148609866,,0.0,1.0,0.0,0.5,0.5,0.04955978513901339,3.731\nB,duration_rule,full_train_baseline,validation,13191622,0.0,27586,795014,299186,29470,1151256,0.7145239633930247,0.08441971772367278,0.48348990465507574,0.7265710107841346,0.1437414675323322,0.6050304577196052,0.6050304577196052,0.06641421062065272,3.456\nB,majority,full_train_baseline,test,13191622,0.0,0,771584,0,208525,980109,0.7872430515381452,,0.0,1.0,0.0,0.5,0.5,0.21275694846185475,2.908\nB,duration_rule,full_train_baseline,test,13191622,0.0,172571,559001,212583,35954,980109,0.7464190207415706,0.4480571407800516,0.827579426927227,0.7244849556237558,0.5813613080469412,0.7760321912754914,0.7760321912754914,0.4074865467764144,2.704\nB,logistic_regression,group_sample,validation,795270,13.5,600,1092598,1602,56456,1151256,0.9495698610908434,0.2724795640326976,0.010515984296130118,0.9985359166514348,0.020250430321644336,0.5045259504737825,0.44900943161671913,0.054790438911638734,3.6\nB,logistic_regression,group_sample,test,795270,13.5,177,771533,51,208348,980109,0.7873716086680155,0.7763157894736842,0.0008488190864404748,0.9999339022063702,0.0016957840126848476,0.5003913606464053,0.8224500524391902,0.6231745883963631,3.829\nB,random_forest,group_sample,validation,795270,19.239,586,1084706,9494,56470,1151256,0.9427025787487753,0.05813492063492064,0.010270611329220415,0.9913233412538841,0.017457102001906578,0.5007969762915523,0.3919743896266559,0.04074654677392458,5.548\nB,random_forest,group_sample,test,795270,19.239,417,766721,4863,208108,980109,0.7827068213841522,0.07897727272727273,0.0019997602205970508,0.99369738097213,0.0039007506840345175,0.4978485705963635,0.7985884732334407,0.6035922752334859,5.206\nB,gradient_boosted_trees,group_sample,validation,795270,17.415,470,1080372,13828,56586,1151256,0.9388372351588179,0.03287173031193174,0.008237521031968592,0.987362456589289,0.013173753398548085,0.4977999888106288,0.4240689529095466,0.044368054370374464,4.048\nB,gradient_boosted_trees,group_sample,test,795270,17.415,1218,764563,7021,207307,980109,0.7813222815013432,0.14783347493627869,0.005841026255844623,0.9909005370769741,0.011238028454909487,0.49837078166640936,0.429676530486362,0.21884530784082926,3.894\nB,logistic_regression,full_train_refit,validation,13191622,44.995,72,1093939,261,56984,1151256,0.9502760463354806,0.21621621621621623,0.0012619181155356142,0.9997614695668068,0.0025091916569377408,0.5005116938411712,0.45398883551558955,0.05589106607229337,4.134\nB,logistic_regression,full_train_refit,test,13191622,44.995,787,771493,91,207738,980109,0.7879531766364761,0.8963553530751709,0.0037741278024217718,0.9998820607996018,0.007516606734382984,0.5018280943010118,0.8276010573455483,0.6496362370253705,3.95\nC,majority,full_train_baseline,validation,13522722,0.0,0,351060,0,41842,392902,0.8935052506731959,,0.0,1.0,0.0,0.5,0.5,0.10649474932680414,2.363\nC,duration_rule,full_train_baseline,validation,13522722,0.0,13096,232873,118187,28746,392902,0.6260314276842571,0.09975396662172559,0.31298695091056833,0.6633424485842876,0.15128953068592058,0.48816469974742793,0.48816469974742793,0.10438497230119091,2.078\nC,majority,full_train_baseline,test,13522722,0.0,0,1457560,0,658,1458218,0.9995487643137034,,0.0,1.0,0.0,0.5,0.5,0.000451235686296562,2.034\nC,duration_rule,full_train_baseline,test,13522722,0.0,71,1131774,325786,587,1458218,0.7761836707543043,0.00021788698723673267,0.10790273556231003,0.7764853590932792,0.00043489579345512455,0.4421940473277946,0.44219404732779466,0.0004260567233281001,2.285\nC,logistic_regression,group_sample,validation,812520,10.976,641,347525,3535,41201,392902,0.8861395462481738,0.15349616858237547,0.015319535395057598,0.9899304962114738,0.02785866400104307,0.5026250158032657,0.4838858048415502,0.10611844996187014,2.27\nC,logistic_regression,group_sample,test,812520,10.976,1,1455269,2291,657,1458218,0.9979783544024282,0.0004363001745200698,0.001519756838905775,0.9984281950657263,0.0006779661016949153,0.4999739759523161,0.3605909537912005,0.0012080643742875576,3.129\nC,random_forest,group_sample,validation,812520,9.08,743,327394,23666,41099,392902,0.8351624578139077,0.030439591953787537,0.01775727737679843,0.9325870221614538,0.022429850115469954,0.47517214976912614,0.4652998271123345,0.09593180138975133,2.729\nC,random_forest,group_sample,test,812520,9.08,16,1445591,11969,642,1458218,0.9913517731916627,0.0013350020859407593,0.0243161094224924,0.991788331183622,0.002531044846950882,0.5080522203030572,0.5062522047297098,0.000545093476709124,3.932\nC,gradient_boosted_trees,group_sample,validation,812520,11.686,1450,322085,28975,40392,392902,0.8234496133895984,0.047658175842235,0.03465417523062951,0.9174642511251638,0.040128966194805375,0.4760592131778967,0.47360582403309415,0.09809767096047926,2.449\nC,gradient_boosted_trees,group_sample,test,812520,11.686,27,1435584,21976,631,1458218,0.9844968310636681,0.001227105394718902,0.041033434650455926,0.9849227476055874,0.002382948678346057,0.5129780911280216,0.4776807109912883,0.0006529601645462665,2.577\nC,logistic_regression,full_train_refit,validation,13522722,29.211,551,347773,3287,41291,392902,0.8865416821497473,0.14356435643564355,0.013168586587639215,0.9906369281604284,0.024124343257443084,0.5019027573740338,0.4837028020579236,0.10551223929699871,2.659\nC,logistic_regression,full_train_refit,test,13522722,29.211,0,1457108,452,658,1458218,0.9992387969425697,0.0,0.0,0.9996898926973846,0.0,0.4998449463486923,0.36451130208365046,0.001251129809765967,3.043\n', 'src/streaming/cluster_scoring.py': '"""Score with the original Phase 5 B preprocessing and frozen K-Means stages."""\n\nimport math\n\nfrom pyspark.ml import PipelineModel\nfrom pyspark.sql import functions as F\n\nfrom src.ml.novelty_scoring import squared_distance\nfrom src.streaming.model_scoring import mounted_summary, probe, verified_model\n\n\ndef clustering(spark, root, names, registered):\n    directory, receipt = mounted_summary(root, "phase5_summary.json", "B", "experiment")\n    if (receipt["features"] != names or receipt["status"] != "validated" or\n            receipt["model"]["sha256"] != registered["model"]["sha256"] or\n            receipt["calibration"] != registered["calibration"] or\n            receipt["reference_occupancy"] != registered["reference_occupancy"]):\n        raise ValueError("Mounted clustering receipt differs from Phase 5 B")\n    pipeline = verified_model(spark, directory / "model", registered["model"]["sha256"], names, "_valid", "_filled")\n    preparation, model = PipelineModel(stages=pipeline.stages[:-1]), pipeline.stages[-1]\n    cutoff = registered["calibration"]["cutoffs"]["0.99"]\n    results = cluster_scores(probe(spark, names), preparation, model, cutoff).select("cluster", "distance").collect()\n    k = registered["selected_k"]\n    if (len(model.clusterCenters()) != k or len(results) != 3 or\n            any(not 0 <= r.cluster < k or not math.isfinite(r.distance) or r.distance < 0 for r in results)):\n        raise ValueError("Invalid clustering synthetic probe")\n    return (preparation, model, cutoff), {"probe_rows": len(results), "k": k, "p99": cutoff,\n                                         "sha256": registered["model"]["sha256"]}\n\n\ndef cluster_scores(frame, preparation, model, cutoff):\n    return (squared_distance(model, preparation.transform(frame))\n            .select(*frame.columns, "cluster", F.col("score").alias("distance"))\n            .withColumn("novel", F.col("distance") > cutoff))\n', 'src/streaming/model_scoring.py': '"""Verify and score the original Phase 4 chronological supervised pipelines."""\n\nfrom hashlib import sha256\nimport json\nimport math\nfrom pathlib import Path\n\nfrom pyspark.ml import PipelineModel\nfrom pyspark.ml.functions import vector_to_array\nfrom pyspark.sql import functions as F\n\n\nSUPERVISED = ("random_forest", "logistic_regression", "duration_rule")\n\n\ndef mounted_summary(root, name, identity, key):\n    found = [path for path in root.rglob(name) if json.loads(path.read_text()).get(key) == identity]\n    if len(found) != 1:\n        raise ValueError(f"Expected one mounted chronological {name}, found {len(found)}")\n    return found[0].parent, json.loads(found[0].read_text())\n\n\ndef verified_model(spark, path, hashes, names, imputer_suffix="", output_suffix="_imputed"):\n    files = {p.relative_to(path).as_posix(): sha256(p.read_bytes()).hexdigest()\n             for p in path.rglob("*") if p.is_file() and not p.name.endswith(".crc")}\n    if not files or files != hashes:\n        raise ValueError(f"Model checksum mismatch for {path.name}")\n    metadata = [path / name for name in files\n                if Path(name).parent.name == "metadata" and Path(name).name.startswith("part-")]\n    if not metadata or any(json.loads(item.read_text())["sparkVersion"] != spark.version for item in metadata):\n        raise ValueError(f"Spark version mismatch for {path.name}")\n    model = PipelineModel.load(str(path))\n    imputer, assembler = next(s for s in model.stages if type(s).__name__ == "ImputerModel"), next(\n        s for s in model.stages if type(s).__name__ == "VectorAssembler")\n    if (imputer.getInputCols() != [n + imputer_suffix for n in names] or\n            assembler.getInputCols() != imputer.getOutputCols() or\n            assembler.getInputCols() != [n + output_suffix for n in names]):\n        raise ValueError(f"Model predictor order mismatch for {path.name}")\n    return model\n\n\ndef probe(spark, names):\n    return spark.createDataFrame([\n        tuple(None for _ in names), tuple(0.0 for _ in names), tuple(float(i + 1) for i in range(len(names))),\n    ], ", ".join(name + " DOUBLE" for name in names))\n\n\ndef supervised(spark, root, names, registered):\n    directory, receipt = mounted_summary(root, "phase4_summary.json", "B", "protocol")\n    if (receipt["registered_predictors"] != names or receipt["baselines"] != registered["baselines"] or\n            receipt["status"] != "validated" or receipt["input"]["receipt_sha256"] != registered["input"]["receipt_sha256"]):\n        raise ValueError("Supervised artifact receipt is inconsistent")\n    models, evidence = {}, {}\n    for algorithm in SUPERVISED[:2]:\n        entry = next(m for m in registered["models"] if m["model"] == algorithm and m["training_cohort"] == "group_sample")\n        mounted = next(m for m in receipt["models"] if m["model"] == algorithm and m["training_cohort"] == "group_sample")\n        if mounted["sha256"] != entry["sha256"]:\n            raise ValueError(f"Mounted {algorithm} is not the pinned Phase 4 pipeline")\n        name = Path(entry["path"]).name\n        model = verified_model(spark, directory / "models" / name, entry["sha256"], names)\n        predictions = model.transform(probe(spark, names)).select(vector_to_array("probability")[1]).collect()\n        if len(predictions) != 3 or any(not math.isfinite(r[0]) or not 0 <= r[0] <= 1 for r in predictions):\n            raise ValueError(f"Invalid {algorithm} synthetic probe")\n        models[algorithm] = model\n        evidence[algorithm] = {"directory": name, "files": len(entry["sha256"]), "probe_rows": len(predictions),\n                               "sha256": entry["sha256"]}\n    cutoff = registered["baselines"]["duration_rule"]["benign_training_median_us"]\n    return models, cutoff, evidence\n\n\ndef supervised_scores(frame, models, duration_cutoff):\n    for name, model in models.items():\n        frame = model.transform(frame).select(*frame.columns, vector_to_array("probability")[1].alias(name))\n    return frame.withColumn("duration_rule", F.when(F.col("flow_duration") <= duration_cutoff, 1.0).otherwise(0.0))\n', 'configs/phase8_hybrid_experiments.yaml': '{\n  "version": "phase8-v1",\n  "phase2": "ramcharan2345/mmds-phase-2-spark-data-engineering/1",\n  "phase4": "ramcharan2345/mmds-phase-4-supervised-attack-detection",\n  "phase5": "ramcharan2345/mmds-phase-5-clustering-pattern-evolution",\n  "chronological_mirror": "ramcharan2345/mmds-phase-6-pinned-chronological-models/1",\n  "versions": {"B": 2, "C": 3},\n  "seed": 23013,\n  "group_sample_permyriad": 625,\n  "minimum_per_class": 100,\n  "score_bins": 2048,\n  "reliability_bins": 10,\n  "probability_epsilon": 1e-6,\n  "budgets": [0.001, 0.005, 0.01, 0.02, 0.05],\n  "primary_budget": 0.01,\n  "review_budget": 0.05,\n  "false_negative_cost": 5,\n  "positive_weights": [1, 2, 5, 10, "inverse"],\n  "fusion_alphas": [0, 0.25, 0.5, 0.75, 1],\n  "hypothetical_prevalences": [0.0001, 0.001, 0.01, 0.1],\n  "reference_sample_permyriad": 125,\n  "rolling": {\n    "R1": {\n      "train": ["2018-02-14", "2018-02-15"],\n      "probability": ["2018-02-16"],\n      "operating": ["2018-02-20"],\n      "test": ["2018-02-21"]\n    },\n    "R2": {\n      "train": ["2018-02-14", "2018-02-15", "2018-02-16", "2018-02-20", "2018-02-21"],\n      "probability": ["2018-02-22"],\n      "operating": ["2018-02-23"],\n      "test": ["2018-02-28"]\n    }\n  },\n  "spark": {"master": "local[4]", "driver_memory": "8g", "shuffle_partitions": 48, "input_partition_bytes": 67108864}\n}\n', 'manifests/phase5_clustering_models.json': '{\n  "status": "validated",\n  "kernel_url": "https://www.kaggle.com/code/ramcharan2345/mmds-phase-5-clustering-pattern-evolution",\n  "experiments": {\n    "G": {\n      "experiment": "G",\n      "config": {\n        "version": "phase5-v1",\n        "seed": 23013,\n        "candidate_k": [\n          3,\n          5,\n          8,\n          12\n        ],\n        "max_iter": 16,\n        "init_steps": 2,\n        "tolerance": 0.0001,\n        "training_permyriad": 250,\n        "internal_permyriad": 12,\n        "reference_percentiles": [\n          0.95,\n          0.99,\n          0.999\n        ],\n        "primary_percentile": 0.99,\n        "quantile_accuracy": 8192,\n        "max_internal_rows": 40000,\n        "rare_support": 100,\n        "window_hours": 3,\n        "lag_reference": "full retained phase4 train-day population",\n        "spark": {\n          "master": "local[4]",\n          "driver_memory": "8g",\n          "shuffle_partitions": 48,\n          "input_partition_bytes": 67108864\n        }\n      },\n      "status": "COMPLETE",\n      "resources": {\n        "spark_version": "4.3.0",\n        "master": "local[4]",\n        "driver_memory": "8g",\n        "shuffle_partitions": 48,\n        "input_partition_bytes": 67108864,\n        "default_parallelism": 4,\n        "cpu_affinity": 4,\n        "scratch_output_free_bytes_start": 20939968512,\n        "memory.max": "32212254720",\n        "cpu.max": "400000 100000",\n        "driver_peak_rss_bytes": 4750106624\n      },\n      "features": [\n        "flow_duration",\n        "tot_fwd_pkts",\n        "tot_bwd_pkts",\n        "totlen_fwd_pkts",\n        "totlen_bwd_pkts",\n        "pkt_len_mean",\n        "pkt_len_std",\n        "flow_iat_mean",\n        "flow_iat_std",\n        "fwd_iat_mean",\n        "bwd_iat_mean",\n        "flow_pkts_s",\n        "flow_byts_s",\n        "down_up_ratio",\n        "syn_flag_cnt",\n        "ack_flag_cnt",\n        "rst_flag_cnt",\n        "fin_flag_cnt"\n      ],\n      "input": {\n        "path": "/kaggle/input/notebooks/ramcharan2345/mmds-phase-2-spark-data-engineering/mmds_phase2/phase2-v1/feature_ready",\n        "days": [\n          "2018-02-14",\n          "2018-02-15",\n          "2018-02-16",\n          "2018-02-20",\n          "2018-02-21",\n          "2018-02-22",\n          "2018-02-23",\n          "2018-02-28",\n          "2018-03-01",\n          "2018-03-02"\n        ],\n        "totals": {\n          "0": 13484708,\n          "1": 2748235\n        },\n        "rows": 16232943,\n        "temporally_flagged": 14,\n        "policy_version": "phase2-v1",\n        "receipt_sha256": "1fa00cd2b9c853ee33eaacee8ca7632dbb41f739573326280d6ffbfb2ccff25a",\n        "producer": {\n          "url": "https://www.kaggle.com/code/ramcharan2345/mmds-phase-2-spark-data-engineering",\n          "kernel_version": 1,\n          "status": "COMPLETE"\n        }\n      },\n      "purge": {\n        "groups_before": 0,\n        "groups_after": 0,\n        "rows": 0\n      },\n      "matched_phase4": null,\n      "preprocessing": {\n        "transform": "signed_log1p",\n        "fit_columns": [\n          "flow_duration",\n          "tot_fwd_pkts",\n          "tot_bwd_pkts",\n          "totlen_fwd_pkts",\n          "totlen_bwd_pkts",\n          "pkt_len_mean",\n          "pkt_len_std",\n          "flow_iat_mean",\n          "flow_iat_std",\n          "fwd_iat_mean",\n          "bwd_iat_mean",\n          "flow_pkts_s",\n          "flow_byts_s",\n          "down_up_ratio",\n          "syn_flag_cnt",\n          "ack_flag_cnt",\n          "rst_flag_cnt",\n          "fin_flag_cnt"\n        ],\n        "active": [\n          "flow_duration",\n          "tot_fwd_pkts",\n          "tot_bwd_pkts",\n          "totlen_fwd_pkts",\n          "totlen_bwd_pkts",\n          "pkt_len_mean",\n          "pkt_len_std",\n          "flow_iat_mean",\n          "flow_iat_std",\n          "fwd_iat_mean",\n          "bwd_iat_mean",\n          "flow_pkts_s",\n          "flow_byts_s",\n          "down_up_ratio",\n          "syn_flag_cnt",\n          "ack_flag_cnt",\n          "rst_flag_cnt",\n          "fin_flag_cnt"\n        ],\n        "dropped": [],\n        "reference_stats": {\n          "flow_duration": {\n            "invalid_or_null": 2,\n            "available": true,\n            "reference_std": 4.947151865641074,\n            "near_constant": false\n          },\n          "tot_fwd_pkts": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.7530213879187553,\n            "near_constant": false\n          },\n          "tot_bwd_pkts": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.9378945538594473,\n            "near_constant": false\n          },\n          "totlen_fwd_pkts": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 2.691333643392363,\n            "near_constant": false\n          },\n          "totlen_bwd_pkts": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 3.173848812379961,\n            "near_constant": false\n          },\n          "pkt_len_mean": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 2.1025627607160313,\n            "near_constant": false\n          },\n          "pkt_len_std": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 2.328633749104362,\n            "near_constant": false\n          },\n          "flow_iat_mean": {\n            "invalid_or_null": 2,\n            "available": true,\n            "reference_std": 4.181293329541656,\n            "near_constant": false\n          },\n          "flow_iat_std": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 6.264059109273554,\n            "near_constant": false\n          },\n          "fwd_iat_mean": {\n            "invalid_or_null": 2,\n            "available": true,\n            "reference_std": 6.0557528119987944,\n            "near_constant": false\n          },\n          "bwd_iat_mean": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 6.163171726202694,\n            "near_constant": false\n          },\n          "flow_pkts_s": {\n            "invalid_or_null": 9,\n            "available": true,\n            "reference_std": 3.8365363784055213,\n            "near_constant": false\n          },\n          "flow_byts_s": {\n            "invalid_or_null": 7,\n            "available": true,\n            "reference_std": 4.9738935827118915,\n            "near_constant": false\n          },\n          "down_up_ratio": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.3569073900673366,\n            "near_constant": false\n          },\n          "syn_flag_cnt": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.15167067548018145,\n            "near_constant": false\n          },\n          "ack_flag_cnt": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.32647293591636467,\n            "near_constant": false\n          },\n          "rst_flag_cnt": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.27492864689025487,\n            "near_constant": false\n          },\n          "fin_flag_cnt": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.042408098008640864,\n            "near_constant": false\n          }\n        },\n        "imputation_surrogates": {\n          "flow_duration_valid": 10.271839059304192,\n          "tot_fwd_pkts_valid": 1.0986122886681096,\n          "tot_bwd_pkts_valid": 0.6931471805599453,\n          "totlen_fwd_pkts_valid": 3.8501476017100584,\n          "totlen_bwd_pkts_valid": 4.6913478822291435,\n          "pkt_len_mean_valid": 4.013375499688434,\n          "pkt_len_std_valid": 3.589566349836834,\n          "flow_iat_mean_valid": 9.528454651548163,\n          "flow_iat_std_valid": 4.70593411340219,\n          "fwd_iat_mean_valid": 8.70167907103957,\n          "bwd_iat_mean_valid": 0.0,\n          "flow_pkts_s_valid": 4.739346173287946,\n          "flow_byts_s_valid": 6.795194361202203,\n          "down_up_ratio_valid": 0.0,\n          "syn_flag_cnt_valid": 0.0,\n          "ack_flag_cnt_valid": 0.0,\n          "rst_flag_cnt_valid": 0.0,\n          "fin_flag_cnt_valid": 0.0\n        },\n        "reference_means": [\n          10.635960082574252,\n          1.4396548986409057,\n          1.0914135214250371,\n          3.7598334915584624,\n          3.948889165036629,\n          3.1733805946445233,\n          3.2033206649848287,\n          9.470806759791927,\n          6.135387665867699,\n          7.7695968071489805,\n          5.05226565531731,\n          5.137223128237749,\n          6.45527291520749,\n          0.3450593185074743,\n          0.03494990928388112,\n          0.23025958254038817,\n          0.13555774713056207,\n          0.00260438917912373\n        ],\n        "reference_scales": [\n          4.947151865641074,\n          0.7530213879187553,\n          0.9378945538594473,\n          2.691333643392363,\n          3.173848812379961,\n          2.1025627607160313,\n          2.328633749104362,\n          4.181293329541656,\n          6.264059109273554,\n          6.0557528119987944,\n          6.163171726202694,\n          3.8365363784055213,\n          4.9738935827118915,\n          0.3569073900673366,\n          0.15167067548018145,\n          0.32647293591636467,\n          0.27492864689025487,\n          0.042408098008640864\n        ]\n      },\n      "training": {\n        "fit_rows": 390702,\n        "internal_rows": 18099,\n        "reference_rows": 16232943,\n        "share_of_reference": 0.024068463740678447\n      },\n      "selected_k": 5,\n      "trials": [\n        {\n          "k": 3,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 11,\n          "wssse": 3963550.706735454,\n          "cluster_sizes": [\n            161216,\n            212405,\n            17081\n          ],\n          "training_rows": 390702,\n          "empty_clusters": 0,\n          "smallest_share": 0.043718742161545115,\n          "fit_seconds": 17.601,\n          "jvm_peak_rss_bytes": 3835715584,\n          "silhouette": 0.5376128609801271,\n          "silhouette_rows": 18099\n        },\n        {\n          "k": 5,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 16,\n          "wssse": 2224306.089655832,\n          "cluster_sizes": [\n            68133,\n            121466,\n            52459,\n            16947,\n            131697\n          ],\n          "training_rows": 390702,\n          "empty_clusters": 0,\n          "smallest_share": 0.04337576976826328,\n          "fit_seconds": 15.082,\n          "jvm_peak_rss_bytes": 3835715584,\n          "silhouette": 0.6178674460772436,\n          "silhouette_rows": 18099\n        },\n        {\n          "k": 8,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 16,\n          "wssse": 1496496.2336061099,\n          "cluster_sizes": [\n            51871,\n            67081,\n            49778,\n            89091,\n            14903,\n            1468,\n            111836,\n            4674\n          ],\n          "training_rows": 390702,\n          "empty_clusters": 0,\n          "smallest_share": 0.003757339353266684,\n          "fit_seconds": 14.047,\n          "jvm_peak_rss_bytes": 3835715584,\n          "silhouette": 0.5663268851635334,\n          "silhouette_rows": 18099\n        },\n        {\n          "k": 12,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 16,\n          "wssse": 1395303.8753451167,\n          "cluster_sizes": [\n            13662,\n            28818,\n            44556,\n            82850,\n            36502,\n            52696,\n            9326,\n            15733,\n            16239,\n            3845,\n            25260,\n            61215\n          ],\n          "training_rows": 390702,\n          "empty_clusters": 0,\n          "smallest_share": 0.0098412600908109,\n          "fit_seconds": 13.797,\n          "jvm_peak_rss_bytes": 3835715584,\n          "silhouette": 0.609291062515786,\n          "silhouette_rows": 18099\n        }\n      ],\n      "robustness": {\n        "paired_rows": 18099,\n        "seed_ari": 0.8954796677466573,\n        "scale_ari": 0.2899832117108664,\n        "alternate_seed": {\n          "k": 5,\n          "seed": 23014,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 10,\n          "wssse": 2375059.0026243413,\n          "cluster_sizes": [\n            17019,\n            112559,\n            133596,\n            126060,\n            1468\n          ],\n          "training_rows": 390702,\n          "empty_clusters": 0,\n          "smallest_share": 0.003757339353266684,\n          "fit_seconds": 9.427,\n          "jvm_peak_rss_bytes": 3835715584\n        },\n        "linear": {\n          "k": 5,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 10,\n          "wssse": 4351926.195591406,\n          "cluster_sizes": [\n            16853,\n            16480,\n            113096,\n            4544,\n            239729\n          ],\n          "training_rows": 390702,\n          "empty_clusters": 0,\n          "smallest_share": 0.011630347425915403,\n          "fit_seconds": 9.35,\n          "jvm_peak_rss_bytes": 3835715584\n        },\n        "linear_silhouette": 0.42688826264849933,\n        "dominance": [\n          {\n            "feature": "pkt_len_mean",\n            "share_between_centroids": 0.07455241217018589\n          },\n          {\n            "feature": "totlen_bwd_pkts",\n            "share_between_centroids": 0.07377299369619311\n          },\n          {\n            "feature": "pkt_len_std",\n            "share_between_centroids": 0.07062813550435584\n          },\n          {\n            "feature": "syn_flag_cnt",\n            "share_between_centroids": 0.069520903400549\n          },\n          {\n            "feature": "totlen_fwd_pkts",\n            "share_between_centroids": 0.06892779538897799\n          },\n          {\n            "feature": "bwd_iat_mean",\n            "share_between_centroids": 0.067890089838014\n          },\n          {\n            "feature": "flow_duration",\n            "share_between_centroids": 0.06287261838219879\n          },\n          {\n            "feature": "flow_pkts_s",\n            "share_between_centroids": 0.06149442711609902\n          },\n          {\n            "feature": "flow_byts_s",\n            "share_between_centroids": 0.061485581507726456\n          },\n          {\n            "feature": "tot_bwd_pkts",\n            "share_between_centroids": 0.05867352115926721\n          },\n          {\n            "feature": "flow_iat_mean",\n            "share_between_centroids": 0.05824763952026458\n          },\n          {\n            "feature": "flow_iat_std",\n            "share_between_centroids": 0.057153496203609\n          },\n          {\n            "feature": "fwd_iat_mean",\n            "share_between_centroids": 0.05669788728773676\n          },\n          {\n            "feature": "ack_flag_cnt",\n            "share_between_centroids": 0.053910587465197723\n          },\n          {\n            "feature": "tot_fwd_pkts",\n            "share_between_centroids": 0.051573294462222294\n          },\n          {\n            "feature": "down_up_ratio",\n            "share_between_centroids": 0.03299269798319865\n          },\n          {\n            "feature": "rst_flag_cnt",\n            "share_between_centroids": 0.018241696706969762\n          },\n          {\n            "feature": "fin_flag_cnt",\n            "share_between_centroids": 0.0013642222072339724\n          }\n        ]\n      },\n      "calibration": {\n        "rows": 18099,\n        "cutoffs": {\n          "0.95": 13.37902829503273,\n          "0.99": 31.042924705057366,\n          "0.999": 263.37528662305107\n        },\n        "accuracy": 8192,\n        "reference_flags": {\n          "0.95": 905,\n          "0.99": 181,\n          "0.999": 20\n        }\n      },\n      "reference_occupancy": [\n        3313334,\n        4931789,\n        2115505,\n        602891,\n        5269424\n      ],\n      "retained_rows": 16232943,\n      "model": {\n        "path": "/kaggle/working/mmds_phase5/model",\n        "reload_rows": 32,\n        "sha256": {\n          "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "metadata/part-00000-0c87b28e-f308-4221-be39-19552f15cae3-c000.txt": "30384fd3941c9003c0bbacc7539717ed7e2fec48b80eb398139246fded412610",\n          "stages/0_SQLTransformer_5ca5a59fde15/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/0_SQLTransformer_5ca5a59fde15/metadata/part-00000-f99cdd32-8a0c-4326-8417-8d2ca3e2cc68-c000.txt": "a364092902ecb06292cf90c86f13b5c411ea08f9ad8a3f23c1bb998942ff32ff",\n          "stages/1_Imputer_dedda7ea3349/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/1_Imputer_dedda7ea3349/data/part-00000-683c9d4f-e991-4b63-b365-8b9b862e5c47-c000.snappy.parquet": "94a3393daa471262d9eef2aaef21eaa4909412c36d022de91248ebe2872b9a95",\n          "stages/1_Imputer_dedda7ea3349/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/1_Imputer_dedda7ea3349/metadata/part-00000-5049af53-561a-4641-b5e7-3c612198d0da-c000.txt": "7c60af0aafa39b4fe5733a5d5c5cae4ce489b8501ec29dd38e1bd7af8f6fa351",\n          "stages/2_VectorAssembler_087035b852be/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/2_VectorAssembler_087035b852be/metadata/part-00000-6f5448aa-4153-4c62-81e1-9fda6620230c-c000.txt": "4cc48d033c823c24464618d6572157be7a0f8fe59c659a5d4fb7a564e219c774",\n          "stages/3_StandardScaler_0d1d4f35be62/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/3_StandardScaler_0d1d4f35be62/data/part-00000-4b1de19e-2265-4edf-b7f2-896c2a90e676-c000.snappy.parquet": "304920b1e927c87dd9c996297666ae6a500243c0dc670a7fa2990feae29cb355",\n          "stages/3_StandardScaler_0d1d4f35be62/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/3_StandardScaler_0d1d4f35be62/metadata/part-00000-0f5661c4-36ce-4d1d-aede-76690ea637c3-c000.txt": "594b3b43be877f97a70a89e0565da67ca71e492cbd583d013b08eb6b2350023d",\n          "stages/4_KMeans_aaeaaa224ba4/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/4_KMeans_aaeaaa224ba4/data/part-00000-bff25530-0601-4c4c-8719-355553c18780-c000.snappy.parquet": "f2571f1c6417de4aa7554eee7a685e880e9421740bea8544315d1ed56a1a385a",\n          "stages/4_KMeans_aaeaaa224ba4/data/part-00001-bff25530-0601-4c4c-8719-355553c18780-c000.snappy.parquet": "4a52b62afb4efaa4d9e2108ababc2bab8710cd84c98731786d330ecee8212c3e",\n          "stages/4_KMeans_aaeaaa224ba4/data/part-00002-bff25530-0601-4c4c-8719-355553c18780-c000.snappy.parquet": "5c35ee55ec66c7e7e68a0b4237e53e979c756110af7b50864765805bd5e63874",\n          "stages/4_KMeans_aaeaaa224ba4/data/part-00003-bff25530-0601-4c4c-8719-355553c18780-c000.snappy.parquet": "8c2357343b3ca6dca74eeec01b686e9faacee83b77af8b8f6103af121a0011ea",\n          "stages/4_KMeans_aaeaaa224ba4/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/4_KMeans_aaeaaa224ba4/metadata/part-00000-7939bfff-34cf-4268-9988-9efb2f086114-c000.txt": "ea809e26eae2de98fbb124fc66a5dc0b2a50003f3ba6493078ea119475de9a12"\n        },\n        "bytes": 26027\n      },\n      "elapsed_seconds": 578.076,\n      "figures": [\n        "phase5_g_cluster_sizes.png",\n        "phase5_g_cluster_attack_share.png",\n        "phase5_g_family_heatmap.png",\n        "phase5_g_daily_occupancy.png",\n        "phase5_g_temporal_shift.png",\n        "phase5_g_centroid_profiles.png",\n        "phase5_g_distance_histogram.png",\n        "phase5_g_daily_novelty.png",\n        "phase5_g_binary_novelty.png"\n      ],\n      "version": 1,\n      "log_events": 702,\n      "downloaded_reload": {\n        "probes": 3,\n        "pipeline_stages": 5,\n        "verified": true\n      }\n    },\n    "B": {\n      "experiment": "B",\n      "config": {\n        "version": "phase5-v1",\n        "seed": 23013,\n        "candidate_k": [\n          3,\n          5,\n          8,\n          12\n        ],\n        "max_iter": 16,\n        "init_steps": 2,\n        "tolerance": 0.0001,\n        "training_permyriad": 250,\n        "internal_permyriad": 12,\n        "reference_percentiles": [\n          0.95,\n          0.99,\n          0.999\n        ],\n        "primary_percentile": 0.99,\n        "quantile_accuracy": 8192,\n        "max_internal_rows": 40000,\n        "rare_support": 100,\n        "window_hours": 3,\n        "lag_reference": "full retained phase4 train-day population",\n        "spark": {\n          "master": "local[4]",\n          "driver_memory": "8g",\n          "shuffle_partitions": 48,\n          "input_partition_bytes": 67108864\n        }\n      },\n      "status": "COMPLETE",\n      "resources": {\n        "spark_version": "4.3.0",\n        "master": "local[4]",\n        "driver_memory": "8g",\n        "shuffle_partitions": 48,\n        "input_partition_bytes": 67108864,\n        "default_parallelism": 4,\n        "cpu_affinity": 4,\n        "scratch_output_free_bytes_start": 20939968512,\n        "memory.max": "32212254720",\n        "cpu.max": "400000 100000",\n        "driver_peak_rss_bytes": 5523873792\n      },\n      "features": [\n        "flow_duration",\n        "tot_fwd_pkts",\n        "tot_bwd_pkts",\n        "totlen_fwd_pkts",\n        "totlen_bwd_pkts",\n        "pkt_len_mean",\n        "pkt_len_std",\n        "flow_iat_mean",\n        "flow_iat_std",\n        "fwd_iat_mean",\n        "bwd_iat_mean",\n        "flow_pkts_s",\n        "flow_byts_s",\n        "down_up_ratio",\n        "syn_flag_cnt",\n        "ack_flag_cnt",\n        "rst_flag_cnt",\n        "fin_flag_cnt"\n      ],\n      "input": {\n        "path": "/kaggle/input/notebooks/ramcharan2345/mmds-phase-2-spark-data-engineering/mmds_phase2/phase2-v1/feature_ready",\n        "days": [\n          "2018-02-14",\n          "2018-02-15",\n          "2018-02-16",\n          "2018-02-20",\n          "2018-02-21",\n          "2018-02-22",\n          "2018-02-23",\n          "2018-02-28",\n          "2018-03-01",\n          "2018-03-02"\n        ],\n        "totals": {\n          "0": 13484708,\n          "1": 2748235\n        },\n        "rows": 16232943,\n        "temporally_flagged": 14,\n        "policy_version": "phase2-v1",\n        "receipt_sha256": "1fa00cd2b9c853ee33eaacee8ca7632dbb41f739573326280d6ffbfb2ccff25a",\n        "producer": {\n          "url": "https://www.kaggle.com/code/ramcharan2345/mmds-phase-2-spark-data-engineering",\n          "kernel_version": 1,\n          "status": "COMPLETE"\n        }\n      },\n      "purge": {\n        "groups_before": 136985,\n        "groups_after": 0,\n        "rows": 909956\n      },\n      "matched_phase4": {\n        "protocol": "B",\n        "kaggle_version": "phase4-v1",\n        "checked": "matched split rules and family/binary totals",\n        "purged_rows": 909956\n      },\n      "preprocessing": {\n        "transform": "signed_log1p",\n        "fit_columns": [\n          "flow_duration",\n          "tot_fwd_pkts",\n          "tot_bwd_pkts",\n          "totlen_fwd_pkts",\n          "totlen_bwd_pkts",\n          "pkt_len_mean",\n          "pkt_len_std",\n          "flow_iat_mean",\n          "flow_iat_std",\n          "fwd_iat_mean",\n          "bwd_iat_mean",\n          "flow_pkts_s",\n          "flow_byts_s",\n          "down_up_ratio",\n          "syn_flag_cnt",\n          "ack_flag_cnt",\n          "rst_flag_cnt",\n          "fin_flag_cnt"\n        ],\n        "active": [\n          "flow_duration",\n          "tot_fwd_pkts",\n          "tot_bwd_pkts",\n          "totlen_fwd_pkts",\n          "totlen_bwd_pkts",\n          "pkt_len_mean",\n          "pkt_len_std",\n          "flow_iat_mean",\n          "flow_iat_std",\n          "fwd_iat_mean",\n          "bwd_iat_mean",\n          "flow_pkts_s",\n          "flow_byts_s",\n          "down_up_ratio",\n          "syn_flag_cnt",\n          "ack_flag_cnt",\n          "rst_flag_cnt",\n          "fin_flag_cnt"\n        ],\n        "dropped": [],\n        "reference_stats": {\n          "flow_duration": {\n            "invalid_or_null": 2,\n            "available": true,\n            "reference_std": 4.922656482085623,\n            "near_constant": false\n          },\n          "tot_fwd_pkts": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.7472669479236256,\n            "near_constant": false\n          },\n          "tot_bwd_pkts": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.9304382766169819,\n            "near_constant": false\n          },\n          "totlen_fwd_pkts": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 2.6960342991497512,\n            "near_constant": false\n          },\n          "totlen_bwd_pkts": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 3.1685009030872178,\n            "near_constant": false\n          },\n          "pkt_len_mean": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 2.1105019819450357,\n            "near_constant": false\n          },\n          "pkt_len_std": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 2.3422247994945606,\n            "near_constant": false\n          },\n          "flow_iat_mean": {\n            "invalid_or_null": 2,\n            "available": true,\n            "reference_std": 4.160112954121857,\n            "near_constant": false\n          },\n          "flow_iat_std": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 6.278238645416323,\n            "near_constant": false\n          },\n          "fwd_iat_mean": {\n            "invalid_or_null": 2,\n            "available": true,\n            "reference_std": 6.013749976297104,\n            "near_constant": false\n          },\n          "bwd_iat_mean": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 6.151272138517253,\n            "near_constant": false\n          },\n          "flow_pkts_s": {\n            "invalid_or_null": 7,\n            "available": true,\n            "reference_std": 3.840823380586106,\n            "near_constant": false\n          },\n          "flow_byts_s": {\n            "invalid_or_null": 5,\n            "available": true,\n            "reference_std": 4.985637883459185,\n            "near_constant": false\n          },\n          "down_up_ratio": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.3555765408371492,\n            "near_constant": false\n          },\n          "syn_flag_cnt": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.1506367131351718,\n            "near_constant": false\n          },\n          "ack_flag_cnt": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.3294001170930551,\n            "near_constant": false\n          },\n          "rst_flag_cnt": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.27314421194577493,\n            "near_constant": false\n          },\n          "fin_flag_cnt": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.042873913468058716,\n            "near_constant": false\n          }\n        },\n        "imputation_surrogates": {\n          "flow_duration_valid": 10.290483144360632,\n          "tot_fwd_pkts_valid": 1.0986122886681096,\n          "tot_bwd_pkts_valid": 0.6931471805599453,\n          "totlen_fwd_pkts_valid": 3.828641396489095,\n          "totlen_bwd_pkts_valid": 4.6913478822291435,\n          "pkt_len_mean_valid": 4.013375499688434,\n          "pkt_len_std_valid": 3.589566349852494,\n          "flow_iat_mean_valid": 9.566615235893575,\n          "flow_iat_std_valid": 4.7100445409878064,\n          "fwd_iat_mean_valid": 8.769040810858803,\n          "bwd_iat_mean_valid": 0.0,\n          "flow_pkts_s_valid": 4.726508071127536,\n          "flow_byts_s_valid": 6.752625541472987,\n          "down_up_ratio_valid": 0.0,\n          "syn_flag_cnt_valid": 0.0,\n          "ack_flag_cnt_valid": 0.0,\n          "rst_flag_cnt_valid": 0.0,\n          "fin_flag_cnt_valid": 0.0\n        },\n        "reference_means": [\n          10.62231300764484,\n          1.4363484754094262,\n          1.0798649837457388,\n          3.743066179733096,\n          3.946154886258588,\n          3.1716542466921442,\n          3.210042151135082,\n          9.465636467739502,\n          6.1415758897976875,\n          7.789394109351316,\n          5.044616165162009,\n          5.1301504432809875,\n          6.431092524929756,\n          0.3392943193650265,\n          0.03444876676732746,\n          0.2388277905479885,\n          0.1332528752834452,\n          0.0026621383032741515\n        ],\n        "reference_scales": [\n          4.922656482085623,\n          0.7472669479236256,\n          0.9304382766169819,\n          2.6960342991497512,\n          3.1685009030872178,\n          2.1105019819450357,\n          2.3422247994945606,\n          4.160112954121857,\n          6.278238645416323,\n          6.013749976297104,\n          6.151272138517253,\n          3.840823380586106,\n          4.985637883459185,\n          0.3555765408371492,\n          0.1506367131351718,\n          0.3294001170930551,\n          0.27314421194577493,\n          0.042873913468058716\n        ]\n      },\n      "training": {\n        "fit_rows": 318175,\n        "internal_rows": 14426,\n        "reference_rows": 13191622,\n        "share_of_reference": 0.024119475224502338\n      },\n      "selected_k": 12,\n      "trials": [\n        {\n          "k": 3,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 12,\n          "wssse": 2623770.7352790604,\n          "cluster_sizes": [\n            114305,\n            93389,\n            110481\n          ],\n          "training_rows": 318175,\n          "empty_clusters": 0,\n          "smallest_share": 0.2935145753123281,\n          "fit_seconds": 16.727,\n          "jvm_peak_rss_bytes": 5518626816,\n          "silhouette": 0.5905867376279668,\n          "silhouette_rows": 14426\n        },\n        {\n          "k": 5,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 16,\n          "wssse": 2043482.959962206,\n          "cluster_sizes": [\n            48817,\n            53488,\n            92308,\n            45527,\n            78035\n          ],\n          "training_rows": 318175,\n          "empty_clusters": 0,\n          "smallest_share": 0.14308792331264242,\n          "fit_seconds": 13.498,\n          "jvm_peak_rss_bytes": 5523873792,\n          "silhouette": 0.3828341860007618,\n          "silhouette_rows": 14426\n        },\n        {\n          "k": 8,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 16,\n          "wssse": 1495383.7629115605,\n          "cluster_sizes": [\n            27677,\n            61084,\n            74004,\n            13520,\n            20308,\n            38236,\n            12669,\n            70677\n          ],\n          "training_rows": 318175,\n          "empty_clusters": 0,\n          "smallest_share": 0.03981771037950813,\n          "fit_seconds": 13.774,\n          "jvm_peak_rss_bytes": 5523873792,\n          "silhouette": 0.5347387017919628,\n          "silhouette_rows": 14426\n        },\n        {\n          "k": 12,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 16,\n          "wssse": 826131.3847364823,\n          "cluster_sizes": [\n            72053,\n            32416,\n            39207,\n            34233,\n            24969,\n            1222,\n            20238,\n            12679,\n            24741,\n            3078,\n            36378,\n            16961\n          ],\n          "training_rows": 318175,\n          "empty_clusters": 0,\n          "smallest_share": 0.003840653728294178,\n          "fit_seconds": 13.601,\n          "jvm_peak_rss_bytes": 5523873792,\n          "silhouette": 0.6288732335989271,\n          "silhouette_rows": 14426\n        }\n      ],\n      "robustness": {\n        "paired_rows": 14426,\n        "seed_ari": 0.8503551767126171,\n        "scale_ari": 0.21828525609211355,\n        "alternate_seed": {\n          "k": 12,\n          "seed": 23014,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 6,\n          "wssse": 824197.6855591865,\n          "cluster_sizes": [\n            72027,\n            54502,\n            16303,\n            16915,\n            24510,\n            34164,\n            1222,\n            20983,\n            28429,\n            15991,\n            13490,\n            19639\n          ],\n          "training_rows": 318175,\n          "empty_clusters": 0,\n          "smallest_share": 0.003840653728294178,\n          "fit_seconds": 8.282,\n          "jvm_peak_rss_bytes": 5523873792\n        },\n        "linear": {\n          "k": 12,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 13,\n          "wssse": 1426972.6993342037,\n          "cluster_sizes": [\n            177802,\n            73,\n            9,\n            2,\n            3597,\n            1222,\n            9,\n            59,\n            28256,\n            1903,\n            93492,\n            11751\n          ],\n          "training_rows": 318175,\n          "empty_clusters": 0,\n          "smallest_share": 6.285848982478196e-06,\n          "fit_seconds": 10.084,\n          "jvm_peak_rss_bytes": 5523873792\n        },\n        "linear_silhouette": 0.5263812389664033,\n        "dominance": [\n          {\n            "feature": "fin_flag_cnt",\n            "share_between_centroids": 0.0649186466394683\n          },\n          {\n            "feature": "syn_flag_cnt",\n            "share_between_centroids": 0.06467752364561152\n          },\n          {\n            "feature": "rst_flag_cnt",\n            "share_between_centroids": 0.06211008232263305\n          },\n          {\n            "feature": "totlen_bwd_pkts",\n            "share_between_centroids": 0.06023091408924288\n          },\n          {\n            "feature": "pkt_len_mean",\n            "share_between_centroids": 0.06006028513836801\n          },\n          {\n            "feature": "fwd_iat_mean",\n            "share_between_centroids": 0.0592081436439001\n          },\n          {\n            "feature": "pkt_len_std",\n            "share_between_centroids": 0.05853597773311246\n          },\n          {\n            "feature": "totlen_fwd_pkts",\n            "share_between_centroids": 0.057959339994185476\n          },\n          {\n            "feature": "bwd_iat_mean",\n            "share_between_centroids": 0.0566050946292509\n          },\n          {\n            "feature": "flow_duration",\n            "share_between_centroids": 0.05623942273305119\n          },\n          {\n            "feature": "tot_bwd_pkts",\n            "share_between_centroids": 0.053953694740570934\n          },\n          {\n            "feature": "flow_pkts_s",\n            "share_between_centroids": 0.05347987389437597\n          },\n          {\n            "feature": "flow_iat_mean",\n            "share_between_centroids": 0.05341174083631976\n          },\n          {\n            "feature": "flow_iat_std",\n            "share_between_centroids": 0.05215410249811013\n          },\n          {\n            "feature": "tot_fwd_pkts",\n            "share_between_centroids": 0.05032721067251503\n          },\n          {\n            "feature": "flow_byts_s",\n            "share_between_centroids": 0.050274400604878325\n          },\n          {\n            "feature": "ack_flag_cnt",\n            "share_between_centroids": 0.046186617034507854\n          },\n          {\n            "feature": "down_up_ratio",\n            "share_between_centroids": 0.039666929149898084\n          }\n        ]\n      },\n      "calibration": {\n        "rows": 14426,\n        "cutoffs": {\n          "0.95": 8.508047634376355,\n          "0.99": 16.67940564007534,\n          "0.999": 35.317581821449224\n        },\n        "accuracy": 8192,\n        "reference_flags": {\n          "0.95": 721,\n          "0.99": 143,\n          "0.999": 16\n        }\n      },\n      "reference_occupancy": [\n        2946330,\n        1352784,\n        1953604,\n        1358963,\n        1004912,\n        62483,\n        806475,\n        453967,\n        988482,\n        122476,\n        1460945,\n        680201\n      ],\n      "retained_rows": 15322987,\n      "model": {\n        "path": "/kaggle/working/mmds_phase5/model",\n        "reload_rows": 32,\n        "sha256": {\n          "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "metadata/part-00000-2c0d5e62-616a-4fc8-86af-688d8012e022-c000.txt": "c81ff6454592ebfb717ba9d802bcfc73d514e52328a55f7dcc106acc27560ad8",\n          "stages/0_SQLTransformer_c1b46ce0d74d/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/0_SQLTransformer_c1b46ce0d74d/metadata/part-00000-a8d654ae-ad50-4736-8af3-d221a4522fe2-c000.txt": "c2f875fe3ca0ba7a7f2525d5b085a87f21af7e6c5ab95792418c2f1885fb8277",\n          "stages/1_Imputer_ded6c4fea5cf/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/1_Imputer_ded6c4fea5cf/data/part-00000-fb2041e8-1fed-41d8-8b3b-65fc52e86496-c000.snappy.parquet": "9acec6d9df3d2655f3b8388f76a96fa6ade0f42a231f65a2b156629ce12f381c",\n          "stages/1_Imputer_ded6c4fea5cf/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/1_Imputer_ded6c4fea5cf/metadata/part-00000-8e883650-80b9-4d1a-a475-fc501557da88-c000.txt": "6282b8ca24229c2232c480a74a042661120568614ef32325223e4d1c5ac5dd14",\n          "stages/2_VectorAssembler_09b31eb4789b/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/2_VectorAssembler_09b31eb4789b/metadata/part-00000-7bf6b8c7-cfe0-4be3-8aad-57fe393e6647-c000.txt": "7761b8b94869ce3c7868118827c1538608bfa271171e0f5cc7b7c0b7f59dda53",\n          "stages/3_StandardScaler_140d8fdfd488/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/3_StandardScaler_140d8fdfd488/data/part-00000-8c91b2ae-2f4e-4d80-b87f-af347bc89a75-c000.snappy.parquet": "5d40366e6580ccde43123e9f86beb790d91159bef135c806f94425a64857015a",\n          "stages/3_StandardScaler_140d8fdfd488/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/3_StandardScaler_140d8fdfd488/metadata/part-00000-b0ea8d09-fb66-4124-bcb2-765abceea562-c000.txt": "bf39baf525e8a513c90ac1deab4bbb394a4382de91a4a3cb4e4e891eea40cbf8",\n          "stages/4_KMeans_827890df35c6/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/4_KMeans_827890df35c6/data/part-00000-9b1c962b-9b22-4a80-bdeb-b7164a439669-c000.snappy.parquet": "292de4cee2346e4b1020503e93e2c96a96282d815f46bdeb9b91671ec991ef5d",\n          "stages/4_KMeans_827890df35c6/data/part-00001-9b1c962b-9b22-4a80-bdeb-b7164a439669-c000.snappy.parquet": "2e119fb76d03b9d67d89562135ad5d63bf27a2495cc10f212024eba6609ff785",\n          "stages/4_KMeans_827890df35c6/data/part-00002-9b1c962b-9b22-4a80-bdeb-b7164a439669-c000.snappy.parquet": "05eebc0eb5be2c9849e72daf13e63a30719b8b56abfd3850e1a9cf5411c2a89a",\n          "stages/4_KMeans_827890df35c6/data/part-00003-9b1c962b-9b22-4a80-bdeb-b7164a439669-c000.snappy.parquet": "39ba8b20846d0e2fb2afb79073094501af559e0bd52defa4ce55c92c93222532",\n          "stages/4_KMeans_827890df35c6/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/4_KMeans_827890df35c6/metadata/part-00000-c17af254-4358-4e37-b28e-62545b7ff7f7-c000.txt": "8e82b0ef89fcfa118bcacd2b0b2862cc88104fdbf882126fd209e411efa73fc8"\n        },\n        "bytes": 27179\n      },\n      "elapsed_seconds": 652.492,\n      "figures": [\n        "phase5_b_cluster_sizes.png",\n        "phase5_b_cluster_attack_share.png",\n        "phase5_b_family_heatmap.png",\n        "phase5_b_daily_occupancy.png",\n        "phase5_b_temporal_shift.png",\n        "phase5_b_centroid_profiles.png",\n        "phase5_b_distance_histogram.png",\n        "phase5_b_daily_novelty.png",\n        "phase5_b_binary_novelty.png",\n        "phase5_b_supervised_comparison.png"\n      ],\n      "version": 2,\n      "log_events": 735,\n      "downloaded_reload": {\n        "probes": 3,\n        "pipeline_stages": 5,\n        "verified": true\n      }\n    },\n    "C": {\n      "experiment": "C",\n      "config": {\n        "version": "phase5-v1",\n        "seed": 23013,\n        "candidate_k": [\n          3,\n          5,\n          8,\n          12\n        ],\n        "max_iter": 16,\n        "init_steps": 2,\n        "tolerance": 0.0001,\n        "training_permyriad": 250,\n        "internal_permyriad": 12,\n        "reference_percentiles": [\n          0.95,\n          0.99,\n          0.999\n        ],\n        "primary_percentile": 0.99,\n        "quantile_accuracy": 8192,\n        "max_internal_rows": 40000,\n        "rare_support": 100,\n        "window_hours": 3,\n        "lag_reference": "full retained phase4 train-day population",\n        "spark": {\n          "master": "local[4]",\n          "driver_memory": "8g",\n          "shuffle_partitions": 48,\n          "input_partition_bytes": 67108864\n        }\n      },\n      "status": "COMPLETE",\n      "resources": {\n        "spark_version": "4.3.0",\n        "master": "local[4]",\n        "driver_memory": "8g",\n        "shuffle_partitions": 48,\n        "input_partition_bytes": 67108864,\n        "default_parallelism": 4,\n        "cpu_affinity": 4,\n        "scratch_output_free_bytes_start": 20939968512,\n        "memory.max": "32212254720",\n        "cpu.max": "400000 100000",\n        "driver_peak_rss_bytes": 4567228416\n      },\n      "features": [\n        "flow_duration",\n        "tot_fwd_pkts",\n        "tot_bwd_pkts",\n        "totlen_fwd_pkts",\n        "totlen_bwd_pkts",\n        "pkt_len_mean",\n        "pkt_len_std",\n        "flow_iat_mean",\n        "flow_iat_std",\n        "fwd_iat_mean",\n        "bwd_iat_mean",\n        "flow_pkts_s",\n        "flow_byts_s",\n        "down_up_ratio",\n        "syn_flag_cnt",\n        "ack_flag_cnt",\n        "rst_flag_cnt",\n        "fin_flag_cnt"\n      ],\n      "input": {\n        "path": "/kaggle/input/notebooks/ramcharan2345/mmds-phase-2-spark-data-engineering/mmds_phase2/phase2-v1/feature_ready",\n        "days": [\n          "2018-02-14",\n          "2018-02-15",\n          "2018-02-16",\n          "2018-02-20",\n          "2018-02-21",\n          "2018-02-22",\n          "2018-02-23",\n          "2018-02-28",\n          "2018-03-01",\n          "2018-03-02"\n        ],\n        "totals": {\n          "0": 13484708,\n          "1": 2748235\n        },\n        "rows": 16232943,\n        "temporally_flagged": 14,\n        "policy_version": "phase2-v1",\n        "receipt_sha256": "1fa00cd2b9c853ee33eaacee8ca7632dbb41f739573326280d6ffbfb2ccff25a",\n        "producer": {\n          "url": "https://www.kaggle.com/code/ramcharan2345/mmds-phase-2-spark-data-engineering",\n          "kernel_version": 1,\n          "status": "COMPLETE"\n        }\n      },\n      "purge": {\n        "groups_before": 142016,\n        "groups_after": 0,\n        "rows": 859101\n      },\n      "matched_phase4": {\n        "protocol": "C",\n        "kaggle_version": "phase4-v1",\n        "checked": "matched split rules and family/binary totals",\n        "purged_rows": 859101\n      },\n      "preprocessing": {\n        "transform": "signed_log1p",\n        "fit_columns": [\n          "flow_duration",\n          "tot_fwd_pkts",\n          "tot_bwd_pkts",\n          "totlen_fwd_pkts",\n          "totlen_bwd_pkts",\n          "pkt_len_mean",\n          "pkt_len_std",\n          "flow_iat_mean",\n          "flow_iat_std",\n          "fwd_iat_mean",\n          "bwd_iat_mean",\n          "flow_pkts_s",\n          "flow_byts_s",\n          "down_up_ratio",\n          "syn_flag_cnt",\n          "ack_flag_cnt",\n          "rst_flag_cnt",\n          "fin_flag_cnt"\n        ],\n        "active": [\n          "flow_duration",\n          "tot_fwd_pkts",\n          "tot_bwd_pkts",\n          "totlen_fwd_pkts",\n          "totlen_bwd_pkts",\n          "pkt_len_mean",\n          "pkt_len_std",\n          "flow_iat_mean",\n          "flow_iat_std",\n          "fwd_iat_mean",\n          "bwd_iat_mean",\n          "flow_pkts_s",\n          "flow_byts_s",\n          "down_up_ratio",\n          "syn_flag_cnt",\n          "ack_flag_cnt",\n          "rst_flag_cnt",\n          "fin_flag_cnt"\n        ],\n        "dropped": [],\n        "reference_stats": {\n          "flow_duration": {\n            "invalid_or_null": 1,\n            "available": true,\n            "reference_std": 4.869136559640692,\n            "near_constant": false\n          },\n          "tot_fwd_pkts": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.7456033837944145,\n            "near_constant": false\n          },\n          "tot_bwd_pkts": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.9269881226653868,\n            "near_constant": false\n          },\n          "totlen_fwd_pkts": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 2.6984430402165542,\n            "near_constant": false\n          },\n          "totlen_bwd_pkts": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 3.155665732457084,\n            "near_constant": false\n          },\n          "pkt_len_mean": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 2.1063324960911207,\n            "near_constant": false\n          },\n          "pkt_len_std": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 2.3411929848797985,\n            "near_constant": false\n          },\n          "flow_iat_mean": {\n            "invalid_or_null": 1,\n            "available": true,\n            "reference_std": 4.093219288443269,\n            "near_constant": false\n          },\n          "flow_iat_std": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 6.251545398563713,\n            "near_constant": false\n          },\n          "fwd_iat_mean": {\n            "invalid_or_null": 1,\n            "available": true,\n            "reference_std": 5.957646239775045,\n            "near_constant": false\n          },\n          "bwd_iat_mean": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 6.135597572259061,\n            "near_constant": false\n          },\n          "flow_pkts_s": {\n            "invalid_or_null": 6,\n            "available": true,\n            "reference_std": 3.792265648088158,\n            "near_constant": false\n          },\n          "flow_byts_s": {\n            "invalid_or_null": 5,\n            "available": true,\n            "reference_std": 4.950192311247816,\n            "near_constant": false\n          },\n          "down_up_ratio": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.35636566021738614,\n            "near_constant": false\n          },\n          "syn_flag_cnt": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.14768589130246518,\n            "near_constant": false\n          },\n          "ack_flag_cnt": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.3276131152096508,\n            "near_constant": false\n          },\n          "rst_flag_cnt": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.2803962629750321,\n            "near_constant": false\n          },\n          "fin_flag_cnt": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.0437856949005969,\n            "near_constant": false\n          }\n        },\n        "imputation_surrogates": {\n          "flow_duration_valid": 10.176030606145105,\n          "tot_fwd_pkts_valid": 1.0986122886681096,\n          "tot_bwd_pkts_valid": 0.6931471805599453,\n          "totlen_fwd_pkts_valid": 3.8501476017100584,\n          "totlen_bwd_pkts_valid": 4.727387818712341,\n          "pkt_len_mean_valid": 4.031286426195792,\n          "pkt_len_std_valid": 3.5895663498534005,\n          "flow_iat_mean_valid": 9.478074806618267,\n          "flow_iat_std_valid": 4.992953920547845,\n          "fwd_iat_mean_valid": 8.564458383883352,\n          "bwd_iat_mean_valid": 0.0,\n          "flow_pkts_s_valid": 4.782380974650369,\n          "flow_byts_s_valid": 6.818838181259726,\n          "down_up_ratio_valid": 0.0,\n          "syn_flag_cnt_valid": 0.0,\n          "ack_flag_cnt_valid": 0.0,\n          "rst_flag_cnt_valid": 0.0,\n          "fin_flag_cnt_valid": 0.0\n        },\n        "reference_means": [\n          10.592229755632966,\n          1.4377055249204316,\n          1.0922496697488846,\n          3.7688694528572517,\n          3.9684904565450854,\n          3.181751208753493,\n          3.229336814616343,\n          9.42202464458342,\n          6.200338594152421,\n          7.74548661256098,\n          5.107133262762929,\n          5.150160890621316,\n          6.445406910744578,\n          0.34414538869010725,\n          0.033041776321702514,\n          0.2335106354098546,\n          0.142879132693334,\n          0.0027770336890388905\n        ],\n        "reference_scales": [\n          4.869136559640692,\n          0.7456033837944145,\n          0.9269881226653868,\n          2.6984430402165542,\n          3.155665732457084,\n          2.1063324960911207,\n          2.3411929848797985,\n          4.093219288443269,\n          6.251545398563713,\n          5.957646239775045,\n          6.135597572259061,\n          3.792265648088158,\n          4.950192311247816,\n          0.35636566021738614,\n          0.14768589130246518,\n          0.3276131152096508,\n          0.2803962629750321,\n          0.0437856949005969\n        ]\n      },\n      "training": {\n        "fit_rows": 323731,\n        "internal_rows": 15229,\n        "reference_rows": 13522722,\n        "share_of_reference": 0.02393978076307418\n      },\n      "selected_k": 12,\n      "trials": [\n        {\n          "k": 3,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 9,\n          "wssse": 2679253.389944727,\n          "cluster_sizes": [\n            114634,\n            93969,\n            115128\n          ],\n          "training_rows": 323731,\n          "empty_clusters": 0,\n          "smallest_share": 0.2902687725302798,\n          "fit_seconds": 17.529,\n          "jvm_peak_rss_bytes": 3822231552,\n          "silhouette": 0.576516009487571,\n          "silhouette_rows": 15229\n        },\n        {\n          "k": 5,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 16,\n          "wssse": 2126996.270917423,\n          "cluster_sizes": [\n            113300,\n            23166,\n            114475,\n            33764,\n            39026\n          ],\n          "training_rows": 323731,\n          "empty_clusters": 0,\n          "smallest_share": 0.0715594119809348,\n          "fit_seconds": 14.797,\n          "jvm_peak_rss_bytes": 3822231552,\n          "silhouette": 0.5301041497841288,\n          "silhouette_rows": 15229\n        },\n        {\n          "k": 8,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 16,\n          "wssse": 1494304.9269670998,\n          "cluster_sizes": [\n            33142,\n            44336,\n            72006,\n            92349,\n            3193,\n            49749,\n            16776,\n            12180\n          ],\n          "training_rows": 323731,\n          "empty_clusters": 0,\n          "smallest_share": 0.009863127102439988,\n          "fit_seconds": 15.078,\n          "jvm_peak_rss_bytes": 3822231552,\n          "silhouette": 0.5244041845116935,\n          "silhouette_rows": 15229\n        },\n        {\n          "k": 12,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 11,\n          "wssse": 856400.4034315124,\n          "cluster_sizes": [\n            14243,\n            73439,\n            17883,\n            31132,\n            42530,\n            21892,\n            1297,\n            45116,\n            29322,\n            12352,\n            14601,\n            19924\n          ],\n          "training_rows": 323731,\n          "empty_clusters": 0,\n          "smallest_share": 0.004006412731557991,\n          "fit_seconds": 12.038,\n          "jvm_peak_rss_bytes": 3822231552,\n          "silhouette": 0.6029164097465304,\n          "silhouette_rows": 15229\n        }\n      ],\n      "robustness": {\n        "paired_rows": 15229,\n        "seed_ari": 0.7973700368914762,\n        "scale_ari": 0.4201543819333072,\n        "alternate_seed": {\n          "k": 12,\n          "seed": 23014,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 9,\n          "wssse": 842656.9529380443,\n          "cluster_sizes": [\n            54239,\n            73342,\n            42519,\n            21892,\n            20582,\n            1297,\n            16207,\n            14455,\n            35981,\n            12293,\n            13091,\n            17833\n          ],\n          "training_rows": 323731,\n          "empty_clusters": 0,\n          "smallest_share": 0.004006412731557991,\n          "fit_seconds": 10.729,\n          "jvm_peak_rss_bytes": 3822231552\n        },\n        "linear": {\n          "k": 12,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 16,\n          "wssse": 1390337.5175443299,\n          "cluster_sizes": [\n            17831,\n            10793,\n            1,\n            129627,\n            93981,\n            70,\n            1297,\n            59,\n            1828,\n            5727,\n            2936,\n            59581\n          ],\n          "training_rows": 323731,\n          "empty_clusters": 0,\n          "smallest_share": 3.0889843728280578e-06,\n          "fit_seconds": 11.83,\n          "jvm_peak_rss_bytes": 3822231552\n        },\n        "linear_silhouette": 0.5941173468373271,\n        "dominance": [\n          {\n            "feature": "fin_flag_cnt",\n            "share_between_centroids": 0.06512712921503346\n          },\n          {\n            "feature": "pkt_len_mean",\n            "share_between_centroids": 0.06047417836859744\n          },\n          {\n            "feature": "totlen_bwd_pkts",\n            "share_between_centroids": 0.06031783885207001\n          },\n          {\n            "feature": "fwd_iat_mean",\n            "share_between_centroids": 0.06002741298235094\n          },\n          {\n            "feature": "pkt_len_std",\n            "share_between_centroids": 0.05903461994649836\n          },\n          {\n            "feature": "totlen_fwd_pkts",\n            "share_between_centroids": 0.05887461070819318\n          },\n          {\n            "feature": "flow_iat_std",\n            "share_between_centroids": 0.0586375097921818\n          },\n          {\n            "feature": "ack_flag_cnt",\n            "share_between_centroids": 0.05840868791776876\n          },\n          {\n            "feature": "bwd_iat_mean",\n            "share_between_centroids": 0.0566957798062644\n          },\n          {\n            "feature": "tot_bwd_pkts",\n            "share_between_centroids": 0.05657030407659864\n          },\n          {\n            "feature": "flow_duration",\n            "share_between_centroids": 0.055108822297945156\n          },\n          {\n            "feature": "syn_flag_cnt",\n            "share_between_centroids": 0.053792429388319865\n          },\n          {\n            "feature": "flow_byts_s",\n            "share_between_centroids": 0.05315918925861557\n          },\n          {\n            "feature": "tot_fwd_pkts",\n            "share_between_centroids": 0.05271965026062504\n          },\n          {\n            "feature": "flow_iat_mean",\n            "share_between_centroids": 0.05141474904055002\n          },\n          {\n            "feature": "flow_pkts_s",\n            "share_between_centroids": 0.051252188351870925\n          },\n          {\n            "feature": "down_up_ratio",\n            "share_between_centroids": 0.04843469529817226\n          },\n          {\n            "feature": "rst_flag_cnt",\n            "share_between_centroids": 0.039950204438344156\n          }\n        ]\n      },\n      "calibration": {\n        "rows": 15229,\n        "cutoffs": {\n          "0.95": 8.43342825481104,\n          "0.99": 17.880003619372005,\n          "0.999": 42.38438316542496\n        },\n        "accuracy": 8192,\n        "reference_flags": {\n          "0.95": 761,\n          "0.99": 153,\n          "0.999": 17\n        }\n      },\n      "reference_occupancy": [\n        574002,\n        2964362,\n        718148,\n        1245715,\n        2195139,\n        952319,\n        62942,\n        1796161,\n        1172002,\n        468073,\n        585595,\n        788264\n      ],\n      "retained_rows": 15373842,\n      "model": {\n        "path": "/kaggle/working/mmds_phase5/model",\n        "reload_rows": 32,\n        "sha256": {\n          "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "metadata/part-00000-4b0ee0c7-d84b-45e3-9817-206310f7c865-c000.txt": "9da180e2ce2ef7d0b4804a19a59f622c064abc2bf8890b332304ca3d406a2082",\n          "stages/0_SQLTransformer_de35da705442/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/0_SQLTransformer_de35da705442/metadata/part-00000-65a40e82-9196-455e-8074-c37a6c2bd711-c000.txt": "6d1f231457c948d2f6446014349f659e2e7a9d718039ae040ce58598b33bd503",\n          "stages/1_Imputer_5f45b06a8ef8/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/1_Imputer_5f45b06a8ef8/data/part-00000-9f75178b-2fee-47c9-9f70-3f7efeea53cf-c000.snappy.parquet": "7eedbba3a84d32352b0d3e289fc1e63f361ab9cc197456e3623af4c7410b4929",\n          "stages/1_Imputer_5f45b06a8ef8/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/1_Imputer_5f45b06a8ef8/metadata/part-00000-1036018f-36e2-483e-8cc0-faf518e5e67c-c000.txt": "757957c77896c481bfbb67d4f44d5b59d75415d16553799068c7c13915847962",\n          "stages/2_VectorAssembler_676a48355ddb/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/2_VectorAssembler_676a48355ddb/metadata/part-00000-e14a109c-8af8-4654-8154-6e9161cd951a-c000.txt": "7b960351421f31b84e2e0e38f691e7ce874de3903721b6a00b857f398a995229",\n          "stages/3_StandardScaler_243daebf2f5b/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/3_StandardScaler_243daebf2f5b/data/part-00000-9fe65596-b9fc-4519-888a-2319358c49ee-c000.snappy.parquet": "bb04a58adf0ba3f3273c0b854ea101062fbfc6b515dac68d84ae8b8cac146fdf",\n          "stages/3_StandardScaler_243daebf2f5b/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/3_StandardScaler_243daebf2f5b/metadata/part-00000-9698ab33-c95f-4830-81ac-9682bb640777-c000.txt": "6fb8b912fa0aaf33c40e738c5c860fefc80db86402b48b31028c440aa9eff021",\n          "stages/4_KMeans_66e40960aec2/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/4_KMeans_66e40960aec2/data/part-00000-1b083e1e-938c-4d35-aeda-d1b570609c1a-c000.snappy.parquet": "7b963520f010e29da53b05e34cc93382cabf1174d8ae301724264e97789c3150",\n          "stages/4_KMeans_66e40960aec2/data/part-00001-1b083e1e-938c-4d35-aeda-d1b570609c1a-c000.snappy.parquet": "e2ad708c0125969e4c6dc6f18bf31a639ffc80429115f78fcd365c928e209504",\n          "stages/4_KMeans_66e40960aec2/data/part-00002-1b083e1e-938c-4d35-aeda-d1b570609c1a-c000.snappy.parquet": "a2306ebace3f4a8c6f9cde43dea2e44dc407900a01ca7d4b35c87ea0d425b48e",\n          "stages/4_KMeans_66e40960aec2/data/part-00003-1b083e1e-938c-4d35-aeda-d1b570609c1a-c000.snappy.parquet": "d05afce580ec2aafd1da15bac100217174f16571502763502c691713827cbcb8",\n          "stages/4_KMeans_66e40960aec2/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/4_KMeans_66e40960aec2/metadata/part-00000-db49e59c-444c-443d-8273-1424303b6238-c000.txt": "df1f0c73cd78654cdecf72fb888de40dfd16bd7a2913c4e67a85ddb35e4b91ea"\n        },\n        "bytes": 27181\n      },\n      "elapsed_seconds": 657.111,\n      "figures": [\n        "phase5_c_cluster_sizes.png",\n        "phase5_c_cluster_attack_share.png",\n        "phase5_c_family_heatmap.png",\n        "phase5_c_daily_occupancy.png",\n        "phase5_c_temporal_shift.png",\n        "phase5_c_centroid_profiles.png",\n        "phase5_c_distance_histogram.png",\n        "phase5_c_daily_novelty.png",\n        "phase5_c_binary_novelty.png",\n        "phase5_c_supervised_comparison.png"\n      ],\n      "version": 3,\n      "log_events": 783,\n      "downloaded_reload": {\n        "probes": 3,\n        "pipeline_stages": 5,\n        "verified": true\n      }\n    }\n  }\n}\n', 'reports/phase5_novelty_metrics.csv': 'experiment,split_rank,percentile,threshold,scope,family,rows,attacks,benign,tp,fp,flags,flag_rate,benign_fpr,attack_recall,precision,support_status\nG,0,0.95,13.37902829503273,binary,All,16232943,2748235,13484708,20455,771700,792155,0.04879922266714052,0.05722778720903708,0.007442958844494739,0.025821966660565167,observed\nG,0,0.95,13.37902829503273,family,Bot,286191,286191,0,1168,0,1168,0.004081190533594697,,0.004081190533594697,,observed\nG,0,0.95,13.37902829503273,family,Credential brute force,380949,380949,0,0,0,0,0.0,,0.0,,observed\nG,0,0.95,13.37902829503273,family,DDoS,1263933,1263933,0,2527,0,2527,0.0019993148370997514,,0.0019993148370997514,,observed\nG,0,0.95,13.37902829503273,family,DoS,654300,654300,0,5110,0,5110,0.007809873146874522,,0.007809873146874522,,observed\nG,0,0.95,13.37902829503273,family,Infiltration,161934,161934,0,11281,0,11281,0.0696641841737992,,0.0696641841737992,,observed\nG,0,0.95,13.37902829503273,family,Web attack,928,928,0,369,0,369,0.39762931034482757,,0.39762931034482757,,observed\nG,0,0.99,31.042924705057366,binary,All,16232943,2748235,13484708,10142,183442,193584,0.011925379150287166,0.01360370576804481,0.0036903685456301954,0.052390693445739316,observed\nG,0,0.99,31.042924705057366,family,Bot,286191,286191,0,330,0,330,0.0011530760925395978,,0.0011530760925395978,,observed\nG,0,0.99,31.042924705057366,family,Credential brute force,380949,380949,0,0,0,0,0.0,,0.0,,observed\nG,0,0.99,31.042924705057366,family,DDoS,1263933,1263933,0,2527,0,2527,0.0019993148370997514,,0.0019993148370997514,,observed\nG,0,0.99,31.042924705057366,family,DoS,654300,654300,0,3813,0,3813,0.0058276020174232,,0.0058276020174232,,observed\nG,0,0.99,31.042924705057366,family,Infiltration,161934,161934,0,3364,0,3364,0.020773895537688193,,0.020773895537688193,,observed\nG,0,0.99,31.042924705057366,family,Web attack,928,928,0,108,0,108,0.11637931034482758,,0.11637931034482758,,observed\nG,0,0.999,263.37528662305107,binary,All,16232943,2748235,13484708,1621,26198,27819,0.0017137373056752556,0.0019427932736845322,0.0005898331110694682,0.0582695280204177,observed\nG,0,0.999,263.37528662305107,family,Bot,286191,286191,0,0,0,0,0.0,,0.0,,observed\nG,0,0.999,263.37528662305107,family,Credential brute force,380949,380949,0,0,0,0,0.0,,0.0,,observed\nG,0,0.999,263.37528662305107,family,DDoS,1263933,1263933,0,0,0,0,0.0,,0.0,,observed\nG,0,0.999,263.37528662305107,family,DoS,654300,654300,0,1276,0,1276,0.0019501757603545774,,0.0019501757603545774,,observed\nG,0,0.999,263.37528662305107,family,Infiltration,161934,161934,0,345,0,345,0.0021304976101374634,,0.0021304976101374634,,observed\nG,0,0.999,263.37528662305107,family,Web attack,928,928,0,0,0,0,0.0,,0.0,,observed\nB,0,0.95,8.508047634376355,binary,All,13191622,2299544,10892078,14567,562170,576737,0.04371994588686668,0.0516127409296922,0.006334734190778694,0.025257613088808244,observed\nB,0,0.95,8.508047634376355,family,Bot,0,0,0,0,0,0,,,,,absent\nB,0,0.95,8.508047634376355,family,Credential brute force,380949,380949,0,7,0,7,1.837516307957233e-05,,1.837516307957233e-05,,observed\nB,0,0.95,8.508047634376355,family,DDoS,1263933,1263933,0,2527,0,2527,0.0019993148370997514,,0.0019993148370997514,,observed\nB,0,0.95,8.508047634376355,family,DoS,654300,654300,0,11923,0,11923,0.018222527892404097,,0.018222527892404097,,observed\nB,0,0.95,8.508047634376355,family,Infiltration,0,0,0,0,0,0,,,,,absent\nB,0,0.95,8.508047634376355,family,Web attack,362,362,0,110,0,110,0.30386740331491713,,0.30386740331491713,,observed\nB,1,0.95,8.508047634376355,binary,All,1151256,57056,1094200,6783,80604,87387,0.07590579332485564,0.07366477791994151,0.11888320246775098,0.07762024099694463,observed\nB,1,0.95,8.508047634376355,family,Bot,0,0,0,0,0,0,,,,,absent\nB,1,0.95,8.508047634376355,family,Credential brute force,0,0,0,0,0,0,,,,,absent\nB,1,0.95,8.508047634376355,family,DDoS,0,0,0,0,0,0,,,,,absent\nB,1,0.95,8.508047634376355,family,DoS,0,0,0,0,0,0,,,,,absent\nB,1,0.95,8.508047634376355,family,Infiltration,56633,56633,0,6514,0,6514,0.11502127734712976,,0.11502127734712976,,observed\nB,1,0.95,8.508047634376355,family,Web attack,423,423,0,269,0,269,0.6359338061465721,,0.6359338061465721,,observed\nB,2,0.95,8.508047634376355,binary,All,980109,208525,771584,8692,53399,62091,0.06335111706963205,0.06920698200066357,0.04168325140870399,0.1399880820086647,observed\nB,2,0.95,8.508047634376355,family,Bot,143274,143274,0,1202,0,1202,0.008389519382442034,,0.008389519382442034,,observed\nB,2,0.95,8.508047634376355,family,Credential brute force,0,0,0,0,0,0,,,,,absent\nB,2,0.95,8.508047634376355,family,DDoS,0,0,0,0,0,0,,,,,absent\nB,2,0.95,8.508047634376355,family,DoS,0,0,0,0,0,0,,,,,absent\nB,2,0.95,8.508047634376355,family,Infiltration,65251,65251,0,7490,0,7490,0.11478751283505234,,0.11478751283505234,,observed\nB,2,0.95,8.508047634376355,family,Web attack,0,0,0,0,0,0,,,,,absent\nB,0,0.99,16.67940564007534,binary,All,13191622,2299544,10892078,4903,127316,132219,0.010022952446636205,0.011688862308918463,0.0021321618546981487,0.0370824162941786,observed\nB,0,0.99,16.67940564007534,family,Bot,0,0,0,0,0,0,,,,,absent\nB,0,0.99,16.67940564007534,family,Credential brute force,380949,380949,0,0,0,0,0.0,,0.0,,observed\nB,0,0.99,16.67940564007534,family,DDoS,1263933,1263933,0,2527,0,2527,0.0019993148370997514,,0.0019993148370997514,,observed\nB,0,0.99,16.67940564007534,family,DoS,654300,654300,0,2272,0,2272,0.0034724132660858935,,0.0034724132660858935,,observed\nB,0,0.99,16.67940564007534,family,Infiltration,0,0,0,0,0,0,,,,,absent\nB,0,0.99,16.67940564007534,family,Web attack,362,362,0,104,0,104,0.287292817679558,,0.287292817679558,,observed\nB,1,0.99,16.67940564007534,binary,All,1151256,57056,1094200,2469,22625,25094,0.021797063381211478,0.020677207091939317,0.043273275378575435,0.09839005339921894,observed\nB,1,0.99,16.67940564007534,family,Bot,0,0,0,0,0,0,,,,,absent\nB,1,0.99,16.67940564007534,family,Credential brute force,0,0,0,0,0,0,,,,,absent\nB,1,0.99,16.67940564007534,family,DDoS,0,0,0,0,0,0,,,,,absent\nB,1,0.99,16.67940564007534,family,DoS,0,0,0,0,0,0,,,,,absent\nB,1,0.99,16.67940564007534,family,Infiltration,56633,56633,0,2207,0,2207,0.03897021171401833,,0.03897021171401833,,observed\nB,1,0.99,16.67940564007534,family,Web attack,423,423,0,262,0,262,0.6193853427895981,,0.6193853427895981,,observed\nB,2,0.99,16.67940564007534,binary,All,980109,208525,771584,4437,16605,21042,0.021469040688331605,0.02152066398473789,0.0212780242177197,0.21086398631308811,observed\nB,2,0.99,16.67940564007534,family,Bot,143274,143274,0,987,0,987,0.006888898195066795,,0.006888898195066795,,observed\nB,2,0.99,16.67940564007534,family,Credential brute force,0,0,0,0,0,0,,,,,absent\nB,2,0.99,16.67940564007534,family,DDoS,0,0,0,0,0,0,,,,,absent\nB,2,0.99,16.67940564007534,family,DoS,0,0,0,0,0,0,,,,,absent\nB,2,0.99,16.67940564007534,family,Infiltration,65251,65251,0,3450,0,3450,0.05287275290800141,,0.05287275290800141,,observed\nB,2,0.99,16.67940564007534,family,Web attack,0,0,0,0,0,0,,,,,absent\nB,0,0.999,35.317581821449224,binary,All,13191622,2299544,10892078,3794,11329,15123,0.0011464094407799131,0.0010401137413815803,0.001649892326478641,0.25087614891225285,observed\nB,0,0.999,35.317581821449224,family,Bot,0,0,0,0,0,0,,,,,absent\nB,0,0.999,35.317581821449224,family,Credential brute force,380949,380949,0,0,0,0,0.0,,0.0,,observed\nB,0,0.999,35.317581821449224,family,DDoS,1263933,1263933,0,2527,0,2527,0.0019993148370997514,,0.0019993148370997514,,observed\nB,0,0.999,35.317581821449224,family,DoS,654300,654300,0,1267,0,1267,0.0019364206021702582,,0.0019364206021702582,,observed\nB,0,0.999,35.317581821449224,family,Infiltration,0,0,0,0,0,0,,,,,absent\nB,0,0.999,35.317581821449224,family,Web attack,362,362,0,0,0,0,0.0,,0.0,,observed\nB,1,0.999,35.317581821449224,binary,All,1151256,57056,1094200,69,2070,2139,0.0018579707727907607,0.0018917930908426248,0.0012093381940549636,0.03225806451612903,observed\nB,1,0.999,35.317581821449224,family,Bot,0,0,0,0,0,0,,,,,absent\nB,1,0.999,35.317581821449224,family,Credential brute force,0,0,0,0,0,0,,,,,absent\nB,1,0.999,35.317581821449224,family,DDoS,0,0,0,0,0,0,,,,,absent\nB,1,0.999,35.317581821449224,family,DoS,0,0,0,0,0,0,,,,,absent\nB,1,0.999,35.317581821449224,family,Infiltration,56633,56633,0,69,0,69,0.0012183709144844878,,0.0012183709144844878,,observed\nB,1,0.999,35.317581821449224,family,Web attack,423,423,0,0,0,0,0.0,,0.0,,observed\nB,2,0.999,35.317581821449224,binary,All,980109,208525,771584,455,1557,2012,0.0020528328991979462,0.0020179267584605177,0.0021819925668385086,0.2261431411530815,observed\nB,2,0.999,35.317581821449224,family,Bot,143274,143274,0,223,0,223,0.0015564582548124573,,0.0015564582548124573,,observed\nB,2,0.999,35.317581821449224,family,Credential brute force,0,0,0,0,0,0,,,,,absent\nB,2,0.999,35.317581821449224,family,DDoS,0,0,0,0,0,0,,,,,absent\nB,2,0.999,35.317581821449224,family,DoS,0,0,0,0,0,0,,,,,absent\nB,2,0.999,35.317581821449224,family,Infiltration,65251,65251,0,232,0,232,0.003555501065117776,,0.003555501065117776,,observed\nB,2,0.999,35.317581821449224,family,Web attack,0,0,0,0,0,0,,,,,absent\nC,0,0.95,8.43342825481104,binary,All,13522722,2678436,10844286,25388,612614,638002,0.047179998228167375,0.05649187046523856,0.009478665908014976,0.03979297870539591,observed\nC,0,0.95,8.43342825481104,family,Bot,286191,286191,0,1164,0,1164,0.004067213853685126,,0.004067213853685126,,observed\nC,0,0.95,8.43342825481104,family,Credential brute force,380949,380949,0,3170,0,3170,0.008321323851749184,,0.008321323851749184,,observed\nC,0,0.95,8.43342825481104,family,DDoS,1263933,1263933,0,2527,0,2527,0.0019993148370997514,,0.0019993148370997514,,observed\nC,0,0.95,8.43342825481104,family,DoS,654300,654300,0,10849,0,10849,0.016581079015742014,,0.016581079015742014,,observed\nC,0,0.95,8.43342825481104,family,Infiltration,93063,93063,0,7678,0,7678,0.08250325048622976,,0.08250325048622976,,observed\nC,0,0.95,8.43342825481104,family,Web attack,0,0,0,0,0,0,,,,,absent\nC,1,0.95,8.43342825481104,binary,All,392902,41842,351060,5307,39791,45098,0.11478180309593741,0.1133452971002108,0.12683428134410402,0.11767705884961639,observed\nC,1,0.95,8.43342825481104,family,Bot,0,0,0,0,0,0,,,,,absent\nC,1,0.95,8.43342825481104,family,Credential brute force,0,0,0,0,0,0,,,,,absent\nC,1,0.95,8.43342825481104,family,DDoS,0,0,0,0,0,0,,,,,absent\nC,1,0.95,8.43342825481104,family,DoS,0,0,0,0,0,0,,,,,absent\nC,1,0.95,8.43342825481104,family,Infiltration,41842,41842,0,5307,0,5307,0.12683428134410402,,0.12683428134410402,,observed\nC,1,0.95,8.43342825481104,family,Web attack,0,0,0,0,0,0,,,,,absent\nC,2,0.95,8.43342825481104,binary,All,1458218,658,1457560,369,74098,74467,0.051067124394294956,0.05083701528582014,0.560790273556231,0.004955215061705185,observed\nC,2,0.95,8.43342825481104,family,Bot,0,0,0,0,0,0,,,,,absent\nC,2,0.95,8.43342825481104,family,Credential brute force,0,0,0,0,0,0,,,,,absent\nC,2,0.95,8.43342825481104,family,DDoS,0,0,0,0,0,0,,,,,absent\nC,2,0.95,8.43342825481104,family,DoS,0,0,0,0,0,0,,,,,absent\nC,2,0.95,8.43342825481104,family,Infiltration,0,0,0,0,0,0,,,,,absent\nC,2,0.95,8.43342825481104,family,Web attack,658,658,0,369,0,369,0.560790273556231,,0.560790273556231,,observed\nC,0,0.99,17.880003619372005,binary,All,13522722,2678436,10844286,10252,144574,154826,0.01144932211133232,0.013331813638998454,0.00382760685713603,0.06621626858537971,observed\nC,0,0.99,17.880003619372005,family,Bot,286191,286191,0,779,0,779,0.002721958412388929,,0.002721958412388929,,observed\nC,0,0.99,17.880003619372005,family,Credential brute force,380949,380949,0,0,0,0,0.0,,0.0,,observed\nC,0,0.99,17.880003619372005,family,DDoS,1263933,1263933,0,2527,0,2527,0.0019993148370997514,,0.0019993148370997514,,observed\nC,0,0.99,17.880003619372005,family,DoS,654300,654300,0,3919,0,3919,0.005989607213816292,,0.005989607213816292,,observed\nC,0,0.99,17.880003619372005,family,Infiltration,93063,93063,0,3027,0,3027,0.032526353115631346,,0.032526353115631346,,observed\nC,0,0.99,17.880003619372005,family,Web attack,0,0,0,0,0,0,,,,,absent\nC,1,0.99,17.880003619372005,binary,All,392902,41842,351060,1521,11805,13326,0.03391685458460379,0.03362673047342335,0.03635103484537068,0.11413777577667718,observed\nC,1,0.99,17.880003619372005,family,Bot,0,0,0,0,0,0,,,,,absent\nC,1,0.99,17.880003619372005,family,Credential brute force,0,0,0,0,0,0,,,,,absent\nC,1,0.99,17.880003619372005,family,DDoS,0,0,0,0,0,0,,,,,absent\nC,1,0.99,17.880003619372005,family,DoS,0,0,0,0,0,0,,,,,absent\nC,1,0.99,17.880003619372005,family,Infiltration,41842,41842,0,1521,0,1521,0.03635103484537068,,0.03635103484537068,,observed\nC,1,0.99,17.880003619372005,family,Web attack,0,0,0,0,0,0,,,,,absent\nC,2,0.99,17.880003619372005,binary,All,1458218,658,1457560,367,24062,24429,0.01675263918015002,0.01650841131754439,0.5577507598784195,0.015023128249212002,observed\nC,2,0.99,17.880003619372005,family,Bot,0,0,0,0,0,0,,,,,absent\nC,2,0.99,17.880003619372005,family,Credential brute force,0,0,0,0,0,0,,,,,absent\nC,2,0.99,17.880003619372005,family,DDoS,0,0,0,0,0,0,,,,,absent\nC,2,0.99,17.880003619372005,family,DoS,0,0,0,0,0,0,,,,,absent\nC,2,0.99,17.880003619372005,family,Infiltration,0,0,0,0,0,0,,,,,absent\nC,2,0.99,17.880003619372005,family,Web attack,658,658,0,367,0,367,0.5577507598784195,,0.5577507598784195,,observed\nC,0,0.999,42.38438316542496,binary,All,13522722,2678436,10844286,4049,11044,15093,0.0011161214435969327,0.0010184165190774202,0.0015117030983753206,0.26827005896773337,observed\nC,0,0.999,42.38438316542496,family,Bot,286191,286191,0,153,0,153,0.0005346080065410863,,0.0005346080065410863,,observed\nC,0,0.999,42.38438316542496,family,Credential brute force,380949,380949,0,0,0,0,0.0,,0.0,,observed\nC,0,0.999,42.38438316542496,family,DDoS,1263933,1263933,0,2523,0,2523,0.0019961501123872865,,0.0019961501123872865,,observed\nC,0,0.999,42.38438316542496,family,DoS,654300,654300,0,1267,0,1267,0.0019364206021702582,,0.0019364206021702582,,observed\nC,0,0.999,42.38438316542496,family,Infiltration,93063,93063,0,106,0,106,0.0011390133565434169,,0.0011390133565434169,,observed\nC,0,0.999,42.38438316542496,family,Web attack,0,0,0,0,0,0,,,,,absent\nC,1,0.999,42.38438316542496,binary,All,392902,41842,351060,39,475,514,0.0013082142620806207,0.0013530450635219052,0.0009320778165479662,0.07587548638132295,observed\nC,1,0.999,42.38438316542496,family,Bot,0,0,0,0,0,0,,,,,absent\nC,1,0.999,42.38438316542496,family,Credential brute force,0,0,0,0,0,0,,,,,absent\nC,1,0.999,42.38438316542496,family,DDoS,0,0,0,0,0,0,,,,,absent\nC,1,0.999,42.38438316542496,family,DoS,0,0,0,0,0,0,,,,,absent\nC,1,0.999,42.38438316542496,family,Infiltration,41842,41842,0,39,0,39,0.0009320778165479662,,0.0009320778165479662,,observed\nC,1,0.999,42.38438316542496,family,Web attack,0,0,0,0,0,0,,,,,absent\nC,2,0.999,42.38438316542496,binary,All,1458218,658,1457560,0,2038,2038,0.0013975962441829686,0.0013982271741815088,0.0,0.0,observed\nC,2,0.999,42.38438316542496,family,Bot,0,0,0,0,0,0,,,,,absent\nC,2,0.999,42.38438316542496,family,Credential brute force,0,0,0,0,0,0,,,,,absent\nC,2,0.999,42.38438316542496,family,DDoS,0,0,0,0,0,0,,,,,absent\nC,2,0.999,42.38438316542496,family,DoS,0,0,0,0,0,0,,,,,absent\nC,2,0.999,42.38438316542496,family,Infiltration,0,0,0,0,0,0,,,,,absent\nC,2,0.999,42.38438316542496,family,Web attack,658,658,0,0,0,0,0.0,,0.0,,observed\n', 'src/ml/phase8_baselines.py': '"""Reconcile retrospective baselines from the original saved pipelines."""\n\nimport csv\nimport json\nimport math\nfrom pathlib import Path\n\nfrom pyspark import StorageLevel\nfrom pyspark.ml import PipelineModel\nfrom pyspark.ml.functions import vector_to_array\nfrom pyspark.sql import functions as F\n\nfrom src.ml.baselines import baseline_score\nfrom src.ml.evaluate_supervised import evaluate\nfrom src.ml.novelty_scoring import novelty_metrics\nfrom src.ml.split_protocols import assign_population, crossing_groups, group_summary, mark_retention\nfrom src.ml.train_clustering import split_counts, verify_phase4_match\nfrom src.sql.traffic_analytics import require, write_rows\nfrom src.streaming.cluster_scoring import cluster_scores\nfrom src.streaming.model_scoring import mounted_summary, probe, verified_model\n\n\ndef json_file(project, name):\n    return json.loads((project / name).read_text())\n\n\ndef rows_file(project, name):\n    with (project / name).open(newline="") as handle:\n        return list(csv.DictReader(handle))\n\n\ndef retain(assigned):\n    crossings = crossing_groups(assigned, "predictor_key").persist(StorageLevel.DISK_ONLY)\n    try:\n        before = group_summary(crossings)\n        after_assignment = mark_retention(assigned, crossings, before["groups"])\n        # Materialize the small predictor projection before releasing the group join.\n        retained = after_assignment.where("retained").drop("retained").persist(StorageLevel.DISK_ONLY)\n        retained.count()\n        return retained, before\n    finally:\n        crossings.unpersist()\n\n\ndef benchmark_population(data, protocol, design, days, previous, output):\n    assigned = assign_population(data, protocol, design, days)\n    active, crossing = retain(assigned)\n    counts = split_counts(active)\n    verify_phase4_match(counts, protocol, previous)\n    excluded = data.count() - sum(row["count"] for row in counts)\n    require(excluded == previous["protocols"][protocol]["overlap"]["purged_rows"], "Benchmark population differs")\n    write_rows(output / "phase8_split_diagnostics.csv", counts)\n    return active, {"purged_rows": excluded, "prepurge_predictor_groups": crossing}\n\n\ndef load_models(spark, input_root, protocol, names, previous, clustering):\n    directory, receipt = mounted_summary(input_root, "phase4_summary.json", protocol, "protocol")\n    require(receipt["baselines"] == previous["baselines"] and receipt["registered_predictors"] == names and\n            receipt["input"]["receipt_sha256"] == previous["input"]["receipt_sha256"], "Phase 4 identity differs")\n    models, manifest = {}, {}\n    for item in previous["models"]:\n        if item["training_cohort"] != "group_sample":\n            continue\n        name = item["model"]\n        saved = next(meta for meta in receipt["models"] if meta["model"] == name and meta["training_cohort"] == "group_sample")\n        require(saved["sha256"] == item["sha256"], "Phase 4 receipt checksums differ")\n        models[name] = verified_model(spark, directory / "models" / Path(item["path"]).name, item["sha256"], names)\n        scores = models[name].transform(probe(spark, names)).select(vector_to_array("probability")[1]).collect()\n        require(len(scores) == 3 and all(math.isfinite(r[0]) and 0 <= r[0] <= 1 for r in scores), "Probability probe failed")\n        manifest[name] = {"files": item["model_files"], "sha256": item["sha256"], "probe_rows": len(scores)}\n    directory, receipt = mounted_summary(input_root, "phase5_summary.json", protocol, "experiment")\n    require(receipt["features"] == names and receipt["model"]["sha256"] == clustering["model"]["sha256"] and\n            receipt["calibration"] == clustering["calibration"], "Phase 5 receipt differs")\n    pipeline = verified_model(spark, directory / "model", clustering["model"]["sha256"], names, "_valid", "_filled")\n    km = (PipelineModel(stages=pipeline.stages[:-1]), pipeline.stages[-1], clustering["calibration"]["cutoffs"]["0.99"])\n    scores = cluster_scores(probe(spark, names), *km).select("cluster", "distance").collect()\n    k = len(km[1].clusterCenters())\n    require(k == clustering["selected_k"] and len(scores) == 3 and\n            all(0 <= r.cluster < k and math.isfinite(r.distance) for r in scores), "K-Means synthetic probe failed")\n    return models, km, manifest | {"kmeans": {"sha256": clustering["model"]["sha256"], "p99": km[2], "probe_rows": len(scores)}}\n\n\ndef reconcile(project, protocol, active, models, km, before, design, output):\n    expected = rows_file(project, "reports/phase4_metrics.csv")\n    expected_novelty = rows_file(project, "reports/phase5_novelty_metrics.csv")\n    result = []\n    for rank, partition in ((1, "validation"), (2, "test")):\n        frame = active.where(F.col("split_rank") == rank)\n        scores = {name: models[name].transform(frame).withColumn("score", vector_to_array("probability")[1]) for name in models}\n        scores.update({name: baseline_score(frame, name, params) for name, params in before["baselines"].items()})\n        for name, scored in scores.items():\n            current, _ = evaluate(scored, design)\n            earlier = next(r for r in expected if r["protocol"] == protocol and r["split"] == partition and\n                           r["model"] == name and r["training_cohort"] != "full_train_refit")\n            checks = ("rows", "tp", "tn", "fp", "fn", "pr_auc", "roc_auc")\n            agree = all(current[k] is not None and math.isclose(float(earlier[k]), current[k], abs_tol=1e-12)\n                        if earlier[k] else current[k] is None for k in checks)\n            result.append({"population": protocol, "partition": partition, "baseline": name, "matches": agree,\n                           "rows": current["rows"], "tp": current["tp"], "fp": current["fp"],\n                           "pr_auc": current["pr_auc"], "roc_auc": current["roc_auc"], "seconds": current["prediction_seconds"]})\n            write_rows(output / "phase8_baseline_reconciliation.csv", result)\n            require(agree, f"Phase 4 baseline discrepancy: {protocol}/{partition}/{name}")\n            print(f"PHASE8 {protocol} reconciled {partition} {name} {current[\'rows\']}", flush=True)\n        measured = novelty_metrics(cluster_scores(frame, *km).withColumnRenamed("distance", "score"), {"0.99": km[2]}, [], 100)\n        require(len(measured) == 1, "Unexpected novelty result groups")\n        current = measured[0]\n        earlier = next(r for r in expected_novelty if r["experiment"] == protocol and int(r["split_rank"]) == rank and\n                       r["percentile"] == "0.99" and r["scope"] == "binary")\n        agree = all(current[k] == int(earlier[k]) for k in ("rows", "tp", "fp", "flags"))\n        result.append({"population": protocol, "partition": partition, "baseline": "kmeans_p99", "matches": agree,\n                       "rows": current["rows"], "tp": current["tp"], "fp": current["fp"],\n                       "pr_auc": None, "roc_auc": None, "seconds": None})\n        write_rows(output / "phase8_baseline_reconciliation.csv", result)\n        require(agree, f"Phase 5 baseline discrepancy: {protocol}/{partition}")\n        print(f"PHASE8 {protocol} reconciled {partition} K-Means P99 {current[\'rows\']}", flush=True)\n    return result\n', 'src/ml/run_phase8.py': '"""Run a bounded phase 8 stage against the verified Phase 2 snapshot."""\n\nfrom time import monotonic\n\nfrom src.ingest.dataset_audit import atomic_json, spark_session\nfrom src.ingest.parquet_pipeline import driver_peak_bytes, resource_limits\nfrom src.ml.feature_registry import predictor_input, registry\nfrom src.ml.phase8_baselines import benchmark_population, json_file, load_models, reconcile\nfrom src.ml.train_supervised import verified_input\nfrom src.sql.traffic_analytics import locate_producer, require\n\n\ndef run(project, input_root, output, scratch, mode, population):\n    start = monotonic()\n    output.mkdir(parents=True, exist_ok=True)\n    scratch.mkdir(parents=True, exist_ok=True)\n    config = json_file(project, "configs/phase8_hybrid_experiments.yaml")\n    design = json_file(project, "configs/ml_experiments.yaml")\n    parquet = json_file(project, "manifests/parquet_layers.json")\n    phase4 = json_file(project, "manifests/phase4_models.json")\n    phase5 = json_file(project, "manifests/phase5_clustering_models.json")\n    policy = json_file(project, "configs/cleaning.yaml")\n    summary = {"status": "running", "stage": mode, "population": population, "protocol": config["version"]}\n    receipt_path = output / "phase8_summary.json"\n    atomic_json(receipt_path, summary)\n    spark = spark_session(config, scratch)\n    spark.conf.set("spark.sql.sources.partitionColumnTypeInference.enabled", "false")\n    spark.sparkContext.setLogLevel("WARN")\n    try:\n        require(spark.version == "4.3.0", "Unexpected Spark version")\n        receipt = locate_producer(input_root, parquet["independent_mount"]["summary_sha256"])\n        frame, summary["input"] = verified_input(spark, receipt, parquet, policy)\n        names = registry(design, parquet["feature_columns"])\n        summary["predictors"] = names\n        summary["resources"] = resource_limits(spark, config, output)\n        prepared = predictor_input(frame, names, design["version"])\n        if population in ("B", "C"):\n            active, summary["split"] = benchmark_population(prepared, population, design, summary["input"]["days"], phase4, output)\n            saved, km, summary["baseline_models"] = load_models(\n                spark, input_root, population, names, phase4["protocols"][population], phase5["experiments"][population])\n            atomic_json(receipt_path, summary)\n            if mode == "baseline":\n                result = reconcile(project, population, active, saved, km, phase4["protocols"][population], design, output)\n                summary["baseline_checks"] = len(result)\n            else:\n                from src.ml.hybrid_detection import benchmark\n                benchmark(project, output, spark, active, saved, km, population, names, config, summary)\n        else:\n            from src.ml.hybrid_detection import rolling\n            rolling(project, output, spark, prepared, population, names, config, summary)\n        summary.update(status=summary.get("outcome", "validated"), elapsed_seconds=round(monotonic() - start, 3),\n                       driver_peak_rss_bytes=driver_peak_bytes(spark))\n        atomic_json(receipt_path, summary)\n        print(f"PHASE8 complete {mode} {population} status {summary[\'status\']}", flush=True)\n    finally:\n        spark.stop()\n', 'src/ml/hybrid_detection.py': '"""Fit and freeze study decisions before scoring each Phase 8 evaluation cohort."""\n\nfrom collections import defaultdict\nfrom hashlib import sha256\nfrom time import monotonic\n\nfrom pyspark import StorageLevel\nfrom pyspark.ml import PipelineModel\nfrom pyspark.ml.clustering import KMeans\nfrom pyspark.ml.functions import vector_to_array\nfrom pyspark.sql import functions as F\n\nfrom src.ingest.dataset_audit import atomic_json\nfrom src.ml.clustering_features import fit_transform\nfrom src.ml.cost_sensitive_training import fit_candidate, preprocess, weights, with_preprocessor\nfrom src.ml.model_registry import capture\nfrom src.ml.novelty_normalization import calibrated_column, percentile_column, reference_grid\nfrom src.ml.operational_metrics import burden\nfrom src.ml.phase8_baselines import json_file, retain\nfrom src.ml.phase8_evaluation import bucket, choose_method, eligible, histograms, supports\nfrom src.ml.probability_calibration import isotonic, platt, reliability\nfrom src.ml.risk_triage import summary as triage_summary\nfrom src.ml.score_fusion import learned\nfrom src.ml.split_protocols import crossing_groups, group_sample, group_summary, label_conflicts\nfrom src.ml.threshold_calibration import at_threshold, curves, thresholds, threshold_value\nfrom src.sql.traffic_analytics import compact, require, write_rows\nfrom src.streaming.cluster_scoring import cluster_scores\nfrom src.streaming.model_scoring import probe\n\n\ndef disk(frame):\n    return frame.persist(StorageLevel.DISK_ONLY)\n\n\ndef chronology(plan):\n    roles = ("train", "probability", "operating", "test")\n    periods = [plan[name] for name in roles]\n    return all(periods) and all(max(a) < min(b) for a, b in zip(periods, periods[1:]))\n\n\ndef overlap(frame, key):\n    return group_summary(crossing_groups(frame, key))\n\n\ndef split_validation(frame, rolling, seed):\n    salt = F.pmod(F.xxhash64("predictor_key", F.lit(seed), F.lit("phase8-validation-roles")), F.lit(10000))\n    rank = F.col("split_rank")\n    middle = (F.when(salt < 6500, "probability").otherwise("meta") if rolling else\n              F.when(salt < 3500, "probability").when(salt < 5500, "meta").otherwise("operating"))\n    return frame.withColumn("role", F.when(rank == 0, "train").when(rank == 1, middle)\n                            .when(rank == 2, "operating" if rolling else "test").otherwise("test"))\n\n\ndef rolling(project, output, spark, prepared, population, names, config, summary):\n    plan = config["rolling"][population]\n    require(chronology(plan), "Rolling origin is out of order")\n    pairs = [value for rank, role in enumerate(("train", "probability", "operating", "test"))\n             for day in plan[role] for value in (F.lit(day), F.lit(rank))]\n    lookup = F.create_map(*pairs)\n    assigned = prepared.withColumn("split_rank", lookup[F.col("source_day")]).where("split_rank IS NOT NULL")\n    active, crossing = retain(assigned)\n    summary["split"] = {"dates": plan, "prepurge_predictor_groups": crossing, "purged_rows": assigned.count()-active.count()}\n    frame = disk(split_validation(active, True, config["seed"]))\n    try:\n        study(output, spark, frame, {}, None, population, names, config, summary)\n    finally:\n        active.unpersist()\n        frame.unpersist()\n\n\ndef benchmark(project, output, spark, active, saved, km, population, names, config, summary):\n    frame = disk(split_validation(active, False, config["seed"]))\n    try:\n        study(output, spark, frame, saved, km, population, names, config, summary)\n    finally:\n        active.unpersist()\n        frame.unpersist()\n\n\ndef model_score(frame, model, name):\n    return model.transform(frame).select(*frame.columns, vector_to_array("probability")[1].alias(name))\n\n\ndef sample_reference(train, config):\n    code = F.pmod(F.xxhash64("predictor_key", F.lit(config["seed"]), F.lit("phase8-novelty-reference")), F.lit(10000))\n    return train.where(code < config["reference_sample_permyriad"])\n\n\ndef fit_cluster(sample, names, config):\n    started = monotonic()\n    prep, details = fit_transform(sample.select(*names), names)\n    model = KMeans(k=12, maxIter=12, seed=config["seed"], featuresCol="features", predictionCol="cluster").fit(\n        prep.transform(sample.select(*names)).select("features"))\n    return prep, model, details, monotonic()-started\n\n\ndef train_candidates(sample, operating, prep, train_support, config, population, output):\n    bins = config["score_bins"]\n    prepared = disk(prep.transform(sample).select("binary_label", "scaled_features", "features"))\n    scoring = disk(prep.transform(operating).select("binary_label", "scaled_features", "features"))\n    fit_rows = prepared.count()\n    all_results, selected = [], {}\n    grid = weights(train_support, config["positive_weights"])\n    try:\n        for algorithm in ("lr", "rf", "gbt"):\n            candidates, key_metrics = {}, {}\n            candidates_for_algorithm = grid if algorithm != "gbt" else {k: grid[k] for k in ("1", "5")}\n            for key, weight in candidates_for_algorithm.items():\n                name = f"{algorithm}_w{key}"\n                estimator, timing = fit_candidate(prepared, algorithm, weight, config["seed"])\n                model = with_preprocessor(prep, estimator)\n                scoring_started = monotonic()\n                hs, _, _ = histograms(model_score(scoring, estimator, name), [name], bins)\n                index, _ = thresholds(hs[name], config)[f"budget_{config[\'primary_budget\']:g}"]\n                result = at_threshold(hs[name], index, bins)\n                item = {"population": population, "model": name, "weight": weight, "train_rows": fit_rows,\n                        "threshold": threshold_value(index, bins), "scoring_seconds": monotonic()-scoring_started,\n                        **timing, **result}\n                all_results.append(item)\n                candidates[name] = model\n                key_metrics[name] = (result, timing["fit_seconds"], 1 if key == "1" else 2)\n                write_rows(output / "phase8_cost_sensitive_results.csv", all_results)\n                print(f"PHASE8 {population} fitted {name} on {fit_rows}", flush=True)\n            chosen = choose_method(key_metrics)\n            unweighted = f"{algorithm}_w1"\n            weighted_best = choose_method({key: row for key, row in key_metrics.items() if key != unweighted})\n            for name in {chosen, unweighted, weighted_best}:\n                selected[name] = candidates[name]\n    finally:\n        prepared.unpersist()\n        scoring.unpersist()\n    return selected, all_results\n\n\ndef encode_columns(frame, models, km, grid, base, calibration, bins):\n    for key, model in models.items():\n        frame = model_score(frame, model, key)\n    frame = cluster_scores(frame, *km).withColumn("novelty", percentile_column(F.col("distance"), grid))\n    frame = frame.withColumn("supervised", calibrated_column(F.col(base), calibration["platt"], bins) if calibration else F.col(base))\n    if calibration:\n        frame = frame.withColumn("platt", F.col("supervised"))\n        if "isotonic" in calibration:\n            frame = frame.withColumn("isotonic", calibrated_column(F.col(base), calibration["isotonic"], bins))\n    return frame\n\n\ndef blend_name(alpha):\n    return f"blend_{alpha:g}".replace(".", "_")\n\n\ndef fusion_columns(frame, config, gates, meta):\n    s, n = F.col("supervised"), F.col("novelty")\n    first = F.col("supervised") >= gates["supervised"]\n    second = F.col("novelty") >= gates["novelty"]\n    frame = frame.withColumn("or", (first | second).cast("double")).withColumn("and", (first & second).cast("double"))\n    for alpha in config["fusion_alphas"]:\n        frame = frame.withColumn(blend_name(alpha), alpha * s + (1-alpha) * n)\n    if meta:\n        a, b = meta["coefficients"]\n        z = meta["intercept"] + a*s + b*n\n        frame = frame.withColumn("learned", 1 / (1 + F.exp(-F.greatest(F.lit(-35.0), F.least(F.lit(35.0), z)))))\n    return frame\n\n\ndef inference(frame, model, km, config, frozen):\n    """Return Phase 8 risk evidence from fitted artifacts and a frozen receipt."""\n    base = frozen["base"]\n    winner = frozen["selected_hybrid"]\n    bins = frozen["bins"]\n    scored = fusion_columns(encode_columns(frame, {base: model}, km, frozen["reference_grid"], base,\n                                           frozen["calibration"], bins), config, frozen["gates"], frozen["learned"])\n    high = frozen["operating_bins"][winner]\n    lower = frozen["lower_bins"][winner]\n    vote = bucket(F.col(winner), bins)\n    present = F.greatest(*[F.col(name).isNotNull().cast("int") for name in frozen["feature_order"]])\n    decision = F.when((present == 0) | (vote < 0), "ABSTAIN").when(vote >= high, "HIGH_RISK") \\\n        .when(vote >= lower, "REVIEW_REQUIRED").otherwise("LOW_RISK")\n    return scored.withColumn("risk_category", decision).select(\n        F.lit(frozen["protocol"] + "-" + frozen["population"]).alias("model_version"),\n        F.col(base).alias("supervised_score"), F.col("supervised").alias("calibrated_probability"),\n        F.col("cluster").alias("cluster_id"), F.col("distance").alias("centroid_distance_squared"),\n        F.col("novelty").alias("novelty_percentile"), F.col(winner).alias("hybrid_score"),\n        F.lit(threshold_value(high, bins)).alias("decision_threshold"), "risk_category",\n        (F.col("risk_category") == "REVIEW_REQUIRED").alias("review_required"),\n        F.when(F.col("risk_category") == "ABSTAIN", "unsupported_input").alias("abstention_reason"))\n\n\ndef record(output, name, rows):\n    if rows:\n        keys = dict.fromkeys(key for row in rows for key in row)\n        write_rows(output / f"phase8_{name}.csv", [{key: row.get(key) for key in keys} for row in rows])\n\n\ndef study(output, spark, frame, previous, km, population, names, config, summary):\n    start = monotonic()\n    bins = config["score_bins"]\n    support = supports(frame)\n    total = sum(sum(group.values()) for group in support.values())\n    summary["roles"] = support\n    counts = compact(frame.groupBy("source_day", "role", "binary_label", "attack_family").count()\n                     .orderBy("source_day", "role", "binary_label", "attack_family"), 2000)\n    record(output, "split_diagnostics", [{"population": population, **row} for row in counts])\n    minimum = config["minimum_per_class"]\n    unsupported = [role for role in ("train", "probability", "operating", "test") if not eligible(support.get(role, {}), minimum)]\n    if unsupported:\n        summary.update(outcome="unsupported", unsupported_roles=unsupported)\n        return\n    # Predictor keys are deterministically derived from cleaned canonical inputs.\n    roles = {"train": 0, "probability": 1, "meta": 2, "operating": 3, "test": 4}\n    lookup = F.create_map(*[x for k, i in roles.items() for x in (F.lit(k), F.lit(i))])\n    keys = frame.select(lookup[F.col("role")].alias("split_rank"), "predictor_key", "record_fingerprint", "split_feature_key")\n    summary["postpurge_overlap"] = {k: overlap(keys, k) for k in ("predictor_key", "record_fingerprint", "split_feature_key")}\n    require(not any(v["groups"] for v in summary["postpurge_overlap"].values()), "Retained roles overlap on source keys")\n    summary["conflicting_predictor_groups"] = group_summary(label_conflicts(frame))\n    train = frame.where("role = \'train\'")\n    sample = disk(group_sample(train, config).where("sampled"))\n    operating = frame.where("role = \'operating\'")\n    prep, prep_details = preprocess(sample, names)\n    summary["preprocessing"] = prep_details\n    chosen, cost = train_candidates(sample, operating, prep, support["train"], config, population, output)\n    variants = previous | chosen\n    train_sample_rows = sample.count()\n    # Diagnostic only: hold the benchmark populations fixed after imputation.\n    transformed = prep.transform(frame).select(lookup[F.col("role")].alias("split_rank"),\n            F.sha2(F.to_json(vector_to_array("features")), 256).alias("imputed_key"))\n    summary["imputed_overlap"] = overlap(transformed, "imputed_key")\n    reference = disk(sample_reference(train, config))\n    new_cluster = None\n    if km is None:\n        transform, centers, details, fit_time = fit_cluster(sample, names, config)\n        new_cluster = PipelineModel(stages=[*transform.stages, centers])\n        km = (transform, centers, 0.0)\n        summary["rolling_cluster"] = {"fit_seconds": fit_time, "preprocessing": details}\n    ref_scores = disk(cluster_scores(reference, *km).select("distance"))\n    grid = reference_grid(ref_scores)\n    bounds = dict(zip(("0.95", "0.99", "0.999"), ref_scores.approxQuantile("distance", [.95, .99, .999], .0001)))\n    summary["reference"] = {"rows": ref_scores.count(), "grid": grid, "thresholds": bounds, "original_p99": km[2] if previous else None}\n    ref_scores.unpersist()\n    reference.unpersist()\n    sample.unpersist()\n    primary = f"budget_{config[\'primary_budget\']:g}"\n    review = f"budget_{config[\'review_budget\']:g}"\n    # Pick the base classifier by operating data prior to probability calibration.\n    op_raw = disk(encode_columns(operating, variants, km, grid, next(iter(variants)), {}, bins))\n    op_base, _, _ = histograms(op_raw, list(variants), bins)\n    fit_cost = {r["model"]: r["fit_seconds"] for r in cost}\n    base = choose_method({name: (thresholds(hist, config)[primary][1], fit_cost.get(name, 0), 1)\n                          for name, hist in op_base.items()})\n    calibration_started = monotonic()\n    prob_frame = frame.where("role = \'probability\'")\n    prob_hist, _, _ = histograms(model_score(prob_frame, variants[base], base), [base], bins)\n    calibration = {"platt": platt(prob_hist[base], bins)}\n    if len(prob_hist[base]) >= 3:\n        calibration["isotonic"] = isotonic(prob_hist[base])\n    op = op_raw.withColumn("supervised", calibrated_column(F.col(base), calibration["platt"], bins))\n    for name, spec in calibration.items():\n        op = op.withColumn(name, calibrated_column(F.col(base), spec, bins))\n    gate_hist, _, _ = histograms(op, ["supervised", "novelty"], bins)\n    gates = {name: threshold_value(thresholds(hist, config)[primary][0], bins) for name, hist in gate_hist.items()}\n    calibrated_seconds = monotonic()-calibration_started\n    meta_started = monotonic()\n    meta = None\n    if eligible(support.get("meta", {}), minimum):\n        fitted_meta = encode_columns(frame.where("role = \'meta\'"), {base: variants[base]}, km, grid, base, calibration, bins)\n        meta = learned(fitted_meta.select("role", "supervised", "novelty", "binary_label"))\n    fusion_seconds = monotonic()-meta_started\n    op = fusion_columns(op, config, gates, meta)\n    methods = [*variants, *calibration, "supervised", "novelty", "or", "and",\n               *[blend_name(alpha) for alpha in config["fusion_alphas"]], *(["learned"] if meta else [])]\n    op_hist, _, _ = histograms(op, methods, bins)\n    selection = {method: thresholds(hist, config) for method, hist in op_hist.items()}\n    for name in ("and", "or"):\n        literal = at_threshold(op_hist[name], bins // 2, bins)\n        selection[name]["literal"] = (bins // 2, literal)\n    valid_hybrid = {name: (selection[name]["literal" if name in ("and", "or") else primary][1],\n                           int(name == "learned"), 2)\n                    for name in methods if (name in ("and", "or", "learned") or name.startswith("blend_"))\n                    and selection[name]["literal" if name in ("and", "or") else primary][1]["fpr"] <= config["primary_budget"]}\n    winner = choose_method(valid_hybrid)\n    selected_threshold = lambda name: selection[name]["literal" if name in ("and", "or") else primary][0]\n    threshold_records = [{"population": population, "method": name, "policy": rule, "bin": value,\n                          "threshold": threshold_value(value, bins), **row}\n                         for name, rules in selection.items() for rule, (value, row) in rules.items()]\n    record(output, "threshold_selection", threshold_records)\n    record(output, "fusion_selection", [{"population": population, "method": name, "validation_eligible": name in valid_hybrid,\n                                        "selected": name == winner, "validation_recall": selection[name]["literal" if name in ("and", "or") else primary][1]["recall"],\n                                        "validation_fpr": selection[name]["literal" if name in ("and", "or") else primary][1]["fpr"]}\n                                      for name in methods if name in ("supervised", "novelty", "and", "or", "learned") or name.startswith("blend_")])\n    freeze = {"population": population, "protocol": config["version"], "base": base, "selected_hybrid": winner,\n              "calibration": calibration, "gates": gates, "learned": meta, "feature_order": names,\n              "bins": bins, "reference_grid": grid,\n              "operating_bins": {name: selected_threshold(name) for name in methods},\n              "lower_bins": {name: min(selected_threshold(name), selection[name][review][0]) for name in methods}}\n    atomic_json(output / "phase8_frozen.json", freeze)\n    summary["frozen_sha256"] = sha256((output / "phase8_frozen.json").read_bytes()).hexdigest()\n    print(f"PHASE8 {population} frozen {base} and {winner} before evaluation", flush=True)\n    op_raw.unpersist()\n    test = frame.where("role = \'test\'")\n    scoring_started = monotonic()\n    scored = disk(fusion_columns(encode_columns(test, variants, km, grid, base, calibration, bins), config, gates, meta)\n                  .select(*methods, "source_day", "attack_family", "binary_label", "distance", "cluster",\n                          F.greatest(*[F.col(name).isNotNull().cast("int") for name in names]).alias("observed_predictors")))\n    hist, slices, inference_seconds = histograms(scored, methods, bins, sliced=True)\n    require(all(sum(sum(x) for x in h.values()) == sum(support["test"].values()) for h in hist.values()), "Test accounting differs")\n    comparison, budget_rows, families, robustness, curve_rows = [], [], [], [], []\n    family_catalog = ("DDoS", "DoS", "Credential brute force", "Bot", "Infiltration", "Web attack")\n    for name in methods:\n        outcome = at_threshold(hist[name], selected_threshold(name), bins)\n        comparison.append({"population": population, "method": name, "selection": "literal" if name in ("or", "and") else primary,\n                           "selected_hybrid": name == winner, "threshold": threshold_value(selected_threshold(name), bins),\n                           "evaluation_coverage": outcome["rows"] / total, **outcome, **burden(outcome, config["hypothetical_prevalences"])})\n        for rule, (index, val) in selection[name].items():\n            result = at_threshold(hist[name], index, bins)\n            budget_rows.append({"population": population, "method": name, "policy": rule, "threshold": threshold_value(index, bins),\n                                "validation_fpr": val["fpr"], "validation_recall": val["recall"], **result})\n        for partition, distributions in (("operating", op_hist), ("test", hist)):\n            points = curves(distributions[name], bins)\n            for i in range(0, bins+1, max(1, bins//128)):\n                row = points[i]\n                curve_rows.append({"population": population, "partition": partition, "method": name,\n                                   "threshold": threshold_value(i, bins),\n                                   **{key: row[key] for key in ("fpr", "recall", "precision")}})\n        for (method, kind, category), histogram in slices.items():\n            if method != name:\n                continue\n            row = at_threshold(histogram, selected_threshold(name), bins)\n            if kind == "source_day":\n                robustness.append({"population": population, "method": name, "capture_day": category, **row})\n            elif category != "Benign":\n                families.append({"population": population, "method": name, "family": category, "support": row["tp"] + row["fn"],\n                                 "seen_in_training": any(c["role"] == "train" and c["attack_family"] == category for c in counts),\n                                 "recall": row["recall"], "detected": row["tp"]})\n        present_families = {row["family"] for row in families if row["method"] == name}\n        for family in family_catalog:\n            if family not in present_families:\n                families.append({"population": population, "method": name, "family": family, "support": 0,\n                                 "seen_in_training": any(c["role"] == "train" and c["attack_family"] == family for c in counts),\n                                 "recall": None, "detected": 0})\n    record(output, "model_comparison", comparison)\n    record(output, "false_positive_budgets", budget_rows)\n    record(output, "robustness", robustness)\n    record(output, "score_curves", curve_rows)\n    distances = []\n    for label, cutoff in bounds.items():\n        value = F.col("distance") > cutoff\n        for row in compact(scored.groupBy("binary_label").agg(F.count("*").alias("rows"),\n                                               F.count(F.when(value, 1)).alias("flags")), 2):\n            distances.append({"population": population, "percentile": label, "cutoff": cutoff, **row})\n    record(output, "reference_novelty_flags", distances)\n    paired = compact(scored.select("source_day", "attack_family", "binary_label",\n                                  (bucket(F.col("supervised"), bins) >= selected_threshold("supervised")).alias("supervised_positive"),\n                                  (bucket(F.col(winner), bins) >= selected_threshold(winner)).alias("hybrid_positive"))\n                     .groupBy("source_day", "attack_family", "binary_label", "supervised_positive", "hybrid_positive").count(), 400)\n    record(output, "paired_disagreement", [{"population": population, **row} for row in paired])\n    specs = {"raw": base, **{name: name for name in calibration}}\n    cal_rows, cal_bins = [], []\n    for method, col in specs.items():\n        for partition, distributions in (("operating", op_hist), ("test", hist)):\n            stats, rel = reliability(distributions[col], bins, config["reliability_bins"], config["probability_epsilon"])\n            cal_rows.append({"population": population, "partition": partition, "method": method, **stats})\n            for i, (n, y, q) in enumerate(rel):\n                cal_bins.append({"population": population, "partition": partition, "method": method, "bin": i,\n                                 "rows": n, "attack_rate": y/n if n else None, "mean_probability": q/n if n else None})\n    record(output, "calibration_metrics", cal_rows)\n    record(output, "reliability", cal_bins)\n    triage = []\n    vote = bucket(F.col(winner), bins)\n    for rule in ("primary", *[f"budget_{p:g}" for p in config["budgets"]]):\n        high = selected_threshold(winner)\n        lower = freeze["lower_bins"][winner] if rule == "primary" else min(high, selection[winner][rule][0])\n        category = F.when(F.col("observed_predictors") == 0, "ABSTAIN").when(vote >= high, "HIGH_RISK") \\\n            .when(vote >= lower, "REVIEW_REQUIRED").otherwise("LOW_RISK")\n        result = defaultdict(lambda: [0, 0])\n        for row in compact(scored.select("binary_label", category.alias("risk")).groupBy("binary_label", "risk").count(), 12):\n            result[row["risk"]][row["binary_label"]] = row["count"]\n        triage.append({"population": population, "method": winner, "policy": rule,\n                       "high": threshold_value(high, bins), "lower": threshold_value(lower, bins), **triage_summary(result)})\n        if rule == "primary":\n            review_counts = {row["attack_family"]: row["count"] for row in compact(scored.where("binary_label = 1")\n                             .withColumn("risk", category).where("risk IN (\'REVIEW_REQUIRED\',\'ABSTAIN\')")\n                             .groupBy("attack_family").count(), 20)}\n    for row in families:\n        row["hybrid_referred"] = review_counts.get(row["family"], 0) if row["method"] == winner else None\n    record(output, "family_recall", families)\n    record(output, "triage_metrics", triage)\n    best_lr = choose_method({name: (selection[name][primary][1], 0, 1) for name in variants\n                             if name.startswith("lr_") or name == "logistic_regression"})\n    weighted_names = {name: (selection[name][primary][1], fit_cost.get(name, 0), 1) for name in chosen\n                      if not name.endswith("_w1")}\n    best_weight = choose_method(weighted_names)\n    original = "random_forest" if previous else "rf_w1"\n    ablation = [("A", original, bins // 2), ("B", original, selected_threshold(original)), ("C", "platt", selected_threshold("platt")),\n                ("D", best_weight, selected_threshold(best_weight)), ("E", "novelty", selected_threshold("novelty")),\n                ("F", winner, selected_threshold(winner)), ("G", winner, selected_threshold(winner))]\n    record(output, "ablation_results", [{"population": population, "component": ab, "method": name, "threshold": threshold_value(idx, bins),\n                                       **at_threshold(hist[name], idx, bins),\n                                       "review_rate": triage[0]["review_rate"] if ab == "G" else None,\n                                       "automatic_coverage": triage[0]["automatic_coverage"] if ab == "G" else None}\n                                      for ab, name, idx in ablation])\n    # Independently time the same retained workload under base and selected hybrid scoring.\n    def measured(combined):\n        tick = monotonic()\n        if combined:\n            result = fusion_columns(encode_columns(test, {base: variants[base]}, km, grid, base, calibration, bins),\n                                    config, gates, meta)\n            column = winner\n        else:\n            result, column = model_score(test, variants[base], "base_runtime"), "base_runtime"\n        rows = result.agg(F.count(column).alias("rows"), F.sum(column).alias("check")).first()\n        require(rows["rows"] == sum(support["test"].values()), "Runtime scoring lost rows")\n        elapsed = monotonic() - tick\n        return elapsed, rows["rows"]\n    base_time, runtime_rows = measured(False)\n    hybrid_time, _ = measured(True)\n    saved_meta = {}\n    for name in sorted({base, best_weight, best_lr} & set(chosen)):\n        saved_meta[name] = capture(chosen[name], name, output, probe(spark, names))\n    if new_cluster:\n        saved_meta["kmeans"] = capture(new_cluster, "kmeans", output, probe(spark, names), kmeans=True)\n    reopened = PipelineModel.load(str(output / saved_meta[base]["artifact"])) if base in saved_meta else variants[base]\n    reopened_km = km\n    if new_cluster:\n        reopened_cluster = PipelineModel.load(str(output / saved_meta["kmeans"]["artifact"]))\n        reopened_km = (PipelineModel(stages=reopened_cluster.stages[:-1]), reopened_cluster.stages[-1], km[2])\n    probes = probe(spark, names)\n    old = inference(probes, variants[base], km, config, freeze).collect()\n    new = inference(probes, reopened, reopened_km, config, json_file(output, "phase8_frozen.json")).collect()\n    require(old == new and len(new) == 3, "Unified inference reload differs")\n    summary.update(registry=saved_meta, best_lr=best_lr, best_weighted=best_weight, selected=winner, supervised_base=base,\n                   training_sample_rows=train_sample_rows, scored_rows=sum(support["test"].values()), inference_reload_probes=len(new),\n                   preparation_seconds=monotonic()-start)\n    record(output, "performance", [{"population": population, "stage": name, "rows": n, "seconds": elapsed,\n                                    "rows_per_second": n/elapsed if n else None}\n                                   for name, n, elapsed in (("supervised_matched", runtime_rows, base_time),\n                                               ("hybrid_matched", runtime_rows, hybrid_time), ("calibration", 0, calibrated_seconds),\n                                               ("learned_fusion", 0, fusion_seconds),\n                                               ("all_candidate_scoring_and_aggregation", runtime_rows, monotonic()-scoring_started))])\n    scored.unpersist()\n    print(f"PHASE8 {population} evaluated {summary[\'scored_rows\']} and verified {len(saved_meta)} saved models", flush=True)\n', 'src/ml/threshold_calibration.py': '"""Tie-preserving histogram thresholds and validation-only selection."""\n\nfrom math import sqrt\n\nfrom src.ml.evaluate_supervised import binned_auc, confusion_metrics, rate\n\n\ndef metrics(tp, tn, fp, fn):\n    result = confusion_metrics(tp, tn, fp, fn)\n    denom = sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))\n    return result | {"fpr": rate(fp, fp + tn), "fnr": rate(fn, tp + fn),\n                     "f2": rate(5 * tp, 5 * tp + fp + 4 * fn),\n                     "mcc": rate(tp * tn - fp * fn, denom)}\n\n\ndef curves(hist, bins):\n    negative = sum(x[0] for x in hist.values())\n    positive = sum(x[1] for x in hist.values())\n    tp = fp = 0\n    result = {bins: metrics(0, negative, 0, positive)}\n    for index in reversed(range(bins)):\n        no, yes = hist.get(index, (0, 0))\n        tp += yes\n        fp += no\n        result[index] = metrics(tp, negative - fp, fp, positive - tp)\n    return result\n\n\ndef objective(row, key):\n    value = row[key]\n    return -1.0 if value is None else value\n\n\ndef choose(points, budget):\n    feasible = [(i, row) for i, row in points.items() if row["fpr"] is not None and row["fpr"] <= budget]\n    if not feasible:\n        raise ValueError("No benign support for operating threshold selection")\n    return max(feasible, key=lambda item: (objective(item[1], "recall"), objective(item[1], "precision"),\n                                           -item[1]["fpr"], item[0]))\n\n\ndef thresholds(hist, config):\n    bins = config["score_bins"]\n    series = curves(hist, bins)\n    if not any(row["recall"] is not None for row in series.values()):\n        raise ValueError("No attack support for operating threshold selection")\n    choices = {"fixed_0.5": (bins // 2, series[bins // 2])}\n    choices.update({f"budget_{budget:g}": choose(series, budget) for budget in config["budgets"]})\n    for name in ("f1", "f2", "balanced_accuracy"):\n        choices[name] = max(series.items(), key=lambda item: (objective(item[1], name), item[0]))\n    choices["cost"] = min(series.items(), key=lambda item: (\n        item[1]["fp"] + config["false_negative_cost"] * item[1]["fn"], -item[0]))\n    return choices\n\n\ndef at_threshold(hist, index, bins):\n    negatives = sum(x[0] for x in hist.values())\n    positives = sum(x[1] for x in hist.values())\n    tn_fp, tp = (sum(x[position] for key, x in hist.items() if key >= index) for position in (0, 1))\n    return metrics(tp, negatives - tn_fp, tn_fp, positives - tp) | binned_auc(hist)\n\n\ndef threshold_value(index, bins):\n    """One step above 1 represents the explicit never-positive threshold."""\n    return (index if index < bins else bins + 1) / bins\n', 'src/ml/probability_calibration.py': '"""Calibrators fitted on compact binned probability-role predictions only."""\n\nfrom bisect import bisect_left\nfrom math import exp, log, log1p\n\n\ndef sigmoid(z):\n    return 1 / (1 + exp(-max(-35, min(35, z))))\n\n\ndef center(bucket, bins):\n    return (bucket + .5) / bins\n\n\ndef platt(histogram, bins):\n    # Newton updates with ridge shrinkage and standardized score coordinates.\n    count = sum(sum(v) for v in histogram.values())\n    if not count or not all(sum(v[i] for v in histogram.values()) for i in range(2)):\n        raise ValueError("Calibration requires both classes")\n    points = [(2 * center(i, bins) - 1, negative + positive, positive)\n              for i, (negative, positive) in histogram.items()]\n    ridge = max(1e-6, count * 1e-5)\n    a = b = 0.0\n    for _ in range(40):\n        g0, g1, h00, h01, h11 = 0.0, ridge * b, 1e-8, 0.0, ridge\n        for x, n, yes in points:\n            p = sigmoid(a + b * x)\n            res = p * n - yes\n            curvature = n * p * (1 - p)\n            g0 += res\n            g1 += res * x\n            h00 += curvature\n            h01 += curvature * x\n            h11 += curvature * x * x\n        determinant = h00 * h11 - h01 * h01\n        da, db = (h11 * g0 - h01 * g1) / determinant, (h00 * g1 - h01 * g0) / determinant\n        shrink = max(1, abs(da) / 2, abs(db) / 2)\n        a, b = a - da / shrink, b - db / shrink\n        if max(abs(da), abs(db)) < 1e-8:\n            break\n    return {"method": "platt", "intercept": a - b, "slope": 2 * b, "input": "histogram_bin_midpoint"}\n\n\ndef isotonic(histogram):\n    if len(histogram) < 3 or not all(sum(v[i] for v in histogram.values()) for i in range(2)):\n        raise ValueError("Isotonic calibration requires both classes and three score bins")\n    blocks = []\n    for key, (negative, positive) in sorted(histogram.items()):\n        blocks.append([key, negative + positive, positive])\n        while len(blocks) >= 2 and blocks[-2][2] * blocks[-1][1] > blocks[-1][2] * blocks[-2][1]:\n            high, n, yes = blocks.pop()\n            blocks[-1] = [high, blocks[-1][1] + n, blocks[-1][2] + yes]\n    return {"method": "isotonic", "upper_bins": [x[0] for x in blocks],\n            "probabilities": [x[2] / x[1] for x in blocks]}\n\n\ndef probability(spec, bucket, bins):\n    if spec["method"] == "platt":\n        return sigmoid(spec["intercept"] + spec["slope"] * center(bucket, bins))\n    return spec["probabilities"][min(bisect_left(spec["upper_bins"], bucket), len(spec["upper_bins"]) - 1)]\n\n\ndef reliability(hist, bins, reliability_bins, epsilon):\n    counts = [[0, 0, 0.0] for _ in range(reliability_bins)]\n    brier = loss = total = 0\n    for bucket, (negative, positive) in hist.items():\n        score = center(bucket, bins)\n        n = negative + positive\n        total += n\n        group = counts[min(int(score * reliability_bins), reliability_bins - 1)]\n        group[0] += n\n        group[1] += positive\n        group[2] += n * score\n        brier += negative * score**2 + positive * (1-score)**2\n        p = max(epsilon, min(1-epsilon, score))\n        loss -= positive * log(p) + negative * log1p(-p)\n    ece = sum(abs(p - yes) for _, yes, p in counts)\n    return ({"rows": total, "brier": brier/total, "log_loss": loss/total, "ece": ece/total} if total else\n            {"rows": 0, "brier": None, "log_loss": None, "ece": None}), counts\n', 'src/ml/cost_sensitive_training.py': '"""Fit bounded weighted Spark classifiers on a common grouped training sample."""\n\nfrom time import monotonic\n\nfrom pyspark.ml import PipelineModel\nfrom pyspark.ml.classification import GBTClassifier, LogisticRegression, RandomForestClassifier\nfrom pyspark.sql import functions as F\n\nfrom src.ingest.parquet_pipeline import driver_peak_bytes\nfrom src.ml.preprocessing import fit_preprocessing\n\n\ndef weights(training_support, grid):\n    negative, positive = (training_support.get(i, 0) for i in (0, 1))\n    if not min(negative, positive):\n        raise ValueError("Weights require both classes in the training population")\n    return {str(key): negative/positive if key == "inverse" else float(key) for key in grid}\n\n\ndef preprocess(sample, names):\n    _, scaled, details = fit_preprocessing(sample, names)\n    return scaled, details\n\n\ndef fit_candidate(sample, algorithm, weight, seed):\n    common = {"labelCol": "binary_label", "weightCol": "class_weight"}\n    if algorithm == "lr":\n        estimator = LogisticRegression(featuresCol="scaled_features", maxIter=12, regParam=.05, standardization=False, **common)\n    elif algorithm == "rf":\n        estimator = RandomForestClassifier(featuresCol="features", seed=seed, numTrees=12, maxDepth=5,\n                                           maxBins=24, minInstancesPerNode=30, maxMemoryInMB=192, **common)\n    elif algorithm == "gbt":\n        estimator = GBTClassifier(featuresCol="features", seed=seed, maxIter=6, maxDepth=3,\n                                  maxBins=24, minInstancesPerNode=30, maxMemoryInMB=192, **common)\n    else:\n        raise ValueError(f"Unknown algorithm {algorithm}")\n    if not estimator.hasParam("weightCol"):\n        raise ValueError(f"Spark {algorithm} does not support weights")\n    weighted = sample.withColumn("class_weight", F.when(F.col("binary_label") == 1, weight).otherwise(1.0))\n    start = monotonic()\n    fitted = estimator.fit(weighted)\n    return fitted, {"fit_seconds": round(monotonic()-start, 3), "peak_jvm_rss_bytes": driver_peak_bytes(sample.sparkSession)}\n\n\ndef with_preprocessor(prep, fitted):\n    return PipelineModel(stages=[*prep.stages, fitted])\n', 'src/ml/novelty_normalization.py': '"""Reference-only quantile normalization of squared centroid distance."""\n\nfrom bisect import bisect_left\n\n\ndef percentile(grid, distance):\n    if not grid or distance is None:\n        return None\n    return bisect_left(grid, distance) / len(grid)\n\n\ndef reference_grid(frame):\n    grid = frame.approxQuantile("distance", [i / 256 for i in range(257)], 0.001)\n    if len(grid) != 257:\n        raise ValueError("No defined reference distance sample")\n    return grid\n\n\ndef step_expression(value, bounds, outputs, tail):\n    """Construct a balanced binary decision tree on numeric Spark columns."""\n    from pyspark.sql import functions as F\n\n    if not bounds:\n        return F.lit(float(tail))\n    i = len(bounds) // 2\n    return F.when(value <= bounds[i], step_expression(value, bounds[:i], outputs[:i], outputs[i])).otherwise(\n        step_expression(value, bounds[i+1:], outputs[i+1:], tail))\n\n\ndef percentile_column(column, grid):\n    from pyspark.sql import functions as F\n\n    unique = sorted(set(grid))\n    return F.when(column.isNotNull(),\n                  step_expression(column, unique, [percentile(grid, x) for x in unique], 1.0))\n\n\ndef calibrated_column(column, spec, bins):\n    from pyspark.sql import functions as F\n\n    bucket = F.least(F.floor(column * bins), F.lit(bins - 1))\n    if spec["method"] == "platt":\n        raw = F.lit(spec["intercept"]) + F.lit(spec["slope"]) * (bucket + .5) / bins\n        return 1 / (1 + F.exp(-F.greatest(F.lit(-35.0), F.least(F.lit(35.0), raw))))\n    return step_expression(bucket, spec["upper_bins"], spec["probabilities"], spec["probabilities"][-1])\n', 'src/ml/score_fusion.py': '"""Transparent supervised and novelty fusion over bounded scalar scores."""\n\nfrom src.ml.probability_calibration import sigmoid\n\n\ndef weighted(supervised, novelty, alpha):\n    if not 0 <= alpha <= 1:\n        raise ValueError("Fusion weight is outside [0,1]")\n    return alpha * supervised + (1 - alpha) * novelty\n\n\ndef learned(frame):\n    from pyspark.ml.classification import LogisticRegression\n    from pyspark.ml.feature import VectorAssembler\n    from pyspark.sql import functions as F\n\n    if frame.where(F.col("role").isNull() | (F.col("role") != "meta")).limit(1).count():\n        raise ValueError("Learned fusion requires a disjoint meta-training role")\n    vector = VectorAssembler(inputCols=["supervised", "novelty"], outputCol="meta_features").transform(frame)\n    model = LogisticRegression(featuresCol="meta_features", labelCol="binary_label", maxIter=16, regParam=.05).fit(vector)\n    return {"intercept": model.intercept, "coefficients": [float(x) for x in model.coefficients],\n            "features": ["supervised", "novelty"], "reg_param": .05}\n\n\ndef learned_score(spec, supervised, novelty):\n    return sigmoid(spec["intercept"] + spec["coefficients"][0] * supervised + spec["coefficients"][1] * novelty)\n', 'src/ml/risk_triage.py': '"""High and lower frozen thresholds for selective automatic decisions."""\n\nfrom math import isfinite\n\nfrom src.ml.evaluate_supervised import rate\n\n\ndef risk_category(score, high, lower):\n    if score is None or not isfinite(score):\n        return "ABSTAIN"\n    if score >= high:\n        return "HIGH_RISK"\n    if score >= lower:\n        return "REVIEW_REQUIRED"\n    return "LOW_RISK"\n\n\ndef summary(counts):\n    high, low, review, abstain = [counts.get(k, (0, 0)) for k in\n                                 ("HIGH_RISK", "LOW_RISK", "REVIEW_REQUIRED", "ABSTAIN")]\n    auto = sum(high) + sum(low)\n    total = auto + sum(review) + sum(abstain)\n    attacks = sum(x[1] for x in (high, low, review, abstain))\n    negatives = total - attacks\n    return {"rows": total, "automatic_coverage": rate(auto, total), "review_rate": rate(sum(review), total),\n            "abstain_rate": rate(sum(abstain), total), "automatic_error_rate": rate(high[0] + low[1], auto),\n            "high_precision": rate(high[1], sum(high)), "detected_attack_rate": rate(high[1], attacks),\n            "referred_or_detected_attack_rate": rate(high[1] + review[1] + abstain[1], attacks),\n            "benign_review_burden": rate(review[0], negatives),\n            "review_per_100000": 100_000 * sum(review)/total if total else None,\n            "category_counts": counts}\n', 'src/ml/phase8_evaluation.py': '"""Bounded Spark aggregation for candidate ranking and matched evaluation."""\n\nfrom collections import defaultdict\nfrom time import monotonic\n\nfrom pyspark.sql import functions as F\n\nfrom src.sql.traffic_analytics import compact, require\n\n\ndef bucket(column, bins):\n    return F.when(column.between(0.0, 1.0) & ~F.isnan(column),\n                  F.least(F.floor(column * bins).cast("int"), F.lit(bins - 1))).otherwise(-1)\n\n\ndef histograms(frame, names, bins, sliced=False):\n    started = monotonic()\n    keys = ["source_day", "attack_family"] if sliced else []\n    entries = [F.struct(F.lit(name).alias("method"), bucket(F.col(name), bins).alias("bucket")) for name in names]\n    data = frame.select("binary_label", *keys, F.explode(F.array(*entries)).alias("score"))\n    rows = compact(data.groupBy("binary_label", *keys, "score.method", "score.bucket").count(), 300000)\n    full = {name: defaultdict(lambda: [0, 0]) for name in names}\n    parts = defaultdict(lambda: defaultdict(lambda: [0, 0]))\n    for row in rows:\n        require(row["binary_label"] in (0, 1) and row["bucket"] >= 0, "Invalid probability, distance score or label")\n        method, index, label = row["method"], row["bucket"], row["binary_label"]\n        full[method][index][label] += row["count"]\n        for key in keys:\n            parts[method, key, row[key]][index][label] += row["count"]\n    return full, parts, monotonic()-started\n\n\ndef supports(frame):\n    output = defaultdict(dict)\n    for row in compact(frame.groupBy("role", "binary_label").count(), 40):\n        output[row["role"]][row["binary_label"]] = row["count"]\n    return dict(output)\n\n\ndef eligible(support, minimum):\n    return min(support.get(k, 0) for k in (0, 1)) >= minimum\n\n\ndef choose_method(candidates):\n    def key(item):\n        name, (row, cost, complexity) = item\n        return (row["recall"], row["precision"] if row["precision"] is not None else -1,\n                -row["fpr"], -cost, -complexity, name)\n    return max(candidates.items(), key=key)[0]\n', 'src/ml/operational_metrics.py': '"""Translate bounded evaluation rates into workload and hypothetical PPV."""\n\n\ndef scenario_ppv(tpr, fpr, prevalence):\n    if tpr is None or fpr is None:\n        return None\n    yes, false = tpr * prevalence, fpr * (1-prevalence)\n    return yes/(yes + false) if yes + false else None\n\n\ndef burden(row, prevalences):\n    fpr = row["fpr"]\n    return {f"false_per_{n}": fpr*n if fpr is not None else None for n in (10_000, 100_000, 1_000_000)} | {\n        "detected_per_100000": row["tp"] / row["rows"] * 100_000 if row["rows"] else None,\n        "scenario_ppv": {str(p): scenario_ppv(row["recall"], fpr, p) for p in prevalences},\n    }\n', 'src/ml/model_registry.py': '"""Save selected Spark pipelines and verify same-probe scoring and file hashes."""\n\nfrom hashlib import sha256\nfrom pathlib import Path\n\nfrom pyspark.ml import PipelineModel\nfrom pyspark.ml.functions import vector_to_array\n\n\ndef capture(model, name, output, probes, kmeans=False):\n    target = output / "models" / name\n    model.write().save(str(target))\n    opened = PipelineModel.load(str(target))\n\n    def score(pipeline):\n        scored = pipeline.transform(probes)\n        if kmeans:\n            return [(r[0], tuple(r[1])) for r in scored.select("cluster", "features").collect()]\n        return [tuple(r[0]) for r in scored.select(vector_to_array("probability")).collect()]\n\n    before, after = score(model), score(opened)\n    if not before or before != after:\n        raise ValueError(f"Reloaded model {name} scored differently")\n    payloads = [p for p in Path(target).rglob("*") if p.is_file() and not p.name.endswith(".crc")]\n    return {"artifact": target.relative_to(output).as_posix(), "reload_probe_rows": len(after),\n            "preprocessing_stages": [type(stage).__name__ for stage in model.stages[:-1]],\n            "bytes": sum(p.stat().st_size for p in payloads),\n            "sha256": {p.relative_to(target).as_posix(): sha256(p.read_bytes()).hexdigest() for p in sorted(payloads)}}\n'}
for relative, source in sources.items():
    target = project / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")
sys.path.insert(0, str(project))


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'river==0.22.0'], check=True)
sources={'src/drift/run_synthetic.py': '"""Calibrate on independent stationarity and evaluate bounded synthetic shifts."""\nfrom bisect import bisect_left\nimport csv,json,pickle\nfrom pathlib import Path\nfrom time import perf_counter\nfrom src.drift.advanced_detectors import create,update,divergences\nfrom src.drift.synthetic_generators import generate,SCENARIOS\nfrom src.drift.drift_evaluation import evaluate\nfrom src.drift.label_availability import LabelSchedule\n\n\ndef write(path,rows):\n    keys=list(dict.fromkeys(k for row in rows for k in row))\n    with path.open(\'w\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=keys,lineterminator=\'\\n\');w.writeheader();w.writerows(rows)\n\n\ndef run(output):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    start=perf_counter();calibration=[];frozen={};bins=64\n    for name in (\'ADWIN\',\'PageHinkley\',\'KSWIN\',\'DDM\'):\n        options=[]\n        for sensitivity in (\'conservative\',\'sensitive\') if name!=\'DDM\' else (\'conservative\',):\n            total=0;seconds=0.\n            for seed in (13013,13014,13015):\n                detector=create(name,sensitivity,seed);tick=perf_counter();alerts=[]\n                for row in generate(\'stationary\',seed):\n                    value=row[\'error\'] if name==\'DDM\' else row[\'signal\']\n                    if update(detector,value,name==\'DDM\'):alerts.append(row[\'index\'])\n                elapsed=perf_counter()-tick;seconds+=elapsed\n                stats=evaluate(alerts,[],4096);total+=stats[\'false_alarms\']\n                calibration.append({\'detector\':name,\'sensitivity\':sensitivity,\'seed\':seed,\'seconds\':elapsed,**stats})\n            rate=total/(3*(4096-256))*1000\n            options.append((rate>1,rate,seconds,sensitivity))\n        frozen[name]=min(options)[-1]\n    # Frozen-reference bins and distribution thresholds use independent calibration.\n    reference=[r[\'signal\'] for r in list(generate(\'stationary\',13013))[:256]]\n    ordered=sorted(reference);edges=[ordered[int(i*len(ordered)/10)] for i in range(1,10)]\n    def histogram(values):\n        counts=[0]*10\n        for x in values:counts[bisect_left(edges,x)]+=1\n        return counts\n    reference_counts=histogram(reference);distributions=[]\n    for seed in (13013,13014,13015):\n        values=[r[\'signal\'] for r in generate(\'stationary\',seed)]\n        distributions.extend(divergences(reference_counts,histogram(values[i:i+bins])) for i in range(256,len(values),bins))\n    for index,name in enumerate((\'JS\',\'PSI\')):\n        values=sorted(row[index] for row in distributions);frozen[name]=values[int(.99*(len(values)-1))]\n    frozen.update(reference_bins=edges,reference_counts=reference_counts,window_size=bins)\n    (output/\'frozen.json\').write_text(json.dumps(frozen,indent=2)+\'\\n\')\n    print(\'PHASE11 synthetic configurations frozen\',flush=True)\n    results=[];events_table=[];signals=[]\n    for kind in SCENARIOS:\n        boundaries=[] if kind==\'stationary\' else [2048,3072] if kind==\'recurring\' else [2048]\n        for seed in (23013,23014,23015):\n            data=list(generate(kind,seed))\n            for name in (\'ADWIN\',\'PageHinkley\',\'KSWIN\',\'DDM\',\'JS\',\'PSI\',\'FixedJS\'):\n                delays=(0,1,3) if name==\'DDM\' else (0,)\n                for delay in delays:\n                    tick=perf_counter();alerts=[];detector=create(name,frozen[name],seed) if name in (\'ADWIN\',\'PageHinkley\',\'KSWIN\',\'DDM\') else None\n                    schedule=LabelSchedule(delay)\n                    for row in data:\n                        i=row[\'index\']\n                        if name==\'DDM\':\n                            schedule.predict(i//bins,i,row[\'error\'])\n                            for identifier,error,released in schedule.release(i//bins):\n                                assert released<=i//bins\n                                if update(detector,error,True):alerts.append(i)\n                        elif detector:\n                            if update(detector,row[\'signal\']):alerts.append(i)\n                        elif i>=256+bins-1 and (i+1)%bins==0:\n                            js,psi=divergences(reference_counts,histogram([r[\'signal\'] for r in data[i+1-bins:i+1]]))\n                            if (js if name!=\'PSI\' else psi)>(.1 if name==\'FixedJS\' else frozen[name]):alerts.append(i)\n                    elapsed=perf_counter()-tick\n                    relevant=(kind in (\'error\',\'concept\')) if name==\'DDM\' else (kind not in (\'error\',\'concept\'))\n                    stats=evaluate(alerts,boundaries if relevant else [],4096)\n                    results.append({\'scenario\':kind,\'seed\':seed,\'detector\':name,\'label_delay_windows\':delay,\'seconds\':elapsed,\'updates_per_second\':4096/elapsed,\'state_bytes\':len(pickle.dumps(detector)) if detector else 0,**stats})\n                    events_table.extend({\'scenario\':kind,\'seed\':seed,\'detector\':name,\'label_delay_windows\':delay,\'alert_index\':i} for i in alerts)\n            if seed==23013:signals.extend({\'scenario\':kind,**r} for r in data[::16])\n            print(\'PHASE11 evaluated\',kind,seed,flush=True)\n    write(output/\'synthetic_metrics.csv\',results);write(output/\'alerts.csv\',events_table);write(output/\'calibration.csv\',calibration);write(output/\'signals.csv\',signals)\n    (output/\'receipt.json\').write_text(json.dumps({\'status\':\'validated\',\'stage\':\'11B\',\'rows\':len(results),\'seconds\':perf_counter()-start,\'synthetic_only\':True,\'pending_end_labels_note\':\'Labels released after the final observation remain pending and are excluded from detector updates\'},indent=2)+\'\\n\')\n    print(\'PHASE11 synthetic complete\',len(results),flush=True)\n', 'src/drift/signal_registry.py': '"""Information availability and unit declarations for Phase 11 signals."""\nSIGNALS={\n \'synthetic_signal\':{\'unit\':\'observation\',\'labels_required\':False},\n \'released_error\':{\'unit\':\'released observation\',\'labels_required\':True},\n \'rf_score_histogram\':{\'unit\':\'historical replay window\',\'labels_required\':False},\n \'feature_js\':{\'unit\':\'historical replay window\',\'labels_required\':False},\n \'historical_error_rate\':{\'unit\':\'released historical replay window\',\'labels_required\':True},\n}\n', 'src/drift/drift_evaluation.py': '"""One-alert-per-event matching with separate repeated and stationary alerts."""\n\n\ndef evaluate(alerts,events,length,warmup=256,tolerance=512):\n    matched=set();delays=[];false=repeat=0\n    eligible=[i for i in alerts if i>=warmup]\n    for index in eligible:\n        candidates=[event for event in events if event<=index<event+tolerance]\n        pending=[event for event in candidates if event not in matched]\n        if pending:\n            event=min(pending);matched.add(event);delays.append(index-event)\n        elif candidates:repeat+=1\n        else:false+=1\n    return {\'true_episodes\':len(events),\'detected_episodes\':len(matched),\'misses\':len(events)-len(matched),\n            \'recall\':len(matched)/len(events) if events else None,\'mean_delay_observations\':sum(delays)/len(delays) if delays else None,\n            \'false_alarms\':false,\'repeated_episode_alerts\':repeat,\'false_per_1000\':1000*false/(length-warmup),\n            \'alert_precision\':len(matched)/len(eligible) if eligible else None,\'alerts\':len(eligible)}\n', 'src/drift/synthetic_generators.py': '"""Bounded declared synthetic mechanisms, independent of real attack labels."""\nimport math,random\n\nSCENARIOS=(\'stationary\',\'abrupt\',\'gradual\',\'incremental\',\'recurring\',\'variance\',\'correlation\',\'prior\',\'missingness\',\'error\',\'concept\')\n\n\ndef generate(kind,seed,size=4096,magnitude=1.):\n    if kind not in SCENARIOS:raise ValueError(\'Unknown synthetic scenario\')\n    rng=random.Random(seed)\n    for i in range(size):\n        progress=max(0.,min(1.,(i-2048)/1024))\n        changed=i>=2048\n        x,y=rng.gauss(0,.5),rng.gauss(0,.5)\n        mean=0.\n        if kind==\'abrupt\':mean=magnitude if changed else 0.\n        if kind==\'gradual\':mean=magnitude if rng.random()<progress else 0.\n        if kind==\'incremental\':mean=magnitude*progress\n        if kind==\'recurring\':mean=magnitude if 2048<=i<3072 else 0.\n        if kind==\'variance\' and changed:x*=3*magnitude\n        if kind==\'correlation\' and changed:y=.9*x+.1*y\n        missing=kind==\'missingness\' and rng.random() < (.3 if changed else .01)\n        prior=int(rng.random() < (.65 if kind==\'prior\' and changed else .2))\n        truth=int(x>0)\n        if kind==\'concept\' and changed:truth=1-truth\n        error=int((int(x>0)!=truth) if kind==\'concept\' else rng.random() < (.45 if kind==\'error\' and changed else .05))\n        feature=1/(1+math.exp(-(x+mean)))\n        signal=(1/(1+math.exp(-x*y*4)) if kind==\'correlation\' else float(missing) if kind==\'missingness\' else float(prior) if kind==\'prior\' else feature)\n        yield {\'index\':i,\'signal\':signal,\'error\':error,\'missing\':missing,\'prior\':prior}\n', 'src/drift/robustness.py': '"""Prespecified distribution-window sensitivity and label-free ensemble ablation."""\nfrom bisect import bisect_left\nfrom collections import defaultdict\nimport csv,json\nfrom pathlib import Path\nfrom time import perf_counter\nfrom src.drift.advanced_detectors import divergences\nfrom src.drift.synthetic_generators import generate\nfrom src.drift.drift_evaluation import evaluate\nfrom src.drift.run_synthetic import write\n\n\ndef run(primary,output):\n    primary,output=Path(primary),Path(output);output.mkdir(parents=True,exist_ok=True)\n    frozen=json.loads((primary/\'frozen.json\').read_text());edges=frozen[\'reference_bins\'];ref=frozen[\'reference_counts\']\n    def histogram(values):\n        result=[0]*10\n        for x in values:result[bisect_left(edges,x)]+=1\n        return result\n    records=[]\n    for window in (32,64,128):\n        calibration=[]\n        for seed in (13013,13014,13015):\n            values=[r[\'signal\'] for r in generate(\'stationary\',seed)]\n            calibration.extend(divergences(ref,histogram(values[i:i+window])) for i in range(256,4096,window))\n        cutoffs=[sorted(v[i] for v in calibration)[int(.99*(len(calibration)-1))] for i in (0,1)]\n        for magnitude in (.5,1.):\n            for seed in (23013,23014,23015):\n                values=[r[\'signal\'] for r in generate(\'abrupt\',seed,magnitude=magnitude)]\n                alerts=[[],[]];tick=perf_counter()\n                for end in range(256+window,4097,window):\n                    scores=divergences(ref,histogram(values[end-window:end]))\n                    for i in (0,1):\n                        if scores[i]>cutoffs[i]:alerts[i].append(end-1)\n                elapsed=perf_counter()-tick\n                for i,name in enumerate((\'JS\',\'PSI\')):\n                    records.append({\'detector\':name,\'window\':window,\'magnitude\':magnitude,\'seed\':seed,\'threshold\':cutoffs[i],\'seconds\':elapsed,**evaluate(alerts[i],[2048],4096)})\n    write(output/\'sensitivity.csv\',records)\n    groups=defaultdict(list)\n    for row in csv.DictReader((primary/\'alerts.csv\').open()):\n        if row[\'detector\'] in (\'ADWIN\',\'PageHinkley\',\'KSWIN\',\'JS\',\'PSI\'):\n            groups[row[\'scenario\'],int(row[\'seed\'])].append((int(row[\'alert_index\']),row[\'detector\']))\n    ablations=[]\n    for kind in (\'stationary\',\'abrupt\',\'gradual\',\'incremental\',\'recurring\',\'variance\',\'correlation\',\'prior\',\'missingness\',\'error\',\'concept\'):\n        boundaries=[] if kind in (\'stationary\',\'error\',\'concept\') else [2048,3072] if kind==\'recurring\' else [2048]\n        for seed in (23013,23014,23015):\n            recent=[];alerts=[];last=-65\n            for index,name in sorted(groups[kind,seed]):\n                recent=[(i,n) for i,n in recent if i>=index-64];recent.append((index,name))\n                if len({n for _,n in recent})>=2 and index-last>=64:\n                    alerts.append(index);last=index\n            ablations.append({\'scenario\':kind,\'seed\':seed,\'detector\':\'two-family-64-observation-ensemble\',**evaluate(alerts,boundaries,4096)})\n            ablations.append({\'scenario\':kind,\'seed\':seed,\'detector\':\'no_detector\',**evaluate([],boundaries,4096)})\n    write(output/\'ablations.csv\',ablations)\n    (output/\'receipt.json\').write_text(json.dumps({\'status\':\'validated\',\'stage\':\'11D\',\'sensitivity_rows\':len(records),\'ablation_rows\':len(ablations),\'ensemble\':\'At least two distinct label-free detector families within 64 observations with 64-observation cooldown\'},indent=2)+\'\\n\')\n    print(\'PHASE11 robustness complete\',flush=True)\n', 'src/drift/historical_monitoring.py': '"""Monitor saved Spark replay aggregates with window-level information clocks."""\nimport csv,json,pickle\nfrom pathlib import Path\nfrom time import perf_counter\nfrom src.drift.advanced_detectors import create,update,divergences\nfrom src.drift.label_availability import LabelSchedule\nfrom src.drift.run_synthetic import write\n\n\ndef run(root,output):\n    root,output=Path(root),Path(output);output.mkdir(parents=True,exist_ok=True)\n    rows=list(csv.DictReader((root/\'reports/phase6_window_metrics.csv\').open()))\n    rows=[r for r in rows if r[\'scope\'] in (\'intraday\',\'capture_only\')]\n    rows.sort(key=lambda r:int(json.loads(r[\'batch_ids\'])[0]))\n    assert len(rows)==49 and sum(int(r[\'rows\']) for r in rows)==15322987\n    histograms=[json.loads(r[\'rf_score_counts_32\']) for r in rows]\n    assert all(sum(h)==int(r[\'rows\']) for h,r in zip(histograms,rows))\n    reference=[sum(h[i] for h in histograms[:3]) for i in range(32)]\n    calibration=[divergences(reference,h) for h in histograms[3:10]]\n    js_threshold=max(p[0] for p in calibration);psi_threshold=max(p[1] for p in calibration)\n    frozen={\'reference_batch_indices\':[0,1,2],\'calibration_batch_indices\':list(range(3,10)),\n            \'evaluation_batch_indices\':list(range(10,49)),\'reference_histogram\':reference,\'JS\':js_threshold,\'PSI\':psi_threshold,\n            \'unit\':\'replay window\',\'label_delay_windows\':[0,1,3],\'DDM\':\'unsupported for historical binary errors because saved aggregate counts omit within-window error ordering\'}\n    (output/\'frozen.json\').write_text(json.dumps(frozen,indent=2)+\'\\n\')\n    records=[];performance=[]\n    for delay in (0,1,3):\n        detectors={name:create(name) for name in (\'ADWIN\',\'PageHinkley\',\'KSWIN\')}\n        label_detectors={name:create(name) for name in (\'ADWIN\',\'PageHinkley\')}\n        scheduler=LabelSchedule(delay)\n        for index,(row,histogram) in enumerate(zip(rows,histograms)):\n            tick=perf_counter();n=int(row[\'rows\']);mean_score=sum((i+.5)/32*c for i,c in enumerate(histogram))/n\n            if n<100:\n                records.append({\'batch\':index,\'source_day\':row[\'source_day\'],\'detector\':\'support_check\',\'signal\':\'RF histogram\',\'value\':n,\'threshold\':100,\'alert\':False,\'label_delay\':None,\'label_source_batch\':None,\'evaluation\':index>=10,\'unit\':\'replay window\'})\n                continue\n            metrics=json.loads(row[\'classifier_metrics\'])[\'random_forest\'];error=(metrics[\'fp\']+metrics[\'fn\'])/n\n            scheduler.predict(index,index,error)\n            js,psi=divergences(reference,histogram)\n            if index>=10 and delay==0:\n                for name,value,threshold in [(\'JS\',js,js_threshold),(\'PSI\',psi,psi_threshold),(\'FixedJS\',js,.1)]:\n                    records.append({\'batch\':index,\'source_day\':row[\'source_day\'],\'detector\':name,\'signal\':\'RF histogram\',\'value\':value,\'threshold\':threshold,\'alert\':value>threshold,\'label_delay\':None,\'label_source_batch\':None,\'evaluation\':True,\'unit\':\'replay window\'})\n            for name,detector in detectors.items():\n                alerted=update(detector,mean_score)\n                if delay==0:\n                    records.append({\'batch\':index,\'source_day\':row[\'source_day\'],\'detector\':name,\'signal\':\'RF score mean\',\'value\':mean_score,\'threshold\':None,\'alert\':alerted,\'label_delay\':None,\'label_source_batch\':None,\'evaluation\':index>=10,\'unit\':\'replay window\'})\n            for source_batch,released_error,released_at in scheduler.release(index):\n                assert released_at<=index and source_batch<=index-delay\n                for name,detector in label_detectors.items():\n                    alerted=update(detector,released_error)\n                    records.append({\'batch\':index,\'source_day\':row[\'source_day\'],\'detector\':name,\'signal\':\'released RF aggregate error rate\',\'value\':released_error,\'threshold\':None,\'alert\':alerted,\'label_delay\':delay,\'label_source_batch\':source_batch,\'evaluation\':index>=10,\'unit\':\'released replay window\'})\n            performance.append({\'batch\':index,\'delay\':delay,\'rows_represented\':n,\'seconds\':perf_counter()-tick,\'state_bytes\':sum(len(pickle.dumps(x)) for x in (*detectors.values(),*label_detectors.values()))})\n    write(output/\'historical_alerts.csv\',records);write(output/\'performance.csv\',performance)\n    (output/\'receipt.json\').write_text(json.dumps({\'status\':\'validated\',\'stage\':\'11C\',\'windows\':49,\'rows_represented\':15322987,\'evaluation_windows\':39,\'historical_true_drift_labels\':False,\'unit\':\'aggregate window\',\'labels_released_after_end\':\'pending and excluded\'},indent=2)+\'\\n\')\n    print(\'PHASE11 historical complete\',len(records),flush=True)\n', 'src/drift/advanced_detectors.py': '"""Version-pinned detector adapters with explicit input semantics."""\nfrom math import isfinite,log\nfrom river import drift\n\n\ndef create(name,sensitivity=\'conservative\',seed=23013):\n    sensitive=sensitivity==\'sensitive\'\n    if name==\'ADWIN\':return drift.ADWIN(delta=.01 if sensitive else .001)\n    if name==\'PageHinkley\':return drift.PageHinkley(delta=.005,threshold=10 if sensitive else 30,min_instances=30,mode=\'both\')\n    if name==\'KSWIN\':return drift.KSWIN(alpha=.01 if sensitive else .001,window_size=100,stat_size=30,seed=seed)\n    if name==\'DDM\':return drift.binary.DDM(warm_start=30,warning_threshold=2.,drift_threshold=3.)\n    raise ValueError(f\'Unknown detector {name}\')\n\n\ndef update(detector,value,binary=False):\n    if not isfinite(value):return False\n    if binary and value not in (0,1):raise ValueError(\'DDM requires a released binary error\')\n    detector.update(value)\n    return bool(detector.drift_detected)\n\n\ndef divergences(reference,current,smoothing=1e-6):\n    if len(reference)!=len(current) or not reference or min(reference+current)<0 or not sum(reference) or not sum(current):\n        raise ValueError(\'Histograms require nonnegative counts and positive support\')\n    def normalize(values):\n        denominator=sum(values)+len(values)*smoothing\n        return [(v+smoothing)/denominator for v in values]\n    p,q=normalize(reference),normalize(current)\n    js=sum(.5*a*log(a/((a+b)/2),2)+.5*b*log(b/((a+b)/2),2) for a,b in zip(p,q))\n    psi=sum((b-a)*log(b/a) for a,b in zip(p,q))\n    return js,psi\n', 'src/drift/label_availability.py': '"""Release labels on a simulated window clock."""\nfrom collections import defaultdict\n\n\nclass LabelSchedule:\n    def __init__(self,delay):\n        if delay<0:raise ValueError(\'Label delay must be nonnegative\')\n        self.delay=delay\n        self.pending=defaultdict(list)\n\n    def predict(self,window,identifier,error):\n        self.pending[window+self.delay].append((identifier,error))\n\n    def release(self,window):\n        released=[]\n        for time in sorted(k for k in self.pending if k<=window):\n            released.extend((identifier,error,time) for identifier,error in self.pending.pop(time))\n        return released\n', 'src/drift/drift_state.py': '"""Trusted local snapshots with idempotent batch updates."""\nimport pickle\nfrom src.drift.advanced_detectors import create,update\n\n\nclass DetectorState:\n    def __init__(self,name,**parameters):\n        self.name=name\n        self.parameters=parameters\n        self.detector=create(name,**parameters)\n        self.batch_ids=set()\n        self.observations=0\n\n    def batch(self,identifier,values):\n        if identifier in self.batch_ids:return []\n        alerts=[]\n        for value in values:\n            if update(self.detector,value,self.name==\'DDM\'):alerts.append(self.observations)\n            self.observations+=1\n        self.batch_ids.add(identifier)\n        return alerts\n\n    def snapshot(self):return pickle.dumps(self,protocol=5)\n\n    @staticmethod\n    def restore(content):\n        # Read only trusted experiment snapshots, never uploaded or arbitrary bytes.\n        value=pickle.loads(content)\n        if not isinstance(value,DetectorState):raise ValueError(\'Incompatible state snapshot\')\n        return value\n', 'reports/phase6_window_metrics.csv': 'scope,source_day,split_rank,batch_ids,window_hour,wall_start,wall_end,rows,eligible,attacks,benign,mean_distance,distance_p95_upper,cluster_counts,rf_score_counts_32,classifier_metrics,novelty_metrics\nintraday,2018-02-14,0,[0],0,00:00,03:00,235564,235564,120348,115216,3.207205195739007,8,"[33417,4454,79054,6654,7877,602,11643,4073,10,688,43271,43821]","[105741,2793,666,671,959,527,68,124,58725,6,22,3234,18,3,37,5,281,0,21,57,32,47,219,127,40,592,447,23,5471,822,6307,47479]","{""random_forest"":{""tp"":60229,""tn"":113480,""fp"":1736,""fn"":60119,""rows"":235564,""accuracy"":0.7374174322052606,""precision"":0.9719841846203502,""recall"":0.5004570080101041,""specificity"":0.9849326482432995,""f1"":0.6607208482115922,""balanced_accuracy"":0.7426948281267018,""roc_auc"":0.9753951546016368,""pr_auc"":0.9743296868027806,""predicted_positive"":61965,""predicted_rate"":0.263049532186582},""logistic_regression"":{""tp"":58594,""tn"":113948,""fp"":1268,""fn"":61754,""rows"":235564,""accuracy"":0.7324633645208946,""precision"":0.9788179479469447,""recall"":0.48687140625519326,""specificity"":0.9889945840855436,""f1"":0.65028577770379,""balanced_accuracy"":0.7379329951703685,""roc_auc"":0.7919539596237084,""pr_auc"":0.8123720690272532,""predicted_positive"":59862,""predicted_rate"":0.2541220220407193},""duration_rule"":{""tp"":60151,""tn"":56486,""fp"":58730,""fn"":60197,""rows"":235564,""accuracy"":0.4951393251940025,""precision"":0.5059765648000942,""recall"":0.49980888755941105,""specificity"":0.4902617691987224,""f1"":0.5028738154655163,""balanced_accuracy"":0.4950353283790667,""roc_auc"":0.4950353283790667,""pr_auc"":0.508435724854289,""predicted_positive"":118881,""predicted_rate"":0.5046653987875906}}","{""rows"":235564,""attacks"":120348,""benign"":115216,""tp"":0,""fp"":975,""flags"":975,""flag_rate"":0.0041390025640590244,""benign_fpr"":0.00846236633800861,""attack_recall"":0.0,""precision"":0.0}"\nintraday,2018-02-14,0,[1],3,03:00,06:00,142824,142824,67241,75583,3.1928255018318707,8,"[20474,3193,46928,4331,5969,396,8061,2557,9,437,25312,25157]","[68777,1599,395,424,609,297,36,62,33007,7,20,2694,18,5,18,5,159,2,17,39,16,40,128,125,26,331,247,18,3013,617,3422,26651]","{""random_forest"":{""tp"":33614,""tn"":74346,""fp"":1237,""fn"":33627,""rows"":142824,""accuracy"":0.7558953677253123,""precision"":0.9645060399988522,""recall"":0.4999033327880311,""specificity"":0.9836338859267295,""f1"":0.6585040943462759,""balanced_accuracy"":0.7417686093573803,""roc_auc"":0.9709962104751702,""pr_auc"":0.9656683303590666,""predicted_positive"":34851,""predicted_rate"":0.2440136111577886},""logistic_regression"":{""tp"":32871,""tn"":74832,""fp"":751,""fn"":34370,""rows"":142824,""accuracy"":0.7540959502604604,""precision"":0.9776634346558801,""recall"":0.488853526866049,""specificity"":0.9900639032586693,""f1"":0.6517950090717112,""balanced_accuracy"":0.7394587150623592,""roc_auc"":0.7744729882714148,""pr_auc"":0.7848017687896421,""predicted_positive"":33622,""predicted_rate"":0.2354086147986333},""duration_rule"":{""tp"":33618,""tn"":37848,""fp"":37735,""fn"":33623,""rows"":142824,""accuracy"":0.5003780877163502,""precision"":0.4711504772048828,""recall"":0.49996282030308886,""specificity"":0.5007475225910588,""f1"":0.4851292263734361,""balanced_accuracy"":0.5003551714470739,""roc_auc"":0.5003551714470739,""pr_auc"":0.4709733377935096,""predicted_positive"":71353,""predicted_rate"":0.49958690416176554}}","{""rows"":142824,""attacks"":67241,""benign"":75583,""tp"":0,""fp"":687,""flags"":687,""flag_rate"":0.00481011594689968,""benign_fpr"":0.009089345487741953,""attack_recall"":0.0,""precision"":0.0}"\nintraday,2018-02-14,0,[2],6,06:00,09:00,67293,67293,0,67293,2.74195891695915,12,"[19146,3993,10706,3021,5323,308,5606,2438,5,263,9352,7132]","[59283,2948,353,448,515,275,29,44,413,6,10,2076,12,2,15,2,108,0,0,5,2,15,96,22,11,6,15,9,0,573,0,0]","{""random_forest"":{""tp"":0,""tn"":66431,""fp"":862,""fn"":0,""rows"":67293,""accuracy"":0.9871903466928209,""precision"":0.0,""recall"":null,""specificity"":0.9871903466928209,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":862,""predicted_rate"":0.012809653307179052},""logistic_regression"":{""tp"":0,""tn"":66438,""fp"":855,""fn"":0,""rows"":67293,""accuracy"":0.9872943693994918,""precision"":0.0,""recall"":null,""specificity"":0.9872943693994918,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":855,""predicted_rate"":0.012705630600508225},""duration_rule"":{""tp"":0,""tn"":34584,""fp"":32709,""fn"":0,""rows"":67293,""accuracy"":0.5139316125005573,""precision"":0.0,""recall"":null,""specificity"":0.5139316125005573,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":32709,""predicted_rate"":0.48606838749944276}}","{""rows"":67293,""attacks"":0,""benign"":67293,""tp"":0,""fp"":679,""flags"":679,""flag_rate"":0.010090202547070274,""benign_fpr"":0.010090202547070274,""attack_recall"":null,""precision"":0.0}"\nintraday,2018-02-14,0,[3],9,09:00,12:00,498994,498994,172390,326604,3.8788482988118758,8,"[91290,14650,226304,22080,24369,1943,33140,11975,28,1638,38235,33342]","[293325,8271,1746,1934,2552,2293,134,346,1116,17,63,10326,52,7,132,10,738,4,6,26,11,39,626,18040,33,20,154513,90,6,2518,0,0]","{""random_forest"":{""tp"":172385,""tn"":322319,""fp"":4285,""fn"":5,""rows"":498994,""accuracy"":0.9914027022369006,""precision"":0.9757457406464028,""recall"":0.9999709959974477,""specificity"":0.9868801361893914,""f1"":0.9877098493095743,""balanced_accuracy"":0.9934255660934195,""roc_auc"":0.9917807218313937,""pr_auc"":0.983051765435146,""predicted_positive"":176670,""predicted_rate"":0.35405235333490986},""logistic_regression"":{""tp"":135411,""tn"":323087,""fp"":3517,""fn"":36979,""rows"":498994,""accuracy"":0.9188447155677223,""precision"":0.9746847287803755,""recall"":0.7854921979233134,""specificity"":0.9892316076961702,""f1"":0.8699207883900063,""balanced_accuracy"":0.8873619028097418,""roc_auc"":0.9442705764621226,""pr_auc"":0.9126354238803605,""predicted_positive"":138928,""predicted_rate"":0.27841617334076163},""duration_rule"":{""tp"":172390,""tn"":163330,""fp"":163274,""fn"":0,""rows"":498994,""accuracy"":0.6727936608456214,""precision"":0.5135790552457219,""recall"":1.0,""specificity"":0.500085730732018,""f1"":0.6786286497104639,""balanced_accuracy"":0.750042865366009,""roc_auc"":0.750042865366009,""pr_auc"":0.5135790552457219,""predicted_positive"":335664,""predicted_rate"":0.6726814350473151}}","{""rows"":498994,""attacks"":172390,""benign"":326604,""tp"":0,""fp"":2286,""flags"":2286,""flag_rate"":0.004581217409427769,""benign_fpr"":0.0069993019068964255,""attack_recall"":0.0,""precision"":0.0}"\nintraday,2018-02-14,0,[4],12,12:00,15:00,103895,103895,20970,82925,3.280413910377969,8,"[24160,2982,34826,4994,5384,497,8113,3142,7,510,10670,8610]","[75190,2009,495,506,682,357,42,82,301,10,13,2221,23,5,25,3,186,0,1,6,3,11,118,2242,10,4,18785,12,0,553,0,0]","{""random_forest"":{""tp"":20969,""tn"":81963,""fp"":962,""fn"":1,""rows"":103895,""accuracy"":0.9907310265171567,""precision"":0.9561351511558981,""recall"":0.9999523128278494,""specificity"":0.9883991558637323,""f1"":0.9775529707932216,""balanced_accuracy"":0.9941757343457909,""roc_auc"":0.9930057223168922,""pr_auc"":0.9702153387116714,""predicted_positive"":21931,""predicted_rate"":0.21108811781125175},""logistic_regression"":{""tp"":16222,""tn"":81981,""fp"":944,""fn"":4748,""rows"":103895,""accuracy"":0.9452139178978777,""precision"":0.9450075731096353,""recall"":0.7735813066285169,""specificity"":0.9886162194754297,""f1"":0.8507447031676106,""balanced_accuracy"":0.8810987630519733,""roc_auc"":0.9441105701197675,""pr_auc"":0.8518304356725963,""predicted_positive"":17166,""predicted_rate"":0.16522450551037104},""duration_rule"":{""tp"":20970,""tn"":40734,""fp"":42191,""fn"":0,""rows"":103895,""accuracy"":0.5939073102651715,""precision"":0.33200867624008484,""recall"":1.0,""specificity"":0.49121495327102804,""f1"":0.49850827875574993,""balanced_accuracy"":0.7456074766355141,""roc_auc"":0.7456074766355141,""pr_auc"":0.33200867624008484,""predicted_positive"":63161,""predicted_rate"":0.6079310842677703}}","{""rows"":103895,""attacks"":20970,""benign"":82925,""tp"":0,""fp"":606,""flags"":606,""flag_rate"":0.00583281197362722,""benign_fpr"":0.007307808260476334,""attack_recall"":0.0,""precision"":0.0}"\ncapture_only,2018-02-14,0,[5],24,,,5,0,0,5,34.352271346839274,64,"[0,5,0,0,0,0,0,0,0,0,0,0]","[0,0,0,0,4,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]","{""random_forest"":{""tp"":0,""tn"":5,""fp"":0,""fn"":0,""rows"":5,""accuracy"":1.0,""precision"":null,""recall"":null,""specificity"":1.0,""f1"":null,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":0,""predicted_rate"":0.0},""logistic_regression"":{""tp"":0,""tn"":5,""fp"":0,""fn"":0,""rows"":5,""accuracy"":1.0,""precision"":null,""recall"":null,""specificity"":1.0,""f1"":null,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":0,""predicted_rate"":0.0},""duration_rule"":{""tp"":0,""tn"":5,""fp"":0,""fn"":0,""rows"":5,""accuracy"":1.0,""precision"":null,""recall"":null,""specificity"":1.0,""f1"":null,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":0,""predicted_rate"":0.0}}","{""rows"":5,""attacks"":0,""benign"":5,""tp"":0,""fp"":4,""flags"":4,""flag_rate"":0.8,""benign_fpr"":0.8,""attack_recall"":null,""precision"":0.0}"\nintraday,2018-02-15,0,[6],0,00:00,03:00,197798,197798,0,197798,2.6252858619461485,12,"[62723,8172,32674,9846,16773,1270,20276,8309,2070,2070,15428,18187]","[174022,4672,4718,840,2161,1192,85,686,810,13,54,5172,34,1,202,6,781,1,2,15,10,21,384,81,13,1,25,119,16,1661,0,0]","{""random_forest"":{""tp"":0,""tn"":194668,""fp"":3130,""fn"":0,""rows"":197798,""accuracy"":0.9841757752858977,""precision"":0.0,""recall"":null,""specificity"":0.9841757752858977,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":3130,""predicted_rate"":0.015824224714102265},""logistic_regression"":{""tp"":0,""tn"":194899,""fp"":2899,""fn"":0,""rows"":197798,""accuracy"":0.9853436334037756,""precision"":0.0,""recall"":null,""specificity"":0.9853436334037756,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":2899,""predicted_rate"":0.014656366596224431},""duration_rule"":{""tp"":0,""tn"":94963,""fp"":102835,""fn"":0,""rows"":197798,""accuracy"":0.4801009110304452,""precision"":0.0,""recall"":null,""specificity"":0.4801009110304452,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":102835,""predicted_rate"":0.5198990889695548}}","{""rows"":197798,""attacks"":0,""benign"":197798,""tp"":0,""fp"":1898,""flags"":1898,""flag_rate"":0.00959564808542048,""benign_fpr"":0.00959564808542048,""attack_recall"":null,""precision"":0.0}"\nintraday,2018-02-15,0,[7],3,03:00,06:00,191551,191551,0,191551,2.5986818768986453,12,"[58693,7798,32206,15833,13174,1088,17681,7320,2332,2012,16792,16622]","[172422,4438,1163,616,2035,1070,83,577,892,7,42,5299,23,1,170,4,704,1,4,12,4,17,318,87,8,3,44,94,6,1407,0,0]","{""random_forest"":{""tp"":0,""tn"":188842,""fp"":2709,""fn"":0,""rows"":191551,""accuracy"":0.9858575522967774,""precision"":0.0,""recall"":null,""specificity"":0.9858575522967774,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":2709,""predicted_rate"":0.01414244770322264},""logistic_regression"":{""tp"":0,""tn"":188712,""fp"":2839,""fn"":0,""rows"":191551,""accuracy"":0.9851788818643599,""precision"":0.0,""recall"":null,""specificity"":0.9851788818643599,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":2839,""predicted_rate"":0.014821118135640117},""duration_rule"":{""tp"":0,""tn"":94920,""fp"":96631,""fn"":0,""rows"":191551,""accuracy"":0.4955338265005142,""precision"":0.0,""recall"":null,""specificity"":0.4955338265005142,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":96631,""predicted_rate"":0.5044661734994857}}","{""rows"":191551,""attacks"":0,""benign"":191551,""tp"":0,""fp"":1653,""flags"":1653,""flag_rate"":0.008629555575277602,""benign_fpr"":0.008629555575277602,""attack_recall"":null,""precision"":0.0}"\nintraday,2018-02-15,0,[8],6,06:00,09:00,87581,87581,0,87581,2.823440781448174,12,"[24353,6765,13379,4948,8401,368,7208,2530,663,832,9759,8375]","[74837,4021,1765,339,787,393,66,230,353,2,20,2554,2,1,101,1,342,1,0,3,2,18,115,29,5,1,10,51,2,1530,0,0]","{""random_forest"":{""tp"":0,""tn"":85472,""fp"":2109,""fn"":0,""rows"":87581,""accuracy"":0.9759194345805597,""precision"":0.0,""recall"":null,""specificity"":0.9759194345805597,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":2109,""predicted_rate"":0.02408056541944029},""logistic_regression"":{""tp"":0,""tn"":86581,""fp"":1000,""fn"":0,""rows"":87581,""accuracy"":0.9885819983786438,""precision"":0.0,""recall"":null,""specificity"":0.9885819983786438,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":1000,""predicted_rate"":0.011418001621356231},""duration_rule"":{""tp"":0,""tn"":47815,""fp"":39766,""fn"":0,""rows"":87581,""accuracy"":0.5459517475251482,""precision"":0.0,""recall"":null,""specificity"":0.5459517475251482,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":39766,""predicted_rate"":0.45404825247485187}}","{""rows"":87581,""attacks"":0,""benign"":87581,""tp"":0,""fp"":1002,""flags"":1002,""flag_rate"":0.011440837624598943,""benign_fpr"":0.011440837624598943,""attack_recall"":null,""precision"":0.0}"\nintraday,2018-02-15,0,[9],9,09:00,12:00,463466,463466,52498,410968,2.8568289082871705,12,"[131583,19597,68893,17626,49526,2787,48662,18134,3273,6926,57121,39338]","[362124,14664,8081,2405,4840,3170,1904,2035,2068,718,2943,13950,755,829,2761,2201,2744,819,6291,848,2626,9470,2665,101,1136,1050,6774,222,19,3253,0,0]","{""random_forest"":{""tp"":31487,""tn"":404437,""fp"":6531,""fn"":21011,""rows"":463466,""accuracy"":0.9405738500774599,""precision"":0.8282129517596928,""recall"":0.5997752295325536,""specificity"":0.9841082517373615,""f1"":0.6957223032392064,""balanced_accuracy"":0.7919417406349576,""roc_auc"":0.96804320540587,""pr_auc"":0.732580537380859,""predicted_positive"":38018,""predicted_rate"":0.08202974975510609},""logistic_regression"":{""tp"":3,""tn"":405049,""fp"":5919,""fn"":52495,""rows"":463466,""accuracy"":0.8739627070809941,""precision"":0.0005065856129685917,""recall"":5.714503409653701e-05,""specificity"":0.985597418777131,""f1"":0.00010270455323519343,""balanced_accuracy"":0.49282728190561376,""roc_auc"":0.49402533070373295,""pr_auc"":0.10727661583028984,""predicted_positive"":5922,""predicted_rate"":0.01277763633146768},""duration_rule"":{""tp"":1132,""tn"":196426,""fp"":214542,""fn"":51366,""rows"":463466,""accuracy"":0.426262120630208,""precision"":0.005248662332965494,""recall"":0.021562726199093298,""specificity"":0.4779593544996204,""f1"":0.008442342973912265,""balanced_accuracy"":0.24976104034935684,""roc_auc"":0.2497610403493569,""pr_auc"":0.11094331187578287,""predicted_positive"":215674,""predicted_rate"":0.4653502090768255}}","{""rows"":463466,""attacks"":52498,""benign"":410968,""tp"":1005,""fp"":3810,""flags"":4815,""flag_rate"":0.010389111606892414,""benign_fpr"":0.009270794806408285,""attack_recall"":0.0191435864223399,""precision"":0.2087227414330218}"\nintraday,2018-02-15,0,[10],12,12:00,15:00,108179,108179,0,108179,2.6699542806657184,12,"[34886,4625,18068,3624,9711,639,10599,4114,701,1247,8326,11639]","[94537,2814,2477,537,1172,923,37,348,382,12,22,2876,33,3,139,7,443,0,1,10,3,17,237,37,8,4,22,75,5,998,0,0]","{""random_forest"":{""tp"":0,""tn"":106319,""fp"":1860,""fn"":0,""rows"":108179,""accuracy"":0.9828062747853096,""precision"":0.0,""recall"":null,""specificity"":0.9828062747853096,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":1860,""predicted_rate"":0.017193725214690467},""logistic_regression"":{""tp"":0,""tn"":106682,""fp"":1497,""fn"":0,""rows"":108179,""accuracy"":0.9861618243836604,""precision"":0.0,""recall"":null,""specificity"":0.9861618243836604,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":1497,""predicted_rate"":0.013838175616339585},""duration_rule"":{""tp"":0,""tn"":52430,""fp"":55749,""fn"":0,""rows"":108179,""accuracy"":0.48465968441194685,""precision"":0.0,""recall"":null,""specificity"":0.48465968441194685,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":55749,""predicted_rate"":0.5153403155880532}}","{""rows"":108179,""attacks"":0,""benign"":108179,""tp"":0,""fp"":1080,""flags"":1080,""flag_rate"":0.009983453350465432,""benign_fpr"":0.009983453350465432,""attack_recall"":null,""precision"":0.0}"\nintraday,2018-02-16,0,[11],0,00:00,03:00,908339,908339,461912,446427,2.26882214194008,8,"[57,418616,21930,0,19,1267,13901,2,2,1,452368,176]","[453718,11647,1574,1450,783,417,1177,414,2767,209,599,239,107,155,3389,1495,7767,1648,2477,171,3,6138,2281,0,0,3475,2948,20732,251,380308,0,0]","{""random_forest"":{""tp"":426511,""tn"":444739,""fp"":1688,""fn"":35401,""rows"":908339,""accuracy"":0.9591683281241915,""precision"":0.9960579076550856,""recall"":0.9233598607526975,""specificity"":0.9962188666904107,""f1"":0.9583321630673028,""balanced_accuracy"":0.9597893637215541,""roc_auc"":0.9956582721897103,""pr_auc"":0.9936235099414321,""predicted_positive"":428199,""predicted_rate"":0.47140880222031645},""logistic_regression"":{""tp"":85,""tn"":446427,""fp"":0,""fn"":461827,""rows"":908339,""accuracy"":0.49156977736285684,""precision"":1.0,""recall"":0.00018401773497982298,""specificity"":1.0,""f1"":0.00036796775736638984,""balanced_accuracy"":0.50009200886749,""roc_auc"":0.9552564625505485,""pr_auc"":0.955211859272473,""predicted_positive"":85,""predicted_rate"":9.357739786577478e-05},""duration_rule"":{""tp"":309904,""tn"":442905,""fp"":3522,""fn"":152008,""rows"":908339,""accuracy"":0.8287753801168947,""precision"":0.9887628977812944,""recall"":0.6709156722492596,""specificity"":0.9921106922296367,""f1"":0.7994036149395489,""balanced_accuracy"":0.8315131822394481,""roc_auc"":0.8315131822394481,""pr_auc"":0.8307237371398359,""predicted_positive"":313426,""predicted_rate"":0.3450539941585686}}","{""rows"":908339,""attacks"":461912,""benign"":446427,""tp"":1267,""fp"":790,""flags"":2057,""flag_rate"":0.0022645730283517495,""benign_fpr"":0.001769606229013926,""attack_recall"":0.00274294670846395,""precision"":0.6159455517744288}"\nintraday,2018-02-16,0,[12],6,06:00,09:00,50,50,0,50,7.5114732309319745,64,"[2,2,8,0,26,0,0,0,0,4,0,8]","[32,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0,8,0,0,0,0,0,0,0,0]","{""random_forest"":{""tp"":0,""tn"":38,""fp"":12,""fn"":0,""rows"":50,""accuracy"":0.76,""precision"":0.0,""recall"":null,""specificity"":0.76,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":12,""predicted_rate"":0.24},""logistic_regression"":{""tp"":0,""tn"":50,""fp"":0,""fn"":0,""rows"":50,""accuracy"":1.0,""precision"":null,""recall"":null,""specificity"":1.0,""f1"":null,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":0,""predicted_rate"":0.0},""duration_rule"":{""tp"":0,""tn"":40,""fp"":10,""fn"":0,""rows"":50,""accuracy"":0.8,""precision"":0.0,""recall"":null,""specificity"":0.8,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":10,""predicted_rate"":0.2}}","{""rows"":50,""attacks"":0,""benign"":50,""tp"":0,""fp"":4,""flags"":4,""flag_rate"":0.08,""benign_fpr"":0.08,""attack_recall"":null,""precision"":0.0}"\nintraday,2018-02-16,0,[13],9,09:00,12:00,140132,140132,139890,242,6.341606056938444,8,"[10,3,139899,0,105,0,2,29,2,9,62,11]","[195,33,0,3,0,2,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,15159,0,0,124593,146,0,0,0,0]","{""random_forest"":{""tp"":139890,""tn"":233,""fp"":9,""fn"":0,""rows"":140132,""accuracy"":0.9999357748408644,""precision"":0.999935667874681,""recall"":1.0,""specificity"":0.9628099173553719,""f1"":0.9999678329026517,""balanced_accuracy"":0.981404958677686,""roc_auc"":0.9979859027370384,""pr_auc"":0.9999930323611592,""predicted_positive"":139899,""predicted_rate"":0.9983372819912654},""logistic_regression"":{""tp"":63052,""tn"":242,""fp"":0,""fn"":76838,""rows"":140132,""accuracy"":0.4516741358148032,""precision"":1.0,""recall"":0.4507255700907856,""specificity"":1.0,""f1"":0.621379507445477,""balanced_accuracy"":0.7253627850453928,""roc_auc"":0.9900561627819734,""pr_auc"":0.9999799623197178,""predicted_positive"":63052,""predicted_rate"":0.4499471926469329},""duration_rule"":{""tp"":139890,""tn"":194,""fp"":48,""fn"":0,""rows"":140132,""accuracy"":0.9996574658179431,""precision"":0.9996569909531364,""recall"":1.0,""specificity"":0.8016528925619835,""f1"":0.9998284660577211,""balanced_accuracy"":0.9008264462809917,""roc_auc"":0.9008264462809917,""pr_auc"":0.9996569909531364,""predicted_positive"":139938,""predicted_rate"":0.9986155910141866}}","{""rows"":140132,""attacks"":139890,""benign"":242,""tp"":0,""fp"":5,""flags"":5,""flag_rate"":3.568064396426227e-05,""benign_fpr"":0.02066115702479339,""attack_recall"":0.0,""precision"":0.0}"\nintraday,2018-02-16,0,[14],12,12:00,15:00,53,53,0,53,3.6926296579931757,16,"[4,3,2,0,21,0,0,5,1,1,15,1]","[43,5,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0]","{""random_forest"":{""tp"":0,""tn"":52,""fp"":1,""fn"":0,""rows"":53,""accuracy"":0.9811320754716981,""precision"":0.0,""recall"":null,""specificity"":0.9811320754716981,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":1,""predicted_rate"":0.018867924528301886},""logistic_regression"":{""tp"":0,""tn"":53,""fp"":0,""fn"":0,""rows"":53,""accuracy"":1.0,""precision"":null,""recall"":null,""specificity"":1.0,""f1"":null,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":0,""predicted_rate"":0.0},""duration_rule"":{""tp"":0,""tn"":40,""fp"":13,""fn"":0,""rows"":53,""accuracy"":0.7547169811320755,""precision"":0.0,""recall"":null,""specificity"":0.7547169811320755,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":13,""predicted_rate"":0.24528301886792453}}","{""rows"":53,""attacks"":0,""benign"":53,""tp"":0,""fp"":0,""flags"":0,""flag_rate"":0.0,""benign_fpr"":0.0,""attack_recall"":null,""precision"":null}"\nintraday,2018-02-20,0,[15],0,00:00,03:00,1643268,1643268,797,1642471,2.5897112529844657,12,"[492016,62613,242108,245193,88561,10794,122617,81496,37942,21741,153077,85110]","[1492981,38938,14928,8813,15759,16376,640,2270,13917,116,470,17898,54,21,416,24,5352,70,114,337,283,486,3068,510,281,67,385,264,164,8266,0,0]","{""random_forest"":{""tp"":782,""tn"":1623606,""fp"":18865,""fn"":15,""rows"":1643268,""accuracy"":0.9885106994111733,""precision"":0.03980251437878556,""recall"":0.9811794228356336,""specificity"":0.9885142568727241,""f1"":0.07650166307963216,""balanced_accuracy"":0.9848468398541789,""roc_auc"":0.9954370858278397,""pr_auc"":0.07229430899138825,""predicted_positive"":19647,""predicted_rate"":0.011956053425247738},""logistic_regression"":{""tp"":662,""tn"":1613269,""fp"":29202,""fn"":135,""rows"":1643268,""accuracy"":0.9821471604144911,""precision"":0.022167157781944818,""recall"":0.8306148055207027,""specificity"":0.9822206906545078,""f1"":0.04318189230618701,""balanced_accuracy"":0.9064177480876052,""roc_auc"":0.8924253638568028,""pr_auc"":0.71162897972887,""predicted_positive"":29864,""predicted_rate"":0.01817354199071606},""duration_rule"":{""tp"":0,""tn"":789255,""fp"":853216,""fn"":797,""rows"":1643268,""accuracy"":0.480295971198855,""precision"":0.0,""recall"":0.0,""specificity"":0.4805290321716487,""f1"":0.0,""balanced_accuracy"":0.24026451608582436,""roc_auc"":0.24026451608582433,""pr_auc"":0.00048500914032282015,""predicted_positive"":853216,""predicted_rate"":0.5192190196608222}}","{""rows"":1643268,""attacks"":797,""benign"":1642471,""tp"":797,""fp"":21312,""flags"":22109,""flag_rate"":0.013454287432116976,""benign_fpr"":0.012975571562602931,""attack_recall"":1.0,""precision"":0.03604866796327288}"\nintraday,2018-02-20,0,[16],3,03:00,06:00,1945025,1945025,0,1945025,2.567411974596214,12,"[560325,75316,276532,339407,124128,11875,135266,90178,44400,27675,170846,89077]","[1750711,46680,17460,8730,18544,36698,795,2665,17219,176,587,22364,49,22,460,32,5489,7,130,176,144,294,3412,640,258,37,1530,328,35,9353,0,0]","{""random_forest"":{""tp"":0,""tn"":1923192,""fp"":21833,""fn"":0,""rows"":1945025,""accuracy"":0.9887749514787728,""precision"":0.0,""recall"":null,""specificity"":0.9887749514787728,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":21833,""predicted_rate"":0.011225048521227233},""logistic_regression"":{""tp"":0,""tn"":1909423,""fp"":35602,""fn"":0,""rows"":1945025,""accuracy"":0.9816958650917084,""precision"":0.0,""recall"":null,""specificity"":0.9816958650917084,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":35602,""predicted_rate"":0.018304134908291667},""duration_rule"":{""tp"":0,""tn"":977619,""fp"":967406,""fn"":0,""rows"":1945025,""accuracy"":0.5026254161257567,""precision"":0.0,""recall"":null,""specificity"":0.5026254161257567,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":967406,""predicted_rate"":0.49737458387424327}}","{""rows"":1945025,""attacks"":0,""benign"":1945025,""tp"":0,""fp"":26389,""flags"":26389,""flag_rate"":0.01356743486587576,""benign_fpr"":0.01356743486587576,""attack_recall"":null,""precision"":0.0}"\nintraday,2018-02-20,0,[17],6,06:00,09:00,479872,479872,0,479872,2.746510604826952,12,"[146697,23612,66634,74371,29979,2623,27001,21751,14162,6154,45015,21873]","[425694,16648,4715,3448,4479,7612,524,1766,3805,14,147,4598,6,5,116,2,1181,6,18,14,49,34,850,94,23,6,116,104,13,3785,0,0]","{""random_forest"":{""tp"":0,""tn"":473579,""fp"":6293,""fn"":0,""rows"":479872,""accuracy"":0.9868860862896772,""precision"":0.0,""recall"":null,""specificity"":0.9868860862896772,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":6293,""predicted_rate"":0.013113913710322753},""logistic_regression"":{""tp"":0,""tn"":472150,""fp"":7722,""fn"":0,""rows"":479872,""accuracy"":0.9839082088556949,""precision"":0.0,""recall"":null,""specificity"":0.9839082088556949,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":7722,""predicted_rate"":0.01609179114430515},""duration_rule"":{""tp"":0,""tn"":235582,""fp"":244290,""fn"":0,""rows"":479872,""accuracy"":0.4909267471325687,""precision"":0.0,""recall"":null,""specificity"":0.4909267471325687,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":244290,""predicted_rate"":0.5090732528674313}}","{""rows"":479872,""attacks"":0,""benign"":479872,""tp"":0,""fp"":8202,""flags"":8202,""flag_rate"":0.017092057882101892,""benign_fpr"":0.017092057882101892,""attack_recall"":null,""precision"":0.0}"\nintraday,2018-02-20,0,[18],9,09:00,12:00,3220536,3220536,575394,2645142,2.6310661322360938,8,"[802526,104254,388887,396648,430785,16803,189433,134032,346540,34223,243224,133181]","[2401616,56952,22590,17072,25872,45117,896,2931,22724,207,830,123293,94,28,536,25,13298,12,11438,1684,603,5919,29153,919,18742,8962,106431,377,44,12883,70,289218]","{""random_forest"":{""tp"":473461,""tn"":2618850,""fp"":26292,""fn"":101933,""rows"":3220536,""accuracy"":0.9601851989855105,""precision"":0.9473900106652686,""recall"":0.8228466059778169,""specificity"":0.9900602689761079,""f1"":0.8807372387217748,""balanced_accuracy"":0.9064534374769624,""roc_auc"":0.9953507102112253,""pr_auc"":0.9729905047150752,""predicted_positive"":499753,""predicted_rate"":0.1551769643314032},""logistic_regression"":{""tp"":0,""tn"":2598586,""fp"":46556,""fn"":575394,""rows"":3220536,""accuracy"":0.8068799727747182,""precision"":0.0,""recall"":0.0,""specificity"":0.9823994326202525,""f1"":0.0,""balanced_accuracy"":0.49119971631012627,""roc_auc"":0.8317907629600536,""pr_auc"":0.36039233562086564,""predicted_positive"":46556,""predicted_rate"":0.014455978756331244},""duration_rule"":{""tp"":828,""tn"":1252354,""fp"":1392788,""fn"":574566,""rows"":3220536,""accuracy"":0.38912218338810683,""precision"":0.0005941378399788751,""recall"":0.0014390139626064922,""specificity"":0.47345435519151713,""f1"":0.0008410317875480572,""balanced_accuracy"":0.2374466845770618,""roc_auc"":0.23744668457706183,""pr_auc"":0.17840780338123532,""predicted_positive"":1393616,""predicted_rate"":0.4327279682636679}}","{""rows"":3220536,""attacks"":575394,""benign"":2645142,""tp"":0,""fp"":35967,""flags"":35967,""flag_rate"":0.01116801675249089,""benign_fpr"":0.013597379649183295,""attack_recall"":0.0,""precision"":0.0}"\nintraday,2018-02-20,0,[19],12,12:00,15:00,660047,660047,0,660047,2.5951684262653654,12,"[172280,30820,100391,88003,42530,3856,47973,27941,12911,9683,78660,44999]","[600653,13804,6885,4022,5299,8545,291,780,3469,40,130,8963,46,11,134,11,1498,4,45,97,172,169,1107,344,36,9,102,88,13,3280,0,0]","{""random_forest"":{""tp"":0,""tn"":653083,""fp"":6964,""fn"":0,""rows"":660047,""accuracy"":0.9894492361907561,""precision"":0.0,""recall"":null,""specificity"":0.9894492361907561,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":6964,""predicted_rate"":0.010550763809243887},""logistic_regression"":{""tp"":0,""tn"":651730,""fp"":8317,""fn"":0,""rows"":660047,""accuracy"":0.9873993821652094,""precision"":0.0,""recall"":null,""specificity"":0.9873993821652094,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":8317,""predicted_rate"":0.012600617834790552},""duration_rule"":{""tp"":0,""tn"":341585,""fp"":318462,""fn"":0,""rows"":660047,""accuracy"":0.5175161768783132,""precision"":0.0,""recall"":null,""specificity"":0.5175161768783132,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":318462,""predicted_rate"":0.4824838231216868}}","{""rows"":660047,""attacks"":0,""benign"":660047,""tp"":0,""fp"":10297,""flags"":10297,""flag_rate"":0.015600404213639332,""benign_fpr"":0.015600404213639332,""attack_recall"":null,""precision"":0.0}"\nintraday,2018-02-21,0,[20],0,00:00,03:00,1044705,1044705,686012,358693,1.8344673895362171,8,"[1,521731,543,0,15,0,3,2,522275,5,127,3]","[358908,393,0,12,76,0,1,2,4,1,29,0,0,2,272,0,39,0,0,0,3,122,0,754,0,317,2,6459,87,520623,345,156254]","{""random_forest"":{""tp"":685004,""tn"":358692,""fp"":1,""fn"":1008,""rows"":1044705,""accuracy"":0.9990341771121991,""precision"":0.9999985401566412,""recall"":0.9985306379480242,""specificity"":0.9999972121005986,""f1"":0.9992640499716634,""balanced_accuracy"":0.9992639250243114,""roc_auc"":0.9999996092823272,""pr_auc"":0.9999997731679439,""predicted_positive"":685005,""predicted_rate"":0.6556922767671256},""logistic_regression"":{""tp"":74,""tn"":358693,""fp"":0,""fn"":685938,""rows"":1044705,""accuracy"":0.34341464815426365,""precision"":1.0,""recall"":0.00010786983318076069,""specificity"":1.0,""f1"":0.0002157163970697551,""balanced_accuracy"":0.5000539349165903,""roc_auc"":0.9495237831353035,""pr_auc"":0.9762139052091648,""predicted_positive"":74,""predicted_rate"":7.083339315883431e-05},""duration_rule"":{""tp"":665706,""tn"":15049,""fp"":343644,""fn"":20306,""rows"":1044705,""accuracy"":0.651624142700571,""precision"":0.6595393074751077,""recall"":0.97039993469502,""specificity"":0.04195509809224044,""f1"":0.785326083750845,""balanced_accuracy"":0.5061775163936302,""roc_auc"":0.5061775163936302,""pr_auc"":0.6594539668686336,""predicted_positive"":1009350,""predicted_rate"":0.9661579106063434}}","{""rows"":1044705,""attacks"":686012,""benign"":358693,""tp"":0,""fp"":82,""flags"":82,""flag_rate"":7.84910572841137e-05,""benign_fpr"":0.00022860775091791588,""attack_recall"":0.0,""precision"":0.0}"\nintraday,2018-02-21,0,[21],6,06:00,09:00,145,145,0,145,12.87489744099472,32,"[4,22,25,1,15,0,13,1,1,2,61,0]","[38,78,0,0,1,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,1,0,0,22,0,0,0,0,0,2,0,0]","{""random_forest"":{""tp"":0,""tn"":120,""fp"":25,""fn"":0,""rows"":145,""accuracy"":0.8275862068965517,""precision"":0.0,""recall"":null,""specificity"":0.8275862068965517,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":25,""predicted_rate"":0.1724137931034483},""logistic_regression"":{""tp"":0,""tn"":145,""fp"":0,""fn"":0,""rows"":145,""accuracy"":1.0,""precision"":null,""recall"":null,""specificity"":1.0,""f1"":null,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":0,""predicted_rate"":0.0},""duration_rule"":{""tp"":0,""tn"":115,""fp"":30,""fn"":0,""rows"":145,""accuracy"":0.7931034482758621,""precision"":0.0,""recall"":null,""specificity"":0.7931034482758621,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":30,""predicted_rate"":0.20689655172413793}}","{""rows"":145,""attacks"":0,""benign"":145,""tp"":0,""fp"":65,""flags"":65,""flag_rate"":0.4482758620689655,""benign_fpr"":0.4482758620689655,""attack_recall"":null,""precision"":0.0}"\nintraday,2018-02-21,0,[22],9,09:00,12:00,3725,3725,1730,1995,94.19667557451992,256,"[7,43,64,1,1664,1,34,3,5,10,175,1718]","[1662,233,10,9,7,2,3,0,1,0,0,2,9,1,8,1,1,2,0,10,0,6,3,47,6,1,34,85,59,1523,0,0]","{""random_forest"":{""tp"":1711,""tn"":1929,""fp"":66,""fn"":19,""rows"":3725,""accuracy"":0.9771812080536912,""precision"":0.9628587507034327,""recall"":0.9890173410404625,""specificity"":0.9669172932330827,""f1"":0.975762760193898,""balanced_accuracy"":0.9779673171367727,""roc_auc"":0.9991478696741855,""pr_auc"":0.9990376598449916,""predicted_positive"":1777,""predicted_rate"":0.4770469798657718},""logistic_regression"":{""tp"":1708,""tn"":1994,""fp"":1,""fn"":22,""rows"":3725,""accuracy"":0.9938255033557047,""precision"":0.9994148624926857,""recall"":0.9872832369942196,""specificity"":0.9994987468671679,""f1"":0.9933120093050305,""balanced_accuracy"":0.9933909919306938,""roc_auc"":0.9991228070175439,""pr_auc"":0.9990656771471047,""predicted_positive"":1709,""predicted_rate"":0.45879194630872483},""duration_rule"":{""tp"":0,""tn"":1924,""fp"":71,""fn"":1730,""rows"":3725,""accuracy"":0.516510067114094,""precision"":0.0,""recall"":0.0,""specificity"":0.9644110275689223,""f1"":0.0,""balanced_accuracy"":0.48220551378446114,""roc_auc"":0.48220551378446114,""pr_auc"":0.46442953020134226,""predicted_positive"":71,""predicted_rate"":0.019060402684563757}}","{""rows"":3725,""attacks"":1730,""benign"":1995,""tp"":1730,""fp"":634,""flags"":2364,""flag_rate"":0.6346308724832215,""benign_fpr"":0.31779448621553885,""attack_recall"":1.0,""precision"":0.7318104906937394}"\nintraday,2018-02-22,0,[23],0,00:00,03:00,202297,202297,81,202216,2.361799413681827,12,"[54031,7217,31004,20455,25888,1044,20128,7075,236,1211,16405,17603]","[171453,4118,880,1264,1540,13371,97,192,647,13,42,5939,14,4,40,4,422,2,3,21,12,28,399,92,14,5,48,27,1,1605,0,0]","{""random_forest"":{""tp"":6,""tn"":199543,""fp"":2673,""fn"":75,""rows"":202297,""accuracy"":0.9864160121010198,""precision"":0.0022396416573348264,""recall"":0.07407407407407407,""specificity"":0.9867814614076037,""f1"":0.004347826086956522,""balanced_accuracy"":0.5304277677408389,""roc_auc"":0.5107200490173812,""pr_auc"":0.000852695781671382,""predicted_positive"":2679,""predicted_rate"":0.013242905233394464},""logistic_regression"":{""tp"":1,""tn"":200129,""fp"":2087,""fn"":80,""rows"":202297,""accuracy"":0.9892880270097926,""precision"":0.0004789272030651341,""recall"":0.012345679012345678,""specificity"":0.989679352771294,""f1"":0.0009220839096357768,""balanced_accuracy"":0.5010125158918198,""roc_auc"":0.46483567015737237,""pr_auc"":0.0006334256336091404,""predicted_positive"":2088,""predicted_rate"":0.010321458054246974},""duration_rule"":{""tp"":39,""tn"":106216,""fp"":96000,""fn"":42,""rows"":202297,""accuracy"":0.525242588866864,""precision"":0.0004060850279573923,""recall"":0.48148148148148145,""specificity"":0.5252601178937374,""f1"":0.0008114856429463171,""balanced_accuracy"":0.5033707996876094,""roc_auc"":0.5033707996876093,""pr_auc"":0.0004031379564423075,""predicted_positive"":96039,""predicted_rate"":0.4747425814520235}}","{""rows"":202297,""attacks"":81,""benign"":202216,""tp"":37,""fp"":1519,""flags"":1556,""flag_rate"":0.007691661270310484,""benign_fpr"":0.007511769592910551,""attack_recall"":0.4567901234567901,""precision"":0.02377892030848329}"\nintraday,2018-02-22,0,[24],3,03:00,06:00,228910,228910,31,228879,2.316283773026146,8,"[58196,8161,34181,27125,30456,1252,23257,7218,317,1282,19702,17763]","[193634,4602,902,1221,1748,15478,113,238,618,32,67,7382,16,58,62,21,457,6,3,17,9,38,384,96,21,13,55,31,6,1582,0,0]","{""random_forest"":{""tp"":0,""tn"":226161,""fp"":2718,""fn"":31,""rows"":228910,""accuracy"":0.9879909134594382,""precision"":0.0,""recall"":0.0,""specificity"":0.9881247296606503,""f1"":0.0,""balanced_accuracy"":0.4940623648303252,""roc_auc"":0.8305998845142715,""pr_auc"":0.00048791027067639903,""predicted_positive"":2718,""predicted_rate"":0.01187366213795815},""logistic_regression"":{""tp"":2,""tn"":226668,""fp"":2211,""fn"":29,""rows"":228910,""accuracy"":0.9902144947796077,""precision"":0.0009037505648441031,""recall"":0.06451612903225806,""specificity"":0.9903398739071737,""f1"":0.0017825311942959,""balanced_accuracy"":0.5274280014697159,""roc_auc"":0.7352548867559121,""pr_auc"":0.0004620452484811368,""predicted_positive"":2213,""predicted_rate"":0.009667554934253636},""duration_rule"":{""tp"":11,""tn"":125102,""fp"":103777,""fn"":20,""rows"":228910,""accuracy"":0.5465597833209558,""precision"":0.00010598527768142753,""recall"":0.3548387096774194,""specificity"":0.5465857505494169,""f1"":0.00021190726167657172,""balanced_accuracy"":0.45071223011341816,""roc_auc"":0.4507122301134182,""pr_auc"":0.00012497826150221193,""predicted_positive"":103788,""predicted_rate"":0.45340089991699795}}","{""rows"":228910,""attacks"":31,""benign"":228879,""tp"":0,""fp"":1643,""flags"":1643,""flag_rate"":0.007177493337993098,""benign_fpr"":0.007178465477391985,""attack_recall"":0.0,""precision"":0.0}"\nintraday,2018-02-22,0,[25],6,06:00,09:00,74421,74421,0,74421,2.75287751495796,12,"[17468,4541,10302,9612,10480,301,5461,1803,60,392,7032,6969]","[60202,3153,327,890,464,5844,36,86,523,4,20,1947,4,0,18,3,116,0,0,2,3,9,115,18,4,3,7,16,2,605,0,0]","{""random_forest"":{""tp"":0,""tn"":73521,""fp"":900,""fn"":0,""rows"":74421,""accuracy"":0.987906639255049,""precision"":0.0,""recall"":null,""specificity"":0.987906639255049,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":900,""predicted_rate"":0.012093360744951022},""logistic_regression"":{""tp"":0,""tn"":73512,""fp"":909,""fn"":0,""rows"":74421,""accuracy"":0.9877857056475995,""precision"":0.0,""recall"":null,""specificity"":0.9877857056475995,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":909,""predicted_rate"":0.012214294352400533},""duration_rule"":{""tp"":0,""tn"":44140,""fp"":30281,""fn"":0,""rows"":74421,""accuracy"":0.5931121592023757,""precision"":0.0,""recall"":null,""specificity"":0.5931121592023757,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":30281,""predicted_rate"":0.4068878407976243}}","{""rows"":74421,""attacks"":0,""benign"":74421,""tp"":0,""fp"":1033,""flags"":1033,""flag_rate"":0.013880490721704894,""benign_fpr"":0.013880490721704894,""attack_recall"":null,""precision"":0.0}"\nintraday,2018-02-22,0,[26],9,09:00,12:00,419123,419123,250,418873,2.3041542595098554,8,"[108185,15219,59388,52538,58714,2041,38358,13074,407,2595,30300,38304]","[347835,9893,1796,2495,2938,37046,204,424,1576,48,91,9543,35,13,125,16,797,1,3,34,13,67,734,121,49,8,39,96,14,3069,0,0]","{""random_forest"":{""tp"":20,""tn"":413848,""fp"":5025,""fn"":230,""rows"":419123,""accuracy"":0.9874619145215128,""precision"":0.003964321110009911,""recall"":0.08,""specificity"":0.9880035237410862,""f1"":0.007554296506137866,""balanced_accuracy"":0.5340017618705432,""roc_auc"":0.6006061264392788,""pr_auc"":0.001352525979096928,""predicted_positive"":5045,""predicted_rate"":0.012037039246235592},""logistic_regression"":{""tp"":12,""tn"":414639,""fp"":4234,""fn"":238,""rows"":419123,""accuracy"":0.9893301011874795,""precision"":0.002826189354686764,""recall"":0.048,""specificity"":0.9898919242825391,""f1"":0.005338078291814947,""balanced_accuracy"":0.5189459621412695,""roc_auc"":0.6095773993549357,""pr_auc"":0.0012941087909108648,""predicted_positive"":4246,""predicted_rate"":0.010130677629240104},""duration_rule"":{""tp"":105,""tn"":231032,""fp"":187841,""fn"":145,""rows"":419123,""accuracy"":0.5514777284949764,""precision"":0.0005586711076585827,""recall"":0.42,""specificity"":0.5515561996118156,""f1"":0.001115857935344003,""balanced_accuracy"":0.4857780998059078,""roc_auc"":0.4857780998059077,""pr_auc"":0.000580602358914159,""predicted_positive"":187946,""predicted_rate"":0.4484268341274518}}","{""rows"":419123,""attacks"":250,""benign"":418873,""tp"":67,""fp"":3667,""flags"":3734,""flag_rate"":0.008909079196321843,""benign_fpr"":0.008754443470932717,""attack_recall"":0.268,""precision"":0.017943224424209963}"\nintraday,2018-02-22,0,[27],12,12:00,15:00,123815,123815,0,123815,2.3471486956947527,12,"[33796,4368,18668,12652,15019,728,12039,4765,123,865,9610,11182]","[105181,2498,540,807,969,8347,58,145,478,2,20,3083,9,0,27,3,261,1,1,15,7,11,236,44,8,3,20,15,1,1025,0,0]","{""random_forest"":{""tp"":0,""tn"":122167,""fp"":1648,""fn"":0,""rows"":123815,""accuracy"":0.9866898194887533,""precision"":0.0,""recall"":null,""specificity"":0.9866898194887533,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":1648,""predicted_rate"":0.013310180511246618},""logistic_regression"":{""tp"":0,""tn"":122390,""fp"":1425,""fn"":0,""rows"":123815,""accuracy"":0.9884908936720107,""precision"":0.0,""recall"":null,""specificity"":0.9884908936720107,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":1425,""predicted_rate"":0.011509106327989339},""duration_rule"":{""tp"":0,""tn"":64159,""fp"":59656,""fn"":0,""rows"":123815,""accuracy"":0.5181843879982232,""precision"":0.0,""recall"":null,""specificity"":0.5181843879982232,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":59656,""predicted_rate"":0.48181561200177686}}","{""rows"":123815,""attacks"":0,""benign"":123815,""tp"":0,""fp"":1019,""flags"":1019,""flag_rate"":0.008230020595242903,""benign_fpr"":0.008230020595242903,""attack_recall"":null,""precision"":0.0}"\ncapture_only,2018-02-22,0,[28],24,,,9,0,0,9,26.37461894259326,64,"[0,9,0,0,0,0,0,0,0,0,0,0]","[0,0,0,0,9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]","{""random_forest"":{""tp"":0,""tn"":9,""fp"":0,""fn"":0,""rows"":9,""accuracy"":1.0,""precision"":null,""recall"":null,""specificity"":1.0,""f1"":null,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":0,""predicted_rate"":0.0},""logistic_regression"":{""tp"":0,""tn"":9,""fp"":0,""fn"":0,""rows"":9,""accuracy"":1.0,""precision"":null,""recall"":null,""specificity"":1.0,""f1"":null,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":0,""predicted_rate"":0.0},""duration_rule"":{""tp"":0,""tn"":9,""fp"":0,""fn"":0,""rows"":9,""accuracy"":1.0,""precision"":null,""recall"":null,""specificity"":1.0,""f1"":null,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":0,""predicted_rate"":0.0}}","{""rows"":9,""attacks"":0,""benign"":9,""tp"":0,""fp"":8,""flags"":8,""flag_rate"":0.8888888888888888,""benign_fpr"":0.8888888888888888,""attack_recall"":null,""precision"":0.0}"\nintraday,2018-02-23,1,[29],0,00:00,03:00,123497,123497,111,123386,2.800460416385407,12,"[22988,6146,1255,18133,20395,324,18738,2002,24,1266,15889,16337]","[101674,3345,830,1201,340,9450,85,65,32,8,39,5392,13,8,7,6,236,2,0,26,9,6,275,0,3,5,15,1,0,424,0,0]","{""random_forest"":{""tp"":5,""tn"":122389,""fp"":997,""fn"":106,""rows"":123497,""accuracy"":0.9910686089540637,""precision"":0.00499001996007984,""recall"":0.04504504504504504,""specificity"":0.9919196667369069,""f1"":0.008984725965858042,""balanced_accuracy"":0.518482355890976,""roc_auc"":0.3520212625054342,""pr_auc"":0.0013324212534051249,""predicted_positive"":1002,""predicted_rate"":0.008113557414350147},""logistic_regression"":{""tp"":0,""tn"":123385,""fp"":1,""fn"":111,""rows"":123497,""accuracy"":0.9990930953788351,""precision"":0.0,""recall"":0.0,""specificity"":0.9999918953527953,""f1"":0.0,""balanced_accuracy"":0.49999594767639766,""roc_auc"":0.3279648807382907,""pr_auc"":0.03555075558853786,""predicted_positive"":1,""predicted_rate"":8.097362688972202e-06},""duration_rule"":{""tp"":34,""tn"":95436,""fp"":27950,""fn"":77,""rows"":123497,""accuracy"":0.7730552159161761,""precision"":0.0012149799885648942,""recall"":0.3063063063063063,""specificity"":0.7734751106284343,""f1"":0.002420359494571988,""balanced_accuracy"":0.5398907084673703,""roc_auc"":0.5398907084673703,""pr_auc"":0.0009956529595842505,""predicted_positive"":27984,""predicted_rate"":0.2265965974881981}}","{""rows"":123497,""attacks"":111,""benign"":123386,""tp"":75,""fp"":1702,""flags"":1777,""flag_rate"":0.014389013498303602,""benign_fpr"":0.01379410954241162,""attack_recall"":0.6756756756756757,""precision"":0.04220596510973551}"\nintraday,2018-02-23,1,[30],3,03:00,06:00,171752,171752,33,171719,2.5110008379418303,8,"[27749,7087,1401,35387,34748,353,22363,2167,32,1244,19733,19488]","[134903,4447,973,1282,433,21606,104,73,49,26,44,6538,31,8,4,7,272,4,0,21,7,35,325,1,24,12,16,1,0,506,0,0]","{""random_forest"":{""tp"":1,""tn"":170496,""fp"":1223,""fn"":32,""rows"":171752,""accuracy"":0.9926929526293726,""precision"":0.0008169934640522876,""recall"":0.030303030303030304,""specificity"":0.9928778993588362,""f1"":0.0015910898965791568,""balanced_accuracy"":0.5115904648309333,""roc_auc"":0.8651200243103294,""pr_auc"":0.0010763453608567711,""predicted_positive"":1224,""predicted_rate"":0.007126554567050166},""logistic_regression"":{""tp"":0,""tn"":171689,""fp"":30,""fn"":33,""rows"":171752,""accuracy"":0.999633192044343,""precision"":0.0,""recall"":0.0,""specificity"":0.9998252959777311,""f1"":0.0,""balanced_accuracy"":0.49991264798886553,""roc_auc"":0.7062836272154991,""pr_auc"":0.0006843151706038894,""predicted_positive"":30,""predicted_rate"":0.00017467045507475894},""duration_rule"":{""tp"":2,""tn"":138459,""fp"":33260,""fn"":31,""rows"":171752,""accuracy"":0.80616819600354,""precision"":6.01286753652817e-05,""recall"":0.06060606060606061,""specificity"":0.8063114739778359,""f1"":0.00012013815888271512,""balanced_accuracy"":0.4334587672919482,""roc_auc"":0.43345876729194827,""pr_auc"":0.00018413696572060133,""predicted_positive"":33262,""predicted_rate"":0.19366295588988774}}","{""rows"":171752,""attacks"":33,""benign"":171719,""tp"":0,""fp"":1804,""flags"":1804,""flag_rate"":0.010503516698495505,""benign_fpr"":0.01050553520577222,""attack_recall"":0.0,""precision"":0.0}"\nintraday,2018-02-23,1,[31],6,06:00,09:00,74913,74913,0,74913,3.0760414505060294,12,"[12900,4855,678,12097,14224,161,8988,1057,16,657,9489,9791]","[56458,4798,850,710,470,8097,59,377,19,18,46,2310,63,2,6,5,121,5,2,11,18,12,142,0,4,1,6,2,0,301,0,0]","{""random_forest"":{""tp"":0,""tn"":74288,""fp"":625,""fn"":0,""rows"":74913,""accuracy"":0.991656988773644,""precision"":0.0,""recall"":null,""specificity"":0.991656988773644,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":625,""predicted_rate"":0.008343011226355905},""logistic_regression"":{""tp"":0,""tn"":74910,""fp"":3,""fn"":0,""rows"":74913,""accuracy"":0.9999599535461134,""precision"":0.0,""recall"":null,""specificity"":0.9999599535461134,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":3,""predicted_rate"":4.004645388650835e-05},""duration_rule"":{""tp"":0,""tn"":60040,""fp"":14873,""fn"":0,""rows"":74913,""accuracy"":0.8014630304486537,""precision"":0.0,""recall"":null,""specificity"":0.8014630304486537,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":14873,""predicted_rate"":0.19853696955134623}}","{""rows"":74913,""attacks"":0,""benign"":74913,""tp"":0,""fp"":1287,""flags"":1287,""flag_rate"":0.017179928717312082,""benign_fpr"":0.017179928717312082,""attack_recall"":null,""precision"":0.0}"\nintraday,2018-02-23,1,[32],9,09:00,12:00,263902,263902,279,263623,2.8221632569059407,12,"[44396,12801,3085,44180,48889,682,34852,3858,419,2450,31654,36636]","[207582,10389,1642,2402,870,28861,224,363,124,68,108,8990,44,10,17,11,586,3,15,31,33,97,562,0,91,13,66,0,0,700,0,0]","{""random_forest"":{""tp"":3,""tn"":261429,""fp"":2194,""fn"":276,""rows"":263902,""accuracy"":0.9906404650211064,""precision"":0.0013654984069185253,""recall"":0.010752688172043012,""specificity"":0.9916775091702924,""f1"":0.0024232633279483036,""balanced_accuracy"":0.5012150986711678,""roc_auc"":0.6295787033881621,""pr_auc"":0.0023241886996228384,""predicted_positive"":2197,""predicted_rate"":0.008325060060173852},""logistic_regression"":{""tp"":0,""tn"":263619,""fp"":4,""fn"":279,""rows"":263902,""accuracy"":0.9989276322271146,""precision"":0.0,""recall"":0.0,""specificity"":0.9999848268170834,""f1"":0.0,""balanced_accuracy"":0.4999924134085417,""roc_auc"":0.23992452048493224,""pr_auc"":0.0010713593418610625,""predicted_positive"":4,""predicted_rate"":1.5157141666224583e-05},""duration_rule"":{""tp"":7,""tn"":209002,""fp"":54621,""fn"":272,""rows"":263902,""accuracy"":0.7919947556289835,""precision"":0.00012813941568426448,""recall"":0.025089605734767026,""specificity"":0.792806393979281,""f1"":0.0002549765967909374,""balanced_accuracy"":0.408947999857024,""roc_auc"":0.408947999857024,""pr_auc"":0.0010339006007218732,""predicted_positive"":54628,""predicted_rate"":0.20700108373562914}}","{""rows"":263902,""attacks"":279,""benign"":263623,""tp"":187,""fp"":3553,""flags"":3740,""flag_rate"":0.014171927457919986,""benign_fpr"":0.013477579725592987,""attack_recall"":0.6702508960573477,""precision"":0.05}"\nintraday,2018-02-23,1,[33],12,12:00,15:00,75402,75402,0,75402,2.7068829789398876,12,"[13677,3382,710,11560,12833,188,10876,1226,15,700,9308,10927]","[61466,2053,428,719,142,6927,71,95,29,10,25,2792,10,1,1,2,174,0,1,12,4,8,177,0,5,2,16,1,0,231,0,0]","{""random_forest"":{""tp"":0,""tn"":74771,""fp"":631,""fn"":0,""rows"":75402,""accuracy"":0.9916315217102994,""precision"":0.0,""recall"":null,""specificity"":0.9916315217102994,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":631,""predicted_rate"":0.008368478289700538},""logistic_regression"":{""tp"":0,""tn"":75402,""fp"":0,""fn"":0,""rows"":75402,""accuracy"":1.0,""precision"":null,""recall"":null,""specificity"":1.0,""f1"":null,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":0,""predicted_rate"":0.0},""duration_rule"":{""tp"":0,""tn"":58635,""fp"":16767,""fn"":0,""rows"":75402,""accuracy"":0.7776318930532347,""precision"":0.0,""recall"":null,""specificity"":0.7776318930532347,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":16767,""predicted_rate"":0.22236810694676534}}","{""rows"":75402,""attacks"":0,""benign"":75402,""tp"":0,""fp"":941,""flags"":941,""flag_rate"":0.01247977507227925,""benign_fpr"":0.01247977507227925,""attack_recall"":null,""precision"":0.0}"\nintraday,2018-02-28,1,[34],0,00:00,03:00,100780,100780,21476,79304,4.012043669571349,16,"[24771,4624,13718,23310,4695,368,11607,4446,4210,1670,5147,2214]","[79470,5108,1581,2188,9764,145,274,281,431,125,146,141,3,3,0,17,532,0,0,3,5,20,263,4,36,16,59,0,0,165,0,0]","{""random_forest"":{""tp"":220,""tn"":78421,""fp"":883,""fn"":21256,""rows"":100780,""accuracy"":0.7803234768803334,""precision"":0.19945602901178605,""recall"":0.010243993294840752,""specificity"":0.9888656309896096,""f1"":0.01948713406262456,""balanced_accuracy"":0.49955481214222514,""roc_auc"":0.39587064291380075,""pr_auc"":0.1784106015216903,""predicted_positive"":1103,""predicted_rate"":0.010944631871403055},""logistic_regression"":{""tp"":262,""tn"":79078,""fp"":226,""fn"":21214,""rows"":100780,""accuracy"":0.7872593768604882,""precision"":0.5368852459016393,""recall"":0.012199664742037623,""specificity"":0.9971502067991527,""f1"":0.023857220906938627,""balanced_accuracy"":0.5046749357705952,""roc_auc"":0.4629796372579082,""pr_auc"":0.21489501202036557,""predicted_positive"":488,""predicted_rate"":0.0048422306013097835},""duration_rule"":{""tp"":10382,""tn"":46364,""fp"":32940,""fn"":11094,""rows"":100780,""accuracy"":0.5630680690613217,""precision"":0.23964729236877338,""recall"":0.4834233563047122,""specificity"":0.5846363361242812,""f1"":0.32044198895027626,""balanced_accuracy"":0.5340298462144968,""roc_auc"":0.5340298462144967,""pr_auc"":0.225932463756517,""predicted_positive"":43322,""predicted_rate"":0.4298670371105378}}","{""rows"":100780,""attacks"":21476,""benign"":79304,""tp"":927,""fp"":3400,""flags"":4327,""flag_rate"":0.04293510617185949,""benign_fpr"":0.04287299505699586,""attack_recall"":0.04316446265598808,""precision"":0.2142361913565981}"\nintraday,2018-02-28,1,[35],3,03:00,06:00,94037,94037,0,94037,3.209842185796914,16,"[25369,3715,3618,21820,5392,338,13273,4665,1826,2481,8975,2565]","[85723,4146,1020,630,420,210,259,36,358,8,484,86,0,2,3,4,350,0,0,10,15,15,174,0,16,2,15,0,0,51,0,0]","{""random_forest"":{""tp"":0,""tn"":93389,""fp"":648,""fn"":0,""rows"":94037,""accuracy"":0.9931090953560833,""precision"":0.0,""recall"":null,""specificity"":0.9931090953560833,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":648,""predicted_rate"":0.0068909046439167565},""logistic_regression"":{""tp"":0,""tn"":93505,""fp"":532,""fn"":0,""rows"":94037,""accuracy"":0.9943426523602412,""precision"":0.0,""recall"":null,""specificity"":0.9943426523602412,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":532,""predicted_rate"":0.005657347639758818},""duration_rule"":{""tp"":0,""tn"":60156,""fp"":33881,""fn"":0,""rows"":94037,""accuracy"":0.6397056477769388,""precision"":0.0,""recall"":null,""specificity"":0.6397056477769388,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":33881,""predicted_rate"":0.3602943522230611}}","{""rows"":94037,""attacks"":0,""benign"":94037,""tp"":0,""fp"":2452,""flags"":2452,""flag_rate"":0.02607484288099365,""benign_fpr"":0.02607484288099365,""attack_recall"":null,""precision"":0.0}"\nintraday,2018-02-28,1,[36],6,06:00,09:00,31948,31948,0,31948,3.644009399954314,16,"[10797,1375,1928,8137,1420,110,2959,1406,1473,460,982,901]","[28772,1754,294,330,158,8,39,98,156,0,33,1,0,4,3,0,140,0,0,0,1,8,86,0,2,0,2,0,0,59,0,0]","{""random_forest"":{""tp"":0,""tn"":31650,""fp"":298,""fn"":0,""rows"":31948,""accuracy"":0.9906723425566546,""precision"":0.0,""recall"":null,""specificity"":0.9906723425566546,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":298,""predicted_rate"":0.009327657443345436},""logistic_regression"":{""tp"":0,""tn"":31745,""fp"":203,""fn"":0,""rows"":31948,""accuracy"":0.9936459246275198,""precision"":0.0,""recall"":null,""specificity"":0.9936459246275198,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":203,""predicted_rate"":0.006354075372480281},""duration_rule"":{""tp"":0,""tn"":18115,""fp"":13833,""fn"":0,""rows"":31948,""accuracy"":0.5670151496181295,""precision"":0.0,""recall"":null,""specificity"":0.5670151496181295,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":13833,""predicted_rate"":0.4329848503818705}}","{""rows"":31948,""attacks"":0,""benign"":31948,""tp"":0,""fp"":933,""flags"":933,""flag_rate"":0.029203706022286214,""benign_fpr"":0.029203706022286214,""attack_recall"":null,""precision"":0.0}"\nintraday,2018-02-28,1,[37],9,09:00,12:00,180108,180108,33295,146813,3.815580174050861,16,"[48151,7624,19956,46861,7469,792,19305,7954,8291,3073,6786,3846]","[150194,9182,2559,2619,11137,177,490,541,648,147,208,192,2,0,3,11,1002,0,1,12,9,22,440,13,42,5,88,0,0,364,0,0]","{""random_forest"":{""tp"":351,""tn"":145166,""fp"":1647,""fn"":32944,""rows"":180108,""accuracy"":0.8079430119705954,""precision"":0.17567567567567569,""recall"":0.010542123441958252,""specificity"":0.9887816474017969,""f1"":0.019890629869945883,""balanced_accuracy"":0.49966188542187756,""roc_auc"":0.42208757170048744,""pr_auc"":0.16046950818346628,""predicted_positive"":1998,""predicted_rate"":0.011093343993603838},""logistic_regression"":{""tp"":325,""tn"":146328,""fp"":485,""fn"":32970,""rows"":180108,""accuracy"":0.8142503386856775,""precision"":0.4012345679012346,""recall"":0.009761225409220604,""specificity"":0.9966964778323445,""f1"":0.019058789033866003,""balanced_accuracy"":0.5032288516207826,""roc_auc"":0.47700558253476893,""pr_auc"":0.18584225576477784,""predicted_positive"":810,""predicted_rate"":0.004497301619028583},""duration_rule"":{""tp"":16387,""tn"":86124,""fp"":60689,""fn"":16908,""rows"":180108,""accuracy"":0.5691640571212828,""precision"":0.21260833463075407,""recall"":0.49217600240276316,""specificity"":0.5866238003446561,""f1"":0.2969439436083754,""balanced_accuracy"":0.5393999013737096,""roc_auc"":0.5393999013737096,""pr_auc"":0.1985177273451294,""predicted_positive"":77076,""predicted_rate"":0.42794323405956425}}","{""rows"":180108,""attacks"":33295,""benign"":146813,""tp"":1149,""fp"":5391,""flags"":6540,""flag_rate"":0.03631154640549004,""benign_fpr"":0.036720181455320716,""attack_recall"":0.03450968613905992,""precision"":0.1756880733944954}"\nintraday,2018-02-28,1,[38],12,12:00,15:00,34917,34917,1862,33055,3.713102508409444,16,"[7663,1825,2159,10198,1842,106,3817,1283,1940,700,2651,733]","[30719,2275,511,448,159,25,46,164,126,18,35,32,0,0,5,0,152,0,0,2,2,2,77,2,30,4,37,0,0,46,0,0]","{""random_forest"":{""tp"":6,""tn"":32707,""fp"":348,""fn"":1856,""rows"":34917,""accuracy"":0.9368788842111293,""precision"":0.01694915254237288,""recall"":0.00322234156820623,""specificity"":0.9894720919679323,""f1"":0.005415162454873646,""balanced_accuracy"":0.49634721676806925,""roc_auc"":0.4749142910434242,""pr_auc"":0.056196513928580924,""predicted_positive"":354,""predicted_rate"":0.010138328035054558},""logistic_regression"":{""tp"":13,""tn"":32937,""fp"":118,""fn"":1849,""rows"":34917,""accuracy"":0.943666408912564,""precision"":0.09923664122137404,""recall"":0.006981740064446832,""specificity"":0.996430192104069,""f1"":0.013045659809332665,""balanced_accuracy"":0.5017059660842579,""roc_auc"":0.48450722772529786,""pr_auc"":0.05413939893009304,""predicted_positive"":131,""predicted_rate"":0.0037517541598648223},""duration_rule"":{""tp"":774,""tn"":22683,""fp"":10372,""fn"":1088,""rows"":34917,""accuracy"":0.6717931093736575,""precision"":0.0694419522698726,""recall"":0.41568206229860366,""specificity"":0.6862199364695205,""f1"":0.11900369003690037,""balanced_accuracy"":0.5509509993840621,""roc_auc"":0.5509509993840621,""pr_auc"":0.06002538099777212,""predicted_positive"":11146,""predicted_rate"":0.3192141363805596}}","{""rows"":34917,""attacks"":1862,""benign"":33055,""tp"":131,""fp"":1162,""flags"":1293,""flag_rate"":0.03703067273820775,""benign_fpr"":0.03515353199213432,""attack_recall"":0.07035445757250268,""precision"":0.10131477184841454}"\nintraday,2018-03-01,2,[39],0,00:00,03:00,55756,55756,24213,31543,3.6601133625591937,16,"[13425,2414,5655,12745,3344,161,5916,2021,1589,1086,5454,1946]","[45409,2526,782,841,4954,518,65,79,37,51,37,56,5,13,1,7,149,0,0,4,5,12,132,1,33,1,31,1,0,6,0,0]","{""random_forest"":{""tp"":171,""tn"":31339,""fp"":204,""fn"":24042,""rows"":55756,""accuracy"":0.565140971375278,""precision"":0.456,""recall"":0.007062321893197869,""specificity"":0.993532637986241,""f1"":0.013909224011713031,""balanced_accuracy"":0.5002974799397194,""roc_auc"":0.5874467389493931,""pr_auc"":0.5765823685076248,""predicted_positive"":375,""predicted_rate"":0.006725733553339551},""logistic_regression"":{""tp"":0,""tn"":31542,""fp"":1,""fn"":24213,""rows"":55756,""accuracy"":0.5657149006384963,""precision"":0.0,""recall"":0.0,""specificity"":0.9999682972450306,""f1"":0.0,""balanced_accuracy"":0.4999841486225153,""roc_auc"":0.578089248496478,""pr_auc"":0.49254327068599296,""predicted_positive"":1,""predicted_rate"":1.7935289475572134e-05},""duration_rule"":{""tp"":10967,""tn"":21304,""fp"":10239,""fn"":13246,""rows"":55756,""accuracy"":0.5787897266661884,""precision"":0.5171649533150995,""recall"":0.45293850410936276,""specificity"":0.6753954918682433,""f1"":0.4829256478566239,""balanced_accuracy"":0.5641669979888031,""roc_auc"":0.5641669979888031,""pr_auc"":0.4718147647257581,""predicted_positive"":21206,""predicted_rate"":0.3803357486189827}}","{""rows"":55756,""attacks"":24213,""benign"":31543,""tp"":1107,""fp"":866,""flags"":1973,""flag_rate"":0.035386326135303826,""benign_fpr"":0.027454585803506324,""attack_recall"":0.04571924172964936,""precision"":0.5610745058286872}"\nintraday,2018-03-01,2,[40],3,03:00,06:00,60647,60647,13902,46745,3.5709036696746437,16,"[14169,2875,2798,14975,4498,178,7081,2233,1366,1158,6408,2908]","[52539,2855,933,684,2180,878,47,51,26,30,30,44,0,1,0,0,205,1,0,7,6,12,85,9,11,1,8,0,0,4,0,0]","{""random_forest"":{""tp"":115,""tn"":46511,""fp"":234,""fn"":13787,""rows"":60647,""accuracy"":0.7688096690685442,""precision"":0.32951289398280803,""recall"":0.008272191051647245,""specificity"":0.9949941170178629,""f1"":0.016139218300470142,""balanced_accuracy"":0.5016331540347551,""roc_auc"":0.5019807509433846,""pr_auc"":0.25516299769894196,""predicted_positive"":349,""predicted_rate"":0.005754612759081241},""logistic_regression"":{""tp"":0,""tn"":46730,""fp"":15,""fn"":13902,""rows"":60647,""accuracy"":0.7705245106930269,""precision"":0.0,""recall"":0.0,""specificity"":0.9996791100652476,""f1"":0.0,""balanced_accuracy"":0.4998395550326238,""roc_auc"":0.5093909378854309,""pr_auc"":0.23937813485731743,""predicted_positive"":15,""predicted_rate"":0.00024733292660807625},""duration_rule"":{""tp"":5456,""tn"":32886,""fp"":13859,""fn"":8446,""rows"":60647,""accuracy"":0.6322159381337906,""precision"":0.2824747605487963,""recall"":0.39246151632858584,""specificity"":0.7035190929511178,""f1"":0.3285064876418701,""balanced_accuracy"":0.5479903046398518,""roc_auc"":0.5479903046398518,""pr_auc"":0.25012539939165557,""predicted_positive"":19315,""predicted_rate"":0.31848236516233286}}","{""rows"":60647,""attacks"":13902,""benign"":46745,""tp"":839,""fp"":1560,""flags"":2399,""flag_rate"":0.03955677939551833,""benign_fpr"":0.033372553214247515,""attack_recall"":0.06035102862897425,""precision"":0.34972905377240515}"\nintraday,2018-03-01,2,[41],6,06:00,09:00,19227,19227,0,19227,3.3961587884260687,16,"[6131,940,391,4336,1626,67,1560,751,807,340,1442,836]","[16807,1007,226,229,113,626,40,99,2,1,4,2,1,1,0,1,41,1,0,0,3,0,20,0,3,0,0,0,0,0,0,0]","{""random_forest"":{""tp"":0,""tn"":19159,""fp"":68,""fn"":0,""rows"":19227,""accuracy"":0.9964633068081344,""precision"":0.0,""recall"":null,""specificity"":0.9964633068081344,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":68,""predicted_rate"":0.0035366931918656055},""logistic_regression"":{""tp"":0,""tn"":19225,""fp"":2,""fn"":0,""rows"":19227,""accuracy"":0.9998959796120039,""precision"":0.0,""recall"":null,""specificity"":0.9998959796120039,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":2,""predicted_rate"":0.00010402038799604722},""duration_rule"":{""tp"":0,""tn"":11911,""fp"":7316,""fn"":0,""rows"":19227,""accuracy"":0.6194934207104592,""precision"":0.0,""recall"":null,""specificity"":0.6194934207104592,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":7316,""predicted_rate"":0.38050657928954074}}","{""rows"":19227,""attacks"":0,""benign"":19227,""tp"":0,""fp"":648,""flags"":648,""flag_rate"":0.0337026057107193,""benign_fpr"":0.0337026057107193,""attack_recall"":null,""precision"":0.0}"\nintraday,2018-03-01,2,[42],9,09:00,12:00,81607,81607,27136,54471,3.936902634452723,16,"[18517,3981,10666,16892,6089,201,8417,3018,2394,1436,6994,3002]","[62774,3954,1151,1274,9453,1941,115,147,42,108,53,114,1,4,3,0,220,1,2,5,8,17,121,15,24,2,44,0,0,14,0,0]","{""random_forest"":{""tp"":129,""tn"":54127,""fp"":344,""fn"":27007,""rows"":81607,""accuracy"":0.6648449275184727,""precision"":0.2727272727272727,""recall"":0.0047538325471698116,""specificity"":0.9936847129665326,""f1"":0.00934477887645333,""balanced_accuracy"":0.4992192727568512,""roc_auc"":0.6747028256856773,""pr_auc"":0.577301782958321,""predicted_positive"":473,""predicted_rate"":0.005796071415442303},""logistic_regression"":{""tp"":0,""tn"":54471,""fp"":0,""fn"":27136,""rows"":81607,""accuracy"":0.6674795054345828,""precision"":null,""recall"":0.0,""specificity"":1.0,""f1"":0.0,""balanced_accuracy"":0.5,""roc_auc"":0.6280727386573712,""pr_auc"":0.42756211291305046,""predicted_positive"":0,""predicted_rate"":0.0},""duration_rule"":{""tp"":14894,""tn"":36238,""fp"":18233,""fn"":12242,""rows"":81607,""accuracy"":0.6265638977048538,""precision"":0.44960304283514957,""recall"":0.5488649764150944,""specificity"":0.6652714288336913,""f1"":0.4942999850654631,""balanced_accuracy"":0.6070682026243928,""roc_auc"":0.607068202624393,""pr_auc"":0.39678300466010297,""predicted_positive"":33127,""predicted_rate"":0.4059333145440955}}","{""rows"":81607,""attacks"":27136,""benign"":54471,""tp"":1504,""fp"":1769,""flags"":3273,""flag_rate"":0.04010685357873712,""benign_fpr"":0.03247599640175506,""attack_recall"":0.055424528301886794,""precision"":0.45951726245035135}"\nintraday,2018-03-01,2,[43],12,12:00,15:00,14236,14236,0,14236,2.9425680273898696,16,"[1968,1034,148,3923,1745,16,1225,349,143,239,3006,440]","[12307,599,115,80,157,919,2,3,4,0,5,1,0,0,0,0,25,1,0,0,0,1,10,0,0,1,4,0,0,2,0,0]","{""random_forest"":{""tp"":0,""tn"":14192,""fp"":44,""fn"":0,""rows"":14236,""accuracy"":0.9969092441697106,""precision"":0.0,""recall"":null,""specificity"":0.9969092441697106,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":44,""predicted_rate"":0.0030907558302894073},""logistic_regression"":{""tp"":0,""tn"":14236,""fp"":0,""fn"":0,""rows"":14236,""accuracy"":1.0,""precision"":null,""recall"":null,""specificity"":1.0,""f1"":null,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":0,""predicted_rate"":0.0},""duration_rule"":{""tp"":0,""tn"":11771,""fp"":2465,""fn"":0,""rows"":14236,""accuracy"":0.8268474290531048,""precision"":0.0,""recall"":null,""specificity"":0.8268474290531048,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":2465,""predicted_rate"":0.1731525709468952}}","{""rows"":14236,""attacks"":0,""benign"":14236,""tp"":0,""fp"":363,""flags"":363,""flag_rate"":0.02549873559988761,""benign_fpr"":0.02549873559988761,""attack_recall"":null,""precision"":0.0}"\nintraday,2018-03-02,2,[44],0,00:00,03:00,162217,162217,36880,125337,2.3667197198868437,12,"[24030,4922,1796,29743,15754,337,13294,3680,36837,1504,17447,12873]","[110141,2468,787,812,449,43022,82,35,17,1,23,3737,3,2,4,0,280,0,1,23,8,23,246,1,14,4,24,0,0,10,0,0]","{""random_forest"":{""tp"":0,""tn"":124703,""fp"":634,""fn"":36880,""rows"":162217,""accuracy"":0.7687418704574737,""precision"":0.0,""recall"":0.0,""specificity"":0.994941637345716,""f1"":0.0,""balanced_accuracy"":0.497470818672858,""roc_auc"":0.9129311954147324,""pr_auc"":0.7701694022941614,""predicted_positive"":634,""predicted_rate"":0.003908344994667637},""logistic_regression"":{""tp"":3,""tn"":125337,""fp"":0,""fn"":36877,""rows"":162217,""accuracy"":0.7726687091981728,""precision"":1.0,""recall"":8.134490238611714e-05,""specificity"":1.0,""f1"":0.00016267657186237562,""balanced_accuracy"":0.5000406724511931,""roc_auc"":0.9715695028502501,""pr_auc"":0.871392368415642,""predicted_positive"":3,""predicted_rate"":1.8493746031550332e-05},""duration_rule"":{""tp"":36673,""tn"":95366,""fp"":29971,""fn"":207,""rows"":162217,""accuracy"":0.8139652440866247,""precision"":0.5502820959126102,""recall"":0.9943872017353579,""specificity"":0.7608766764802094,""f1"":0.7084927166647348,""balanced_accuracy"":0.8776319391077836,""roc_auc"":0.8776319391077836,""pr_auc"":0.5484695419957853,""predicted_positive"":66644,""predicted_rate"":0.41083240350888006}}","{""rows"":162217,""attacks"":36880,""benign"":125337,""tp"":92,""fp"":2232,""flags"":2324,""flag_rate"":0.014326488592440989,""benign_fpr"":0.017807989659876972,""attack_recall"":0.002494577006507592,""precision"":0.039586919104991396}"\nintraday,2018-03-02,2,[45],3,03:00,06:00,201728,201728,33879,167849,2.278970068147466,8,"[29523,5613,1987,47304,26764,347,16259,4188,33498,1268,20877,14100]","[146161,3214,867,1014,889,44152,76,47,14,13,30,4497,4,7,10,1,274,2,3,9,8,36,289,1,38,11,51,1,0,9,0,0]","{""random_forest"":{""tp"":1,""tn"":167118,""fp"":731,""fn"":33878,""rows"":201728,""accuracy"":0.828437301713198,""precision"":0.001366120218579235,""recall"":2.951680982319431e-05,""specificity"":0.9956448951140608,""f1"":5.778509722342608e-05,""balanced_accuracy"":0.49783720596194203,""roc_auc"":0.7750229602010034,""pr_auc"":0.524918668453082,""predicted_positive"":732,""predicted_rate"":0.0036286484771573604},""logistic_regression"":{""tp"":159,""tn"":167819,""fp"":30,""fn"":33720,""rows"":201728,""accuracy"":0.832695510786802,""precision"":0.8412698412698413,""recall"":0.0046931727618878954,""specificity"":0.9998212679253377,""f1"":0.009334272631208172,""balanced_accuracy"":0.5022572203436128,""roc_auc"":0.9753458866282624,""pr_auc"":0.8269895619452455,""predicted_positive"":189,""predicted_rate"":0.0009369051395939086},""duration_rule"":{""tp"":32966,""tn"":131305,""fp"":36544,""fn"":913,""rows"":201728,""accuracy"":0.8143192814086294,""precision"":0.47426269601496185,""recall"":0.9730511526314236,""specificity"":0.7822805021179751,""f1"":0.6377080733927207,""balanced_accuracy"":0.8776658273746993,""roc_auc"":0.8776658273746993,""pr_auc"":0.4660077592637903,""predicted_positive"":69510,""predicted_rate"":0.3445728902284264}}","{""rows"":201728,""attacks"":33879,""benign"":167849,""tp"":470,""fp"":2035,""flags"":2505,""flag_rate"":0.01241771097715736,""benign_fpr"":0.01212399239792909,""attack_recall"":0.013872900616901326,""precision"":0.18762475049900199}"\nintraday,2018-03-02,2,[46],6,06:00,09:00,28429,28429,0,28429,3.809510501281392,16,"[8239,1693,1226,3124,2686,228,2655,989,165,279,2957,4188]","[22828,2807,283,499,279,221,24,40,0,0,1,564,1,2,2,2,328,1,27,17,98,132,89,0,126,3,44,2,0,9,0,0]","{""random_forest"":{""tp"":0,""tn"":27553,""fp"":876,""fn"":0,""rows"":28429,""accuracy"":0.9691863941749622,""precision"":0.0,""recall"":null,""specificity"":0.9691863941749622,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":876,""predicted_rate"":0.030813605825037813},""logistic_regression"":{""tp"":0,""tn"":28429,""fp"":0,""fn"":0,""rows"":28429,""accuracy"":1.0,""precision"":null,""recall"":null,""specificity"":1.0,""f1"":null,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":0,""predicted_rate"":0.0},""duration_rule"":{""tp"":0,""tn"":17761,""fp"":10668,""fn"":0,""rows"":28429,""accuracy"":0.6247493756375532,""precision"":0.0,""recall"":null,""specificity"":0.6247493756375532,""f1"":0.0,""balanced_accuracy"":null,""roc_auc"":null,""pr_auc"":null,""predicted_positive"":10668,""predicted_rate"":0.37525062436244677}}","{""rows"":28429,""attacks"":0,""benign"":28429,""tp"":0,""fp"":904,""flags"":904,""flag_rate"":0.031798515600267335,""benign_fpr"":0.031798515600267335,""attack_recall"":null,""precision"":0.0}"\nintraday,2018-03-02,2,[47],9,09:00,12:00,277973,277973,56595,221378,2.628968410047066,12,"[53730,10169,4237,37486,23441,702,27088,7942,56694,3087,26000,27397]","[194013,8061,1659,2256,1659,62405,162,122,83,11,61,6018,11,9,10,4,622,2,41,28,28,72,529,1,62,3,20,1,0,20,0,0]","{""random_forest"":{""tp"":1,""tn"":219950,""fp"":1428,""fn"":56594,""rows"":277973,""accuracy"":0.7912674972029657,""precision"":0.0006997900629811056,""recall"":1.7669405424507466e-05,""specificity"":0.9935494945297184,""f1"":3.446849579484351e-05,""balanced_accuracy"":0.49678358196757144,""roc_auc"":0.9328428772334671,""pr_auc"":0.7878814742518745,""predicted_positive"":1429,""predicted_rate"":0.005140787054857846},""logistic_regression"":{""tp"":15,""tn"":221375,""fp"":3,""fn"":56580,""rows"":277973,""accuracy"":0.7964442589747925,""precision"":0.8333333333333334,""recall"":0.00026504108136761196,""specificity"":0.9999864485179196,""f1"":0.000529913624079275,""balanced_accuracy"":0.5001257447996437,""roc_auc"":0.9548064135805653,""pr_auc"":0.7629313855585642,""predicted_positive"":18,""predicted_rate"":6.475449054404565e-05},""duration_rule"":{""tp"":55722,""tn"":153567,""fp"":67811,""fn"":873,""rows"":277973,""accuracy"":0.7529112539707093,""precision"":0.45106975464046045,""recall"":0.984574609064405,""specificity"":0.6936868162148,""f1"":0.6186933736009949,""balanced_accuracy"":0.8391307126396025,""roc_auc"":0.8391307126396025,""pr_auc"":0.44725242012729466,""predicted_positive"":123533,""predicted_rate"":0.44440647113208837}}","{""rows"":277973,""attacks"":56595,""benign"":221378,""tp"":425,""fp"":4826,""flags"":5251,""flag_rate"":0.018890323880376873,""benign_fpr"":0.021799817506707984,""attack_recall"":0.007509497305415673,""precision"":0.08093696438773566}"\nintraday,2018-03-02,2,[48],12,12:00,15:00,78289,78289,15920,62369,2.4428727987801397,12,"[12534,2483,805,14234,7646,154,6891,1821,15992,738,8434,6557]","[54815,1350,362,530,163,18739,25,48,18,3,6,1924,1,0,5,0,126,0,4,9,7,7,127,1,0,2,12,0,0,5,0,0]","{""random_forest"":{""tp"":0,""tn"":62069,""fp"":300,""fn"":15920,""rows"":78289,""accuracy"":0.792818914534609,""precision"":0.0,""recall"":0.0,""specificity"":0.9951899180682711,""f1"":0.0,""balanced_accuracy"":0.49759495903413553,""roc_auc"":0.9175543738671229,""pr_auc"":0.7550621516915319,""predicted_positive"":300,""predicted_rate"":0.0038319559580528555},""logistic_regression"":{""tp"":0,""tn"":62369,""fp"":0,""fn"":15920,""rows"":78289,""accuracy"":0.7966508704926618,""precision"":null,""recall"":0.0,""specificity"":1.0,""f1"":0.0,""balanced_accuracy"":0.5,""roc_auc"":0.9741655922874651,""pr_auc"":0.8604551255118941,""predicted_positive"":0,""predicted_rate"":0.0},""duration_rule"":{""tp"":15893,""tn"":46892,""fp"":15477,""fn"":27,""rows"":78289,""accuracy"":0.8019645160878285,""precision"":0.506630538731272,""recall"":0.9983040201005026,""specificity"":0.7518478731421059,""f1"":0.6721505603721717,""balanced_accuracy"":0.8750759466213043,""roc_auc"":0.8750759466213043,""pr_auc"":0.5061161795573369,""predicted_positive"":31370,""predicted_rate"":0.4006948613470602}}","{""rows"":78289,""attacks"":15920,""benign"":62369,""tp"":0,""fp"":1402,""flags"":1402,""flag_rate"":0.017908007510633678,""benign_fpr"":0.022479116227613078,""attack_recall"":0.0,""precision"":0.0}"\ndaily,2018-02-14,0,"[0,1,2,3,4,5]",,,,1048575,1048570,380949,667626,3.5024115854320548,8,"[188487,29277,397818,41080,48922,3746,66563,24185,59,3536,126840,118062]","[602316,17620,3655,3983,5321,3750,309,658,93562,46,128,20551,123,22,227,25,1472,6,45,133,64,152,1187,20556,120,953,174007,152,8490,5083,9729,74130]","{""random_forest"":{""tp"":287197,""tn"":658544,""fp"":9082,""fn"":93752,""rows"":1048575,""accuracy"":0.9019297618196124,""precision"":0.9693464605996375,""recall"":0.7538988158519907,""specificity"":0.9863965753281029,""f1"":0.8481545358431725,""balanced_accuracy"":0.8701476955900468,""roc_auc"":0.9832948558649439,""pr_auc"":0.9665730386605506,""predicted_positive"":296279,""predicted_rate"":0.28255394225496505},""logistic_regression"":{""tp"":243098,""tn"":660291,""fp"":7335,""fn"":137851,""rows"":1048575,""accuracy"":0.8615397086522185,""precision"":0.970710729017342,""recall"":0.6381379134739821,""specificity"":0.98901330984713,""f1"":0.7700504607353393,""balanced_accuracy"":0.813575611660556,""roc_auc"":0.8650399175827749,""pr_auc"":0.8259938715417812,""predicted_positive"":250433,""predicted_rate"":0.23883174784827027},""duration_rule"":{""tp"":287129,""tn"":332987,""fp"":334639,""fn"":93820,""rows"":1048575,""accuracy"":0.5913892663853325,""precision"":0.46179443136346676,""recall"":0.7537203142677892,""specificity"":0.4987627803590633,""f1"":0.5727019687509038,""balanced_accuracy"":0.6262415473134262,""roc_auc"":0.6262415473134262,""pr_auc"":0.43753765362849584,""predicted_positive"":621768,""predicted_rate"":0.5929647378585223}}","{""rows"":1048575,""attacks"":380949,""benign"":667626,""tp"":0,""fp"":5237,""flags"":5237,""flag_rate"":0.004994397158047827,""benign_fpr"":0.007844212178674887,""attack_recall"":0.0,""precision"":0.0}"\ndaily,2018-02-15,0,"[6,7,8,9,10]",,,,1048575,1048575,52498,996077,2.7439260168699504,12,"[312238,46957,165220,51877,97585,6152,104426,40407,9039,13087,107426,94161]","[877942,30609,18204,4737,10995,6748,2175,3876,4505,752,3081,29851,847,835,3373,2219,5014,822,6298,888,2645,9543,3719,335,1170,1059,6875,561,48,8849,0,0]","{""random_forest"":{""tp"":31487,""tn"":979738,""fp"":16339,""fn"":21011,""rows"":1048575,""accuracy"":0.964380230312567,""precision"":0.6583657424831681,""recall"":0.5997752295325536,""specificity"":0.9835966496566029,""f1"":0.627706231808939,""balanced_accuracy"":0.7916859395945782,""roc_auc"":0.9672345691522293,""pr_auc"":0.5353383853333865,""predicted_positive"":47826,""predicted_rate"":0.0456104713539804},""logistic_regression"":{""tp"":3,""tn"":981923,""fp"":14154,""fn"":52495,""rows"":1048575,""accuracy"":0.9364384998688696,""precision"":0.00021190930281839374,""recall"":5.714503409653701e-05,""specificity"":0.985790255171036,""f1"":9.001575275673243e-05,""balanced_accuracy"":0.49292370010256625,""roc_auc"":0.4928897544743445,""pr_auc"":0.0473540355835173,""predicted_positive"":14157,""predicted_rate"":0.013501180173092054},""duration_rule"":{""tp"":1132,""tn"":486554,""fp"":509523,""fn"":51366,""rows"":1048575,""accuracy"":0.46509405621915456,""precision"":0.002216760826781291,""recall"":0.021562726199093298,""specificity"":0.4884702688647564,""f1"":0.0040202218580030645,""balanced_accuracy"":0.25501649753192485,""roc_auc"":0.2550164975319249,""pr_auc"":0.049034281060429605,""predicted_positive"":510655,""predicted_rate"":0.48699902248289345}}","{""rows"":1048575,""attacks"":52498,""benign"":996077,""tp"":1005,""fp"":9443,""flags"":10448,""flag_rate"":0.009963998760222206,""benign_fpr"":0.009480190788463141,""attack_recall"":0.0191435864223399,""precision"":0.0961906584992343}"\ndaily,2018-02-16,0,"[11,12,13,14]",,,,1048574,1048574,601802,446772,2.8134331564505968,8,"[73,418624,161839,0,171,1267,13903,36,5,15,452445,196]","[453988,11691,1574,1457,783,419,1177,414,2767,209,599,239,107,155,3389,1495,7771,1648,2478,171,3,6138,2281,15168,0,3475,127541,20878,251,380308,0,0]","{""random_forest"":{""tp"":566401,""tn"":445062,""fp"":1710,""fn"":35401,""rows"":1048574,""accuracy"":0.9646081249392031,""precision"":0.9969900248366956,""recall"":0.9411750044034417,""specificity"":0.9961725443850554,""f1"":0.968278837828112,""balanced_accuracy"":0.9686737743942486,""roc_auc"":0.9957762051655167,""pr_auc"":0.9947737965160105,""predicted_positive"":568111,""predicted_rate"":0.5417939029577312},""logistic_regression"":{""tp"":63137,""tn"":446772,""fp"":0,""fn"":538665,""rows"":1048574,""accuracy"":0.4862880445252314,""precision"":1.0,""recall"":0.10491324389084782,""specificity"":1.0,""f1"":0.18990313397168762,""balanced_accuracy"":0.5524566219454239,""roc_auc"":0.93811626276652,""pr_auc"":0.9651962099013892,""predicted_positive"":63137,""predicted_rate"":0.06021225016069443},""duration_rule"":{""tp"":449794,""tn"":443179,""fp"":3593,""fn"":152008,""rows"":1048574,""accuracy"":0.851607039655761,""precision"":0.9920752028620141,""recall"":0.7474119394751098,""specificity"":0.9919578666523416,""f1"":0.8525373179591523,""balanced_accuracy"":0.8696849030637257,""roc_auc"":0.8696849030637256,""pr_auc"":0.8864552534660108,""predicted_positive"":453387,""predicted_rate"":0.4323843620002022}}","{""rows"":1048574,""attacks"":601802,""benign"":446772,""tp"":1267,""fp"":799,""flags"":2066,""flag_rate"":0.001970294895734588,""benign_fpr"":0.0017883842317781777,""attack_recall"":0.0021053436180006047,""precision"":0.6132623426911907}"\ndaily,2018-02-20,0,"[15,16,17,18,19]",,,,7948748,7948748,576191,7372557,2.6109294162572616,12,"[2173844,296615,1074552,1143622,715983,45951,522290,355398,455955,99476,690822,374240]","[6671655,173022,66578,42085,69953,114348,3146,10412,61134,553,2164,177116,249,87,1662,94,26818,99,11745,2308,1251,6902,37590,2507,19340,9081,108564,1161,269,37567,70,289218]","{""random_forest"":{""tp"":474243,""tn"":7292310,""fp"":80247,""fn"":101948,""rows"":7948748,""accuracy"":0.9770787802053859,""precision"":0.8552778228642537,""recall"":0.8230656153948951,""specificity"":0.9891154452925898,""f1"":0.8388625969658993,""balanced_accuracy"":0.9060905303437424,""roc_auc"":0.9949418188054512,""pr_auc"":0.9278941299433678,""predicted_positive"":554490,""predicted_rate"":0.06975815562400518},""logistic_regression"":{""tp"":662,""tn"":7245158,""fp"":127399,""fn"":575529,""rows"":7948748,""accuracy"":0.9115674569127112,""precision"":0.0051694114523547375,""recall"":0.001148924575357824,""specificity"":0.9827198351942209,""f1"":0.00188000886046472,""balanced_accuracy"":0.4919343798847894,""roc_auc"":0.830631962200307,""pr_auc"":0.17922906759952514,""predicted_positive"":128061,""predicted_rate"":0.016110839090634148},""duration_rule"":{""tp"":828,""tn"":3596395,""fp"":3776162,""fn"":575363,""rows"":7948748,""accuracy"":0.45255215035122515,""precision"":0.00021922218486149025,""recall"":0.0014370234870034416,""specificity"":0.48780836824998436,""f1"":0.0003804114738165034,""balanced_accuracy"":0.2446226958684939,""roc_auc"":0.2446226958684939,""pr_auc"":0.07238441878817171,""predicted_positive"":3776990,""predicted_rate"":0.4751679132361474}}","{""rows"":7948748,""attacks"":576191,""benign"":7372557,""tp"":797,""fp"":102167,""flags"":102964,""flag_rate"":0.012953486511334867,""benign_fpr"":0.013857742978453744,""attack_recall"":0.0013832218830214286,""precision"":0.007740569519443689}"\ndaily,2018-02-21,0,"[20,21,22]",,,,1048575,1048575,687742,360833,2.1641053151462364,8,"[12,521796,632,2,1694,1,50,6,522281,17,363,1721]","[360608,704,10,21,84,2,4,2,5,1,29,2,9,3,283,1,40,2,0,10,4,128,3,823,6,318,36,6544,146,522148,345,156254]","{""random_forest"":{""tp"":686715,""tn"":360741,""fp"":92,""fn"":1027,""rows"":1048575,""accuracy"":0.9989328374222158,""precision"":0.9998660467933496,""recall"":0.9985067074571569,""specificity"":0.9997450344064983,""f1"":0.9991859147982356,""balanced_accuracy"":0.9991258709318276,""roc_auc"":0.9999957109406793,""pr_auc"":0.9999965588783885,""predicted_positive"":686807,""predicted_rate"":0.6549908208759507},""logistic_regression"":{""tp"":1782,""tn"":360832,""fp"":1,""fn"":685960,""rows"":1048575,""accuracy"":0.34581598836516225,""precision"":0.9994391475042064,""recall"":0.0025910879370461596,""specificity"":0.9999972286348533,""f1"":0.0051687756063957075,""balanced_accuracy"":0.5012941582859497,""roc_auc"":0.9498658413060886,""pr_auc"":0.9762475537104661,""predicted_positive"":1783,""predicted_rate"":0.0017004029277829435},""duration_rule"":{""tp"":665706,""tn"":17088,""fp"":343745,""fn"":22036,""rows"":1048575,""accuracy"":0.6511637221944067,""precision"":0.6594733176746568,""recall"":0.9679589148256178,""specificity"":0.04735708762779457,""f1"":0.7844788424180397,""balanced_accuracy"":0.5076580012267062,""roc_auc"":0.5076580012267062,""pr_auc"":0.6593582642107831,""predicted_positive"":1009451,""predicted_rate"":0.9626884104618172}}","{""rows"":1048575,""attacks"":687742,""benign"":360833,""tp"":1730,""fp"":781,""flags"":2511,""flag_rate"":0.002394678492239468,""benign_fpr"":0.002164436179617718,""attack_recall"":0.002515478188041446,""precision"":0.688968538430904}"\ndaily,2018-02-22,0,"[23,24,25,26,27,28]",,,,1048575,1048566,362,1048213,2.3550542260211933,8,"[271676,39515,153543,122382,140557,5366,99243,33935,1143,6345,83049,91821]","[878305,24264,4445,6677,7668,80086,508,1085,3842,99,240,27894,78,75,272,47,2053,10,10,89,44,153,1868,371,96,32,169,185,24,7886,0,0]","{""random_forest"":{""tp"":26,""tn"":1035249,""fp"":12964,""fn"":336,""rows"":1048575,""accuracy"":0.9873161194955058,""precision"":0.0020015396458814474,""recall"":0.0718232044198895,""specificity"":0.9876322846597018,""f1"":0.0038945476333133613,""balanced_accuracy"":0.5297277445397957,""roc_auc"":0.6004566121538085,""pr_auc"":0.0007208002208471149,""predicted_positive"":12990,""predicted_rate"":0.01238824118446463},""logistic_regression"":{""tp"":15,""tn"":1037347,""fp"":10866,""fn"":347,""rows"":1048575,""accuracy"":0.9893064396919629,""precision"":0.0013785497656465398,""recall"":0.04143646408839779,""specificity"":0.9896337862629064,""f1"":0.0026683269589967092,""balanced_accuracy"":0.5155351251756521,""roc_auc"":0.5864148427869236,""pr_auc"":0.0006812924254651736,""predicted_positive"":10881,""predicted_rate"":0.010376940133037695},""duration_rule"":{""tp"":155,""tn"":570658,""fp"":477555,""fn"":207,""rows"":1048575,""accuracy"":0.544370216722695,""precision"":0.0003244646333549643,""recall"":0.4281767955801105,""specificity"":0.5444103440808309,""f1"":0.000648437892200338,""balanced_accuracy"":0.4862935698304707,""roc_auc"":0.48629356983047073,""pr_auc"":0.0003363389987506806,""predicted_positive"":477710,""predicted_rate"":0.455580192165558}}","{""rows"":1048575,""attacks"":362,""benign"":1048213,""tp"":104,""fp"":8889,""flags"":8993,""flag_rate"":0.00857640130653506,""benign_fpr"":0.008480146687743807,""attack_recall"":0.287292817679558,""precision"":0.011564550205715557}"\ndaily,2018-02-23,1,"[29,30,31,32,33]",,,,709466,709466,423,709043,2.757612468172272,12,"[121710,34271,7129,121357,131089,1708,95817,10310,506,6317,86073,93179]","[562083,25032,4723,6314,2255,74941,543,973,253,130,262,26022,161,29,35,31,1389,14,18,101,71,158,1481,1,127,33,119,5,0,2162,0,0]","{""random_forest"":{""tp"":9,""tn"":703373,""fp"":5670,""fn"":414,""rows"":709466,""accuracy"":0.9914245362004662,""precision"":0.001584786053882726,""recall"":0.02127659574468085,""specificity"":0.9920033058643835,""f1"":0.0029498525073746312,""balanced_accuracy"":0.5066399508045322,""roc_auc"":0.5751572036185331,""pr_auc"":0.0010836697324813971,""predicted_positive"":5679,""predicted_rate"":0.008004611919387257},""logistic_regression"":{""tp"":0,""tn"":709005,""fp"":38,""fn"":423,""rows"":709466,""accuracy"":0.9993502155142037,""precision"":0.0,""recall"":0.0,""specificity"":0.9999464066354227,""f1"":0.0,""balanced_accuracy"":0.49997320331771133,""roc_auc"":0.3009177098493051,""pr_auc"":0.0015626876435786536,""predicted_positive"":38,""predicted_rate"":5.3561410976706427e-05},""duration_rule"":{""tp"":43,""tn"":561572,""fp"":147471,""fn"":380,""rows"":709466,""accuracy"":0.7916024164653416,""precision"":0.00029149775614517945,""recall"":0.1016548463356974,""specificity"":0.7920140245372989,""f1"":0.0005813285384994964,""balanced_accuracy"":0.44683443543649815,""roc_auc"":0.44683443543649815,""pr_auc"":0.000565246269375203,""predicted_positive"":147514,""predicted_rate"":0.207922578389944}}","{""rows"":709466,""attacks"":423,""benign"":709043,""tp"":262,""fp"":9287,""flags"":9549,""flag_rate"":0.013459418774120253,""benign_fpr"":0.013097936232358264,""attack_recall"":0.6193853427895981,""precision"":0.027437428002932245}"\ndaily,2018-02-28,1,"[34,35,36,37,38]",,,,441790,441790,56633,385157,3.7109563757192565,16,"[116751,19163,41379,110326,20818,1714,50961,19754,17740,8384,24541,10259]","[374878,22465,5965,6215,21638,565,1108,1120,1719,298,906,452,5,9,14,32,2176,0,1,27,32,67,1040,19,126,27,201,0,0,685,0,0]","{""random_forest"":{""tp"":577,""tn"":381333,""fp"":3824,""fn"":56056,""rows"":441790,""accuracy"":0.8644604902781865,""precision"":0.13110656668938878,""recall"":0.010188406053007963,""specificity"":0.9900715811993551,""f1"":0.018907494183569813,""balanced_accuracy"":0.5001299936261815,""roc_auc"":0.4338672051550671,""pr_auc"":0.11390282259584279,""predicted_positive"":4401,""predicted_rate"":0.009961746531157337},""logistic_regression"":{""tp"":600,""tn"":383593,""fp"":1564,""fn"":56033,""rows"":441790,""accuracy"":0.8696281038502456,""precision"":0.27726432532347506,""recall"":0.01059452969116946,""specificity"":0.9959393182520375,""f1"":0.020409204551252613,""balanced_accuracy"":0.5032669239716034,""roc_auc"":0.48591410966703474,""pr_auc"":0.13083911112951085,""predicted_positive"":2164,""predicted_rate"":0.004898254826953983},""duration_rule"":{""tp"":27543,""tn"":233442,""fp"":151715,""fn"":29090,""rows"":441790,""accuracy"":0.5907444713551687,""precision"":0.15365004630197815,""recall"":0.48634188547313406,""specificity"":0.6060956960408352,""f1"":0.23352311025007313,""balanced_accuracy"":0.5462187907569847,""roc_auc"":0.5462187907569847,""pr_auc"":0.14057221704597989,""predicted_positive"":179258,""predicted_rate"":0.40575386495846444}}","{""rows"":441790,""attacks"":56633,""benign"":385157,""tp"":2207,""fp"":13338,""flags"":15545,""flag_rate"":0.03518640077865049,""benign_fpr"":0.034630033986140715,""attack_recall"":0.03897021171401833,""precision"":0.14197491154712127}"\ndaily,2018-03-01,2,"[39,40,41,42,43]",,,,231473,231473,65251,166222,3.6682681446664716,16,"[54210,11244,19658,52871,17302,623,24199,8372,6299,4259,23304,9132]","[189836,10941,3207,3108,16857,4882,269,379,111,190,129,217,7,19,4,8,640,4,2,16,22,42,368,25,71,5,87,1,0,26,0,0]","{""random_forest"":{""tp"":415,""tn"":165328,""fp"":894,""fn"":64836,""rows"":231473,""accuracy"":0.7160359955588773,""precision"":0.3170359052711994,""recall"":0.006360055784585677,""specificity"":0.994621650563704,""f1"":0.012469951923076924,""balanced_accuracy"":0.5004908531741449,""roc_auc"":0.6027127139241765,""pr_auc"":0.43779382552899665,""predicted_positive"":1309,""predicted_rate"":0.005655087202395096},""logistic_regression"":{""tp"":0,""tn"":166204,""fp"":18,""fn"":65251,""rows"":231473,""accuracy"":0.7180275885308438,""precision"":0.0,""recall"":0.0,""specificity"":0.9998917110851753,""f1"":0.0,""balanced_accuracy"":0.49994585554258764,""roc_auc"":0.5737651370741172,""pr_auc"":0.32903961922147545,""predicted_positive"":18,""predicted_rate"":7.776284923079581e-05},""duration_rule"":{""tp"":31317,""tn"":114110,""fp"":52112,""fn"":33934,""rows"":231473,""accuracy"":0.6282676597270525,""precision"":0.3753730717136727,""recall"":0.47994666748402326,""specificity"":0.6864915594806945,""f1"":0.42126715092816786,""balanced_accuracy"":0.5832191134823589,""roc_auc"":0.5832191134823588,""pr_auc"":0.326759306265431,""predicted_positive"":83429,""predicted_rate"":0.360426486026448}}","{""rows"":231473,""attacks"":65251,""benign"":166222,""tp"":3450,""fp"":5206,""flags"":8656,""flag_rate"":0.037395290163431585,""benign_fpr"":0.03131956058764784,""attack_recall"":0.05287275290800141,""precision"":0.3985674676524954}"\ndaily,2018-03-02,2,"[44,45,46,47,48]",,,,748636,748636,143274,605362,2.5032019902045537,12,"[128056,24880,10051,131891,76291,1768,66187,18620,143186,6876,75715,65115]","[527958,17900,3958,5111,3439,168539,369,292,132,28,121,16740,20,20,31,7,1630,5,76,86,149,270,1280,4,240,23,151,4,0,53,0,0]","{""random_forest"":{""tp"":2,""tn"":601393,""fp"":3969,""fn"":143272,""rows"":748636,""accuracy"":0.8033209730763682,""precision"":0.000503651473180559,""recall"":1.395926685930455e-05,""specificity"":0.9934435924289929,""f1"":2.716560833984176e-05,""balanced_accuracy"":0.49672877584792613,""roc_auc"":0.8872055960620285,""pr_auc"":0.7018425503402654,""predicted_positive"":3971,""predicted_rate"":0.0053043134447181275},""logistic_regression"":{""tp"":177,""tn"":605329,""fp"":33,""fn"":143097,""rows"":748636,""accuracy"":0.8088122932907315,""precision"":0.8428571428571429,""recall"":0.0012353951170484526,""specificity"":0.9999454871630529,""f1"":0.0024671740403111146,""balanced_accuracy"":0.5005904411400507,""roc_auc"":0.9628597253500065,""pr_auc"":0.7911617516564654,""predicted_positive"":210,""predicted_rate"":0.00028051015446759175},""duration_rule"":{""tp"":141254,""tn"":444891,""fp"":160471,""fn"":2020,""rows"":748636,""accuracy"":0.7829505928114597,""precision"":0.468154776700638,""recall"":0.9859011404721024,""specificity"":0.7349172891592138,""f1"":0.6348508648334041,""balanced_accuracy"":0.8604092148156581,""roc_auc"":0.8604092148156582,""pr_auc"":0.46425256880007165,""predicted_positive"":301725,""predicted_rate"":0.4030329826511148}}","{""rows"":748636,""attacks"":143274,""benign"":605362,""tp"":987,""fp"":11399,""flags"":12386,""flag_rate"":0.016544756063026623,""benign_fpr"":0.01883005540486519,""attack_recall"":0.006888898195066795,""precision"":0.07968674309704506}"\n', 'reports/phase6_shift_metrics.csv': 'scope,source_day,split_rank,batch_ids,window_hour,wall_start,wall_end,category,detector,value,reference_value,delta,threshold,rows,support,labels_used,version\nintraday,2018-02-14,0,[0],0,00:00,03:00,INPUT_SHIFT,flow_duration,0.08174367039818693,0.0,0.08174367039818693,0.1,235564,supported,False,phase6-v1\nintraday,2018-02-14,0,[0],0,00:00,03:00,INPUT_SHIFT,tot_fwd_pkts,0.13846253385586335,0.0,0.13846253385586335,0.1,235564,supported,False,phase6-v1\nintraday,2018-02-14,0,[0],0,00:00,03:00,INPUT_SHIFT,totlen_fwd_pkts,0.10002639475834188,0.0,0.10002639475834188,0.1,235564,supported,False,phase6-v1\nintraday,2018-02-14,0,[0],0,00:00,03:00,INPUT_SHIFT,pkt_len_mean,0.049499353212927304,0.0,0.049499353212927304,0.1,235564,supported,False,phase6-v1\nintraday,2018-02-14,0,[0],0,00:00,03:00,INPUT_SHIFT,flow_iat_mean,0.06201772582995031,0.0,0.06201772582995031,0.1,235564,supported,False,phase6-v1\nintraday,2018-02-14,0,[0],0,00:00,03:00,INPUT_SHIFT,flow_pkts_s,0.09659227083261325,0.0,0.09659227083261325,0.1,235564,supported,False,phase6-v1\nintraday,2018-02-14,0,[0],0,00:00,03:00,SCORE_SHIFT,random_forest,0.31738802171044905,0.0,0.31738802171044905,0.1,235564,supported,False,phase6-v1\nintraday,2018-02-14,0,[0],0,00:00,03:00,SCORE_SHIFT,logistic_regression,0.29899210916310903,0.0,0.29899210916310903,0.1,235564,supported,False,phase6-v1\nintraday,2018-02-14,0,[0],0,00:00,03:00,SCORE_SHIFT,duration_rule,0.000153869039926661,0.0,0.000153869039926661,0.1,235564,supported,False,phase6-v1\nintraday,2018-02-14,0,[0],0,00:00,03:00,CLUSTER_SHIFT,cluster,0.15584959526184122,0.0,0.15584959526184122,0.1,235564,supported,False,phase6-v1\nintraday,2018-02-14,0,[0],0,00:00,03:00,NOVELTY_RATE,kmeans_p99,0.0041390025640590244,0.010022952446636205,-0.005883949882577181,0.015,235564,supported,False,phase6-v1\nintraday,2018-02-14,0,[0],0,00:00,03:00,PERFORMANCE_WARNING,random_forest_recall,0.5004570080101041,0.88977162428725,0.3893146162771459,0.2,235564,supported,True,phase6-v1\nintraday,2018-02-14,0,[1],3,03:00,06:00,INPUT_SHIFT,flow_duration,0.0722520578832312,0.0,0.0722520578832312,0.1,142824,supported,False,phase6-v1\nintraday,2018-02-14,0,[1],3,03:00,06:00,INPUT_SHIFT,tot_fwd_pkts,0.11400266314882611,0.0,0.11400266314882611,0.1,142824,supported,False,phase6-v1\nintraday,2018-02-14,0,[1],3,03:00,06:00,INPUT_SHIFT,totlen_fwd_pkts,0.08872789853069722,0.0,0.08872789853069722,0.1,142824,supported,False,phase6-v1\nintraday,2018-02-14,0,[1],3,03:00,06:00,INPUT_SHIFT,pkt_len_mean,0.04766719755236898,0.0,0.04766719755236898,0.1,142824,supported,False,phase6-v1\nintraday,2018-02-14,0,[1],3,03:00,06:00,INPUT_SHIFT,flow_iat_mean,0.052167587734622475,0.0,0.052167587734622475,0.1,142824,supported,False,phase6-v1\nintraday,2018-02-14,0,[1],3,03:00,06:00,INPUT_SHIFT,flow_pkts_s,0.08392663610555576,0.0,0.08392663610555576,0.1,142824,supported,False,phase6-v1\nintraday,2018-02-14,0,[1],3,03:00,06:00,SCORE_SHIFT,random_forest,0.29267671260057554,0.0,0.29267671260057554,0.1,142824,supported,False,phase6-v1\nintraday,2018-02-14,0,[1],3,03:00,06:00,SCORE_SHIFT,logistic_regression,0.29106494694856555,0.0,0.29106494694856555,0.1,142824,supported,False,phase6-v1\nintraday,2018-02-14,0,[1],3,03:00,06:00,SCORE_SHIFT,duration_rule,0.00027947180752923694,0.0,0.00027947180752923694,0.1,142824,supported,False,phase6-v1\nintraday,2018-02-14,0,[1],3,03:00,06:00,CLUSTER_SHIFT,cluster,0.14255821833597318,0.0,0.14255821833597318,0.1,142824,supported,False,phase6-v1\nintraday,2018-02-14,0,[1],3,03:00,06:00,NOVELTY_RATE,kmeans_p99,0.00481011594689968,0.010022952446636205,-0.005212836499736525,0.015,142824,supported,False,phase6-v1\nintraday,2018-02-14,0,[1],3,03:00,06:00,PERFORMANCE_WARNING,random_forest_recall,0.4999033327880311,0.88977162428725,0.38986829149921887,0.2,142824,supported,True,phase6-v1\nintraday,2018-02-14,0,[2],6,06:00,09:00,INPUT_SHIFT,flow_duration,0.0015674163923937625,0.0,0.0015674163923937625,0.1,67293,supported,False,phase6-v1\nintraday,2018-02-14,0,[2],6,06:00,09:00,INPUT_SHIFT,tot_fwd_pkts,0.0021305553194844846,0.0,0.0021305553194844846,0.1,67293,supported,False,phase6-v1\nintraday,2018-02-14,0,[2],6,06:00,09:00,INPUT_SHIFT,totlen_fwd_pkts,0.002207834980327465,0.0,0.002207834980327465,0.1,67293,supported,False,phase6-v1\nintraday,2018-02-14,0,[2],6,06:00,09:00,INPUT_SHIFT,pkt_len_mean,0.0038886539959856648,0.0,0.0038886539959856648,0.1,67293,supported,False,phase6-v1\nintraday,2018-02-14,0,[2],6,06:00,09:00,INPUT_SHIFT,flow_iat_mean,0.001344801529490428,0.0,0.001344801529490428,0.1,67293,supported,False,phase6-v1\nintraday,2018-02-14,0,[2],6,06:00,09:00,INPUT_SHIFT,flow_pkts_s,0.0038197475043377075,0.0,0.0038197475043377075,0.1,67293,supported,False,phase6-v1\nintraday,2018-02-14,0,[2],6,06:00,09:00,SCORE_SHIFT,random_forest,0.10596330092623762,0.0,0.10596330092623762,0.1,67293,supported,False,phase6-v1\nintraday,2018-02-14,0,[2],6,06:00,09:00,SCORE_SHIFT,logistic_regression,0.1482632121343371,0.0,0.1482632121343371,0.1,67293,supported,False,phase6-v1\nintraday,2018-02-14,0,[2],6,06:00,09:00,SCORE_SHIFT,duration_rule,0.0007951532393468096,0.0,0.0007951532393468096,0.1,67293,supported,False,phase6-v1\nintraday,2018-02-14,0,[2],6,06:00,09:00,CLUSTER_SHIFT,cluster,0.06271890182211903,0.0,0.06271890182211903,0.1,67293,supported,False,phase6-v1\nintraday,2018-02-14,0,[2],6,06:00,09:00,NOVELTY_RATE,kmeans_p99,0.010090202547070274,0.010022952446636205,6.725010043406865e-05,0.015,67293,supported,False,phase6-v1\nintraday,2018-02-14,0,[2],6,06:00,09:00,PERFORMANCE_WARNING,random_forest_recall,,0.88977162428725,,0.2,67293,insufficient,True,phase6-v1\nintraday,2018-02-14,0,[3],9,09:00,12:00,INPUT_SHIFT,flow_duration,0.08614793898492504,0.0,0.08614793898492504,0.1,498994,supported,False,phase6-v1\nintraday,2018-02-14,0,[3],9,09:00,12:00,INPUT_SHIFT,tot_fwd_pkts,0.05083927792242687,0.0,0.05083927792242687,0.1,498994,supported,False,phase6-v1\nintraday,2018-02-14,0,[3],9,09:00,12:00,INPUT_SHIFT,totlen_fwd_pkts,0.013800843018507902,0.0,0.013800843018507902,0.1,498994,supported,False,phase6-v1\nintraday,2018-02-14,0,[3],9,09:00,12:00,INPUT_SHIFT,pkt_len_mean,0.027713951576847622,0.0,0.027713951576847622,0.1,498994,supported,False,phase6-v1\nintraday,2018-02-14,0,[3],9,09:00,12:00,INPUT_SHIFT,flow_iat_mean,0.08424645861855075,0.0,0.08424645861855075,0.1,498994,supported,False,phase6-v1\nintraday,2018-02-14,0,[3],9,09:00,12:00,INPUT_SHIFT,flow_pkts_s,0.08283400181081106,0.0,0.08283400181081106,0.1,498994,supported,False,phase6-v1\nintraday,2018-02-14,0,[3],9,09:00,12:00,SCORE_SHIFT,random_forest,0.20204430257086337,0.0,0.20204430257086337,0.1,498994,supported,False,phase6-v1\nintraday,2018-02-14,0,[3],9,09:00,12:00,SCORE_SHIFT,logistic_regression,0.2221745164834034,0.0,0.2221745164834034,0.1,498994,supported,False,phase6-v1\nintraday,2018-02-14,0,[3],9,09:00,12:00,SCORE_SHIFT,duration_rule,0.01771113875494037,0.0,0.01771113875494037,0.1,498994,supported,False,phase6-v1\nintraday,2018-02-14,0,[3],9,09:00,12:00,CLUSTER_SHIFT,cluster,0.12843707573767985,0.0,0.12843707573767985,0.1,498994,supported,False,phase6-v1\nintraday,2018-02-14,0,[3],9,09:00,12:00,NOVELTY_RATE,kmeans_p99,0.004581217409427769,0.010022952446636205,-0.0054417350372084365,0.015,498994,supported,False,phase6-v1\nintraday,2018-02-14,0,[3],9,09:00,12:00,PERFORMANCE_WARNING,random_forest_recall,0.9999709959974477,0.88977162428725,-0.11019937171019767,0.2,498994,supported,True,phase6-v1\nintraday,2018-02-14,0,[4],12,12:00,15:00,INPUT_SHIFT,flow_duration,0.034942888551831155,0.0,0.034942888551831155,0.1,103895,supported,False,phase6-v1\nintraday,2018-02-14,0,[4],12,12:00,15:00,INPUT_SHIFT,tot_fwd_pkts,0.023855632165501753,0.0,0.023855632165501753,0.1,103895,supported,False,phase6-v1\nintraday,2018-02-14,0,[4],12,12:00,15:00,INPUT_SHIFT,totlen_fwd_pkts,0.00476023007401489,0.0,0.00476023007401489,0.1,103895,supported,False,phase6-v1\nintraday,2018-02-14,0,[4],12,12:00,15:00,INPUT_SHIFT,pkt_len_mean,0.007593049087240625,0.0,0.007593049087240625,0.1,103895,supported,False,phase6-v1\nintraday,2018-02-14,0,[4],12,12:00,15:00,INPUT_SHIFT,flow_iat_mean,0.03482355887156571,0.0,0.03482355887156571,0.1,103895,supported,False,phase6-v1\nintraday,2018-02-14,0,[4],12,12:00,15:00,INPUT_SHIFT,flow_pkts_s,0.03662467409380238,0.0,0.03662467409380238,0.1,103895,supported,False,phase6-v1\nintraday,2018-02-14,0,[4],12,12:00,15:00,SCORE_SHIFT,random_forest,0.13554030161829797,0.0,0.13554030161829797,0.1,103895,supported,False,phase6-v1\nintraday,2018-02-14,0,[4],12,12:00,15:00,SCORE_SHIFT,logistic_regression,0.16002176267373708,0.0,0.16002176267373708,0.1,103895,supported,False,phase6-v1\nintraday,2018-02-14,0,[4],12,12:00,15:00,SCORE_SHIFT,duration_rule,0.0057723391162049255,0.0,0.0057723391162049255,0.1,103895,supported,False,phase6-v1\nintraday,2018-02-14,0,[4],12,12:00,15:00,CLUSTER_SHIFT,cluster,0.09305707485710397,0.0,0.09305707485710397,0.1,103895,supported,False,phase6-v1\nintraday,2018-02-14,0,[4],12,12:00,15:00,NOVELTY_RATE,kmeans_p99,0.00583281197362722,0.010022952446636205,-0.0041901404730089855,0.015,103895,supported,False,phase6-v1\nintraday,2018-02-14,0,[4],12,12:00,15:00,PERFORMANCE_WARNING,random_forest_recall,0.9999523128278494,0.88977162428725,-0.11018068854059937,0.2,103895,supported,True,phase6-v1\ncapture_only,2018-02-14,0,[5],24,,,INPUT_SHIFT,flow_duration,0.9999887034955526,0.0,0.9999887034955526,0.1,5,insufficient,False,phase6-v1\ncapture_only,2018-02-14,0,[5],24,,,INPUT_SHIFT,tot_fwd_pkts,0.5195308411941477,0.0,0.5195308411941477,0.1,5,insufficient,False,phase6-v1\ncapture_only,2018-02-14,0,[5],24,,,INPUT_SHIFT,totlen_fwd_pkts,0.25515194517376366,0.0,0.25515194517376366,0.1,5,insufficient,False,phase6-v1\ncapture_only,2018-02-14,0,[5],24,,,INPUT_SHIFT,pkt_len_mean,0.42401474484204194,0.0,0.42401474484204194,0.1,5,insufficient,False,phase6-v1\ncapture_only,2018-02-14,0,[5],24,,,INPUT_SHIFT,flow_iat_mean,0.9999887034955526,0.0,0.9999887034955526,0.1,5,insufficient,False,phase6-v1\ncapture_only,2018-02-14,0,[5],24,,,INPUT_SHIFT,flow_pkts_s,0.9741404677609196,0.0,0.9741404677609196,0.1,5,insufficient,False,phase6-v1\ncapture_only,2018-02-14,0,[5],24,,,SCORE_SHIFT,random_forest,0.9986274802292984,0.0,0.9986274802292984,0.1,5,insufficient,False,phase6-v1\ncapture_only,2018-02-14,0,[5],24,,,SCORE_SHIFT,logistic_regression,0.9366184405283419,0.0,0.9366184405283419,0.1,5,insufficient,False,phase6-v1\ncapture_only,2018-02-14,0,[5],24,,,SCORE_SHIFT,duration_rule,0.3267277026444694,0.0,0.3267277026444694,0.1,5,insufficient,False,phase6-v1\ncapture_only,2018-02-14,0,[5],24,,,CLUSTER_SHIFT,cluster,0.7538891939602701,0.0,0.7538891939602701,0.1,5,insufficient,False,phase6-v1\ncapture_only,2018-02-14,0,[5],24,,,NOVELTY_RATE,kmeans_p99,0.8,0.010022952446636205,0.7899770475533638,0.015,5,insufficient,False,phase6-v1\ncapture_only,2018-02-14,0,[5],24,,,PERFORMANCE_WARNING,random_forest_recall,,0.88977162428725,,0.2,5,insufficient,True,phase6-v1\nintraday,2018-02-15,0,[6],0,00:00,03:00,INPUT_SHIFT,flow_duration,0.0042038909450551176,0.0,0.0042038909450551176,0.1,197798,supported,False,phase6-v1\nintraday,2018-02-15,0,[6],0,00:00,03:00,INPUT_SHIFT,tot_fwd_pkts,0.005583463778696918,0.0,0.005583463778696918,0.1,197798,supported,False,phase6-v1\nintraday,2018-02-15,0,[6],0,00:00,03:00,INPUT_SHIFT,totlen_fwd_pkts,0.009366420240293612,0.0,0.009366420240293612,0.1,197798,supported,False,phase6-v1\nintraday,2018-02-15,0,[6],0,00:00,03:00,INPUT_SHIFT,pkt_len_mean,0.012314827300182579,0.0,0.012314827300182579,0.1,197798,supported,False,phase6-v1\nintraday,2018-02-15,0,[6],0,00:00,03:00,INPUT_SHIFT,flow_iat_mean,0.006011818723573317,0.0,0.006011818723573317,0.1,197798,supported,False,phase6-v1\nintraday,2018-02-15,0,[6],0,00:00,03:00,INPUT_SHIFT,flow_pkts_s,0.008767970150816314,0.0,0.008767970150816314,0.1,197798,supported,False,phase6-v1\nintraday,2018-02-15,0,[6],0,00:00,03:00,SCORE_SHIFT,random_forest,0.09728890567215549,0.0,0.09728890567215549,0.1,197798,supported,False,phase6-v1\nintraday,2018-02-15,0,[6],0,00:00,03:00,SCORE_SHIFT,logistic_regression,0.11679909690261799,0.0,0.11679909690261799,0.1,197798,supported,False,phase6-v1\nintraday,2018-02-15,0,[6],0,00:00,03:00,SCORE_SHIFT,duration_rule,2.8954685080020796e-07,0.0,2.8954685080020796e-07,0.1,197798,supported,False,phase6-v1\nintraday,2018-02-15,0,[6],0,00:00,03:00,CLUSTER_SHIFT,cluster,0.053143140707947054,0.0,0.053143140707947054,0.1,197798,supported,False,phase6-v1\nintraday,2018-02-15,0,[6],0,00:00,03:00,NOVELTY_RATE,kmeans_p99,0.00959564808542048,0.010022952446636205,-0.0004273043612157256,0.015,197798,supported,False,phase6-v1\nintraday,2018-02-15,0,[6],0,00:00,03:00,PERFORMANCE_WARNING,random_forest_recall,,0.88977162428725,,0.2,197798,insufficient,True,phase6-v1\nintraday,2018-02-15,0,[7],3,03:00,06:00,INPUT_SHIFT,flow_duration,0.0013668432357526474,0.0,0.0013668432357526474,0.1,191551,supported,False,phase6-v1\nintraday,2018-02-15,0,[7],3,03:00,06:00,INPUT_SHIFT,tot_fwd_pkts,0.0020243551152820858,0.0,0.0020243551152820858,0.1,191551,supported,False,phase6-v1\nintraday,2018-02-15,0,[7],3,03:00,06:00,INPUT_SHIFT,totlen_fwd_pkts,0.0040725375739549315,0.0,0.0040725375739549315,0.1,191551,supported,False,phase6-v1\nintraday,2018-02-15,0,[7],3,03:00,06:00,INPUT_SHIFT,pkt_len_mean,0.0069245147358723376,0.0,0.0069245147358723376,0.1,191551,supported,False,phase6-v1\nintraday,2018-02-15,0,[7],3,03:00,06:00,INPUT_SHIFT,flow_iat_mean,0.0010804496341395647,0.0,0.0010804496341395647,0.1,191551,supported,False,phase6-v1\nintraday,2018-02-15,0,[7],3,03:00,06:00,INPUT_SHIFT,flow_pkts_s,0.007810203158336341,0.0,0.007810203158336341,0.1,191551,supported,False,phase6-v1\nintraday,2018-02-15,0,[7],3,03:00,06:00,SCORE_SHIFT,random_forest,0.09126681546336013,0.0,0.09126681546336013,0.1,191551,supported,False,phase6-v1\nintraday,2018-02-15,0,[7],3,03:00,06:00,SCORE_SHIFT,logistic_regression,0.11107263064289687,0.0,0.11107263064289687,0.1,191551,supported,False,phase6-v1\nintraday,2018-02-15,0,[7],3,03:00,06:00,SCORE_SHIFT,duration_rule,0.00015809542386919975,0.0,0.00015809542386919975,0.1,191551,supported,False,phase6-v1\nintraday,2018-02-15,0,[7],3,03:00,06:00,CLUSTER_SHIFT,cluster,0.040873195661600276,0.0,0.040873195661600276,0.1,191551,supported,False,phase6-v1\nintraday,2018-02-15,0,[7],3,03:00,06:00,NOVELTY_RATE,kmeans_p99,0.008629555575277602,0.010022952446636205,-0.0013933968713586032,0.015,191551,supported,False,phase6-v1\nintraday,2018-02-15,0,[7],3,03:00,06:00,PERFORMANCE_WARNING,random_forest_recall,,0.88977162428725,,0.2,191551,insufficient,True,phase6-v1\nintraday,2018-02-15,0,[8],6,06:00,09:00,INPUT_SHIFT,flow_duration,0.0013769672885188754,0.0,0.0013769672885188754,0.1,87581,supported,False,phase6-v1\nintraday,2018-02-15,0,[8],6,06:00,09:00,INPUT_SHIFT,tot_fwd_pkts,0.0012752584646469126,0.0,0.0012752584646469126,0.1,87581,supported,False,phase6-v1\nintraday,2018-02-15,0,[8],6,06:00,09:00,INPUT_SHIFT,totlen_fwd_pkts,0.003903699719582847,0.0,0.003903699719582847,0.1,87581,supported,False,phase6-v1\nintraday,2018-02-15,0,[8],6,06:00,09:00,INPUT_SHIFT,pkt_len_mean,0.005548505809471474,0.0,0.005548505809471474,0.1,87581,supported,False,phase6-v1\nintraday,2018-02-15,0,[8],6,06:00,09:00,INPUT_SHIFT,flow_iat_mean,0.0018749483658080338,0.0,0.0018749483658080338,0.1,87581,supported,False,phase6-v1\nintraday,2018-02-15,0,[8],6,06:00,09:00,INPUT_SHIFT,flow_pkts_s,0.008050352270742543,0.0,0.008050352270742543,0.1,87581,supported,False,phase6-v1\nintraday,2018-02-15,0,[8],6,06:00,09:00,SCORE_SHIFT,random_forest,0.0990558500623868,0.0,0.0990558500623868,0.1,87581,supported,False,phase6-v1\nintraday,2018-02-15,0,[8],6,06:00,09:00,SCORE_SHIFT,logistic_regression,0.11875666843685266,0.0,0.11875666843685266,0.1,87581,supported,False,phase6-v1\nintraday,2018-02-15,0,[8],6,06:00,09:00,SCORE_SHIFT,duration_rule,0.003072522809250137,0.0,0.003072522809250137,0.1,87581,supported,False,phase6-v1\nintraday,2018-02-15,0,[8],6,06:00,09:00,CLUSTER_SHIFT,cluster,0.03841169634677565,0.0,0.03841169634677565,0.1,87581,supported,False,phase6-v1\nintraday,2018-02-15,0,[8],6,06:00,09:00,NOVELTY_RATE,kmeans_p99,0.011440837624598943,0.010022952446636205,0.0014178851779627372,0.015,87581,supported,False,phase6-v1\nintraday,2018-02-15,0,[8],6,06:00,09:00,PERFORMANCE_WARNING,random_forest_recall,,0.88977162428725,,0.2,87581,insufficient,True,phase6-v1\nintraday,2018-02-15,0,[9],9,09:00,12:00,INPUT_SHIFT,flow_duration,0.0030419557138042655,0.0,0.0030419557138042655,0.1,463466,supported,False,phase6-v1\nintraday,2018-02-15,0,[9],9,09:00,12:00,INPUT_SHIFT,tot_fwd_pkts,0.0036714840479817067,0.0,0.0036714840479817067,0.1,463466,supported,False,phase6-v1\nintraday,2018-02-15,0,[9],9,09:00,12:00,INPUT_SHIFT,totlen_fwd_pkts,0.01264584344493837,0.0,0.01264584344493837,0.1,463466,supported,False,phase6-v1\nintraday,2018-02-15,0,[9],9,09:00,12:00,INPUT_SHIFT,pkt_len_mean,0.004753128861951438,0.0,0.004753128861951438,0.1,463466,supported,False,phase6-v1\nintraday,2018-02-15,0,[9],9,09:00,12:00,INPUT_SHIFT,flow_iat_mean,0.01063231827554148,0.0,0.01063231827554148,0.1,463466,supported,False,phase6-v1\nintraday,2018-02-15,0,[9],9,09:00,12:00,INPUT_SHIFT,flow_pkts_s,0.01042576386978819,0.0,0.01042576386978819,0.1,463466,supported,False,phase6-v1\nintraday,2018-02-15,0,[9],9,09:00,12:00,SCORE_SHIFT,random_forest,0.12291456663631067,0.0,0.12291456663631067,0.1,463466,supported,False,phase6-v1\nintraday,2018-02-15,0,[9],9,09:00,12:00,SCORE_SHIFT,logistic_regression,0.12386781499615045,0.0,0.12386781499615045,0.1,463466,supported,False,phase6-v1\nintraday,2018-02-15,0,[9],9,09:00,12:00,SCORE_SHIFT,duration_rule,0.002098410289779234,0.0,0.002098410289779234,0.1,463466,supported,False,phase6-v1\nintraday,2018-02-15,0,[9],9,09:00,12:00,CLUSTER_SHIFT,cluster,0.056736408877588955,0.0,0.056736408877588955,0.1,463466,supported,False,phase6-v1\nintraday,2018-02-15,0,[9],9,09:00,12:00,NOVELTY_RATE,kmeans_p99,0.010389111606892414,0.010022952446636205,0.0003661591602562089,0.015,463466,supported,False,phase6-v1\nintraday,2018-02-15,0,[9],9,09:00,12:00,PERFORMANCE_WARNING,random_forest_recall,0.5997752295325536,0.88977162428725,0.2899963947546964,0.2,463466,supported,True,phase6-v1\nintraday,2018-02-15,0,[10],12,12:00,15:00,INPUT_SHIFT,flow_duration,0.004999644149504132,0.0,0.004999644149504132,0.1,108179,supported,False,phase6-v1\nintraday,2018-02-15,0,[10],12,12:00,15:00,INPUT_SHIFT,tot_fwd_pkts,0.007162145279589793,0.0,0.007162145279589793,0.1,108179,supported,False,phase6-v1\nintraday,2018-02-15,0,[10],12,12:00,15:00,INPUT_SHIFT,totlen_fwd_pkts,0.010009231704214038,0.0,0.010009231704214038,0.1,108179,supported,False,phase6-v1\nintraday,2018-02-15,0,[10],12,12:00,15:00,INPUT_SHIFT,pkt_len_mean,0.013458818789205436,0.0,0.013458818789205436,0.1,108179,supported,False,phase6-v1\nintraday,2018-02-15,0,[10],12,12:00,15:00,INPUT_SHIFT,flow_iat_mean,0.006745033567036989,0.0,0.006745033567036989,0.1,108179,supported,False,phase6-v1\nintraday,2018-02-15,0,[10],12,12:00,15:00,INPUT_SHIFT,flow_pkts_s,0.010232026358803608,0.0,0.010232026358803608,0.1,108179,supported,False,phase6-v1\nintraday,2018-02-15,0,[10],12,12:00,15:00,SCORE_SHIFT,random_forest,0.0993762820946476,0.0,0.0993762820946476,0.1,108179,supported,False,phase6-v1\nintraday,2018-02-15,0,[10],12,12:00,15:00,SCORE_SHIFT,logistic_regression,0.12111076293634293,0.0,0.12111076293634293,0.1,108179,supported,False,phase6-v1\nintraday,2018-02-15,0,[10],12,12:00,15:00,SCORE_SHIFT,duration_rule,1.1130133816689104e-05,0.0,1.1130133816689104e-05,0.1,108179,supported,False,phase6-v1\nintraday,2018-02-15,0,[10],12,12:00,15:00,CLUSTER_SHIFT,cluster,0.06704315006409914,0.0,0.06704315006409914,0.1,108179,supported,False,phase6-v1\nintraday,2018-02-15,0,[10],12,12:00,15:00,NOVELTY_RATE,kmeans_p99,0.009983453350465432,0.010022952446636205,-3.9499096170772865e-05,0.015,108179,supported,False,phase6-v1\nintraday,2018-02-15,0,[10],12,12:00,15:00,PERFORMANCE_WARNING,random_forest_recall,,0.88977162428725,,0.2,108179,insufficient,True,phase6-v1\nintraday,2018-02-16,0,[11],0,00:00,03:00,INPUT_SHIFT,flow_duration,0.16315079124636614,0.0,0.16315079124636614,0.1,908339,supported,False,phase6-v1\nintraday,2018-02-16,0,[11],0,00:00,03:00,INPUT_SHIFT,tot_fwd_pkts,0.1675627369625387,0.0,0.1675627369625387,0.1,908339,supported,False,phase6-v1\nintraday,2018-02-16,0,[11],0,00:00,03:00,INPUT_SHIFT,totlen_fwd_pkts,0.11373317494188757,0.0,0.11373317494188757,0.1,908339,supported,False,phase6-v1\nintraday,2018-02-16,0,[11],0,00:00,03:00,INPUT_SHIFT,pkt_len_mean,0.17273988193961975,0.0,0.17273988193961975,0.1,908339,supported,False,phase6-v1\nintraday,2018-02-16,0,[11],0,00:00,03:00,INPUT_SHIFT,flow_iat_mean,0.17214372270827885,0.0,0.17214372270827885,0.1,908339,supported,False,phase6-v1\nintraday,2018-02-16,0,[11],0,00:00,03:00,INPUT_SHIFT,flow_pkts_s,0.18366375323008524,0.0,0.18366375323008524,0.1,908339,supported,False,phase6-v1\nintraday,2018-02-16,0,[11],0,00:00,03:00,SCORE_SHIFT,random_forest,0.5293767521063872,0.0,0.5293767521063872,0.1,908339,supported,False,phase6-v1\nintraday,2018-02-16,0,[11],0,00:00,03:00,SCORE_SHIFT,logistic_regression,0.5099917723724287,0.0,0.5099917723724287,0.1,908339,supported,False,phase6-v1\nintraday,2018-02-16,0,[11],0,00:00,03:00,SCORE_SHIFT,duration_rule,0.022428697356306967,0.0,0.022428697356306967,0.1,908339,supported,False,phase6-v1\nintraday,2018-02-16,0,[11],0,00:00,03:00,CLUSTER_SHIFT,cluster,0.5170554138667345,0.0,0.5170554138667345,0.1,908339,supported,False,phase6-v1\nintraday,2018-02-16,0,[11],0,00:00,03:00,NOVELTY_RATE,kmeans_p99,0.0022645730283517495,0.010022952446636205,-0.007758379418284456,0.015,908339,supported,False,phase6-v1\nintraday,2018-02-16,0,[11],0,00:00,03:00,PERFORMANCE_WARNING,random_forest_recall,0.9233598607526975,0.88977162428725,-0.03358823646544751,0.2,908339,supported,True,phase6-v1\nintraday,2018-02-16,0,[12],6,06:00,09:00,INPUT_SHIFT,flow_duration,0.21751421317719638,0.0,0.21751421317719638,0.1,50,insufficient,False,phase6-v1\nintraday,2018-02-16,0,[12],6,06:00,09:00,INPUT_SHIFT,tot_fwd_pkts,0.13858641358030138,0.0,0.13858641358030138,0.1,50,insufficient,False,phase6-v1\nintraday,2018-02-16,0,[12],6,06:00,09:00,INPUT_SHIFT,totlen_fwd_pkts,0.11804434764630976,0.0,0.11804434764630976,0.1,50,insufficient,False,phase6-v1\nintraday,2018-02-16,0,[12],6,06:00,09:00,INPUT_SHIFT,pkt_len_mean,0.2679807755302446,0.0,0.2679807755302446,0.1,50,insufficient,False,phase6-v1\nintraday,2018-02-16,0,[12],6,06:00,09:00,INPUT_SHIFT,flow_iat_mean,0.23997643113900954,0.0,0.23997643113900954,0.1,50,insufficient,False,phase6-v1\nintraday,2018-02-16,0,[12],6,06:00,09:00,INPUT_SHIFT,flow_pkts_s,0.14314144059457187,0.0,0.14314144059457187,0.1,50,insufficient,False,phase6-v1\nintraday,2018-02-16,0,[12],6,06:00,09:00,SCORE_SHIFT,random_forest,0.8205825838863807,0.0,0.8205825838863807,0.1,50,insufficient,False,phase6-v1\nintraday,2018-02-16,0,[12],6,06:00,09:00,SCORE_SHIFT,logistic_regression,0.8846788934079475,0.0,0.8846788934079475,0.1,50,insufficient,False,phase6-v1\nintraday,2018-02-16,0,[12],6,06:00,09:00,SCORE_SHIFT,duration_rule,0.08194972040364977,0.0,0.08194972040364977,0.1,50,insufficient,False,phase6-v1\nintraday,2018-02-16,0,[12],6,06:00,09:00,CLUSTER_SHIFT,cluster,0.43358274733432944,0.0,0.43358274733432944,0.1,50,insufficient,False,phase6-v1\nintraday,2018-02-16,0,[12],6,06:00,09:00,NOVELTY_RATE,kmeans_p99,0.08,0.010022952446636205,0.0699770475533638,0.015,50,insufficient,False,phase6-v1\nintraday,2018-02-16,0,[12],6,06:00,09:00,PERFORMANCE_WARNING,random_forest_recall,,0.88977162428725,,0.2,50,insufficient,True,phase6-v1\nintraday,2018-02-16,0,[13],9,09:00,12:00,INPUT_SHIFT,flow_duration,0.6997486355918936,0.0,0.6997486355918936,0.1,140132,supported,False,phase6-v1\nintraday,2018-02-16,0,[13],9,09:00,12:00,INPUT_SHIFT,tot_fwd_pkts,0.5119273036136904,0.0,0.5119273036136904,0.1,140132,supported,False,phase6-v1\nintraday,2018-02-16,0,[13],9,09:00,12:00,INPUT_SHIFT,totlen_fwd_pkts,0.251087576406683,0.0,0.251087576406683,0.1,140132,supported,False,phase6-v1\nintraday,2018-02-16,0,[13],9,09:00,12:00,INPUT_SHIFT,pkt_len_mean,0.41991667469944505,0.0,0.41991667469944505,0.1,140132,supported,False,phase6-v1\nintraday,2018-02-16,0,[13],9,09:00,12:00,INPUT_SHIFT,flow_iat_mean,0.6907698142032617,0.0,0.6907698142032617,0.1,140132,supported,False,phase6-v1\nintraday,2018-02-16,0,[13],9,09:00,12:00,INPUT_SHIFT,flow_pkts_s,0.6591597938671825,0.0,0.6591597938671825,0.1,140132,supported,False,phase6-v1\nintraday,2018-02-16,0,[13],9,09:00,12:00,SCORE_SHIFT,random_forest,0.908310624631512,0.0,0.908310624631512,0.1,140132,supported,False,phase6-v1\nintraday,2018-02-16,0,[13],9,09:00,12:00,SCORE_SHIFT,logistic_regression,0.8972045194562305,0.0,0.8972045194562305,0.1,140132,supported,False,phase6-v1\nintraday,2018-02-16,0,[13],9,09:00,12:00,SCORE_SHIFT,duration_rule,0.28976140686252705,0.0,0.28976140686252705,0.1,140132,supported,False,phase6-v1\nintraday,2018-02-16,0,[13],9,09:00,12:00,CLUSTER_SHIFT,cluster,0.6742169630083136,0.0,0.6742169630083136,0.1,140132,supported,False,phase6-v1\nintraday,2018-02-16,0,[13],9,09:00,12:00,NOVELTY_RATE,kmeans_p99,3.568064396426227e-05,0.010022952446636205,-0.009987271802671943,0.015,140132,supported,False,phase6-v1\nintraday,2018-02-16,0,[13],9,09:00,12:00,PERFORMANCE_WARNING,random_forest_recall,1.0,0.88977162428725,-0.11022837571275002,0.2,140132,supported,True,phase6-v1\nintraday,2018-02-16,0,[14],12,12:00,15:00,INPUT_SHIFT,flow_duration,0.13487215913434578,0.0,0.13487215913434578,0.1,53,insufficient,False,phase6-v1\nintraday,2018-02-16,0,[14],12,12:00,15:00,INPUT_SHIFT,tot_fwd_pkts,0.08818009129629821,0.0,0.08818009129629821,0.1,53,insufficient,False,phase6-v1\nintraday,2018-02-16,0,[14],12,12:00,15:00,INPUT_SHIFT,totlen_fwd_pkts,0.03495309306242367,0.0,0.03495309306242367,0.1,53,insufficient,False,phase6-v1\nintraday,2018-02-16,0,[14],12,12:00,15:00,INPUT_SHIFT,pkt_len_mean,0.1385485521993055,0.0,0.1385485521993055,0.1,53,insufficient,False,phase6-v1\nintraday,2018-02-16,0,[14],12,12:00,15:00,INPUT_SHIFT,flow_iat_mean,0.15970841535653396,0.0,0.15970841535653396,0.1,53,insufficient,False,phase6-v1\nintraday,2018-02-16,0,[14],12,12:00,15:00,INPUT_SHIFT,flow_pkts_s,0.1789101778967488,0.0,0.1789101778967488,0.1,53,insufficient,False,phase6-v1\nintraday,2018-02-16,0,[14],12,12:00,15:00,SCORE_SHIFT,random_forest,0.6413391719462133,0.0,0.6413391719462133,0.1,53,insufficient,False,phase6-v1\nintraday,2018-02-16,0,[14],12,12:00,15:00,SCORE_SHIFT,logistic_regression,0.7794688977489869,0.0,0.7794688977489869,0.1,53,insufficient,False,phase6-v1\nintraday,2018-02-16,0,[14],12,12:00,15:00,SCORE_SHIFT,duration_rule,0.05831028467072842,0.0,0.05831028467072842,0.1,53,insufficient,False,phase6-v1\nintraday,2018-02-16,0,[14],12,12:00,15:00,CLUSTER_SHIFT,cluster,0.2862385041155628,0.0,0.2862385041155628,0.1,53,insufficient,False,phase6-v1\nintraday,2018-02-16,0,[14],12,12:00,15:00,NOVELTY_RATE,kmeans_p99,0.0,0.010022952446636205,-0.010022952446636205,0.015,53,insufficient,False,phase6-v1\nintraday,2018-02-16,0,[14],12,12:00,15:00,PERFORMANCE_WARNING,random_forest_recall,,0.88977162428725,,0.2,53,insufficient,True,phase6-v1\nintraday,2018-02-20,0,[15],0,00:00,03:00,INPUT_SHIFT,flow_duration,0.001395183874170428,0.0,0.001395183874170428,0.1,1643268,supported,False,phase6-v1\nintraday,2018-02-20,0,[15],0,00:00,03:00,INPUT_SHIFT,tot_fwd_pkts,0.003702718969514454,0.0,0.003702718969514454,0.1,1643268,supported,False,phase6-v1\nintraday,2018-02-20,0,[15],0,00:00,03:00,INPUT_SHIFT,totlen_fwd_pkts,0.0018504216393858694,0.0,0.0018504216393858694,0.1,1643268,supported,False,phase6-v1\nintraday,2018-02-20,0,[15],0,00:00,03:00,INPUT_SHIFT,pkt_len_mean,0.007248446940835527,0.0,0.007248446940835527,0.1,1643268,supported,False,phase6-v1\nintraday,2018-02-20,0,[15],0,00:00,03:00,INPUT_SHIFT,flow_iat_mean,0.002883388631827215,0.0,0.002883388631827215,0.1,1643268,supported,False,phase6-v1\nintraday,2018-02-20,0,[15],0,00:00,03:00,INPUT_SHIFT,flow_pkts_s,0.004037378918458722,0.0,0.004037378918458722,0.1,1643268,supported,False,phase6-v1\nintraday,2018-02-20,0,[15],0,00:00,03:00,SCORE_SHIFT,random_forest,0.08725046684771591,0.0,0.08725046684771591,0.1,1643268,supported,False,phase6-v1\nintraday,2018-02-20,0,[15],0,00:00,03:00,SCORE_SHIFT,logistic_regression,0.07306969606764234,0.0,0.07306969606764234,0.1,1643268,supported,False,phase6-v1\nintraday,2018-02-20,0,[15],0,00:00,03:00,SCORE_SHIFT,duration_rule,1.595568948117149e-09,0.0,1.595568948117149e-09,0.1,1643268,supported,False,phase6-v1\nintraday,2018-02-20,0,[15],0,00:00,03:00,CLUSTER_SHIFT,cluster,0.032314409273023044,0.0,0.032314409273023044,0.1,1643268,supported,False,phase6-v1\nintraday,2018-02-20,0,[15],0,00:00,03:00,NOVELTY_RATE,kmeans_p99,0.013454287432116976,0.010022952446636205,0.003431334985480771,0.015,1643268,supported,False,phase6-v1\nintraday,2018-02-20,0,[15],0,00:00,03:00,PERFORMANCE_WARNING,random_forest_recall,0.9811794228356336,0.88977162428725,-0.09140779854838366,0.2,1643268,supported,True,phase6-v1\nintraday,2018-02-20,0,[16],3,03:00,06:00,INPUT_SHIFT,flow_duration,0.0012461151680122015,0.0,0.0012461151680122015,0.1,1945025,supported,False,phase6-v1\nintraday,2018-02-20,0,[16],3,03:00,06:00,INPUT_SHIFT,tot_fwd_pkts,0.003306850587443663,0.0,0.003306850587443663,0.1,1945025,supported,False,phase6-v1\nintraday,2018-02-20,0,[16],3,03:00,06:00,INPUT_SHIFT,totlen_fwd_pkts,0.0015018709765510614,0.0,0.0015018709765510614,0.1,1945025,supported,False,phase6-v1\nintraday,2018-02-20,0,[16],3,03:00,06:00,INPUT_SHIFT,pkt_len_mean,0.00542288830308653,0.0,0.00542288830308653,0.1,1945025,supported,False,phase6-v1\nintraday,2018-02-20,0,[16],3,03:00,06:00,INPUT_SHIFT,flow_iat_mean,0.0012060262093850432,0.0,0.0012060262093850432,0.1,1945025,supported,False,phase6-v1\nintraday,2018-02-20,0,[16],3,03:00,06:00,INPUT_SHIFT,flow_pkts_s,0.0041224774405451,0.0,0.0041224774405451,0.1,1945025,supported,False,phase6-v1\nintraday,2018-02-20,0,[16],3,03:00,06:00,SCORE_SHIFT,random_forest,0.08498953095911255,0.0,0.08498953095911255,0.1,1945025,supported,False,phase6-v1\nintraday,2018-02-20,0,[16],3,03:00,06:00,SCORE_SHIFT,logistic_regression,0.06593627198429315,0.0,0.06593627198429315,0.1,1945025,supported,False,phase6-v1\nintraday,2018-02-20,0,[16],3,03:00,06:00,SCORE_SHIFT,duration_rule,0.00034581824926594243,0.0,0.00034581824926594243,0.1,1945025,supported,False,phase6-v1\nintraday,2018-02-20,0,[16],3,03:00,06:00,CLUSTER_SHIFT,cluster,0.033812924885288385,0.0,0.033812924885288385,0.1,1945025,supported,False,phase6-v1\nintraday,2018-02-20,0,[16],3,03:00,06:00,NOVELTY_RATE,kmeans_p99,0.01356743486587576,0.010022952446636205,0.0035444824192395544,0.015,1945025,supported,False,phase6-v1\nintraday,2018-02-20,0,[16],3,03:00,06:00,PERFORMANCE_WARNING,random_forest_recall,,0.88977162428725,,0.2,1945025,insufficient,True,phase6-v1\nintraday,2018-02-20,0,[17],6,06:00,09:00,INPUT_SHIFT,flow_duration,0.0032768089447041708,0.0,0.0032768089447041708,0.1,479872,supported,False,phase6-v1\nintraday,2018-02-20,0,[17],6,06:00,09:00,INPUT_SHIFT,tot_fwd_pkts,0.005836741533847538,0.0,0.005836741533847538,0.1,479872,supported,False,phase6-v1\nintraday,2018-02-20,0,[17],6,06:00,09:00,INPUT_SHIFT,totlen_fwd_pkts,0.0008126142087422791,0.0,0.0008126142087422791,0.1,479872,supported,False,phase6-v1\nintraday,2018-02-20,0,[17],6,06:00,09:00,INPUT_SHIFT,pkt_len_mean,0.0056976552159647776,0.0,0.0056976552159647776,0.1,479872,supported,False,phase6-v1\nintraday,2018-02-20,0,[17],6,06:00,09:00,INPUT_SHIFT,flow_iat_mean,0.0021279781852244547,0.0,0.0021279781852244547,0.1,479872,supported,False,phase6-v1\nintraday,2018-02-20,0,[17],6,06:00,09:00,INPUT_SHIFT,flow_pkts_s,0.0035344879570444543,0.0,0.0035344879570444543,0.1,479872,supported,False,phase6-v1\nintraday,2018-02-20,0,[17],6,06:00,09:00,SCORE_SHIFT,random_forest,0.08896910575780646,0.0,0.08896910575780646,0.1,479872,supported,False,phase6-v1\nintraday,2018-02-20,0,[17],6,06:00,09:00,SCORE_SHIFT,logistic_regression,0.07019895477408623,0.0,0.07019895477408623,0.1,479872,supported,False,phase6-v1\nintraday,2018-02-20,0,[17],6,06:00,09:00,SCORE_SHIFT,duration_rule,7.500407895314962e-05,0.0,7.500407895314962e-05,0.1,479872,supported,False,phase6-v1\nintraday,2018-02-20,0,[17],6,06:00,09:00,CLUSTER_SHIFT,cluster,0.024794107344266555,0.0,0.024794107344266555,0.1,479872,supported,False,phase6-v1\nintraday,2018-02-20,0,[17],6,06:00,09:00,NOVELTY_RATE,kmeans_p99,0.017092057882101892,0.010022952446636205,0.007069105435465687,0.015,479872,supported,False,phase6-v1\nintraday,2018-02-20,0,[17],6,06:00,09:00,PERFORMANCE_WARNING,random_forest_recall,,0.88977162428725,,0.2,479872,insufficient,True,phase6-v1\nintraday,2018-02-20,0,[18],9,09:00,12:00,INPUT_SHIFT,flow_duration,0.00906035681708272,0.0,0.00906035681708272,0.1,3220536,supported,False,phase6-v1\nintraday,2018-02-20,0,[18],9,09:00,12:00,INPUT_SHIFT,tot_fwd_pkts,0.0023076173623922266,0.0,0.0023076173623922266,0.1,3220536,supported,False,phase6-v1\nintraday,2018-02-20,0,[18],9,09:00,12:00,INPUT_SHIFT,totlen_fwd_pkts,0.003928751046764825,0.0,0.003928751046764825,0.1,3220536,supported,False,phase6-v1\nintraday,2018-02-20,0,[18],9,09:00,12:00,INPUT_SHIFT,pkt_len_mean,0.007593761255050779,0.0,0.007593761255050779,0.1,3220536,supported,False,phase6-v1\nintraday,2018-02-20,0,[18],9,09:00,12:00,INPUT_SHIFT,flow_iat_mean,0.009522745872359184,0.0,0.009522745872359184,0.1,3220536,supported,False,phase6-v1\nintraday,2018-02-20,0,[18],9,09:00,12:00,INPUT_SHIFT,flow_pkts_s,0.01113484162545415,0.0,0.01113484162545415,0.1,3220536,supported,False,phase6-v1\nintraday,2018-02-20,0,[18],9,09:00,12:00,SCORE_SHIFT,random_forest,0.09144598915989548,0.0,0.09144598915989548,0.1,3220536,supported,False,phase6-v1\nintraday,2018-02-20,0,[18],9,09:00,12:00,SCORE_SHIFT,logistic_regression,0.07881469230130145,0.0,0.07881469230130145,0.1,3220536,supported,False,phase6-v1\nintraday,2018-02-20,0,[18],9,09:00,12:00,SCORE_SHIFT,duration_rule,0.005421387096121395,0.0,0.005421387096121395,0.1,3220536,supported,False,phase6-v1\nintraday,2018-02-20,0,[18],9,09:00,12:00,CLUSTER_SHIFT,cluster,0.027013967773385387,0.0,0.027013967773385387,0.1,3220536,supported,False,phase6-v1\nintraday,2018-02-20,0,[18],9,09:00,12:00,NOVELTY_RATE,kmeans_p99,0.01116801675249089,0.010022952446636205,0.0011450643058546848,0.015,3220536,supported,False,phase6-v1\nintraday,2018-02-20,0,[18],9,09:00,12:00,PERFORMANCE_WARNING,random_forest_recall,0.8228466059778169,0.88977162428725,0.06692501830943309,0.2,3220536,supported,True,phase6-v1\nintraday,2018-02-20,0,[19],12,12:00,15:00,INPUT_SHIFT,flow_duration,0.0011124035235406753,0.0,0.0011124035235406753,0.1,660047,supported,False,phase6-v1\nintraday,2018-02-20,0,[19],12,12:00,15:00,INPUT_SHIFT,tot_fwd_pkts,0.001483388276510131,0.0,0.001483388276510131,0.1,660047,supported,False,phase6-v1\nintraday,2018-02-20,0,[19],12,12:00,15:00,INPUT_SHIFT,totlen_fwd_pkts,0.002154970295701372,0.0,0.002154970295701372,0.1,660047,supported,False,phase6-v1\nintraday,2018-02-20,0,[19],12,12:00,15:00,INPUT_SHIFT,pkt_len_mean,0.0035725778926609464,0.0,0.0035725778926609464,0.1,660047,supported,False,phase6-v1\nintraday,2018-02-20,0,[19],12,12:00,15:00,INPUT_SHIFT,flow_iat_mean,0.001966328650431416,0.0,0.001966328650431416,0.1,660047,supported,False,phase6-v1\nintraday,2018-02-20,0,[19],12,12:00,15:00,INPUT_SHIFT,flow_pkts_s,0.004601710191743528,0.0,0.004601710191743528,0.1,660047,supported,False,phase6-v1\nintraday,2018-02-20,0,[19],12,12:00,15:00,SCORE_SHIFT,random_forest,0.08768669545477173,0.0,0.08768669545477173,0.1,660047,supported,False,phase6-v1\nintraday,2018-02-20,0,[19],12,12:00,15:00,SCORE_SHIFT,logistic_regression,0.07169966221749635,0.0,0.07169966221749635,0.1,660047,supported,False,phase6-v1\nintraday,2018-02-20,0,[19],12,12:00,15:00,SCORE_SHIFT,duration_rule,0.0009761556645019719,0.0,0.0009761556645019719,0.1,660047,supported,False,phase6-v1\nintraday,2018-02-20,0,[19],12,12:00,15:00,CLUSTER_SHIFT,cluster,0.02513175135620887,0.0,0.02513175135620887,0.1,660047,supported,False,phase6-v1\nintraday,2018-02-20,0,[19],12,12:00,15:00,NOVELTY_RATE,kmeans_p99,0.015600404213639332,0.010022952446636205,0.005577451767003127,0.015,660047,supported,False,phase6-v1\nintraday,2018-02-20,0,[19],12,12:00,15:00,PERFORMANCE_WARNING,random_forest_recall,,0.88977162428725,,0.2,660047,insufficient,True,phase6-v1\nintraday,2018-02-21,0,[20],0,00:00,03:00,INPUT_SHIFT,flow_duration,0.29207486798850985,0.0,0.29207486798850985,0.1,1044705,supported,False,phase6-v1\nintraday,2018-02-21,0,[20],0,00:00,03:00,INPUT_SHIFT,tot_fwd_pkts,0.17534169530989305,0.0,0.17534169530989305,0.1,1044705,supported,False,phase6-v1\nintraday,2018-02-21,0,[20],0,00:00,03:00,INPUT_SHIFT,totlen_fwd_pkts,0.07328227277663482,0.0,0.07328227277663482,0.1,1044705,supported,False,phase6-v1\nintraday,2018-02-21,0,[20],0,00:00,03:00,INPUT_SHIFT,pkt_len_mean,0.22997607655239424,0.0,0.22997607655239424,0.1,1044705,supported,False,phase6-v1\nintraday,2018-02-21,0,[20],0,00:00,03:00,INPUT_SHIFT,flow_iat_mean,0.31645157185939415,0.0,0.31645157185939415,0.1,1044705,supported,False,phase6-v1\nintraday,2018-02-21,0,[20],0,00:00,03:00,INPUT_SHIFT,flow_pkts_s,0.36389635875854587,0.0,0.36389635875854587,0.1,1044705,supported,False,phase6-v1\nintraday,2018-02-21,0,[20],0,00:00,03:00,SCORE_SHIFT,random_forest,0.7071764831972546,0.0,0.7071764831972546,0.1,1044705,supported,False,phase6-v1\nintraday,2018-02-21,0,[20],0,00:00,03:00,SCORE_SHIFT,logistic_regression,0.624407527502431,0.0,0.624407527502431,0.1,1044705,supported,False,phase6-v1\nintraday,2018-02-21,0,[20],0,00:00,03:00,SCORE_SHIFT,duration_rule,0.2165079463523171,0.0,0.2165079463523171,0.1,1044705,supported,False,phase6-v1\nintraday,2018-02-21,0,[20],0,00:00,03:00,CLUSTER_SHIFT,cluster,0.6379210861740551,0.0,0.6379210861740551,0.1,1044705,supported,False,phase6-v1\nintraday,2018-02-21,0,[20],0,00:00,03:00,NOVELTY_RATE,kmeans_p99,7.84910572841137e-05,0.010022952446636205,-0.009944461389352092,0.015,1044705,supported,False,phase6-v1\nintraday,2018-02-21,0,[20],0,00:00,03:00,PERFORMANCE_WARNING,random_forest_recall,0.9985306379480242,0.88977162428725,-0.10875901366077423,0.2,1044705,supported,True,phase6-v1\nintraday,2018-02-21,0,[21],6,06:00,09:00,INPUT_SHIFT,flow_duration,0.303368853291959,0.0,0.303368853291959,0.1,145,insufficient,False,phase6-v1\nintraday,2018-02-21,0,[21],6,06:00,09:00,INPUT_SHIFT,tot_fwd_pkts,0.027170735670563183,0.0,0.027170735670563183,0.1,145,insufficient,False,phase6-v1\nintraday,2018-02-21,0,[21],6,06:00,09:00,INPUT_SHIFT,totlen_fwd_pkts,0.26899135965466703,0.0,0.26899135965466703,0.1,145,insufficient,False,phase6-v1\nintraday,2018-02-21,0,[21],6,06:00,09:00,INPUT_SHIFT,pkt_len_mean,0.111690362565066,0.0,0.111690362565066,0.1,145,insufficient,False,phase6-v1\nintraday,2018-02-21,0,[21],6,06:00,09:00,INPUT_SHIFT,flow_iat_mean,0.3085377676882322,0.0,0.3085377676882322,0.1,145,insufficient,False,phase6-v1\nintraday,2018-02-21,0,[21],6,06:00,09:00,INPUT_SHIFT,flow_pkts_s,0.199193001834511,0.0,0.199193001834511,0.1,145,insufficient,False,phase6-v1\nintraday,2018-02-21,0,[21],6,06:00,09:00,SCORE_SHIFT,random_forest,0.7437374686622645,0.0,0.7437374686622645,0.1,145,insufficient,False,phase6-v1\nintraday,2018-02-21,0,[21],6,06:00,09:00,SCORE_SHIFT,logistic_regression,0.8035615958096476,0.0,0.8035615958096476,0.1,145,insufficient,False,phase6-v1\nintraday,2018-02-21,0,[21],6,06:00,09:00,SCORE_SHIFT,duration_rule,0.07799253796990932,0.0,0.07799253796990932,0.1,145,insufficient,False,phase6-v1\nintraday,2018-02-21,0,[21],6,06:00,09:00,CLUSTER_SHIFT,cluster,0.23575821160534927,0.0,0.23575821160534927,0.1,145,insufficient,False,phase6-v1\nintraday,2018-02-21,0,[21],6,06:00,09:00,NOVELTY_RATE,kmeans_p99,0.4482758620689655,0.010022952446636205,0.4382529096223293,0.015,145,insufficient,False,phase6-v1\nintraday,2018-02-21,0,[21],6,06:00,09:00,PERFORMANCE_WARNING,random_forest_recall,,0.88977162428725,,0.2,145,insufficient,True,phase6-v1\nintraday,2018-02-21,0,[22],9,09:00,12:00,INPUT_SHIFT,flow_duration,0.6203716369638027,0.0,0.6203716369638027,0.1,3725,insufficient,False,phase6-v1\nintraday,2018-02-21,0,[22],9,09:00,12:00,INPUT_SHIFT,tot_fwd_pkts,0.4643386474228326,0.0,0.4643386474228326,0.1,3725,insufficient,False,phase6-v1\nintraday,2018-02-21,0,[22],9,09:00,12:00,INPUT_SHIFT,totlen_fwd_pkts,0.5145265336978144,0.0,0.5145265336978144,0.1,3725,insufficient,False,phase6-v1\nintraday,2018-02-21,0,[22],9,09:00,12:00,INPUT_SHIFT,pkt_len_mean,0.39867076184030126,0.0,0.39867076184030126,0.1,3725,insufficient,False,phase6-v1\nintraday,2018-02-21,0,[22],9,09:00,12:00,INPUT_SHIFT,flow_iat_mean,0.24330840685425195,0.0,0.24330840685425195,0.1,3725,insufficient,False,phase6-v1\nintraday,2018-02-21,0,[22],9,09:00,12:00,INPUT_SHIFT,flow_pkts_s,0.24384226164712458,0.0,0.24384226164712458,0.1,3725,insufficient,False,phase6-v1\nintraday,2018-02-21,0,[22],9,09:00,12:00,SCORE_SHIFT,random_forest,0.8690220172838709,0.0,0.8690220172838709,0.1,3725,insufficient,False,phase6-v1\nintraday,2018-02-21,0,[22],9,09:00,12:00,SCORE_SHIFT,logistic_regression,0.815134587133612,0.0,0.815134587133612,0.1,3725,insufficient,False,phase6-v1\nintraday,2018-02-21,0,[22],9,09:00,12:00,SCORE_SHIFT,duration_rule,0.27273097410424596,0.0,0.27273097410424596,0.1,3725,insufficient,False,phase6-v1\nintraday,2018-02-21,0,[22],9,09:00,12:00,CLUSTER_SHIFT,cluster,0.5449211129658454,0.0,0.5449211129658454,0.1,3725,insufficient,False,phase6-v1\nintraday,2018-02-21,0,[22],9,09:00,12:00,NOVELTY_RATE,kmeans_p99,0.6346308724832215,0.010022952446636205,0.6246079200365853,0.015,3725,insufficient,False,phase6-v1\nintraday,2018-02-21,0,[22],9,09:00,12:00,PERFORMANCE_WARNING,random_forest_recall,0.9890173410404625,0.88977162428725,-0.09924571675321248,0.2,3725,insufficient,True,phase6-v1\nintraday,2018-02-22,0,[23],0,00:00,03:00,INPUT_SHIFT,flow_duration,0.007134626723408359,0.0,0.007134626723408359,0.1,202297,supported,False,phase6-v1\nintraday,2018-02-22,0,[23],0,00:00,03:00,INPUT_SHIFT,tot_fwd_pkts,0.0007086127365298193,0.0,0.0007086127365298193,0.1,202297,supported,False,phase6-v1\nintraday,2018-02-22,0,[23],0,00:00,03:00,INPUT_SHIFT,totlen_fwd_pkts,0.0027920829259834383,0.0,0.0027920829259834383,0.1,202297,supported,False,phase6-v1\nintraday,2018-02-22,0,[23],0,00:00,03:00,INPUT_SHIFT,pkt_len_mean,0.004406997338182739,0.0,0.004406997338182739,0.1,202297,supported,False,phase6-v1\nintraday,2018-02-22,0,[23],0,00:00,03:00,INPUT_SHIFT,flow_iat_mean,0.007644551777294596,0.0,0.007644551777294596,0.1,202297,supported,False,phase6-v1\nintraday,2018-02-22,0,[23],0,00:00,03:00,INPUT_SHIFT,flow_pkts_s,0.00747213111496588,0.0,0.00747213111496588,0.1,202297,supported,False,phase6-v1\nintraday,2018-02-22,0,[23],0,00:00,03:00,SCORE_SHIFT,random_forest,0.1044018733893059,0.0,0.1044018733893059,0.1,202297,supported,False,phase6-v1\nintraday,2018-02-22,0,[23],0,00:00,03:00,SCORE_SHIFT,logistic_regression,0.12647172280982463,0.0,0.12647172280982463,0.1,202297,supported,False,phase6-v1\nintraday,2018-02-22,0,[23],0,00:00,03:00,SCORE_SHIFT,duration_rule,0.001430477568406429,0.0,0.001430477568406429,0.1,202297,supported,False,phase6-v1\nintraday,2018-02-22,0,[23],0,00:00,03:00,CLUSTER_SHIFT,cluster,0.06068180766375298,0.0,0.06068180766375298,0.1,202297,supported,False,phase6-v1\nintraday,2018-02-22,0,[23],0,00:00,03:00,NOVELTY_RATE,kmeans_p99,0.007691661270310484,0.010022952446636205,-0.0023312911763257215,0.015,202297,supported,False,phase6-v1\nintraday,2018-02-22,0,[23],0,00:00,03:00,PERFORMANCE_WARNING,random_forest_recall,0.07407407407407407,0.88977162428725,0.8156975502131759,0.2,202297,insufficient,True,phase6-v1\nintraday,2018-02-22,0,[24],3,03:00,06:00,INPUT_SHIFT,flow_duration,0.006779595283946918,0.0,0.006779595283946918,0.1,228910,supported,False,phase6-v1\nintraday,2018-02-22,0,[24],3,03:00,06:00,INPUT_SHIFT,tot_fwd_pkts,0.0012714501963518698,0.0,0.0012714501963518698,0.1,228910,supported,False,phase6-v1\nintraday,2018-02-22,0,[24],3,03:00,06:00,INPUT_SHIFT,totlen_fwd_pkts,0.0023591413776211335,0.0,0.0023591413776211335,0.1,228910,supported,False,phase6-v1\nintraday,2018-02-22,0,[24],3,03:00,06:00,INPUT_SHIFT,pkt_len_mean,0.0012181479960788083,0.0,0.0012181479960788083,0.1,228910,supported,False,phase6-v1\nintraday,2018-02-22,0,[24],3,03:00,06:00,INPUT_SHIFT,flow_iat_mean,0.01013996071103037,0.0,0.01013996071103037,0.1,228910,supported,False,phase6-v1\nintraday,2018-02-22,0,[24],3,03:00,06:00,INPUT_SHIFT,flow_pkts_s,0.008015815708125226,0.0,0.008015815708125226,0.1,228910,supported,False,phase6-v1\nintraday,2018-02-22,0,[24],3,03:00,06:00,SCORE_SHIFT,random_forest,0.10985501448888828,0.0,0.10985501448888828,0.1,228910,supported,False,phase6-v1\nintraday,2018-02-22,0,[24],3,03:00,06:00,SCORE_SHIFT,logistic_regression,0.13075449681974102,0.0,0.13075449681974102,0.1,228910,supported,False,phase6-v1\nintraday,2018-02-22,0,[24],3,03:00,06:00,SCORE_SHIFT,duration_rule,0.0031339760238631408,0.0,0.0031339760238631408,0.1,228910,supported,False,phase6-v1\nintraday,2018-02-22,0,[24],3,03:00,06:00,CLUSTER_SHIFT,cluster,0.05919612672374181,0.0,0.05919612672374181,0.1,228910,supported,False,phase6-v1\nintraday,2018-02-22,0,[24],3,03:00,06:00,NOVELTY_RATE,kmeans_p99,0.007177493337993098,0.010022952446636205,-0.0028454591086431074,0.015,228910,supported,False,phase6-v1\nintraday,2018-02-22,0,[24],3,03:00,06:00,PERFORMANCE_WARNING,random_forest_recall,0.0,0.88977162428725,0.88977162428725,0.2,228910,insufficient,True,phase6-v1\nintraday,2018-02-22,0,[25],6,06:00,09:00,INPUT_SHIFT,flow_duration,0.014270290291490744,0.0,0.014270290291490744,0.1,74421,supported,False,phase6-v1\nintraday,2018-02-22,0,[25],6,06:00,09:00,INPUT_SHIFT,tot_fwd_pkts,0.001683308338409678,0.0,0.001683308338409678,0.1,74421,supported,False,phase6-v1\nintraday,2018-02-22,0,[25],6,06:00,09:00,INPUT_SHIFT,totlen_fwd_pkts,0.0009127549571579219,0.0,0.0009127549571579219,0.1,74421,supported,False,phase6-v1\nintraday,2018-02-22,0,[25],6,06:00,09:00,INPUT_SHIFT,pkt_len_mean,0.006432707847971656,0.0,0.006432707847971656,0.1,74421,supported,False,phase6-v1\nintraday,2018-02-22,0,[25],6,06:00,09:00,INPUT_SHIFT,flow_iat_mean,0.016336634868218767,0.0,0.016336634868218767,0.1,74421,supported,False,phase6-v1\nintraday,2018-02-22,0,[25],6,06:00,09:00,INPUT_SHIFT,flow_pkts_s,0.012424025087162437,0.0,0.012424025087162437,0.1,74421,supported,False,phase6-v1\nintraday,2018-02-22,0,[25],6,06:00,09:00,SCORE_SHIFT,random_forest,0.12479756235240219,0.0,0.12479756235240219,0.1,74421,supported,False,phase6-v1\nintraday,2018-02-22,0,[25],6,06:00,09:00,SCORE_SHIFT,logistic_regression,0.1467131844105315,0.0,0.1467131844105315,0.1,74421,supported,False,phase6-v1\nintraday,2018-02-22,0,[25],6,06:00,09:00,SCORE_SHIFT,duration_rule,0.00917965924231958,0.0,0.00917965924231958,0.1,74421,supported,False,phase6-v1\nintraday,2018-02-22,0,[25],6,06:00,09:00,CLUSTER_SHIFT,cluster,0.053225242572574015,0.0,0.053225242572574015,0.1,74421,supported,False,phase6-v1\nintraday,2018-02-22,0,[25],6,06:00,09:00,NOVELTY_RATE,kmeans_p99,0.013880490721704894,0.010022952446636205,0.003857538275068689,0.015,74421,supported,False,phase6-v1\nintraday,2018-02-22,0,[25],6,06:00,09:00,PERFORMANCE_WARNING,random_forest_recall,,0.88977162428725,,0.2,74421,insufficient,True,phase6-v1\nintraday,2018-02-22,0,[26],9,09:00,12:00,INPUT_SHIFT,flow_duration,0.012307776151505004,0.0,0.012307776151505004,0.1,419123,supported,False,phase6-v1\nintraday,2018-02-22,0,[26],9,09:00,12:00,INPUT_SHIFT,tot_fwd_pkts,0.000149050593360805,0.0,0.000149050593360805,0.1,419123,supported,False,phase6-v1\nintraday,2018-02-22,0,[26],9,09:00,12:00,INPUT_SHIFT,totlen_fwd_pkts,0.0012563080985479977,0.0,0.0012563080985479977,0.1,419123,supported,False,phase6-v1\nintraday,2018-02-22,0,[26],9,09:00,12:00,INPUT_SHIFT,pkt_len_mean,0.0020888354717294075,0.0,0.0020888354717294075,0.1,419123,supported,False,phase6-v1\nintraday,2018-02-22,0,[26],9,09:00,12:00,INPUT_SHIFT,flow_iat_mean,0.01228173771843121,0.0,0.01228173771843121,0.1,419123,supported,False,phase6-v1\nintraday,2018-02-22,0,[26],9,09:00,12:00,INPUT_SHIFT,flow_pkts_s,0.009735878399165822,0.0,0.009735878399165822,0.1,419123,supported,False,phase6-v1\nintraday,2018-02-22,0,[26],9,09:00,12:00,SCORE_SHIFT,random_forest,0.11422660326907083,0.0,0.11422660326907083,0.1,419123,supported,False,phase6-v1\nintraday,2018-02-22,0,[26],9,09:00,12:00,SCORE_SHIFT,logistic_regression,0.1346438304347303,0.0,0.1346438304347303,0.1,419123,supported,False,phase6-v1\nintraday,2018-02-22,0,[26],9,09:00,12:00,SCORE_SHIFT,duration_rule,0.003626693588922685,0.0,0.003626693588922685,0.1,419123,supported,False,phase6-v1\nintraday,2018-02-22,0,[26],9,09:00,12:00,CLUSTER_SHIFT,cluster,0.06420569389130551,0.0,0.06420569389130551,0.1,419123,supported,False,phase6-v1\nintraday,2018-02-22,0,[26],9,09:00,12:00,NOVELTY_RATE,kmeans_p99,0.008909079196321843,0.010022952446636205,-0.0011138732503143626,0.015,419123,supported,False,phase6-v1\nintraday,2018-02-22,0,[26],9,09:00,12:00,PERFORMANCE_WARNING,random_forest_recall,0.08,0.88977162428725,0.80977162428725,0.2,419123,supported,True,phase6-v1\nintraday,2018-02-22,0,[27],12,12:00,15:00,INPUT_SHIFT,flow_duration,0.0076385740175008085,0.0,0.0076385740175008085,0.1,123815,supported,False,phase6-v1\nintraday,2018-02-22,0,[27],12,12:00,15:00,INPUT_SHIFT,tot_fwd_pkts,0.0004533714132531315,0.0,0.0004533714132531315,0.1,123815,supported,False,phase6-v1\nintraday,2018-02-22,0,[27],12,12:00,15:00,INPUT_SHIFT,totlen_fwd_pkts,0.0027121690828708417,0.0,0.0027121690828708417,0.1,123815,supported,False,phase6-v1\nintraday,2018-02-22,0,[27],12,12:00,15:00,INPUT_SHIFT,pkt_len_mean,0.004968973616539608,0.0,0.004968973616539608,0.1,123815,supported,False,phase6-v1\nintraday,2018-02-22,0,[27],12,12:00,15:00,INPUT_SHIFT,flow_iat_mean,0.006495931111625165,0.0,0.006495931111625165,0.1,123815,supported,False,phase6-v1\nintraday,2018-02-22,0,[27],12,12:00,15:00,INPUT_SHIFT,flow_pkts_s,0.0063074219648565945,0.0,0.0063074219648565945,0.1,123815,supported,False,phase6-v1\nintraday,2018-02-22,0,[27],12,12:00,15:00,SCORE_SHIFT,random_forest,0.10261704668772162,0.0,0.10261704668772162,0.1,123815,supported,False,phase6-v1\nintraday,2018-02-22,0,[27],12,12:00,15:00,SCORE_SHIFT,logistic_regression,0.12205657465105801,0.0,0.12205657465105801,0.1,123815,supported,False,phase6-v1\nintraday,2018-02-22,0,[27],12,12:00,15:00,SCORE_SHIFT,duration_rule,0.0010119513380025778,0.0,0.0010119513380025778,0.1,123815,supported,False,phase6-v1\nintraday,2018-02-22,0,[27],12,12:00,15:00,CLUSTER_SHIFT,cluster,0.06127734373897274,0.0,0.06127734373897274,0.1,123815,supported,False,phase6-v1\nintraday,2018-02-22,0,[27],12,12:00,15:00,NOVELTY_RATE,kmeans_p99,0.008230020595242903,0.010022952446636205,-0.0017929318513933024,0.015,123815,supported,False,phase6-v1\nintraday,2018-02-22,0,[27],12,12:00,15:00,PERFORMANCE_WARNING,random_forest_recall,,0.88977162428725,,0.2,123815,insufficient,True,phase6-v1\ncapture_only,2018-02-22,0,[28],24,,,INPUT_SHIFT,flow_duration,0.9999887034955526,0.0,0.9999887034955526,0.1,9,insufficient,False,phase6-v1\ncapture_only,2018-02-22,0,[28],24,,,INPUT_SHIFT,tot_fwd_pkts,0.46673918263721015,0.0,0.46673918263721015,0.1,9,insufficient,False,phase6-v1\ncapture_only,2018-02-22,0,[28],24,,,INPUT_SHIFT,totlen_fwd_pkts,0.25515194517376366,0.0,0.25515194517376366,0.1,9,insufficient,False,phase6-v1\ncapture_only,2018-02-22,0,[28],24,,,INPUT_SHIFT,pkt_len_mean,0.42401474484204194,0.0,0.42401474484204194,0.1,9,insufficient,False,phase6-v1\ncapture_only,2018-02-22,0,[28],24,,,INPUT_SHIFT,flow_iat_mean,0.9999887034955526,0.0,0.9999887034955526,0.1,9,insufficient,False,phase6-v1\ncapture_only,2018-02-22,0,[28],24,,,INPUT_SHIFT,flow_pkts_s,0.9741404677609196,0.0,0.9741404677609196,0.1,9,insufficient,False,phase6-v1\ncapture_only,2018-02-22,0,[28],24,,,SCORE_SHIFT,random_forest,0.9979008758764536,0.0,0.9979008758764536,0.1,9,insufficient,False,phase6-v1\ncapture_only,2018-02-22,0,[28],24,,,SCORE_SHIFT,logistic_regression,0.9355275552466287,0.0,0.9355275552466287,0.1,9,insufficient,False,phase6-v1\ncapture_only,2018-02-22,0,[28],24,,,SCORE_SHIFT,duration_rule,0.3267277026444694,0.0,0.3267277026444694,0.1,9,insufficient,False,phase6-v1\ncapture_only,2018-02-22,0,[28],24,,,CLUSTER_SHIFT,cluster,0.7538891939602701,0.0,0.7538891939602701,0.1,9,insufficient,False,phase6-v1\ncapture_only,2018-02-22,0,[28],24,,,NOVELTY_RATE,kmeans_p99,0.8888888888888888,0.010022952446636205,0.8788659364422526,0.015,9,insufficient,False,phase6-v1\ncapture_only,2018-02-22,0,[28],24,,,PERFORMANCE_WARNING,random_forest_recall,,0.88977162428725,,0.2,9,insufficient,True,phase6-v1\nintraday,2018-02-23,1,[29],0,00:00,03:00,INPUT_SHIFT,flow_duration,0.10374508830059243,0.0,0.10374508830059243,0.1,123497,supported,False,phase6-v1\nintraday,2018-02-23,1,[29],0,00:00,03:00,INPUT_SHIFT,tot_fwd_pkts,0.027358661752303604,0.0,0.027358661752303604,0.1,123497,supported,False,phase6-v1\nintraday,2018-02-23,1,[29],0,00:00,03:00,INPUT_SHIFT,totlen_fwd_pkts,0.03275918643198847,0.0,0.03275918643198847,0.1,123497,supported,False,phase6-v1\nintraday,2018-02-23,1,[29],0,00:00,03:00,INPUT_SHIFT,pkt_len_mean,0.016395959628478708,0.0,0.016395959628478708,0.1,123497,supported,False,phase6-v1\nintraday,2018-02-23,1,[29],0,00:00,03:00,INPUT_SHIFT,flow_iat_mean,0.09481294617208347,0.0,0.09481294617208347,0.1,123497,supported,False,phase6-v1\nintraday,2018-02-23,1,[29],0,00:00,03:00,INPUT_SHIFT,flow_pkts_s,0.09010877771380665,0.0,0.09010877771380665,0.1,123497,supported,False,phase6-v1\nintraday,2018-02-23,1,[29],0,00:00,03:00,SCORE_SHIFT,random_forest,0.20354660539841646,0.0,0.20354660539841646,0.1,123497,supported,False,phase6-v1\nintraday,2018-02-23,1,[29],0,00:00,03:00,SCORE_SHIFT,logistic_regression,0.22752422923647186,0.0,0.22752422923647186,0.1,123497,supported,False,phase6-v1\nintraday,2018-02-23,1,[29],0,00:00,03:00,SCORE_SHIFT,duration_rule,0.06741636512980405,0.0,0.06741636512980405,0.1,123497,supported,False,phase6-v1\nintraday,2018-02-23,1,[29],0,00:00,03:00,CLUSTER_SHIFT,cluster,0.14208265230110742,0.0,0.14208265230110742,0.1,123497,supported,False,phase6-v1\nintraday,2018-02-23,1,[29],0,00:00,03:00,NOVELTY_RATE,kmeans_p99,0.014389013498303602,0.010022952446636205,0.004366061051667397,0.015,123497,supported,False,phase6-v1\nintraday,2018-02-23,1,[29],0,00:00,03:00,PERFORMANCE_WARNING,random_forest_recall,0.04504504504504504,0.88977162428725,0.844726579242205,0.2,123497,supported,True,phase6-v1\nintraday,2018-02-23,1,[30],3,03:00,06:00,INPUT_SHIFT,flow_duration,0.1231882042399762,0.0,0.1231882042399762,0.1,171752,supported,False,phase6-v1\nintraday,2018-02-23,1,[30],3,03:00,06:00,INPUT_SHIFT,tot_fwd_pkts,0.03642722766511153,0.0,0.03642722766511153,0.1,171752,supported,False,phase6-v1\nintraday,2018-02-23,1,[30],3,03:00,06:00,INPUT_SHIFT,totlen_fwd_pkts,0.02963845050049785,0.0,0.02963845050049785,0.1,171752,supported,False,phase6-v1\nintraday,2018-02-23,1,[30],3,03:00,06:00,INPUT_SHIFT,pkt_len_mean,0.01593358946212165,0.0,0.01593358946212165,0.1,171752,supported,False,phase6-v1\nintraday,2018-02-23,1,[30],3,03:00,06:00,INPUT_SHIFT,flow_iat_mean,0.11810283700848322,0.0,0.11810283700848322,0.1,171752,supported,False,phase6-v1\nintraday,2018-02-23,1,[30],3,03:00,06:00,INPUT_SHIFT,flow_pkts_s,0.10876468812409171,0.0,0.10876468812409171,0.1,171752,supported,False,phase6-v1\nintraday,2018-02-23,1,[30],3,03:00,06:00,SCORE_SHIFT,random_forest,0.2361827762286165,0.0,0.2361827762286165,0.1,171752,supported,False,phase6-v1\nintraday,2018-02-23,1,[30],3,03:00,06:00,SCORE_SHIFT,logistic_regression,0.25323957772206424,0.0,0.25323957772206424,0.1,171752,supported,False,phase6-v1\nintraday,2018-02-23,1,[30],3,03:00,06:00,SCORE_SHIFT,duration_rule,0.08570913812474824,0.0,0.08570913812474824,0.1,171752,supported,False,phase6-v1\nintraday,2018-02-23,1,[30],3,03:00,06:00,CLUSTER_SHIFT,cluster,0.16104409896730285,0.0,0.16104409896730285,0.1,171752,supported,False,phase6-v1\nintraday,2018-02-23,1,[30],3,03:00,06:00,NOVELTY_RATE,kmeans_p99,0.010503516698495505,0.010022952446636205,0.0004805642518593001,0.015,171752,supported,False,phase6-v1\nintraday,2018-02-23,1,[30],3,03:00,06:00,PERFORMANCE_WARNING,random_forest_recall,0.030303030303030304,0.88977162428725,0.8594685939842197,0.2,171752,insufficient,True,phase6-v1\nintraday,2018-02-23,1,[31],6,06:00,09:00,INPUT_SHIFT,flow_duration,0.11879164554273607,0.0,0.11879164554273607,0.1,74913,supported,False,phase6-v1\nintraday,2018-02-23,1,[31],6,06:00,09:00,INPUT_SHIFT,tot_fwd_pkts,0.025375400693060934,0.0,0.025375400693060934,0.1,74913,supported,False,phase6-v1\nintraday,2018-02-23,1,[31],6,06:00,09:00,INPUT_SHIFT,totlen_fwd_pkts,0.02103985642643524,0.0,0.02103985642643524,0.1,74913,supported,False,phase6-v1\nintraday,2018-02-23,1,[31],6,06:00,09:00,INPUT_SHIFT,pkt_len_mean,0.012734807044921385,0.0,0.012734807044921385,0.1,74913,supported,False,phase6-v1\nintraday,2018-02-23,1,[31],6,06:00,09:00,INPUT_SHIFT,flow_iat_mean,0.10502644692069132,0.0,0.10502644692069132,0.1,74913,supported,False,phase6-v1\nintraday,2018-02-23,1,[31],6,06:00,09:00,INPUT_SHIFT,flow_pkts_s,0.1055563158552963,0.0,0.1055563158552963,0.1,74913,supported,False,phase6-v1\nintraday,2018-02-23,1,[31],6,06:00,09:00,SCORE_SHIFT,random_forest,0.2232589801161517,0.0,0.2232589801161517,0.1,74913,supported,False,phase6-v1\nintraday,2018-02-23,1,[31],6,06:00,09:00,SCORE_SHIFT,logistic_regression,0.22979034673849372,0.0,0.22979034673849372,0.1,74913,supported,False,phase6-v1\nintraday,2018-02-23,1,[31],6,06:00,09:00,SCORE_SHIFT,duration_rule,0.0828070142545377,0.0,0.0828070142545377,0.1,74913,supported,False,phase6-v1\nintraday,2018-02-23,1,[31],6,06:00,09:00,CLUSTER_SHIFT,cluster,0.14215261246042393,0.0,0.14215261246042393,0.1,74913,supported,False,phase6-v1\nintraday,2018-02-23,1,[31],6,06:00,09:00,NOVELTY_RATE,kmeans_p99,0.017179928717312082,0.010022952446636205,0.0071569762706758765,0.015,74913,supported,False,phase6-v1\nintraday,2018-02-23,1,[31],6,06:00,09:00,PERFORMANCE_WARNING,random_forest_recall,,0.88977162428725,,0.2,74913,insufficient,True,phase6-v1\nintraday,2018-02-23,1,[32],9,09:00,12:00,INPUT_SHIFT,flow_duration,0.11207508237476915,0.0,0.11207508237476915,0.1,263902,supported,False,phase6-v1\nintraday,2018-02-23,1,[32],9,09:00,12:00,INPUT_SHIFT,tot_fwd_pkts,0.03106000957048858,0.0,0.03106000957048858,0.1,263902,supported,False,phase6-v1\nintraday,2018-02-23,1,[32],9,09:00,12:00,INPUT_SHIFT,totlen_fwd_pkts,0.026905279538109728,0.0,0.026905279538109728,0.1,263902,supported,False,phase6-v1\nintraday,2018-02-23,1,[32],9,09:00,12:00,INPUT_SHIFT,pkt_len_mean,0.012865481088421596,0.0,0.012865481088421596,0.1,263902,supported,False,phase6-v1\nintraday,2018-02-23,1,[32],9,09:00,12:00,INPUT_SHIFT,flow_iat_mean,0.10125773598026978,0.0,0.10125773598026978,0.1,263902,supported,False,phase6-v1\nintraday,2018-02-23,1,[32],9,09:00,12:00,INPUT_SHIFT,flow_pkts_s,0.09676107966696455,0.0,0.09676107966696455,0.1,263902,supported,False,phase6-v1\nintraday,2018-02-23,1,[32],9,09:00,12:00,SCORE_SHIFT,random_forest,0.21053377491990075,0.0,0.21053377491990075,0.1,263902,supported,False,phase6-v1\nintraday,2018-02-23,1,[32],9,09:00,12:00,SCORE_SHIFT,logistic_regression,0.22563970188862234,0.0,0.22563970188862234,0.1,263902,supported,False,phase6-v1\nintraday,2018-02-23,1,[32],9,09:00,12:00,SCORE_SHIFT,duration_rule,0.07793360876381704,0.0,0.07793360876381704,0.1,263902,supported,False,phase6-v1\nintraday,2018-02-23,1,[32],9,09:00,12:00,CLUSTER_SHIFT,cluster,0.14278438477483513,0.0,0.14278438477483513,0.1,263902,supported,False,phase6-v1\nintraday,2018-02-23,1,[32],9,09:00,12:00,NOVELTY_RATE,kmeans_p99,0.014171927457919986,0.010022952446636205,0.00414897501128378,0.015,263902,supported,False,phase6-v1\nintraday,2018-02-23,1,[32],9,09:00,12:00,PERFORMANCE_WARNING,random_forest_recall,0.010752688172043012,0.88977162428725,0.879018936115207,0.2,263902,supported,True,phase6-v1\nintraday,2018-02-23,1,[33],12,12:00,15:00,INPUT_SHIFT,flow_duration,0.10733146740599736,0.0,0.10733146740599736,0.1,75402,supported,False,phase6-v1\nintraday,2018-02-23,1,[33],12,12:00,15:00,INPUT_SHIFT,tot_fwd_pkts,0.02763356206292265,0.0,0.02763356206292265,0.1,75402,supported,False,phase6-v1\nintraday,2018-02-23,1,[33],12,12:00,15:00,INPUT_SHIFT,totlen_fwd_pkts,0.030561664520256385,0.0,0.030561664520256385,0.1,75402,supported,False,phase6-v1\nintraday,2018-02-23,1,[33],12,12:00,15:00,INPUT_SHIFT,pkt_len_mean,0.018081607967197738,0.0,0.018081607967197738,0.1,75402,supported,False,phase6-v1\nintraday,2018-02-23,1,[33],12,12:00,15:00,INPUT_SHIFT,flow_iat_mean,0.09611856587826928,0.0,0.09611856587826928,0.1,75402,supported,False,phase6-v1\nintraday,2018-02-23,1,[33],12,12:00,15:00,INPUT_SHIFT,flow_pkts_s,0.09270860444803015,0.0,0.09270860444803015,0.1,75402,supported,False,phase6-v1\nintraday,2018-02-23,1,[33],12,12:00,15:00,SCORE_SHIFT,random_forest,0.20709624512054645,0.0,0.20709624512054645,0.1,75402,supported,False,phase6-v1\nintraday,2018-02-23,1,[33],12,12:00,15:00,SCORE_SHIFT,logistic_regression,0.23431215808067796,0.0,0.23431215808067796,0.1,75402,supported,False,phase6-v1\nintraday,2018-02-23,1,[33],12,12:00,15:00,SCORE_SHIFT,duration_rule,0.06959897852509547,0.0,0.06959897852509547,0.1,75402,supported,False,phase6-v1\nintraday,2018-02-23,1,[33],12,12:00,15:00,CLUSTER_SHIFT,cluster,0.14844919100482917,0.0,0.14844919100482917,0.1,75402,supported,False,phase6-v1\nintraday,2018-02-23,1,[33],12,12:00,15:00,NOVELTY_RATE,kmeans_p99,0.01247977507227925,0.010022952446636205,0.002456822625643045,0.015,75402,supported,False,phase6-v1\nintraday,2018-02-23,1,[33],12,12:00,15:00,PERFORMANCE_WARNING,random_forest_recall,,0.88977162428725,,0.2,75402,insufficient,True,phase6-v1\nintraday,2018-02-28,1,[34],0,00:00,03:00,INPUT_SHIFT,flow_duration,0.03428517184145108,0.0,0.03428517184145108,0.1,100780,supported,False,phase6-v1\nintraday,2018-02-28,1,[34],0,00:00,03:00,INPUT_SHIFT,tot_fwd_pkts,0.018082823214184026,0.0,0.018082823214184026,0.1,100780,supported,False,phase6-v1\nintraday,2018-02-28,1,[34],0,00:00,03:00,INPUT_SHIFT,totlen_fwd_pkts,0.005304469144498084,0.0,0.005304469144498084,0.1,100780,supported,False,phase6-v1\nintraday,2018-02-28,1,[34],0,00:00,03:00,INPUT_SHIFT,pkt_len_mean,0.00339788557583109,0.0,0.00339788557583109,0.1,100780,supported,False,phase6-v1\nintraday,2018-02-28,1,[34],0,00:00,03:00,INPUT_SHIFT,flow_iat_mean,0.043421777173146325,0.0,0.043421777173146325,0.1,100780,supported,False,phase6-v1\nintraday,2018-02-28,1,[34],0,00:00,03:00,INPUT_SHIFT,flow_pkts_s,0.03162277770684536,0.0,0.03162277770684536,0.1,100780,supported,False,phase6-v1\nintraday,2018-02-28,1,[34],0,00:00,03:00,SCORE_SHIFT,random_forest,0.23553405673598768,0.0,0.23553405673598768,0.1,100780,supported,False,phase6-v1\nintraday,2018-02-28,1,[34],0,00:00,03:00,SCORE_SHIFT,logistic_regression,0.22694459356855845,0.0,0.22694459356855845,0.1,100780,supported,False,phase6-v1\nintraday,2018-02-28,1,[34],0,00:00,03:00,SCORE_SHIFT,duration_rule,0.005787918000044544,0.0,0.005787918000044544,0.1,100780,supported,False,phase6-v1\nintraday,2018-02-28,1,[34],0,00:00,03:00,CLUSTER_SHIFT,cluster,0.05268719703617158,0.0,0.05268719703617158,0.1,100780,supported,False,phase6-v1\nintraday,2018-02-28,1,[34],0,00:00,03:00,NOVELTY_RATE,kmeans_p99,0.04293510617185949,0.010022952446636205,0.032912153725223284,0.015,100780,supported,False,phase6-v1\nintraday,2018-02-28,1,[34],0,00:00,03:00,PERFORMANCE_WARNING,random_forest_recall,0.010243993294840752,0.88977162428725,0.8795276309924093,0.2,100780,supported,True,phase6-v1\nintraday,2018-02-28,1,[35],3,03:00,06:00,INPUT_SHIFT,flow_duration,0.034149527528720684,0.0,0.034149527528720684,0.1,94037,supported,False,phase6-v1\nintraday,2018-02-28,1,[35],3,03:00,06:00,INPUT_SHIFT,tot_fwd_pkts,0.019000888011949154,0.0,0.019000888011949154,0.1,94037,supported,False,phase6-v1\nintraday,2018-02-28,1,[35],3,03:00,06:00,INPUT_SHIFT,totlen_fwd_pkts,0.014089259157408024,0.0,0.014089259157408024,0.1,94037,supported,False,phase6-v1\nintraday,2018-02-28,1,[35],3,03:00,06:00,INPUT_SHIFT,pkt_len_mean,0.023439869775902927,0.0,0.023439869775902927,0.1,94037,supported,False,phase6-v1\nintraday,2018-02-28,1,[35],3,03:00,06:00,INPUT_SHIFT,flow_iat_mean,0.04235725446471675,0.0,0.04235725446471675,0.1,94037,supported,False,phase6-v1\nintraday,2018-02-28,1,[35],3,03:00,06:00,INPUT_SHIFT,flow_pkts_s,0.03721871181443862,0.0,0.03721871181443862,0.1,94037,supported,False,phase6-v1\nintraday,2018-02-28,1,[35],3,03:00,06:00,SCORE_SHIFT,random_forest,0.1893984236976497,0.0,0.1893984236976497,0.1,94037,supported,False,phase6-v1\nintraday,2018-02-28,1,[35],3,03:00,06:00,SCORE_SHIFT,logistic_regression,0.18396198584396917,0.0,0.18396198584396917,0.1,94037,supported,False,phase6-v1\nintraday,2018-02-28,1,[35],3,03:00,06:00,SCORE_SHIFT,duration_rule,0.01858290569010692,0.0,0.01858290569010692,0.1,94037,supported,False,phase6-v1\nintraday,2018-02-28,1,[35],3,03:00,06:00,CLUSTER_SHIFT,cluster,0.08777571984868641,0.0,0.08777571984868641,0.1,94037,supported,False,phase6-v1\nintraday,2018-02-28,1,[35],3,03:00,06:00,NOVELTY_RATE,kmeans_p99,0.02607484288099365,0.010022952446636205,0.016051890434357445,0.015,94037,supported,False,phase6-v1\nintraday,2018-02-28,1,[35],3,03:00,06:00,PERFORMANCE_WARNING,random_forest_recall,,0.88977162428725,,0.2,94037,insufficient,True,phase6-v1\nintraday,2018-02-28,1,[36],6,06:00,09:00,INPUT_SHIFT,flow_duration,0.048155380639902885,0.0,0.048155380639902885,0.1,31948,supported,False,phase6-v1\nintraday,2018-02-28,1,[36],6,06:00,09:00,INPUT_SHIFT,tot_fwd_pkts,0.023710181633196617,0.0,0.023710181633196617,0.1,31948,supported,False,phase6-v1\nintraday,2018-02-28,1,[36],6,06:00,09:00,INPUT_SHIFT,totlen_fwd_pkts,0.012295904901005078,0.0,0.012295904901005078,0.1,31948,supported,False,phase6-v1\nintraday,2018-02-28,1,[36],6,06:00,09:00,INPUT_SHIFT,pkt_len_mean,0.02550634952762175,0.0,0.02550634952762175,0.1,31948,supported,False,phase6-v1\nintraday,2018-02-28,1,[36],6,06:00,09:00,INPUT_SHIFT,flow_iat_mean,0.0479801547993115,0.0,0.0479801547993115,0.1,31948,supported,False,phase6-v1\nintraday,2018-02-28,1,[36],6,06:00,09:00,INPUT_SHIFT,flow_pkts_s,0.02866601872988133,0.0,0.02866601872988133,0.1,31948,supported,False,phase6-v1\nintraday,2018-02-28,1,[36],6,06:00,09:00,SCORE_SHIFT,random_forest,0.20580011352608454,0.0,0.20580011352608454,0.1,31948,supported,False,phase6-v1\nintraday,2018-02-28,1,[36],6,06:00,09:00,SCORE_SHIFT,logistic_regression,0.20580077791078455,0.0,0.20580077791078455,0.1,31948,supported,False,phase6-v1\nintraday,2018-02-28,1,[36],6,06:00,09:00,SCORE_SHIFT,duration_rule,0.005389074619819408,0.0,0.005389074619819408,0.1,31948,supported,False,phase6-v1\nintraday,2018-02-28,1,[36],6,06:00,09:00,CLUSTER_SHIFT,cluster,0.08384576859236126,0.0,0.08384576859236126,0.1,31948,supported,False,phase6-v1\nintraday,2018-02-28,1,[36],6,06:00,09:00,NOVELTY_RATE,kmeans_p99,0.029203706022286214,0.010022952446636205,0.01918075357565001,0.015,31948,supported,False,phase6-v1\nintraday,2018-02-28,1,[36],6,06:00,09:00,PERFORMANCE_WARNING,random_forest_recall,,0.88977162428725,,0.2,31948,insufficient,True,phase6-v1\nintraday,2018-02-28,1,[37],9,09:00,12:00,INPUT_SHIFT,flow_duration,0.034741472530050714,0.0,0.034741472530050714,0.1,180108,supported,False,phase6-v1\nintraday,2018-02-28,1,[37],9,09:00,12:00,INPUT_SHIFT,tot_fwd_pkts,0.018336428484568484,0.0,0.018336428484568484,0.1,180108,supported,False,phase6-v1\nintraday,2018-02-28,1,[37],9,09:00,12:00,INPUT_SHIFT,totlen_fwd_pkts,0.010800381950743603,0.0,0.010800381950743603,0.1,180108,supported,False,phase6-v1\nintraday,2018-02-28,1,[37],9,09:00,12:00,INPUT_SHIFT,pkt_len_mean,0.008237669991323046,0.0,0.008237669991323046,0.1,180108,supported,False,phase6-v1\nintraday,2018-02-28,1,[37],9,09:00,12:00,INPUT_SHIFT,flow_iat_mean,0.04305511990272911,0.0,0.04305511990272911,0.1,180108,supported,False,phase6-v1\nintraday,2018-02-28,1,[37],9,09:00,12:00,INPUT_SHIFT,flow_pkts_s,0.023664401708533983,0.0,0.023664401708533983,0.1,180108,supported,False,phase6-v1\nintraday,2018-02-28,1,[37],9,09:00,12:00,SCORE_SHIFT,random_forest,0.22179569998845702,0.0,0.22179569998845702,0.1,180108,supported,False,phase6-v1\nintraday,2018-02-28,1,[37],9,09:00,12:00,SCORE_SHIFT,logistic_regression,0.21385677974095882,0.0,0.21385677974095882,0.1,180108,supported,False,phase6-v1\nintraday,2018-02-28,1,[37],9,09:00,12:00,SCORE_SHIFT,duration_rule,0.006041274911449165,0.0,0.006041274911449165,0.1,180108,supported,False,phase6-v1\nintraday,2018-02-28,1,[37],9,09:00,12:00,CLUSTER_SHIFT,cluster,0.06839329681624329,0.0,0.06839329681624329,0.1,180108,supported,False,phase6-v1\nintraday,2018-02-28,1,[37],9,09:00,12:00,NOVELTY_RATE,kmeans_p99,0.03631154640549004,0.010022952446636205,0.026288593958853832,0.015,180108,supported,False,phase6-v1\nintraday,2018-02-28,1,[37],9,09:00,12:00,PERFORMANCE_WARNING,random_forest_recall,0.010542123441958252,0.88977162428725,0.8792295008452917,0.2,180108,supported,True,phase6-v1\nintraday,2018-02-28,1,[38],12,12:00,15:00,INPUT_SHIFT,flow_duration,0.04684850361066662,0.0,0.04684850361066662,0.1,34917,supported,False,phase6-v1\nintraday,2018-02-28,1,[38],12,12:00,15:00,INPUT_SHIFT,tot_fwd_pkts,0.020320137839482458,0.0,0.020320137839482458,0.1,34917,supported,False,phase6-v1\nintraday,2018-02-28,1,[38],12,12:00,15:00,INPUT_SHIFT,totlen_fwd_pkts,0.025126099605894743,0.0,0.025126099605894743,0.1,34917,supported,False,phase6-v1\nintraday,2018-02-28,1,[38],12,12:00,15:00,INPUT_SHIFT,pkt_len_mean,0.01939905522234577,0.0,0.01939905522234577,0.1,34917,supported,False,phase6-v1\nintraday,2018-02-28,1,[38],12,12:00,15:00,INPUT_SHIFT,flow_iat_mean,0.04629525935448283,0.0,0.04629525935448283,0.1,34917,supported,False,phase6-v1\nintraday,2018-02-28,1,[38],12,12:00,15:00,INPUT_SHIFT,flow_pkts_s,0.03326477539873215,0.0,0.03326477539873215,0.1,34917,supported,False,phase6-v1\nintraday,2018-02-28,1,[38],12,12:00,15:00,SCORE_SHIFT,random_forest,0.2257505616627123,0.0,0.2257505616627123,0.1,34917,supported,False,phase6-v1\nintraday,2018-02-28,1,[38],12,12:00,15:00,SCORE_SHIFT,logistic_regression,0.23074889082451372,0.0,0.23074889082451372,0.1,34917,supported,False,phase6-v1\nintraday,2018-02-28,1,[38],12,12:00,15:00,SCORE_SHIFT,duration_rule,0.02987154105710514,0.0,0.02987154105710514,0.1,34917,supported,False,phase6-v1\nintraday,2018-02-28,1,[38],12,12:00,15:00,CLUSTER_SHIFT,cluster,0.06951977276364725,0.0,0.06951977276364725,0.1,34917,supported,False,phase6-v1\nintraday,2018-02-28,1,[38],12,12:00,15:00,NOVELTY_RATE,kmeans_p99,0.03703067273820775,0.010022952446636205,0.027007720291571547,0.015,34917,supported,False,phase6-v1\nintraday,2018-02-28,1,[38],12,12:00,15:00,PERFORMANCE_WARNING,random_forest_recall,0.00322234156820623,0.88977162428725,0.8865492827190438,0.2,34917,supported,True,phase6-v1\nintraday,2018-03-01,2,[39],0,00:00,03:00,INPUT_SHIFT,flow_duration,0.054936028164035984,0.0,0.054936028164035984,0.1,55756,supported,False,phase6-v1\nintraday,2018-03-01,2,[39],0,00:00,03:00,INPUT_SHIFT,tot_fwd_pkts,0.02255982412204715,0.0,0.02255982412204715,0.1,55756,supported,False,phase6-v1\nintraday,2018-03-01,2,[39],0,00:00,03:00,INPUT_SHIFT,totlen_fwd_pkts,0.007711059046369558,0.0,0.007711059046369558,0.1,55756,supported,False,phase6-v1\nintraday,2018-03-01,2,[39],0,00:00,03:00,INPUT_SHIFT,pkt_len_mean,0.008023773698284931,0.0,0.008023773698284931,0.1,55756,supported,False,phase6-v1\nintraday,2018-03-01,2,[39],0,00:00,03:00,INPUT_SHIFT,flow_iat_mean,0.055383435473121814,0.0,0.055383435473121814,0.1,55756,supported,False,phase6-v1\nintraday,2018-03-01,2,[39],0,00:00,03:00,INPUT_SHIFT,flow_pkts_s,0.05589269331902,0.0,0.05589269331902,0.1,55756,supported,False,phase6-v1\nintraday,2018-03-01,2,[39],0,00:00,03:00,SCORE_SHIFT,random_forest,0.2370221322449676,0.0,0.2370221322449676,0.1,55756,supported,False,phase6-v1\nintraday,2018-03-01,2,[39],0,00:00,03:00,SCORE_SHIFT,logistic_regression,0.2314723283116496,0.0,0.2314723283116496,0.1,55756,supported,False,phase6-v1\nintraday,2018-03-01,2,[39],0,00:00,03:00,SCORE_SHIFT,duration_rule,0.014112920813755682,0.0,0.014112920813755682,0.1,55756,supported,False,phase6-v1\nintraday,2018-03-01,2,[39],0,00:00,03:00,CLUSTER_SHIFT,cluster,0.04579982235154677,0.0,0.04579982235154677,0.1,55756,supported,False,phase6-v1\nintraday,2018-03-01,2,[39],0,00:00,03:00,NOVELTY_RATE,kmeans_p99,0.035386326135303826,0.010022952446636205,0.02536337368866762,0.015,55756,supported,False,phase6-v1\nintraday,2018-03-01,2,[39],0,00:00,03:00,PERFORMANCE_WARNING,random_forest_recall,0.007062321893197869,0.88977162428725,0.8827093023940521,0.2,55756,supported,True,phase6-v1\nintraday,2018-03-01,2,[40],3,03:00,06:00,INPUT_SHIFT,flow_duration,0.06159812809903126,0.0,0.06159812809903126,0.1,60647,supported,False,phase6-v1\nintraday,2018-03-01,2,[40],3,03:00,06:00,INPUT_SHIFT,tot_fwd_pkts,0.019605669659860634,0.0,0.019605669659860634,0.1,60647,supported,False,phase6-v1\nintraday,2018-03-01,2,[40],3,03:00,06:00,INPUT_SHIFT,totlen_fwd_pkts,0.0144008968277637,0.0,0.0144008968277637,0.1,60647,supported,False,phase6-v1\nintraday,2018-03-01,2,[40],3,03:00,06:00,INPUT_SHIFT,pkt_len_mean,0.014703138443197535,0.0,0.014703138443197535,0.1,60647,supported,False,phase6-v1\nintraday,2018-03-01,2,[40],3,03:00,06:00,INPUT_SHIFT,flow_iat_mean,0.06026789439855339,0.0,0.06026789439855339,0.1,60647,supported,False,phase6-v1\nintraday,2018-03-01,2,[40],3,03:00,06:00,INPUT_SHIFT,flow_pkts_s,0.059866134967538606,0.0,0.059866134967538606,0.1,60647,supported,False,phase6-v1\nintraday,2018-03-01,2,[40],3,03:00,06:00,SCORE_SHIFT,random_forest,0.22303454750433901,0.0,0.22303454750433901,0.1,60647,supported,False,phase6-v1\nintraday,2018-03-01,2,[40],3,03:00,06:00,SCORE_SHIFT,logistic_regression,0.214873288689947,0.0,0.214873288689947,0.1,60647,supported,False,phase6-v1\nintraday,2018-03-01,2,[40],3,03:00,06:00,SCORE_SHIFT,duration_rule,0.030099801415957875,0.0,0.030099801415957875,0.1,60647,supported,False,phase6-v1\nintraday,2018-03-01,2,[40],3,03:00,06:00,CLUSTER_SHIFT,cluster,0.06850098661340692,0.0,0.06850098661340692,0.1,60647,supported,False,phase6-v1\nintraday,2018-03-01,2,[40],3,03:00,06:00,NOVELTY_RATE,kmeans_p99,0.03955677939551833,0.010022952446636205,0.02953382694888212,0.015,60647,supported,False,phase6-v1\nintraday,2018-03-01,2,[40],3,03:00,06:00,PERFORMANCE_WARNING,random_forest_recall,0.008272191051647245,0.88977162428725,0.8814994332356028,0.2,60647,supported,True,phase6-v1\nintraday,2018-03-01,2,[41],6,06:00,09:00,INPUT_SHIFT,flow_duration,0.06787365585536687,0.0,0.06787365585536687,0.1,19227,supported,False,phase6-v1\nintraday,2018-03-01,2,[41],6,06:00,09:00,INPUT_SHIFT,tot_fwd_pkts,0.022353899023332242,0.0,0.022353899023332242,0.1,19227,supported,False,phase6-v1\nintraday,2018-03-01,2,[41],6,06:00,09:00,INPUT_SHIFT,totlen_fwd_pkts,0.006423917253302412,0.0,0.006423917253302412,0.1,19227,supported,False,phase6-v1\nintraday,2018-03-01,2,[41],6,06:00,09:00,INPUT_SHIFT,pkt_len_mean,0.024015388739077284,0.0,0.024015388739077284,0.1,19227,supported,False,phase6-v1\nintraday,2018-03-01,2,[41],6,06:00,09:00,INPUT_SHIFT,flow_iat_mean,0.051718345225399136,0.0,0.051718345225399136,0.1,19227,supported,False,phase6-v1\nintraday,2018-03-01,2,[41],6,06:00,09:00,INPUT_SHIFT,flow_pkts_s,0.053686853657448234,0.0,0.053686853657448234,0.1,19227,supported,False,phase6-v1\nintraday,2018-03-01,2,[41],6,06:00,09:00,SCORE_SHIFT,random_forest,0.21553339250597262,0.0,0.21553339250597262,0.1,19227,supported,False,phase6-v1\nintraday,2018-03-01,2,[41],6,06:00,09:00,SCORE_SHIFT,logistic_regression,0.22175521956454583,0.0,0.22175521956454583,0.1,19227,supported,False,phase6-v1\nintraday,2018-03-01,2,[41],6,06:00,09:00,SCORE_SHIFT,duration_rule,0.01407762104814611,0.0,0.01407762104814611,0.1,19227,supported,False,phase6-v1\nintraday,2018-03-01,2,[41],6,06:00,09:00,CLUSTER_SHIFT,cluster,0.07789852191173904,0.0,0.07789852191173904,0.1,19227,supported,False,phase6-v1\nintraday,2018-03-01,2,[41],6,06:00,09:00,NOVELTY_RATE,kmeans_p99,0.0337026057107193,0.010022952446636205,0.023679653264083094,0.015,19227,supported,False,phase6-v1\nintraday,2018-03-01,2,[41],6,06:00,09:00,PERFORMANCE_WARNING,random_forest_recall,,0.88977162428725,,0.2,19227,insufficient,True,phase6-v1\nintraday,2018-03-01,2,[42],9,09:00,12:00,INPUT_SHIFT,flow_duration,0.055884658990169166,0.0,0.055884658990169166,0.1,81607,supported,False,phase6-v1\nintraday,2018-03-01,2,[42],9,09:00,12:00,INPUT_SHIFT,tot_fwd_pkts,0.019160956353388423,0.0,0.019160956353388423,0.1,81607,supported,False,phase6-v1\nintraday,2018-03-01,2,[42],9,09:00,12:00,INPUT_SHIFT,totlen_fwd_pkts,0.0019658474267782483,0.0,0.0019658474267782483,0.1,81607,supported,False,phase6-v1\nintraday,2018-03-01,2,[42],9,09:00,12:00,INPUT_SHIFT,pkt_len_mean,0.0015679592779333448,0.0,0.0015679592779333448,0.1,81607,supported,False,phase6-v1\nintraday,2018-03-01,2,[42],9,09:00,12:00,INPUT_SHIFT,flow_iat_mean,0.050611828831338974,0.0,0.050611828831338974,0.1,81607,supported,False,phase6-v1\nintraday,2018-03-01,2,[42],9,09:00,12:00,INPUT_SHIFT,flow_pkts_s,0.05770354476912878,0.0,0.05770354476912878,0.1,81607,supported,False,phase6-v1\nintraday,2018-03-01,2,[42],9,09:00,12:00,SCORE_SHIFT,random_forest,0.2502567970884075,0.0,0.2502567970884075,0.1,81607,supported,False,phase6-v1\nintraday,2018-03-01,2,[42],9,09:00,12:00,SCORE_SHIFT,logistic_regression,0.25055289943480724,0.0,0.25055289943480724,0.1,81607,supported,False,phase6-v1\nintraday,2018-03-01,2,[42],9,09:00,12:00,SCORE_SHIFT,duration_rule,0.009337957581554063,0.0,0.009337957581554063,0.1,81607,supported,False,phase6-v1\nintraday,2018-03-01,2,[42],9,09:00,12:00,CLUSTER_SHIFT,cluster,0.03492218341763168,0.0,0.03492218341763168,0.1,81607,supported,False,phase6-v1\nintraday,2018-03-01,2,[42],9,09:00,12:00,NOVELTY_RATE,kmeans_p99,0.04010685357873712,0.010022952446636205,0.030083901132100913,0.015,81607,supported,False,phase6-v1\nintraday,2018-03-01,2,[42],9,09:00,12:00,PERFORMANCE_WARNING,random_forest_recall,0.0047538325471698116,0.88977162428725,0.8850177917400802,0.2,81607,supported,True,phase6-v1\nintraday,2018-03-01,2,[43],12,12:00,15:00,INPUT_SHIFT,flow_duration,0.11704854407238105,0.0,0.11704854407238105,0.1,14236,supported,False,phase6-v1\nintraday,2018-03-01,2,[43],12,12:00,15:00,INPUT_SHIFT,tot_fwd_pkts,0.049851494985907324,0.0,0.049851494985907324,0.1,14236,supported,False,phase6-v1\nintraday,2018-03-01,2,[43],12,12:00,15:00,INPUT_SHIFT,totlen_fwd_pkts,0.039134641188486884,0.0,0.039134641188486884,0.1,14236,supported,False,phase6-v1\nintraday,2018-03-01,2,[43],12,12:00,15:00,INPUT_SHIFT,pkt_len_mean,0.017658188240272165,0.0,0.017658188240272165,0.1,14236,supported,False,phase6-v1\nintraday,2018-03-01,2,[43],12,12:00,15:00,INPUT_SHIFT,flow_iat_mean,0.11019749155154619,0.0,0.11019749155154619,0.1,14236,supported,False,phase6-v1\nintraday,2018-03-01,2,[43],12,12:00,15:00,INPUT_SHIFT,flow_pkts_s,0.11379299230734419,0.0,0.11379299230734419,0.1,14236,supported,False,phase6-v1\nintraday,2018-03-01,2,[43],12,12:00,15:00,SCORE_SHIFT,random_forest,0.2724022062357742,0.0,0.2724022062357742,0.1,14236,supported,False,phase6-v1\nintraday,2018-03-01,2,[43],12,12:00,15:00,SCORE_SHIFT,logistic_regression,0.3005374247282516,0.0,0.3005374247282516,0.1,14236,supported,False,phase6-v1\nintraday,2018-03-01,2,[43],12,12:00,15:00,SCORE_SHIFT,duration_rule,0.09873964365364184,0.0,0.09873964365364184,0.1,14236,supported,False,phase6-v1\nintraday,2018-03-01,2,[43],12,12:00,15:00,CLUSTER_SHIFT,cluster,0.13146895665797267,0.0,0.13146895665797267,0.1,14236,supported,False,phase6-v1\nintraday,2018-03-01,2,[43],12,12:00,15:00,NOVELTY_RATE,kmeans_p99,0.02549873559988761,0.010022952446636205,0.015475783153251405,0.015,14236,supported,False,phase6-v1\nintraday,2018-03-01,2,[43],12,12:00,15:00,PERFORMANCE_WARNING,random_forest_recall,,0.88977162428725,,0.2,14236,insufficient,True,phase6-v1\nintraday,2018-03-02,2,[44],0,00:00,03:00,INPUT_SHIFT,flow_duration,0.08776329932462217,0.0,0.08776329932462217,0.1,162217,supported,False,phase6-v1\nintraday,2018-03-02,2,[44],0,00:00,03:00,INPUT_SHIFT,tot_fwd_pkts,0.029034382394606064,0.0,0.029034382394606064,0.1,162217,supported,False,phase6-v1\nintraday,2018-03-02,2,[44],0,00:00,03:00,INPUT_SHIFT,totlen_fwd_pkts,0.08953144226244593,0.0,0.08953144226244593,0.1,162217,supported,False,phase6-v1\nintraday,2018-03-02,2,[44],0,00:00,03:00,INPUT_SHIFT,pkt_len_mean,0.048894463779714965,0.0,0.048894463779714965,0.1,162217,supported,False,phase6-v1\nintraday,2018-03-02,2,[44],0,00:00,03:00,INPUT_SHIFT,flow_iat_mean,0.053822272252248136,0.0,0.053822272252248136,0.1,162217,supported,False,phase6-v1\nintraday,2018-03-02,2,[44],0,00:00,03:00,INPUT_SHIFT,flow_pkts_s,0.08673649398186115,0.0,0.08673649398186115,0.1,162217,supported,False,phase6-v1\nintraday,2018-03-02,2,[44],0,00:00,03:00,SCORE_SHIFT,random_forest,0.2974028422654074,0.0,0.2974028422654074,0.1,162217,supported,False,phase6-v1\nintraday,2018-03-02,2,[44],0,00:00,03:00,SCORE_SHIFT,logistic_regression,0.3111329619969611,0.0,0.3111329619969611,0.1,162217,supported,False,phase6-v1\nintraday,2018-03-02,2,[44],0,00:00,03:00,SCORE_SHIFT,duration_rule,0.008540339326269526,0.0,0.008540339326269526,0.1,162217,supported,False,phase6-v1\nintraday,2018-03-02,2,[44],0,00:00,03:00,CLUSTER_SHIFT,cluster,0.11369215330201375,0.0,0.11369215330201375,0.1,162217,supported,False,phase6-v1\nintraday,2018-03-02,2,[44],0,00:00,03:00,NOVELTY_RATE,kmeans_p99,0.014326488592440989,0.010022952446636205,0.004303536145804783,0.015,162217,supported,False,phase6-v1\nintraday,2018-03-02,2,[44],0,00:00,03:00,PERFORMANCE_WARNING,random_forest_recall,0.0,0.88977162428725,0.88977162428725,0.2,162217,supported,True,phase6-v1\nintraday,2018-03-02,2,[45],3,03:00,06:00,INPUT_SHIFT,flow_duration,0.09800360851396372,0.0,0.09800360851396372,0.1,201728,supported,False,phase6-v1\nintraday,2018-03-02,2,[45],3,03:00,06:00,INPUT_SHIFT,tot_fwd_pkts,0.0343080560171259,0.0,0.0343080560171259,0.1,201728,supported,False,phase6-v1\nintraday,2018-03-02,2,[45],3,03:00,06:00,INPUT_SHIFT,totlen_fwd_pkts,0.0676973277256625,0.0,0.0676973277256625,0.1,201728,supported,False,phase6-v1\nintraday,2018-03-02,2,[45],3,03:00,06:00,INPUT_SHIFT,pkt_len_mean,0.035459160021563546,0.0,0.035459160021563546,0.1,201728,supported,False,phase6-v1\nintraday,2018-03-02,2,[45],3,03:00,06:00,INPUT_SHIFT,flow_iat_mean,0.06590015654081291,0.0,0.06590015654081291,0.1,201728,supported,False,phase6-v1\nintraday,2018-03-02,2,[45],3,03:00,06:00,INPUT_SHIFT,flow_pkts_s,0.08654156388083692,0.0,0.08654156388083692,0.1,201728,supported,False,phase6-v1\nintraday,2018-03-02,2,[45],3,03:00,06:00,SCORE_SHIFT,random_forest,0.27800584732871925,0.0,0.27800584732871925,0.1,201728,supported,False,phase6-v1\nintraday,2018-03-02,2,[45],3,03:00,06:00,SCORE_SHIFT,logistic_regression,0.30168610319009886,0.0,0.30168610319009886,0.1,201728,supported,False,phase6-v1\nintraday,2018-03-02,2,[45],3,03:00,06:00,SCORE_SHIFT,duration_rule,0.022556542014315617,0.0,0.022556542014315617,0.1,201728,supported,False,phase6-v1\nintraday,2018-03-02,2,[45],3,03:00,06:00,CLUSTER_SHIFT,cluster,0.11595358238914047,0.0,0.11595358238914047,0.1,201728,supported,False,phase6-v1\nintraday,2018-03-02,2,[45],3,03:00,06:00,NOVELTY_RATE,kmeans_p99,0.01241771097715736,0.010022952446636205,0.002394758530521155,0.015,201728,supported,False,phase6-v1\nintraday,2018-03-02,2,[45],3,03:00,06:00,PERFORMANCE_WARNING,random_forest_recall,2.951680982319431e-05,0.88977162428725,0.8897421074774268,0.2,201728,supported,True,phase6-v1\nintraday,2018-03-02,2,[46],6,06:00,09:00,INPUT_SHIFT,flow_duration,0.049440865843064334,0.0,0.049440865843064334,0.1,28429,supported,False,phase6-v1\nintraday,2018-03-02,2,[46],6,06:00,09:00,INPUT_SHIFT,tot_fwd_pkts,0.015962933375408585,0.0,0.015962933375408585,0.1,28429,supported,False,phase6-v1\nintraday,2018-03-02,2,[46],6,06:00,09:00,INPUT_SHIFT,totlen_fwd_pkts,0.006767102620663851,0.0,0.006767102620663851,0.1,28429,supported,False,phase6-v1\nintraday,2018-03-02,2,[46],6,06:00,09:00,INPUT_SHIFT,pkt_len_mean,0.03598540874447759,0.0,0.03598540874447759,0.1,28429,supported,False,phase6-v1\nintraday,2018-03-02,2,[46],6,06:00,09:00,INPUT_SHIFT,flow_iat_mean,0.03099238481197185,0.0,0.03099238481197185,0.1,28429,supported,False,phase6-v1\nintraday,2018-03-02,2,[46],6,06:00,09:00,INPUT_SHIFT,flow_pkts_s,0.03416736308685701,0.0,0.03416736308685701,0.1,28429,supported,False,phase6-v1\nintraday,2018-03-02,2,[46],6,06:00,09:00,SCORE_SHIFT,random_forest,0.20811327523451517,0.0,0.20811327523451517,0.1,28429,supported,False,phase6-v1\nintraday,2018-03-02,2,[46],6,06:00,09:00,SCORE_SHIFT,logistic_regression,0.1969863905022168,0.0,0.1969863905022168,0.1,28429,supported,False,phase6-v1\nintraday,2018-03-02,2,[46],6,06:00,09:00,SCORE_SHIFT,duration_rule,0.015185178973816278,0.0,0.015185178973816278,0.1,28429,supported,False,phase6-v1\nintraday,2018-03-02,2,[46],6,06:00,09:00,CLUSTER_SHIFT,cluster,0.07553174387014644,0.0,0.07553174387014644,0.1,28429,supported,False,phase6-v1\nintraday,2018-03-02,2,[46],6,06:00,09:00,NOVELTY_RATE,kmeans_p99,0.031798515600267335,0.010022952446636205,0.02177556315363113,0.015,28429,supported,False,phase6-v1\nintraday,2018-03-02,2,[46],6,06:00,09:00,PERFORMANCE_WARNING,random_forest_recall,,0.88977162428725,,0.2,28429,insufficient,True,phase6-v1\nintraday,2018-03-02,2,[47],9,09:00,12:00,INPUT_SHIFT,flow_duration,0.06565848622817884,0.0,0.06565848622817884,0.1,277973,supported,False,phase6-v1\nintraday,2018-03-02,2,[47],9,09:00,12:00,INPUT_SHIFT,tot_fwd_pkts,0.012006803004513712,0.0,0.012006803004513712,0.1,277973,supported,False,phase6-v1\nintraday,2018-03-02,2,[47],9,09:00,12:00,INPUT_SHIFT,totlen_fwd_pkts,0.06894992942423495,0.0,0.06894992942423495,0.1,277973,supported,False,phase6-v1\nintraday,2018-03-02,2,[47],9,09:00,12:00,INPUT_SHIFT,pkt_len_mean,0.04638655305121537,0.0,0.04638655305121537,0.1,277973,supported,False,phase6-v1\nintraday,2018-03-02,2,[47],9,09:00,12:00,INPUT_SHIFT,flow_iat_mean,0.044753881492224944,0.0,0.044753881492224944,0.1,277973,supported,False,phase6-v1\nintraday,2018-03-02,2,[47],9,09:00,12:00,INPUT_SHIFT,flow_pkts_s,0.07133552415264556,0.0,0.07133552415264556,0.1,277973,supported,False,phase6-v1\nintraday,2018-03-02,2,[47],9,09:00,12:00,SCORE_SHIFT,random_forest,0.2594133924559393,0.0,0.2594133924559393,0.1,277973,supported,False,phase6-v1\nintraday,2018-03-02,2,[47],9,09:00,12:00,SCORE_SHIFT,logistic_regression,0.26198759998960786,0.0,0.26198759998960786,0.1,277973,supported,False,phase6-v1\nintraday,2018-03-02,2,[47],9,09:00,12:00,SCORE_SHIFT,duration_rule,0.004051552455073135,0.0,0.004051552455073135,0.1,277973,supported,False,phase6-v1\nintraday,2018-03-02,2,[47],9,09:00,12:00,CLUSTER_SHIFT,cluster,0.09116421841599051,0.0,0.09116421841599051,0.1,277973,supported,False,phase6-v1\nintraday,2018-03-02,2,[47],9,09:00,12:00,NOVELTY_RATE,kmeans_p99,0.018890323880376873,0.010022952446636205,0.008867371433740667,0.015,277973,supported,False,phase6-v1\nintraday,2018-03-02,2,[47],9,09:00,12:00,PERFORMANCE_WARNING,random_forest_recall,1.7669405424507466e-05,0.88977162428725,0.8897539548818255,0.2,277973,supported,True,phase6-v1\nintraday,2018-03-02,2,[48],12,12:00,15:00,INPUT_SHIFT,flow_duration,0.07754716923514804,0.0,0.07754716923514804,0.1,78289,supported,False,phase6-v1\nintraday,2018-03-02,2,[48],12,12:00,15:00,INPUT_SHIFT,tot_fwd_pkts,0.025433614887397298,0.0,0.025433614887397298,0.1,78289,supported,False,phase6-v1\nintraday,2018-03-02,2,[48],12,12:00,15:00,INPUT_SHIFT,totlen_fwd_pkts,0.08086813210347604,0.0,0.08086813210347604,0.1,78289,supported,False,phase6-v1\nintraday,2018-03-02,2,[48],12,12:00,15:00,INPUT_SHIFT,pkt_len_mean,0.045522785763220476,0.0,0.045522785763220476,0.1,78289,supported,False,phase6-v1\nintraday,2018-03-02,2,[48],12,12:00,15:00,INPUT_SHIFT,flow_iat_mean,0.054285207971550155,0.0,0.054285207971550155,0.1,78289,supported,False,phase6-v1\nintraday,2018-03-02,2,[48],12,12:00,15:00,INPUT_SHIFT,flow_pkts_s,0.08116426018793448,0.0,0.08116426018793448,0.1,78289,supported,False,phase6-v1\nintraday,2018-03-02,2,[48],12,12:00,15:00,SCORE_SHIFT,random_forest,0.28424463223572055,0.0,0.28424463223572055,0.1,78289,supported,False,phase6-v1\nintraday,2018-03-02,2,[48],12,12:00,15:00,SCORE_SHIFT,logistic_regression,0.30222383431655314,0.0,0.30222383431655314,0.1,78289,supported,False,phase6-v1\nintraday,2018-03-02,2,[48],12,12:00,15:00,SCORE_SHIFT,duration_rule,0.010231738218426073,0.0,0.010231738218426073,0.1,78289,supported,False,phase6-v1\nintraday,2018-03-02,2,[48],12,12:00,15:00,CLUSTER_SHIFT,cluster,0.10685278020364793,0.0,0.10685278020364793,0.1,78289,supported,False,phase6-v1\nintraday,2018-03-02,2,[48],12,12:00,15:00,NOVELTY_RATE,kmeans_p99,0.017908007510633678,0.010022952446636205,0.007885055063997472,0.015,78289,supported,False,phase6-v1\nintraday,2018-03-02,2,[48],12,12:00,15:00,PERFORMANCE_WARNING,random_forest_recall,0.0,0.88977162428725,0.88977162428725,0.2,78289,supported,True,phase6-v1\ndaily,2018-02-14,0,"[0,1,2,3,4,5]",,,,INPUT_SHIFT,flow_duration,0.05874575169818049,0.0,0.05874575169818049,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-14,0,"[0,1,2,3,4,5]",,,,INPUT_SHIFT,tot_fwd_pkts,0.05614865517920083,0.0,0.05614865517920083,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-14,0,"[0,1,2,3,4,5]",,,,INPUT_SHIFT,totlen_fwd_pkts,0.026368403654177416,0.0,0.026368403654177416,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-14,0,"[0,1,2,3,4,5]",,,,INPUT_SHIFT,pkt_len_mean,0.020153726002894452,0.0,0.020153726002894452,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-14,0,"[0,1,2,3,4,5]",,,,INPUT_SHIFT,flow_iat_mean,0.053851370582026464,0.0,0.053851370582026464,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-14,0,"[0,1,2,3,4,5]",,,,INPUT_SHIFT,flow_pkts_s,0.056530183915628224,0.0,0.056530183915628224,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-14,0,"[0,1,2,3,4,5]",,,,SCORE_SHIFT,random_forest,0.18172289256027205,0.0,0.18172289256027205,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-14,0,"[0,1,2,3,4,5]",,,,SCORE_SHIFT,logistic_regression,0.19171130946013795,0.0,0.19171130946013795,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-14,0,"[0,1,2,3,4,5]",,,,SCORE_SHIFT,duration_rule,0.00397181261019636,0.0,0.00397181261019636,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-14,0,"[0,1,2,3,4,5]",,,,CLUSTER_SHIFT,cluster,0.11685858988377541,0.0,0.11685858988377541,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-14,0,"[0,1,2,3,4,5]",,,,NOVELTY_RATE,kmeans_p99,0.004994397158047827,0.010022952446636205,-0.0050285552885883785,0.015,1048575,supported,False,phase6-v1\ndaily,2018-02-14,0,"[0,1,2,3,4,5]",,,,PERFORMANCE_WARNING,random_forest_recall,0.7538988158519907,0.88977162428725,0.13587280843525928,0.2,1048575,supported,True,phase6-v1\ndaily,2018-02-15,0,"[6,7,8,9,10]",,,,INPUT_SHIFT,flow_duration,0.0020818376262895654,0.0,0.0020818376262895654,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-15,0,"[6,7,8,9,10]",,,,INPUT_SHIFT,tot_fwd_pkts,0.0032725073350924104,0.0,0.0032725073350924104,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-15,0,"[6,7,8,9,10]",,,,INPUT_SHIFT,totlen_fwd_pkts,0.00841566109676512,0.0,0.00841566109676512,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-15,0,"[6,7,8,9,10]",,,,INPUT_SHIFT,pkt_len_mean,0.006968975025441936,0.0,0.006968975025441936,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-15,0,"[6,7,8,9,10]",,,,INPUT_SHIFT,flow_iat_mean,0.005375598510925321,0.0,0.005375598510925321,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-15,0,"[6,7,8,9,10]",,,,INPUT_SHIFT,flow_pkts_s,0.008461652427748722,0.0,0.008461652427748722,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-15,0,"[6,7,8,9,10]",,,,SCORE_SHIFT,random_forest,0.098192238554924,0.0,0.098192238554924,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-15,0,"[6,7,8,9,10]",,,,SCORE_SHIFT,logistic_regression,0.11139099046290421,0.0,0.11139099046290421,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-15,0,"[6,7,8,9,10]",,,,SCORE_SHIFT,duration_rule,0.0007511972576988886,0.0,0.0007511972576988886,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-15,0,"[6,7,8,9,10]",,,,CLUSTER_SHIFT,cluster,0.04918987975139846,0.0,0.04918987975139846,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-15,0,"[6,7,8,9,10]",,,,NOVELTY_RATE,kmeans_p99,0.009963998760222206,0.010022952446636205,-5.8953686413999035e-05,0.015,1048575,supported,False,phase6-v1\ndaily,2018-02-15,0,"[6,7,8,9,10]",,,,PERFORMANCE_WARNING,random_forest_recall,0.5997752295325536,0.88977162428725,0.2899963947546964,0.2,1048575,supported,True,phase6-v1\ndaily,2018-02-16,0,"[11,12,13,14]",,,,INPUT_SHIFT,flow_duration,0.09293482235050965,0.0,0.09293482235050965,0.1,1048574,supported,False,phase6-v1\ndaily,2018-02-16,0,"[11,12,13,14]",,,,INPUT_SHIFT,tot_fwd_pkts,0.04342154168852974,0.0,0.04342154168852974,0.1,1048574,supported,False,phase6-v1\ndaily,2018-02-16,0,"[11,12,13,14]",,,,INPUT_SHIFT,totlen_fwd_pkts,0.09901716212166706,0.0,0.09901716212166706,0.1,1048574,supported,False,phase6-v1\ndaily,2018-02-16,0,"[11,12,13,14]",,,,INPUT_SHIFT,pkt_len_mean,0.17253965827747944,0.0,0.17253965827747944,0.1,1048574,supported,False,phase6-v1\ndaily,2018-02-16,0,"[11,12,13,14]",,,,INPUT_SHIFT,flow_iat_mean,0.09781172505998244,0.0,0.09781172505998244,0.1,1048574,supported,False,phase6-v1\ndaily,2018-02-16,0,"[11,12,13,14]",,,,INPUT_SHIFT,flow_pkts_s,0.11039671576802518,0.0,0.11039671576802518,0.1,1048574,supported,False,phase6-v1\ndaily,2018-02-16,0,"[11,12,13,14]",,,,SCORE_SHIFT,random_forest,0.5051212652878785,0.0,0.5051212652878785,0.1,1048574,supported,False,phase6-v1\ndaily,2018-02-16,0,"[11,12,13,14]",,,,SCORE_SHIFT,logistic_regression,0.49528383382853713,0.0,0.49528383382853713,0.1,1048574,supported,False,phase6-v1\ndaily,2018-02-16,0,"[11,12,13,14]",,,,SCORE_SHIFT,duration_rule,0.005464762343011055,0.0,0.005464762343011055,0.1,1048574,supported,False,phase6-v1\ndaily,2018-02-16,0,"[11,12,13,14]",,,,CLUSTER_SHIFT,cluster,0.43771024191610974,0.0,0.43771024191610974,0.1,1048574,supported,False,phase6-v1\ndaily,2018-02-16,0,"[11,12,13,14]",,,,NOVELTY_RATE,kmeans_p99,0.001970294895734588,0.010022952446636205,-0.008052657550901617,0.015,1048574,supported,False,phase6-v1\ndaily,2018-02-16,0,"[11,12,13,14]",,,,PERFORMANCE_WARNING,random_forest_recall,0.9411750044034417,0.88977162428725,-0.05140338011619172,0.2,1048574,supported,True,phase6-v1\ndaily,2018-02-20,0,"[15,16,17,18,19]",,,,INPUT_SHIFT,flow_duration,0.0016280384823329115,0.0,0.0016280384823329115,0.1,7948748,supported,False,phase6-v1\ndaily,2018-02-20,0,"[15,16,17,18,19]",,,,INPUT_SHIFT,tot_fwd_pkts,0.0005942569852936249,0.0,0.0005942569852936249,0.1,7948748,supported,False,phase6-v1\ndaily,2018-02-20,0,"[15,16,17,18,19]",,,,INPUT_SHIFT,totlen_fwd_pkts,0.0009168029608377237,0.0,0.0009168029608377237,0.1,7948748,supported,False,phase6-v1\ndaily,2018-02-20,0,"[15,16,17,18,19]",,,,INPUT_SHIFT,pkt_len_mean,0.004847728111824015,0.0,0.004847728111824015,0.1,7948748,supported,False,phase6-v1\ndaily,2018-02-20,0,"[15,16,17,18,19]",,,,INPUT_SHIFT,flow_iat_mean,0.0017370000804255265,0.0,0.0017370000804255265,0.1,7948748,supported,False,phase6-v1\ndaily,2018-02-20,0,"[15,16,17,18,19]",,,,INPUT_SHIFT,flow_pkts_s,0.004814514106456825,0.0,0.004814514106456825,0.1,7948748,supported,False,phase6-v1\ndaily,2018-02-20,0,"[15,16,17,18,19]",,,,SCORE_SHIFT,random_forest,0.06644108801845656,0.0,0.06644108801845656,0.1,7948748,supported,False,phase6-v1\ndaily,2018-02-20,0,"[15,16,17,18,19]",,,,SCORE_SHIFT,logistic_regression,0.053054646477738714,0.0,0.053054646477738714,0.1,7948748,supported,False,phase6-v1\ndaily,2018-02-20,0,"[15,16,17,18,19]",,,,SCORE_SHIFT,duration_rule,0.001403261708533183,0.0,0.001403261708533183,0.1,7948748,supported,False,phase6-v1\ndaily,2018-02-20,0,"[15,16,17,18,19]",,,,CLUSTER_SHIFT,cluster,0.019031858090806073,0.0,0.019031858090806073,0.1,7948748,supported,False,phase6-v1\ndaily,2018-02-20,0,"[15,16,17,18,19]",,,,NOVELTY_RATE,kmeans_p99,0.012953486511334867,0.010022952446636205,0.0029305340646986616,0.015,7948748,supported,False,phase6-v1\ndaily,2018-02-20,0,"[15,16,17,18,19]",,,,PERFORMANCE_WARNING,random_forest_recall,0.8230656153948951,0.88977162428725,0.0667060088923549,0.2,7948748,supported,True,phase6-v1\ndaily,2018-02-21,0,"[20,21,22]",,,,INPUT_SHIFT,flow_duration,0.2820507351875505,0.0,0.2820507351875505,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-21,0,"[20,21,22]",,,,INPUT_SHIFT,tot_fwd_pkts,0.17361909584842872,0.0,0.17361909584842872,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-21,0,"[20,21,22]",,,,INPUT_SHIFT,totlen_fwd_pkts,0.07183150037827149,0.0,0.07183150037827149,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-21,0,"[20,21,22]",,,,INPUT_SHIFT,pkt_len_mean,0.21916610425506866,0.0,0.21916610425506866,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-21,0,"[20,21,22]",,,,INPUT_SHIFT,flow_iat_mean,0.3094882184676699,0.0,0.3094882184676699,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-21,0,"[20,21,22]",,,,INPUT_SHIFT,flow_pkts_s,0.35695503695795777,0.0,0.35695503695795777,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-21,0,"[20,21,22]",,,,SCORE_SHIFT,random_forest,0.7020835215002608,0.0,0.7020835215002608,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-21,0,"[20,21,22]",,,,SCORE_SHIFT,logistic_regression,0.6185490990344056,0.0,0.6185490990344056,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-21,0,"[20,21,22]",,,,SCORE_SHIFT,duration_rule,0.21089022220913314,0.0,0.21089022220913314,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-21,0,"[20,21,22]",,,,CLUSTER_SHIFT,cluster,0.6257344502466008,0.0,0.6257344502466008,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-21,0,"[20,21,22]",,,,NOVELTY_RATE,kmeans_p99,0.002394678492239468,0.010022952446636205,-0.0076282739543967375,0.015,1048575,supported,False,phase6-v1\ndaily,2018-02-21,0,"[20,21,22]",,,,PERFORMANCE_WARNING,random_forest_recall,0.9985067074571569,0.88977162428725,-0.10873508316990688,0.2,1048575,supported,True,phase6-v1\ndaily,2018-02-22,0,"[23,24,25,26,27,28]",,,,INPUT_SHIFT,flow_duration,0.009208081956550243,0.0,0.009208081956550243,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-22,0,"[23,24,25,26,27,28]",,,,INPUT_SHIFT,tot_fwd_pkts,0.0002931218450800485,0.0,0.0002931218450800485,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-22,0,"[23,24,25,26,27,28]",,,,INPUT_SHIFT,totlen_fwd_pkts,0.0015326474263058947,0.0,0.0015326474263058947,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-22,0,"[23,24,25,26,27,28]",,,,INPUT_SHIFT,pkt_len_mean,0.002513642644790454,0.0,0.002513642644790454,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-22,0,"[23,24,25,26,27,28]",,,,INPUT_SHIFT,flow_iat_mean,0.010132679779147901,0.0,0.010132679779147901,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-22,0,"[23,24,25,26,27,28]",,,,INPUT_SHIFT,flow_pkts_s,0.008308537596546716,0.0,0.008308537596546716,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-22,0,"[23,24,25,26,27,28]",,,,SCORE_SHIFT,random_forest,0.10775497121888447,0.0,0.10775497121888447,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-22,0,"[23,24,25,26,27,28]",,,,SCORE_SHIFT,logistic_regression,0.12711763838499127,0.0,0.12711763838499127,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-22,0,"[23,24,25,26,27,28]",,,,SCORE_SHIFT,duration_rule,0.00292954276308513,0.0,0.00292954276308513,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-22,0,"[23,24,25,26,27,28]",,,,CLUSTER_SHIFT,cluster,0.06041831264982809,0.0,0.06041831264982809,0.1,1048575,supported,False,phase6-v1\ndaily,2018-02-22,0,"[23,24,25,26,27,28]",,,,NOVELTY_RATE,kmeans_p99,0.00857640130653506,0.010022952446636205,-0.001446551140101145,0.015,1048575,supported,False,phase6-v1\ndaily,2018-02-22,0,"[23,24,25,26,27,28]",,,,PERFORMANCE_WARNING,random_forest_recall,0.0718232044198895,0.88977162428725,0.8179484198673604,0.2,1048575,supported,True,phase6-v1\ndaily,2018-02-23,1,"[29,30,31,32,33]",,,,INPUT_SHIFT,flow_duration,0.11295211916026414,0.0,0.11295211916026414,0.1,709466,supported,False,phase6-v1\ndaily,2018-02-23,1,"[29,30,31,32,33]",,,,INPUT_SHIFT,tot_fwd_pkts,0.030372202563331636,0.0,0.030372202563331636,0.1,709466,supported,False,phase6-v1\ndaily,2018-02-23,1,"[29,30,31,32,33]",,,,INPUT_SHIFT,totlen_fwd_pkts,0.027901035428867633,0.0,0.027901035428867633,0.1,709466,supported,False,phase6-v1\ndaily,2018-02-23,1,"[29,30,31,32,33]",,,,INPUT_SHIFT,pkt_len_mean,0.013978873193008546,0.0,0.013978873193008546,0.1,709466,supported,False,phase6-v1\ndaily,2018-02-23,1,"[29,30,31,32,33]",,,,INPUT_SHIFT,flow_iat_mean,0.10346465063638031,0.0,0.10346465063638031,0.1,709466,supported,False,phase6-v1\ndaily,2018-02-23,1,"[29,30,31,32,33]",,,,INPUT_SHIFT,flow_pkts_s,0.09848035023502129,0.0,0.09848035023502129,0.1,709466,supported,False,phase6-v1\ndaily,2018-02-23,1,"[29,30,31,32,33]",,,,SCORE_SHIFT,random_forest,0.2116030637630316,0.0,0.2116030637630316,0.1,709466,supported,False,phase6-v1\ndaily,2018-02-23,1,"[29,30,31,32,33]",,,,SCORE_SHIFT,logistic_regression,0.22384354193893377,0.0,0.22384354193893377,0.1,709466,supported,False,phase6-v1\ndaily,2018-02-23,1,"[29,30,31,32,33]",,,,SCORE_SHIFT,duration_rule,0.077415460705203,0.0,0.077415460705203,0.1,709466,supported,False,phase6-v1\ndaily,2018-02-23,1,"[29,30,31,32,33]",,,,CLUSTER_SHIFT,cluster,0.14633877834652026,0.0,0.14633877834652026,0.1,709466,supported,False,phase6-v1\ndaily,2018-02-23,1,"[29,30,31,32,33]",,,,NOVELTY_RATE,kmeans_p99,0.013459418774120253,0.010022952446636205,0.0034364663274840477,0.015,709466,supported,False,phase6-v1\ndaily,2018-02-23,1,"[29,30,31,32,33]",,,,PERFORMANCE_WARNING,random_forest_recall,0.02127659574468085,0.88977162428725,0.8684950285425691,0.2,709466,supported,True,phase6-v1\ndaily,2018-02-28,1,"[34,35,36,37,38]",,,,INPUT_SHIFT,flow_duration,0.03423862054183397,0.0,0.03423862054183397,0.1,441790,supported,False,phase6-v1\ndaily,2018-02-28,1,"[34,35,36,37,38]",,,,INPUT_SHIFT,tot_fwd_pkts,0.017513228758704584,0.0,0.017513228758704584,0.1,441790,supported,False,phase6-v1\ndaily,2018-02-28,1,"[34,35,36,37,38]",,,,INPUT_SHIFT,totlen_fwd_pkts,0.009478829649203222,0.0,0.009478829649203222,0.1,441790,supported,False,phase6-v1\ndaily,2018-02-28,1,"[34,35,36,37,38]",,,,INPUT_SHIFT,pkt_len_mean,0.01051299481376409,0.0,0.01051299481376409,0.1,441790,supported,False,phase6-v1\ndaily,2018-02-28,1,"[34,35,36,37,38]",,,,INPUT_SHIFT,flow_iat_mean,0.041791634557923646,0.0,0.041791634557923646,0.1,441790,supported,False,phase6-v1\ndaily,2018-02-28,1,"[34,35,36,37,38]",,,,INPUT_SHIFT,flow_pkts_s,0.027468242562519552,0.0,0.027468242562519552,0.1,441790,supported,False,phase6-v1\ndaily,2018-02-28,1,"[34,35,36,37,38]",,,,SCORE_SHIFT,random_forest,0.21047561942139512,0.0,0.21047561942139512,0.1,441790,supported,False,phase6-v1\ndaily,2018-02-28,1,"[34,35,36,37,38]",,,,SCORE_SHIFT,logistic_regression,0.19937801439707667,0.0,0.19937801439707667,0.1,441790,supported,False,phase6-v1\ndaily,2018-02-28,1,"[34,35,36,37,38]",,,,SCORE_SHIFT,duration_rule,0.00936787412432031,0.0,0.00936787412432031,0.1,441790,supported,False,phase6-v1\ndaily,2018-02-28,1,"[34,35,36,37,38]",,,,CLUSTER_SHIFT,cluster,0.06219284870286714,0.0,0.06219284870286714,0.1,441790,supported,False,phase6-v1\ndaily,2018-02-28,1,"[34,35,36,37,38]",,,,NOVELTY_RATE,kmeans_p99,0.03518640077865049,0.010022952446636205,0.025163448332014286,0.015,441790,supported,False,phase6-v1\ndaily,2018-02-28,1,"[34,35,36,37,38]",,,,PERFORMANCE_WARNING,random_forest_recall,0.010188406053007963,0.88977162428725,0.879583218234242,0.2,441790,supported,True,phase6-v1\ndaily,2018-03-01,2,"[39,40,41,42,43]",,,,INPUT_SHIFT,flow_duration,0.05741968235080792,0.0,0.05741968235080792,0.1,231473,supported,False,phase6-v1\ndaily,2018-03-01,2,"[39,40,41,42,43]",,,,INPUT_SHIFT,tot_fwd_pkts,0.019340103320166564,0.0,0.019340103320166564,0.1,231473,supported,False,phase6-v1\ndaily,2018-03-01,2,"[39,40,41,42,43]",,,,INPUT_SHIFT,totlen_fwd_pkts,0.007151205478709343,0.0,0.007151205478709343,0.1,231473,supported,False,phase6-v1\ndaily,2018-03-01,2,"[39,40,41,42,43]",,,,INPUT_SHIFT,pkt_len_mean,0.0068610474962121255,0.0,0.0068610474962121255,0.1,231473,supported,False,phase6-v1\ndaily,2018-03-01,2,"[39,40,41,42,43]",,,,INPUT_SHIFT,flow_iat_mean,0.05322549256065831,0.0,0.05322549256065831,0.1,231473,supported,False,phase6-v1\ndaily,2018-03-01,2,"[39,40,41,42,43]",,,,INPUT_SHIFT,flow_pkts_s,0.056869235233643414,0.0,0.056869235233643414,0.1,231473,supported,False,phase6-v1\ndaily,2018-03-01,2,"[39,40,41,42,43]",,,,SCORE_SHIFT,random_forest,0.2316415018524772,0.0,0.2316415018524772,0.1,231473,supported,False,phase6-v1\ndaily,2018-03-01,2,"[39,40,41,42,43]",,,,SCORE_SHIFT,logistic_regression,0.22508458160437042,0.0,0.22508458160437042,0.1,231473,supported,False,phase6-v1\ndaily,2018-03-01,2,"[39,40,41,42,43]",,,,SCORE_SHIFT,duration_rule,0.01855127274659412,0.0,0.01855127274659412,0.1,231473,supported,False,phase6-v1\ndaily,2018-03-01,2,"[39,40,41,42,43]",,,,CLUSTER_SHIFT,cluster,0.045597689946667665,0.0,0.045597689946667665,0.1,231473,supported,False,phase6-v1\ndaily,2018-03-01,2,"[39,40,41,42,43]",,,,NOVELTY_RATE,kmeans_p99,0.037395290163431585,0.010022952446636205,0.02737233771679538,0.015,231473,supported,False,phase6-v1\ndaily,2018-03-01,2,"[39,40,41,42,43]",,,,PERFORMANCE_WARNING,random_forest_recall,0.006360055784585677,0.88977162428725,0.8834115685026643,0.2,231473,supported,True,phase6-v1\ndaily,2018-03-02,2,"[44,45,46,47,48]",,,,INPUT_SHIFT,flow_duration,0.07623472560189469,0.0,0.07623472560189469,0.1,748636,supported,False,phase6-v1\ndaily,2018-03-02,2,"[44,45,46,47,48]",,,,INPUT_SHIFT,tot_fwd_pkts,0.019397701181352592,0.0,0.019397701181352592,0.1,748636,supported,False,phase6-v1\ndaily,2018-03-02,2,"[44,45,46,47,48]",,,,INPUT_SHIFT,totlen_fwd_pkts,0.06942530356886138,0.0,0.06942530356886138,0.1,748636,supported,False,phase6-v1\ndaily,2018-03-02,2,"[44,45,46,47,48]",,,,INPUT_SHIFT,pkt_len_mean,0.04058719082677931,0.0,0.04058719082677931,0.1,748636,supported,False,phase6-v1\ndaily,2018-03-02,2,"[44,45,46,47,48]",,,,INPUT_SHIFT,flow_iat_mean,0.04944001621199456,0.0,0.04944001621199456,0.1,748636,supported,False,phase6-v1\ndaily,2018-03-02,2,"[44,45,46,47,48]",,,,INPUT_SHIFT,flow_pkts_s,0.0753913754349004,0.0,0.0753913754349004,0.1,748636,supported,False,phase6-v1\ndaily,2018-03-02,2,"[44,45,46,47,48]",,,,SCORE_SHIFT,random_forest,0.2660164467586206,0.0,0.2660164467586206,0.1,748636,supported,False,phase6-v1\ndaily,2018-03-02,2,"[44,45,46,47,48]",,,,SCORE_SHIFT,logistic_regression,0.27292369931574334,0.0,0.27292369931574334,0.1,748636,supported,False,phase6-v1\ndaily,2018-03-02,2,"[44,45,46,47,48]",,,,SCORE_SHIFT,duration_rule,0.009827566878720988,0.0,0.009827566878720988,0.1,748636,supported,False,phase6-v1\ndaily,2018-03-02,2,"[44,45,46,47,48]",,,,CLUSTER_SHIFT,cluster,0.09671358856945385,0.0,0.09671358856945385,0.1,748636,supported,False,phase6-v1\ndaily,2018-03-02,2,"[44,45,46,47,48]",,,,NOVELTY_RATE,kmeans_p99,0.016544756063026623,0.010022952446636205,0.0065218036163904175,0.015,748636,supported,False,phase6-v1\ndaily,2018-03-02,2,"[44,45,46,47,48]",,,,PERFORMANCE_WARNING,random_forest_recall,1.395926685930455e-05,0.88977162428725,0.8897576650203907,0.2,748636,supported,True,phase6-v1\n', 'reports/phase6_replay_batches.csv': 'replay_seq,source_day,window_hour,split_rank,rows,bytes,batch_id,temporally_eligible,entered_utc,finished_utc,scoring_seconds,processing_seconds,rows_per_second\n0,2018-02-14,0,0,235564,23025634,0,235564,2026-11-24T04:31:46.596+00:00,2026-11-24T04:32:14.449+00:00,13.200405,27.85338,8457.285973910526\n1,2018-02-14,3,0,142824,14244752,1,142824,2026-11-24T04:32:14.733+00:00,2026-11-24T04:32:28.823+00:00,8.044395,14.089925,10136.604701586419\n2,2018-02-14,6,0,67293,7830067,2,67293,2026-11-24T04:32:29.588+00:00,2026-11-24T04:32:37.332+00:00,4.612171,7.743988,8689.708713391601\n3,2018-02-14,9,0,498994,41269899,3,498994,2026-11-24T04:32:37.519+00:00,2026-11-24T04:33:07.521+00:00,22.320028,30.002314,16631.8504632676\n4,2018-02-14,12,0,103895,10021630,4,103895,2026-11-24T04:33:07.694+00:00,2026-11-24T04:33:16.179+00:00,5.554753,8.485603,12243.67908797996\n5,2018-02-14,24,0,5,10815,5,0,2026-11-24T04:33:16.350+00:00,2026-11-24T04:33:19.196+00:00,1.012241,2.846005,1.7568486351921377\n6,2018-02-15,0,0,197798,22484352,6,197798,2026-11-24T04:33:19.352+00:00,2026-11-24T04:33:33.719+00:00,9.71548,14.366399,13768.098742071692\n7,2018-02-15,3,0,191551,22005602,7,191551,2026-11-24T04:33:33.895+00:00,2026-11-24T04:33:47.172+00:00,9.324984,13.277034,14427.24331352921\n8,2018-02-15,6,0,87581,10213596,8,87581,2026-11-24T04:33:47.343+00:00,2026-11-24T04:33:55.389+00:00,5.184959,8.045681,10885.467619210853\n9,2018-02-15,9,0,463466,53299300,9,463466,2026-11-24T04:33:55.585+00:00,2026-11-24T04:34:23.357+00:00,20.75945,27.772889,16687.712970732\n10,2018-02-15,12,0,108179,12348875,10,108179,2026-11-24T04:34:23.537+00:00,2026-11-24T04:34:32.102+00:00,5.659702,8.564302,12631.385488274469\n11,2018-02-16,0,0,908339,100167851,11,908339,2026-11-24T04:34:32.262+00:00,2026-11-24T04:35:23.756+00:00,39.393513,51.494359,17639.58261913698\n12,2018-02-16,6,0,50,14397,12,50,2026-11-24T04:35:23.938+00:00,2026-11-24T04:35:26.525+00:00,0.776139,2.587771,19.321647858330586\n13,2018-02-16,9,0,140132,1048093,13,140132,2026-11-24T04:35:26.676+00:00,2026-11-24T04:35:36.486+00:00,6.573928,9.809699,14285.045851050068\n14,2018-02-16,12,0,53,17843,14,53,2026-11-24T04:35:36.651+00:00,2026-11-24T04:35:39.160+00:00,0.893947,2.509745,21.117683270611156\n15,2018-02-20,0,0,1643268,189414902,15,1643268,2026-11-24T04:35:39.315+00:00,2026-11-24T04:36:48.970+00:00,53.89013,69.654391,23591.735946697172\n16,2018-02-20,3,0,1945025,224391626,16,1945025,2026-11-24T04:36:49.139+00:00,2026-11-24T04:38:01.963+00:00,55.39407,72.823109,26708.89813287153\n17,2018-02-20,6,0,479872,55340499,17,479872,2026-11-24T04:38:02.131+00:00,2026-11-24T04:38:35.630+00:00,24.541216,33.498691,14325.09706125532\n18,2018-02-20,9,0,3220536,370157176,18,3220536,2026-11-24T04:38:35.825+00:00,2026-11-24T04:40:15.227+00:00,77.074272,99.40203,32399.09687961101\n19,2018-02-20,12,0,660047,76596955,19,660047,2026-11-24T04:40:15.406+00:00,2026-11-24T04:40:53.946+00:00,29.100627,38.539236,17126.6238905203\n20,2018-02-21,0,0,1044705,96429353,20,1044705,2026-11-24T04:40:54.108+00:00,2026-11-24T04:41:53.259+00:00,45.499756,59.150475,17661.81928378428\n21,2018-02-21,6,0,145,26940,21,145,2026-11-24T04:41:53.421+00:00,2026-11-24T04:41:56.135+00:00,0.804269,2.713854,53.42955074222858\n22,2018-02-21,9,0,3725,489284,22,3725,2026-11-24T04:41:56.289+00:00,2026-11-24T04:41:58.927+00:00,0.916892,2.638109,1411.99624427952\n23,2018-02-22,0,0,202297,23478718,23,202297,2026-11-24T04:41:59.083+00:00,2026-11-24T04:42:12.917+00:00,9.643731,13.834059,14623.112421307442\n24,2018-02-22,3,0,228910,26631797,24,228910,2026-11-24T04:42:13.076+00:00,2026-11-24T04:42:28.230+00:00,10.846644,15.153985,15105.597636529268\n25,2018-02-22,6,0,74421,8857431,25,74421,2026-11-24T04:42:28.386+00:00,2026-11-24T04:42:35.191+00:00,4.205315,6.804672,10936.750514940324\n26,2018-02-22,9,0,419123,48729152,26,419123,2026-11-24T04:42:35.346+00:00,2026-11-24T04:43:01.172+00:00,19.137555,25.825593,16228.978749878075\n27,2018-02-22,12,0,123815,14389137,27,123815,2026-11-24T04:43:01.357+00:00,2026-11-24T04:43:11.050+00:00,6.274863,9.693221,12773.359856336712\n28,2018-02-22,24,0,9,11229,28,0,2026-11-24T04:43:11.205+00:00,2026-11-24T04:43:13.545+00:00,0.738853,2.339934,3.8462623304759878\n29,2018-02-23,0,1,123497,15785945,29,123497,2026-11-24T04:43:13.703+00:00,2026-11-24T04:43:23.364+00:00,6.217202,9.661311,12782.633744012588\n30,2018-02-23,3,1,171752,21770970,30,171752,2026-11-24T04:43:23.519+00:00,2026-11-24T04:43:35.462+00:00,8.335926,11.943021,14380.951017334726\n31,2018-02-23,6,1,74913,9515011,31,74913,2026-11-24T04:43:35.623+00:00,2026-11-24T04:43:42.637+00:00,4.241213,7.014455,10679.80334894158\n32,2018-02-23,9,1,263902,33354353,32,263902,2026-11-24T04:43:42.798+00:00,2026-11-24T04:44:00.331+00:00,12.531139,17.533108,15051.638306226143\n33,2018-02-23,12,1,75402,9638159,33,75402,2026-11-24T04:44:00.502+00:00,2026-11-24T04:44:07.579+00:00,4.25029,7.076472,10655.309594950704\n34,2018-02-28,0,1,100780,11569601,34,100780,2026-11-24T04:44:07.786+00:00,2026-11-24T04:44:15.870+00:00,5.245384,8.083361,12467.586193416328\n35,2018-02-28,3,1,94037,11313707,35,94037,2026-11-24T04:44:16.026+00:00,2026-11-24T04:44:23.947+00:00,4.961984,7.921113,11871.690253629762\n36,2018-02-28,6,1,31948,3856925,36,31948,2026-11-24T04:44:24.106+00:00,2026-11-24T04:44:28.428+00:00,2.210153,4.321188,7393.337202639644\n37,2018-02-28,9,1,180108,20968816,37,180108,2026-11-24T04:44:28.601+00:00,2026-11-24T04:44:41.236+00:00,8.612683,12.634521,14255.229778794148\n38,2018-02-28,12,1,34917,4286869,38,34917,2026-11-24T04:44:41.396+00:00,2026-11-24T04:44:46.076+00:00,2.430522,4.680453,7460.17532918288\n39,2018-03-01,0,2,55756,6761025,39,55756,2026-11-24T04:44:46.245+00:00,2026-11-24T04:44:52.042+00:00,3.251138,5.796202,9619.402498394638\n40,2018-03-01,3,2,60647,7794969,40,60647,2026-11-24T04:44:52.194+00:00,2026-11-24T04:44:58.138+00:00,3.382432,5.943712,10203.556296132787\n41,2018-03-01,6,2,19227,2432266,41,19227,2026-11-24T04:44:58.302+00:00,2026-11-24T04:45:01.760+00:00,1.62152,3.458238,5559.767719861964\n42,2018-03-01,9,2,81607,9691355,42,81607,2026-11-24T04:45:01.915+00:00,2026-11-24T04:45:09.199+00:00,4.455186,7.28428,11203.166270379503\n43,2018-03-01,12,2,14236,1842018,43,14236,2026-11-24T04:45:09.365+00:00,2026-11-24T04:45:12.742+00:00,1.557509,3.377165,4215.369992286429\n44,2018-03-02,0,2,162217,19872204,44,162217,2026-11-24T04:45:12.884+00:00,2026-11-24T04:45:24.661+00:00,7.984089,11.776418,13774.731841210121\n45,2018-03-02,3,2,201728,24803522,45,201728,2026-11-24T04:45:24.827+00:00,2026-11-24T04:45:38.379+00:00,9.56091,13.551863,14885.628640136045\n46,2018-03-02,6,2,28429,3523659,46,28429,2026-11-24T04:45:38.544+00:00,2026-11-24T04:45:42.675+00:00,2.128862,4.131421,6881.167520811848\n47,2018-03-02,9,2,277973,34288063,47,277973,2026-11-24T04:45:42.837+00:00,2026-11-24T04:46:00.637+00:00,12.814812,17.800363,15616.14221013358\n48,2018-03-02,12,2,78289,9765685,48,78289,2026-11-24T04:46:00.819+00:00,2026-11-24T04:46:07.909+00:00,4.334429,7.089467,11043.00224544384\n', 'reports/phase11_synthetic/alerts.csv': 'scenario,seed,detector,label_delay_windows,alert_index\nstationary,23013,KSWIN,0,1209\nstationary,23013,JS,0,767\nstationary,23013,PSI,0,767\nstationary,23013,PSI,0,3327\nstationary,23013,PSI,0,3711\nstationary,23014,KSWIN,0,2195\nstationary,23014,KSWIN,0,2943\nstationary,23014,DDM,0,39\nstationary,23014,DDM,0,178\nstationary,23014,DDM,0,261\nstationary,23014,DDM,1,64\nstationary,23014,DDM,1,192\nstationary,23014,DDM,1,320\nstationary,23014,DDM,3,192\nstationary,23014,DDM,3,320\nstationary,23014,DDM,3,448\nstationary,23015,KSWIN,0,1305\nstationary,23015,KSWIN,0,1825\nstationary,23015,KSWIN,0,2789\nstationary,23015,KSWIN,0,3640\nabrupt,23013,ADWIN,0,2111\nabrupt,23013,PageHinkley,0,2190\nabrupt,23013,KSWIN,0,1209\nabrupt,23013,KSWIN,0,2060\nabrupt,23013,KSWIN,0,3030\nabrupt,23013,JS,0,767\nabrupt,23013,JS,0,2111\nabrupt,23013,JS,0,2175\nabrupt,23013,JS,0,2239\nabrupt,23013,JS,0,2303\nabrupt,23013,JS,0,2367\nabrupt,23013,JS,0,2431\nabrupt,23013,JS,0,2495\nabrupt,23013,JS,0,2559\nabrupt,23013,JS,0,2623\nabrupt,23013,JS,0,2687\nabrupt,23013,JS,0,2751\nabrupt,23013,JS,0,2815\nabrupt,23013,JS,0,2879\nabrupt,23013,JS,0,2943\nabrupt,23013,JS,0,3007\nabrupt,23013,JS,0,3071\nabrupt,23013,JS,0,3135\nabrupt,23013,JS,0,3199\nabrupt,23013,JS,0,3263\nabrupt,23013,JS,0,3327\nabrupt,23013,JS,0,3391\nabrupt,23013,JS,0,3455\nabrupt,23013,JS,0,3519\nabrupt,23013,JS,0,3583\nabrupt,23013,JS,0,3647\nabrupt,23013,JS,0,3711\nabrupt,23013,JS,0,3775\nabrupt,23013,JS,0,3839\nabrupt,23013,JS,0,3903\nabrupt,23013,JS,0,3967\nabrupt,23013,JS,0,4031\nabrupt,23013,JS,0,4095\nabrupt,23013,PSI,0,767\nabrupt,23013,PSI,0,2111\nabrupt,23013,PSI,0,2175\nabrupt,23013,PSI,0,2239\nabrupt,23013,PSI,0,2303\nabrupt,23013,PSI,0,2367\nabrupt,23013,PSI,0,2431\nabrupt,23013,PSI,0,2495\nabrupt,23013,PSI,0,2559\nabrupt,23013,PSI,0,2623\nabrupt,23013,PSI,0,2687\nabrupt,23013,PSI,0,2751\nabrupt,23013,PSI,0,2815\nabrupt,23013,PSI,0,2879\nabrupt,23013,PSI,0,2943\nabrupt,23013,PSI,0,3007\nabrupt,23013,PSI,0,3071\nabrupt,23013,PSI,0,3135\nabrupt,23013,PSI,0,3199\nabrupt,23013,PSI,0,3263\nabrupt,23013,PSI,0,3327\nabrupt,23013,PSI,0,3391\nabrupt,23013,PSI,0,3455\nabrupt,23013,PSI,0,3519\nabrupt,23013,PSI,0,3583\nabrupt,23013,PSI,0,3647\nabrupt,23013,PSI,0,3711\nabrupt,23013,PSI,0,3775\nabrupt,23013,PSI,0,3839\nabrupt,23013,PSI,0,3903\nabrupt,23013,PSI,0,3967\nabrupt,23013,PSI,0,4031\nabrupt,23013,PSI,0,4095\nabrupt,23013,FixedJS,0,2111\nabrupt,23013,FixedJS,0,2175\nabrupt,23013,FixedJS,0,2239\nabrupt,23013,FixedJS,0,2303\nabrupt,23013,FixedJS,0,2367\nabrupt,23013,FixedJS,0,2431\nabrupt,23013,FixedJS,0,2495\nabrupt,23013,FixedJS,0,2559\nabrupt,23013,FixedJS,0,2623\nabrupt,23013,FixedJS,0,2687\nabrupt,23013,FixedJS,0,2751\nabrupt,23013,FixedJS,0,2815\nabrupt,23013,FixedJS,0,2879\nabrupt,23013,FixedJS,0,2943\nabrupt,23013,FixedJS,0,3007\nabrupt,23013,FixedJS,0,3071\nabrupt,23013,FixedJS,0,3135\nabrupt,23013,FixedJS,0,3199\nabrupt,23013,FixedJS,0,3263\nabrupt,23013,FixedJS,0,3327\nabrupt,23013,FixedJS,0,3391\nabrupt,23013,FixedJS,0,3455\nabrupt,23013,FixedJS,0,3519\nabrupt,23013,FixedJS,0,3583\nabrupt,23013,FixedJS,0,3647\nabrupt,23013,FixedJS,0,3711\nabrupt,23013,FixedJS,0,3775\nabrupt,23013,FixedJS,0,3839\nabrupt,23013,FixedJS,0,3903\nabrupt,23013,FixedJS,0,3967\nabrupt,23013,FixedJS,0,4031\nabrupt,23013,FixedJS,0,4095\nabrupt,23014,ADWIN,0,2111\nabrupt,23014,PageHinkley,0,2188\nabrupt,23014,KSWIN,0,2067\nabrupt,23014,DDM,0,39\nabrupt,23014,DDM,0,178\nabrupt,23014,DDM,0,261\nabrupt,23014,DDM,1,64\nabrupt,23014,DDM,1,192\nabrupt,23014,DDM,1,320\nabrupt,23014,DDM,3,192\nabrupt,23014,DDM,3,320\nabrupt,23014,DDM,3,448\nabrupt,23014,JS,0,2111\nabrupt,23014,JS,0,2175\nabrupt,23014,JS,0,2239\nabrupt,23014,JS,0,2303\nabrupt,23014,JS,0,2367\nabrupt,23014,JS,0,2431\nabrupt,23014,JS,0,2495\nabrupt,23014,JS,0,2559\nabrupt,23014,JS,0,2623\nabrupt,23014,JS,0,2687\nabrupt,23014,JS,0,2751\nabrupt,23014,JS,0,2815\nabrupt,23014,JS,0,2879\nabrupt,23014,JS,0,2943\nabrupt,23014,JS,0,3007\nabrupt,23014,JS,0,3071\nabrupt,23014,JS,0,3135\nabrupt,23014,JS,0,3199\nabrupt,23014,JS,0,3263\nabrupt,23014,JS,0,3327\nabrupt,23014,JS,0,3391\nabrupt,23014,JS,0,3455\nabrupt,23014,JS,0,3519\nabrupt,23014,JS,0,3583\nabrupt,23014,JS,0,3647\nabrupt,23014,JS,0,3711\nabrupt,23014,JS,0,3775\nabrupt,23014,JS,0,3839\nabrupt,23014,JS,0,3903\nabrupt,23014,JS,0,3967\nabrupt,23014,JS,0,4031\nabrupt,23014,JS,0,4095\nabrupt,23014,PSI,0,2111\nabrupt,23014,PSI,0,2175\nabrupt,23014,PSI,0,2239\nabrupt,23014,PSI,0,2303\nabrupt,23014,PSI,0,2367\nabrupt,23014,PSI,0,2431\nabrupt,23014,PSI,0,2495\nabrupt,23014,PSI,0,2559\nabrupt,23014,PSI,0,2623\nabrupt,23014,PSI,0,2687\nabrupt,23014,PSI,0,2751\nabrupt,23014,PSI,0,2815\nabrupt,23014,PSI,0,2879\nabrupt,23014,PSI,0,2943\nabrupt,23014,PSI,0,3007\nabrupt,23014,PSI,0,3071\nabrupt,23014,PSI,0,3135\nabrupt,23014,PSI,0,3199\nabrupt,23014,PSI,0,3263\nabrupt,23014,PSI,0,3327\nabrupt,23014,PSI,0,3391\nabrupt,23014,PSI,0,3455\nabrupt,23014,PSI,0,3519\nabrupt,23014,PSI,0,3583\nabrupt,23014,PSI,0,3647\nabrupt,23014,PSI,0,3711\nabrupt,23014,PSI,0,3775\nabrupt,23014,PSI,0,3839\nabrupt,23014,PSI,0,3903\nabrupt,23014,PSI,0,3967\nabrupt,23014,PSI,0,4031\nabrupt,23014,PSI,0,4095\nabrupt,23014,FixedJS,0,2111\nabrupt,23014,FixedJS,0,2175\nabrupt,23014,FixedJS,0,2239\nabrupt,23014,FixedJS,0,2303\nabrupt,23014,FixedJS,0,2367\nabrupt,23014,FixedJS,0,2431\nabrupt,23014,FixedJS,0,2495\nabrupt,23014,FixedJS,0,2559\nabrupt,23014,FixedJS,0,2623\nabrupt,23014,FixedJS,0,2687\nabrupt,23014,FixedJS,0,2751\nabrupt,23014,FixedJS,0,2815\nabrupt,23014,FixedJS,0,2879\nabrupt,23014,FixedJS,0,2943\nabrupt,23014,FixedJS,0,3007\nabrupt,23014,FixedJS,0,3071\nabrupt,23014,FixedJS,0,3135\nabrupt,23014,FixedJS,0,3199\nabrupt,23014,FixedJS,0,3263\nabrupt,23014,FixedJS,0,3327\nabrupt,23014,FixedJS,0,3391\nabrupt,23014,FixedJS,0,3455\nabrupt,23014,FixedJS,0,3519\nabrupt,23014,FixedJS,0,3583\nabrupt,23014,FixedJS,0,3647\nabrupt,23014,FixedJS,0,3711\nabrupt,23014,FixedJS,0,3775\nabrupt,23014,FixedJS,0,3839\nabrupt,23014,FixedJS,0,3903\nabrupt,23014,FixedJS,0,3967\nabrupt,23014,FixedJS,0,4031\nabrupt,23014,FixedJS,0,4095\nabrupt,23015,ADWIN,0,2111\nabrupt,23015,PageHinkley,0,2182\nabrupt,23015,KSWIN,0,1305\nabrupt,23015,KSWIN,0,1825\nabrupt,23015,KSWIN,0,2072\nabrupt,23015,KSWIN,0,2356\nabrupt,23015,JS,0,2111\nabrupt,23015,JS,0,2175\nabrupt,23015,JS,0,2239\nabrupt,23015,JS,0,2303\nabrupt,23015,JS,0,2367\nabrupt,23015,JS,0,2431\nabrupt,23015,JS,0,2495\nabrupt,23015,JS,0,2559\nabrupt,23015,JS,0,2623\nabrupt,23015,JS,0,2687\nabrupt,23015,JS,0,2751\nabrupt,23015,JS,0,2815\nabrupt,23015,JS,0,2879\nabrupt,23015,JS,0,2943\nabrupt,23015,JS,0,3007\nabrupt,23015,JS,0,3071\nabrupt,23015,JS,0,3135\nabrupt,23015,JS,0,3199\nabrupt,23015,JS,0,3263\nabrupt,23015,JS,0,3327\nabrupt,23015,JS,0,3391\nabrupt,23015,JS,0,3455\nabrupt,23015,JS,0,3519\nabrupt,23015,JS,0,3583\nabrupt,23015,JS,0,3647\nabrupt,23015,JS,0,3711\nabrupt,23015,JS,0,3775\nabrupt,23015,JS,0,3839\nabrupt,23015,JS,0,3903\nabrupt,23015,JS,0,3967\nabrupt,23015,JS,0,4031\nabrupt,23015,JS,0,4095\nabrupt,23015,PSI,0,2111\nabrupt,23015,PSI,0,2175\nabrupt,23015,PSI,0,2239\nabrupt,23015,PSI,0,2303\nabrupt,23015,PSI,0,2367\nabrupt,23015,PSI,0,2431\nabrupt,23015,PSI,0,2495\nabrupt,23015,PSI,0,2559\nabrupt,23015,PSI,0,2623\nabrupt,23015,PSI,0,2687\nabrupt,23015,PSI,0,2751\nabrupt,23015,PSI,0,2815\nabrupt,23015,PSI,0,2879\nabrupt,23015,PSI,0,2943\nabrupt,23015,PSI,0,3007\nabrupt,23015,PSI,0,3071\nabrupt,23015,PSI,0,3135\nabrupt,23015,PSI,0,3199\nabrupt,23015,PSI,0,3263\nabrupt,23015,PSI,0,3327\nabrupt,23015,PSI,0,3391\nabrupt,23015,PSI,0,3455\nabrupt,23015,PSI,0,3519\nabrupt,23015,PSI,0,3583\nabrupt,23015,PSI,0,3647\nabrupt,23015,PSI,0,3711\nabrupt,23015,PSI,0,3775\nabrupt,23015,PSI,0,3839\nabrupt,23015,PSI,0,3903\nabrupt,23015,PSI,0,3967\nabrupt,23015,PSI,0,4031\nabrupt,23015,PSI,0,4095\nabrupt,23015,FixedJS,0,2111\nabrupt,23015,FixedJS,0,2175\nabrupt,23015,FixedJS,0,2239\nabrupt,23015,FixedJS,0,2303\nabrupt,23015,FixedJS,0,2367\nabrupt,23015,FixedJS,0,2431\nabrupt,23015,FixedJS,0,2495\nabrupt,23015,FixedJS,0,2559\nabrupt,23015,FixedJS,0,2623\nabrupt,23015,FixedJS,0,2687\nabrupt,23015,FixedJS,0,2751\nabrupt,23015,FixedJS,0,2815\nabrupt,23015,FixedJS,0,2879\nabrupt,23015,FixedJS,0,2943\nabrupt,23015,FixedJS,0,3007\nabrupt,23015,FixedJS,0,3071\nabrupt,23015,FixedJS,0,3135\nabrupt,23015,FixedJS,0,3199\nabrupt,23015,FixedJS,0,3263\nabrupt,23015,FixedJS,0,3327\nabrupt,23015,FixedJS,0,3391\nabrupt,23015,FixedJS,0,3455\nabrupt,23015,FixedJS,0,3519\nabrupt,23015,FixedJS,0,3583\nabrupt,23015,FixedJS,0,3647\nabrupt,23015,FixedJS,0,3711\nabrupt,23015,FixedJS,0,3775\nabrupt,23015,FixedJS,0,3839\nabrupt,23015,FixedJS,0,3903\nabrupt,23015,FixedJS,0,3967\nabrupt,23015,FixedJS,0,4031\nabrupt,23015,FixedJS,0,4095\ngradual,23013,ADWIN,0,2463\ngradual,23013,ADWIN,0,2975\ngradual,23013,PageHinkley,0,2646\ngradual,23013,JS,0,1599\ngradual,23013,JS,0,2367\ngradual,23013,JS,0,2431\ngradual,23013,JS,0,2495\ngradual,23013,JS,0,2559\ngradual,23013,JS,0,2623\ngradual,23013,JS,0,2687\ngradual,23013,JS,0,2751\ngradual,23013,JS,0,2815\ngradual,23013,JS,0,2879\ngradual,23013,JS,0,2943\ngradual,23013,JS,0,3007\ngradual,23013,JS,0,3071\ngradual,23013,JS,0,3135\ngradual,23013,JS,0,3199\ngradual,23013,JS,0,3263\ngradual,23013,JS,0,3327\ngradual,23013,JS,0,3391\ngradual,23013,JS,0,3455\ngradual,23013,JS,0,3519\ngradual,23013,JS,0,3583\ngradual,23013,JS,0,3647\ngradual,23013,JS,0,3711\ngradual,23013,JS,0,3775\ngradual,23013,JS,0,3839\ngradual,23013,JS,0,3903\ngradual,23013,JS,0,3967\ngradual,23013,JS,0,4031\ngradual,23013,JS,0,4095\ngradual,23013,PSI,0,2559\ngradual,23013,PSI,0,2879\ngradual,23013,PSI,0,2943\ngradual,23013,PSI,0,3007\ngradual,23013,PSI,0,3071\ngradual,23013,PSI,0,3135\ngradual,23013,PSI,0,3199\ngradual,23013,PSI,0,3263\ngradual,23013,PSI,0,3327\ngradual,23013,PSI,0,3391\ngradual,23013,PSI,0,3455\ngradual,23013,PSI,0,3519\ngradual,23013,PSI,0,3583\ngradual,23013,PSI,0,3647\ngradual,23013,PSI,0,3711\ngradual,23013,PSI,0,3775\ngradual,23013,PSI,0,3839\ngradual,23013,PSI,0,3903\ngradual,23013,PSI,0,3967\ngradual,23013,PSI,0,4031\ngradual,23013,PSI,0,4095\ngradual,23013,FixedJS,0,2367\ngradual,23013,FixedJS,0,2495\ngradual,23013,FixedJS,0,2559\ngradual,23013,FixedJS,0,2623\ngradual,23013,FixedJS,0,2687\ngradual,23013,FixedJS,0,2751\ngradual,23013,FixedJS,0,2815\ngradual,23013,FixedJS,0,2879\ngradual,23013,FixedJS,0,2943\ngradual,23013,FixedJS,0,3007\ngradual,23013,FixedJS,0,3071\ngradual,23013,FixedJS,0,3135\ngradual,23013,FixedJS,0,3199\ngradual,23013,FixedJS,0,3263\ngradual,23013,FixedJS,0,3327\ngradual,23013,FixedJS,0,3391\ngradual,23013,FixedJS,0,3455\ngradual,23013,FixedJS,0,3519\ngradual,23013,FixedJS,0,3583\ngradual,23013,FixedJS,0,3647\ngradual,23013,FixedJS,0,3711\ngradual,23013,FixedJS,0,3775\ngradual,23013,FixedJS,0,3839\ngradual,23013,FixedJS,0,3903\ngradual,23013,FixedJS,0,3967\ngradual,23013,FixedJS,0,4031\ngradual,23013,FixedJS,0,4095\ngradual,23014,ADWIN,0,2463\ngradual,23014,ADWIN,0,3071\ngradual,23014,PageHinkley,0,2636\ngradual,23014,DDM,0,810\ngradual,23014,DDM,0,860\ngradual,23014,DDM,1,832\ngradual,23014,DDM,1,896\ngradual,23014,DDM,3,960\ngradual,23014,DDM,3,1024\ngradual,23014,JS,0,2303\ngradual,23014,JS,0,2431\ngradual,23014,JS,0,2559\ngradual,23014,JS,0,2623\ngradual,23014,JS,0,2687\ngradual,23014,JS,0,2751\ngradual,23014,JS,0,2815\ngradual,23014,JS,0,2879\ngradual,23014,JS,0,2943\ngradual,23014,JS,0,3007\ngradual,23014,JS,0,3071\ngradual,23014,JS,0,3135\ngradual,23014,JS,0,3199\ngradual,23014,JS,0,3263\ngradual,23014,JS,0,3327\ngradual,23014,JS,0,3391\ngradual,23014,JS,0,3455\ngradual,23014,JS,0,3519\ngradual,23014,JS,0,3583\ngradual,23014,JS,0,3647\ngradual,23014,JS,0,3711\ngradual,23014,JS,0,3775\ngradual,23014,JS,0,3839\ngradual,23014,JS,0,3903\ngradual,23014,JS,0,3967\ngradual,23014,JS,0,4031\ngradual,23014,JS,0,4095\ngradual,23014,PSI,0,2751\ngradual,23014,PSI,0,2815\ngradual,23014,PSI,0,2879\ngradual,23014,PSI,0,2943\ngradual,23014,PSI,0,3007\ngradual,23014,PSI,0,3071\ngradual,23014,PSI,0,3135\ngradual,23014,PSI,0,3199\ngradual,23014,PSI,0,3263\ngradual,23014,PSI,0,3327\ngradual,23014,PSI,0,3391\ngradual,23014,PSI,0,3455\ngradual,23014,PSI,0,3519\ngradual,23014,PSI,0,3583\ngradual,23014,PSI,0,3647\ngradual,23014,PSI,0,3711\ngradual,23014,PSI,0,3775\ngradual,23014,PSI,0,3839\ngradual,23014,PSI,0,3903\ngradual,23014,PSI,0,3967\ngradual,23014,PSI,0,4031\ngradual,23014,PSI,0,4095\ngradual,23014,FixedJS,0,2559\ngradual,23014,FixedJS,0,2623\ngradual,23014,FixedJS,0,2687\ngradual,23014,FixedJS,0,2751\ngradual,23014,FixedJS,0,2815\ngradual,23014,FixedJS,0,2879\ngradual,23014,FixedJS,0,2943\ngradual,23014,FixedJS,0,3007\ngradual,23014,FixedJS,0,3071\ngradual,23014,FixedJS,0,3135\ngradual,23014,FixedJS,0,3199\ngradual,23014,FixedJS,0,3263\ngradual,23014,FixedJS,0,3327\ngradual,23014,FixedJS,0,3391\ngradual,23014,FixedJS,0,3455\ngradual,23014,FixedJS,0,3519\ngradual,23014,FixedJS,0,3583\ngradual,23014,FixedJS,0,3647\ngradual,23014,FixedJS,0,3711\ngradual,23014,FixedJS,0,3775\ngradual,23014,FixedJS,0,3839\ngradual,23014,FixedJS,0,3903\ngradual,23014,FixedJS,0,3967\ngradual,23014,FixedJS,0,4031\ngradual,23014,FixedJS,0,4095\ngradual,23015,ADWIN,0,2463\ngradual,23015,ADWIN,0,3071\ngradual,23015,PageHinkley,0,2655\ngradual,23015,KSWIN,0,2612\ngradual,23015,KSWIN,0,3283\ngradual,23015,JS,0,2495\ngradual,23015,JS,0,2559\ngradual,23015,JS,0,2623\ngradual,23015,JS,0,2687\ngradual,23015,JS,0,2751\ngradual,23015,JS,0,2815\ngradual,23015,JS,0,2879\ngradual,23015,JS,0,2943\ngradual,23015,JS,0,3007\ngradual,23015,JS,0,3071\ngradual,23015,JS,0,3135\ngradual,23015,JS,0,3199\ngradual,23015,JS,0,3263\ngradual,23015,JS,0,3327\ngradual,23015,JS,0,3391\ngradual,23015,JS,0,3455\ngradual,23015,JS,0,3519\ngradual,23015,JS,0,3583\ngradual,23015,JS,0,3647\ngradual,23015,JS,0,3711\ngradual,23015,JS,0,3775\ngradual,23015,JS,0,3839\ngradual,23015,JS,0,3903\ngradual,23015,JS,0,3967\ngradual,23015,JS,0,4031\ngradual,23015,JS,0,4095\ngradual,23015,PSI,0,2559\ngradual,23015,PSI,0,2751\ngradual,23015,PSI,0,2815\ngradual,23015,PSI,0,2943\ngradual,23015,PSI,0,3007\ngradual,23015,PSI,0,3071\ngradual,23015,PSI,0,3135\ngradual,23015,PSI,0,3199\ngradual,23015,PSI,0,3263\ngradual,23015,PSI,0,3327\ngradual,23015,PSI,0,3391\ngradual,23015,PSI,0,3455\ngradual,23015,PSI,0,3519\ngradual,23015,PSI,0,3583\ngradual,23015,PSI,0,3647\ngradual,23015,PSI,0,3711\ngradual,23015,PSI,0,3775\ngradual,23015,PSI,0,3839\ngradual,23015,PSI,0,3903\ngradual,23015,PSI,0,3967\ngradual,23015,PSI,0,4031\ngradual,23015,PSI,0,4095\ngradual,23015,FixedJS,0,2495\ngradual,23015,FixedJS,0,2623\ngradual,23015,FixedJS,0,2687\ngradual,23015,FixedJS,0,2751\ngradual,23015,FixedJS,0,2815\ngradual,23015,FixedJS,0,2879\ngradual,23015,FixedJS,0,2943\ngradual,23015,FixedJS,0,3007\ngradual,23015,FixedJS,0,3071\ngradual,23015,FixedJS,0,3135\ngradual,23015,FixedJS,0,3199\ngradual,23015,FixedJS,0,3263\ngradual,23015,FixedJS,0,3327\ngradual,23015,FixedJS,0,3391\ngradual,23015,FixedJS,0,3455\ngradual,23015,FixedJS,0,3519\ngradual,23015,FixedJS,0,3583\ngradual,23015,FixedJS,0,3647\ngradual,23015,FixedJS,0,3711\ngradual,23015,FixedJS,0,3775\ngradual,23015,FixedJS,0,3839\ngradual,23015,FixedJS,0,3903\ngradual,23015,FixedJS,0,3967\ngradual,23015,FixedJS,0,4031\ngradual,23015,FixedJS,0,4095\nincremental,23013,ADWIN,0,2431\nincremental,23013,ADWIN,0,2975\nincremental,23013,PageHinkley,0,2625\nincremental,23013,KSWIN,0,1209\nincremental,23013,KSWIN,0,2503\nincremental,23013,KSWIN,0,2959\nincremental,23013,KSWIN,0,3532\nincremental,23013,JS,0,767\nincremental,23013,JS,0,2367\nincremental,23013,JS,0,2431\nincremental,23013,JS,0,2495\nincremental,23013,JS,0,2559\nincremental,23013,JS,0,2623\nincremental,23013,JS,0,2687\nincremental,23013,JS,0,2751\nincremental,23013,JS,0,2815\nincremental,23013,JS,0,2879\nincremental,23013,JS,0,2943\nincremental,23013,JS,0,3007\nincremental,23013,JS,0,3071\nincremental,23013,JS,0,3135\nincremental,23013,JS,0,3199\nincremental,23013,JS,0,3263\nincremental,23013,JS,0,3327\nincremental,23013,JS,0,3391\nincremental,23013,JS,0,3455\nincremental,23013,JS,0,3519\nincremental,23013,JS,0,3583\nincremental,23013,JS,0,3647\nincremental,23013,JS,0,3711\nincremental,23013,JS,0,3775\nincremental,23013,JS,0,3839\nincremental,23013,JS,0,3903\nincremental,23013,JS,0,3967\nincremental,23013,JS,0,4031\nincremental,23013,JS,0,4095\nincremental,23013,PSI,0,767\nincremental,23013,PSI,0,2367\nincremental,23013,PSI,0,2431\nincremental,23013,PSI,0,2495\nincremental,23013,PSI,0,2623\nincremental,23013,PSI,0,2751\nincremental,23013,PSI,0,2815\nincremental,23013,PSI,0,2879\nincremental,23013,PSI,0,2943\nincremental,23013,PSI,0,3007\nincremental,23013,PSI,0,3071\nincremental,23013,PSI,0,3135\nincremental,23013,PSI,0,3199\nincremental,23013,PSI,0,3263\nincremental,23013,PSI,0,3327\nincremental,23013,PSI,0,3391\nincremental,23013,PSI,0,3455\nincremental,23013,PSI,0,3519\nincremental,23013,PSI,0,3583\nincremental,23013,PSI,0,3647\nincremental,23013,PSI,0,3711\nincremental,23013,PSI,0,3775\nincremental,23013,PSI,0,3839\nincremental,23013,PSI,0,3903\nincremental,23013,PSI,0,3967\nincremental,23013,PSI,0,4031\nincremental,23013,PSI,0,4095\nincremental,23013,FixedJS,0,2367\nincremental,23013,FixedJS,0,2431\nincremental,23013,FixedJS,0,2495\nincremental,23013,FixedJS,0,2559\nincremental,23013,FixedJS,0,2623\nincremental,23013,FixedJS,0,2687\nincremental,23013,FixedJS,0,2751\nincremental,23013,FixedJS,0,2815\nincremental,23013,FixedJS,0,2879\nincremental,23013,FixedJS,0,2943\nincremental,23013,FixedJS,0,3007\nincremental,23013,FixedJS,0,3071\nincremental,23013,FixedJS,0,3135\nincremental,23013,FixedJS,0,3199\nincremental,23013,FixedJS,0,3263\nincremental,23013,FixedJS,0,3327\nincremental,23013,FixedJS,0,3391\nincremental,23013,FixedJS,0,3455\nincremental,23013,FixedJS,0,3519\nincremental,23013,FixedJS,0,3583\nincremental,23013,FixedJS,0,3647\nincremental,23013,FixedJS,0,3711\nincremental,23013,FixedJS,0,3775\nincremental,23013,FixedJS,0,3839\nincremental,23013,FixedJS,0,3903\nincremental,23013,FixedJS,0,3967\nincremental,23013,FixedJS,0,4031\nincremental,23013,FixedJS,0,4095\nincremental,23014,ADWIN,0,2367\nincremental,23014,ADWIN,0,3039\nincremental,23014,PageHinkley,0,2573\nincremental,23014,PageHinkley,0,3868\nincremental,23014,DDM,0,39\nincremental,23014,DDM,0,178\nincremental,23014,DDM,0,261\nincremental,23014,DDM,1,64\nincremental,23014,DDM,1,192\nincremental,23014,DDM,1,320\nincremental,23014,DDM,3,192\nincremental,23014,DDM,3,320\nincremental,23014,DDM,3,448\nincremental,23014,JS,0,2367\nincremental,23014,JS,0,2431\nincremental,23014,JS,0,2495\nincremental,23014,JS,0,2559\nincremental,23014,JS,0,2623\nincremental,23014,JS,0,2687\nincremental,23014,JS,0,2751\nincremental,23014,JS,0,2815\nincremental,23014,JS,0,2879\nincremental,23014,JS,0,2943\nincremental,23014,JS,0,3007\nincremental,23014,JS,0,3071\nincremental,23014,JS,0,3135\nincremental,23014,JS,0,3199\nincremental,23014,JS,0,3263\nincremental,23014,JS,0,3327\nincremental,23014,JS,0,3391\nincremental,23014,JS,0,3455\nincremental,23014,JS,0,3519\nincremental,23014,JS,0,3583\nincremental,23014,JS,0,3647\nincremental,23014,JS,0,3711\nincremental,23014,JS,0,3775\nincremental,23014,JS,0,3839\nincremental,23014,JS,0,3903\nincremental,23014,JS,0,3967\nincremental,23014,JS,0,4031\nincremental,23014,JS,0,4095\nincremental,23014,PSI,0,2367\nincremental,23014,PSI,0,2431\nincremental,23014,PSI,0,2559\nincremental,23014,PSI,0,2623\nincremental,23014,PSI,0,2687\nincremental,23014,PSI,0,2751\nincremental,23014,PSI,0,2815\nincremental,23014,PSI,0,2879\nincremental,23014,PSI,0,2943\nincremental,23014,PSI,0,3007\nincremental,23014,PSI,0,3071\nincremental,23014,PSI,0,3135\nincremental,23014,PSI,0,3199\nincremental,23014,PSI,0,3263\nincremental,23014,PSI,0,3327\nincremental,23014,PSI,0,3391\nincremental,23014,PSI,0,3455\nincremental,23014,PSI,0,3519\nincremental,23014,PSI,0,3583\nincremental,23014,PSI,0,3647\nincremental,23014,PSI,0,3711\nincremental,23014,PSI,0,3775\nincremental,23014,PSI,0,3839\nincremental,23014,PSI,0,3903\nincremental,23014,PSI,0,3967\nincremental,23014,PSI,0,4031\nincremental,23014,PSI,0,4095\nincremental,23014,FixedJS,0,2367\nincremental,23014,FixedJS,0,2495\nincremental,23014,FixedJS,0,2559\nincremental,23014,FixedJS,0,2623\nincremental,23014,FixedJS,0,2687\nincremental,23014,FixedJS,0,2751\nincremental,23014,FixedJS,0,2815\nincremental,23014,FixedJS,0,2879\nincremental,23014,FixedJS,0,2943\nincremental,23014,FixedJS,0,3007\nincremental,23014,FixedJS,0,3071\nincremental,23014,FixedJS,0,3135\nincremental,23014,FixedJS,0,3199\nincremental,23014,FixedJS,0,3263\nincremental,23014,FixedJS,0,3327\nincremental,23014,FixedJS,0,3391\nincremental,23014,FixedJS,0,3455\nincremental,23014,FixedJS,0,3519\nincremental,23014,FixedJS,0,3583\nincremental,23014,FixedJS,0,3647\nincremental,23014,FixedJS,0,3711\nincremental,23014,FixedJS,0,3775\nincremental,23014,FixedJS,0,3839\nincremental,23014,FixedJS,0,3903\nincremental,23014,FixedJS,0,3967\nincremental,23014,FixedJS,0,4031\nincremental,23014,FixedJS,0,4095\nincremental,23015,ADWIN,0,2399\nincremental,23015,ADWIN,0,2975\nincremental,23015,PageHinkley,0,2581\nincremental,23015,PageHinkley,0,3988\nincremental,23015,KSWIN,0,1305\nincremental,23015,KSWIN,0,1825\nincremental,23015,KSWIN,0,2270\nincremental,23015,JS,0,2367\nincremental,23015,JS,0,2495\nincremental,23015,JS,0,2559\nincremental,23015,JS,0,2623\nincremental,23015,JS,0,2687\nincremental,23015,JS,0,2751\nincremental,23015,JS,0,2815\nincremental,23015,JS,0,2879\nincremental,23015,JS,0,2943\nincremental,23015,JS,0,3007\nincremental,23015,JS,0,3071\nincremental,23015,JS,0,3135\nincremental,23015,JS,0,3199\nincremental,23015,JS,0,3263\nincremental,23015,JS,0,3327\nincremental,23015,JS,0,3391\nincremental,23015,JS,0,3455\nincremental,23015,JS,0,3519\nincremental,23015,JS,0,3583\nincremental,23015,JS,0,3647\nincremental,23015,JS,0,3711\nincremental,23015,JS,0,3775\nincremental,23015,JS,0,3839\nincremental,23015,JS,0,3903\nincremental,23015,JS,0,3967\nincremental,23015,JS,0,4031\nincremental,23015,JS,0,4095\nincremental,23015,PSI,0,2559\nincremental,23015,PSI,0,2623\nincremental,23015,PSI,0,2687\nincremental,23015,PSI,0,2751\nincremental,23015,PSI,0,2815\nincremental,23015,PSI,0,2879\nincremental,23015,PSI,0,2943\nincremental,23015,PSI,0,3007\nincremental,23015,PSI,0,3071\nincremental,23015,PSI,0,3135\nincremental,23015,PSI,0,3199\nincremental,23015,PSI,0,3263\nincremental,23015,PSI,0,3327\nincremental,23015,PSI,0,3391\nincremental,23015,PSI,0,3455\nincremental,23015,PSI,0,3519\nincremental,23015,PSI,0,3583\nincremental,23015,PSI,0,3647\nincremental,23015,PSI,0,3711\nincremental,23015,PSI,0,3775\nincremental,23015,PSI,0,3839\nincremental,23015,PSI,0,3903\nincremental,23015,PSI,0,3967\nincremental,23015,PSI,0,4031\nincremental,23015,PSI,0,4095\nincremental,23015,FixedJS,0,2367\nincremental,23015,FixedJS,0,2495\nincremental,23015,FixedJS,0,2559\nincremental,23015,FixedJS,0,2623\nincremental,23015,FixedJS,0,2687\nincremental,23015,FixedJS,0,2751\nincremental,23015,FixedJS,0,2815\nincremental,23015,FixedJS,0,2879\nincremental,23015,FixedJS,0,2943\nincremental,23015,FixedJS,0,3007\nincremental,23015,FixedJS,0,3071\nincremental,23015,FixedJS,0,3135\nincremental,23015,FixedJS,0,3199\nincremental,23015,FixedJS,0,3263\nincremental,23015,FixedJS,0,3327\nincremental,23015,FixedJS,0,3391\nincremental,23015,FixedJS,0,3455\nincremental,23015,FixedJS,0,3519\nincremental,23015,FixedJS,0,3583\nincremental,23015,FixedJS,0,3647\nincremental,23015,FixedJS,0,3711\nincremental,23015,FixedJS,0,3775\nincremental,23015,FixedJS,0,3839\nincremental,23015,FixedJS,0,3903\nincremental,23015,FixedJS,0,3967\nincremental,23015,FixedJS,0,4031\nincremental,23015,FixedJS,0,4095\nrecurring,23013,ADWIN,0,2111\nrecurring,23013,ADWIN,0,3135\nrecurring,23013,PageHinkley,0,2190\nrecurring,23013,PageHinkley,0,3219\nrecurring,23013,KSWIN,0,1209\nrecurring,23013,KSWIN,0,2060\nrecurring,23013,KSWIN,0,3030\nrecurring,23013,KSWIN,0,3130\nrecurring,23013,KSWIN,0,3519\nrecurring,23013,JS,0,767\nrecurring,23013,JS,0,2111\nrecurring,23013,JS,0,2175\nrecurring,23013,JS,0,2239\nrecurring,23013,JS,0,2303\nrecurring,23013,JS,0,2367\nrecurring,23013,JS,0,2431\nrecurring,23013,JS,0,2495\nrecurring,23013,JS,0,2559\nrecurring,23013,JS,0,2623\nrecurring,23013,JS,0,2687\nrecurring,23013,JS,0,2751\nrecurring,23013,JS,0,2815\nrecurring,23013,JS,0,2879\nrecurring,23013,JS,0,2943\nrecurring,23013,JS,0,3007\nrecurring,23013,JS,0,3071\nrecurring,23013,PSI,0,767\nrecurring,23013,PSI,0,2111\nrecurring,23013,PSI,0,2175\nrecurring,23013,PSI,0,2239\nrecurring,23013,PSI,0,2303\nrecurring,23013,PSI,0,2367\nrecurring,23013,PSI,0,2431\nrecurring,23013,PSI,0,2495\nrecurring,23013,PSI,0,2559\nrecurring,23013,PSI,0,2623\nrecurring,23013,PSI,0,2687\nrecurring,23013,PSI,0,2751\nrecurring,23013,PSI,0,2815\nrecurring,23013,PSI,0,2879\nrecurring,23013,PSI,0,2943\nrecurring,23013,PSI,0,3007\nrecurring,23013,PSI,0,3071\nrecurring,23013,PSI,0,3327\nrecurring,23013,PSI,0,3711\nrecurring,23013,FixedJS,0,2111\nrecurring,23013,FixedJS,0,2175\nrecurring,23013,FixedJS,0,2239\nrecurring,23013,FixedJS,0,2303\nrecurring,23013,FixedJS,0,2367\nrecurring,23013,FixedJS,0,2431\nrecurring,23013,FixedJS,0,2495\nrecurring,23013,FixedJS,0,2559\nrecurring,23013,FixedJS,0,2623\nrecurring,23013,FixedJS,0,2687\nrecurring,23013,FixedJS,0,2751\nrecurring,23013,FixedJS,0,2815\nrecurring,23013,FixedJS,0,2879\nrecurring,23013,FixedJS,0,2943\nrecurring,23013,FixedJS,0,3007\nrecurring,23013,FixedJS,0,3071\nrecurring,23014,ADWIN,0,2111\nrecurring,23014,ADWIN,0,3103\nrecurring,23014,PageHinkley,0,2188\nrecurring,23014,PageHinkley,0,3202\nrecurring,23014,KSWIN,0,2067\nrecurring,23014,KSWIN,0,3086\nrecurring,23014,DDM,0,39\nrecurring,23014,DDM,0,178\nrecurring,23014,DDM,0,261\nrecurring,23014,DDM,1,64\nrecurring,23014,DDM,1,192\nrecurring,23014,DDM,1,320\nrecurring,23014,DDM,3,192\nrecurring,23014,DDM,3,320\nrecurring,23014,DDM,3,448\nrecurring,23014,JS,0,2111\nrecurring,23014,JS,0,2175\nrecurring,23014,JS,0,2239\nrecurring,23014,JS,0,2303\nrecurring,23014,JS,0,2367\nrecurring,23014,JS,0,2431\nrecurring,23014,JS,0,2495\nrecurring,23014,JS,0,2559\nrecurring,23014,JS,0,2623\nrecurring,23014,JS,0,2687\nrecurring,23014,JS,0,2751\nrecurring,23014,JS,0,2815\nrecurring,23014,JS,0,2879\nrecurring,23014,JS,0,2943\nrecurring,23014,JS,0,3007\nrecurring,23014,JS,0,3071\nrecurring,23014,PSI,0,2111\nrecurring,23014,PSI,0,2175\nrecurring,23014,PSI,0,2239\nrecurring,23014,PSI,0,2303\nrecurring,23014,PSI,0,2367\nrecurring,23014,PSI,0,2431\nrecurring,23014,PSI,0,2495\nrecurring,23014,PSI,0,2559\nrecurring,23014,PSI,0,2623\nrecurring,23014,PSI,0,2687\nrecurring,23014,PSI,0,2751\nrecurring,23014,PSI,0,2815\nrecurring,23014,PSI,0,2879\nrecurring,23014,PSI,0,2943\nrecurring,23014,PSI,0,3007\nrecurring,23014,PSI,0,3071\nrecurring,23014,FixedJS,0,2111\nrecurring,23014,FixedJS,0,2175\nrecurring,23014,FixedJS,0,2239\nrecurring,23014,FixedJS,0,2303\nrecurring,23014,FixedJS,0,2367\nrecurring,23014,FixedJS,0,2431\nrecurring,23014,FixedJS,0,2495\nrecurring,23014,FixedJS,0,2559\nrecurring,23014,FixedJS,0,2623\nrecurring,23014,FixedJS,0,2687\nrecurring,23014,FixedJS,0,2751\nrecurring,23014,FixedJS,0,2815\nrecurring,23014,FixedJS,0,2879\nrecurring,23014,FixedJS,0,2943\nrecurring,23014,FixedJS,0,3007\nrecurring,23014,FixedJS,0,3071\nrecurring,23015,ADWIN,0,2111\nrecurring,23015,ADWIN,0,3135\nrecurring,23015,PageHinkley,0,2182\nrecurring,23015,PageHinkley,0,3210\nrecurring,23015,KSWIN,0,1305\nrecurring,23015,KSWIN,0,1825\nrecurring,23015,KSWIN,0,2072\nrecurring,23015,KSWIN,0,2356\nrecurring,23015,KSWIN,0,3087\nrecurring,23015,KSWIN,0,3651\nrecurring,23015,KSWIN,0,4072\nrecurring,23015,JS,0,2111\nrecurring,23015,JS,0,2175\nrecurring,23015,JS,0,2239\nrecurring,23015,JS,0,2303\nrecurring,23015,JS,0,2367\nrecurring,23015,JS,0,2431\nrecurring,23015,JS,0,2495\nrecurring,23015,JS,0,2559\nrecurring,23015,JS,0,2623\nrecurring,23015,JS,0,2687\nrecurring,23015,JS,0,2751\nrecurring,23015,JS,0,2815\nrecurring,23015,JS,0,2879\nrecurring,23015,JS,0,2943\nrecurring,23015,JS,0,3007\nrecurring,23015,JS,0,3071\nrecurring,23015,PSI,0,2111\nrecurring,23015,PSI,0,2175\nrecurring,23015,PSI,0,2239\nrecurring,23015,PSI,0,2303\nrecurring,23015,PSI,0,2367\nrecurring,23015,PSI,0,2431\nrecurring,23015,PSI,0,2495\nrecurring,23015,PSI,0,2559\nrecurring,23015,PSI,0,2623\nrecurring,23015,PSI,0,2687\nrecurring,23015,PSI,0,2751\nrecurring,23015,PSI,0,2815\nrecurring,23015,PSI,0,2879\nrecurring,23015,PSI,0,2943\nrecurring,23015,PSI,0,3007\nrecurring,23015,PSI,0,3071\nrecurring,23015,FixedJS,0,2111\nrecurring,23015,FixedJS,0,2175\nrecurring,23015,FixedJS,0,2239\nrecurring,23015,FixedJS,0,2303\nrecurring,23015,FixedJS,0,2367\nrecurring,23015,FixedJS,0,2431\nrecurring,23015,FixedJS,0,2495\nrecurring,23015,FixedJS,0,2559\nrecurring,23015,FixedJS,0,2623\nrecurring,23015,FixedJS,0,2687\nrecurring,23015,FixedJS,0,2751\nrecurring,23015,FixedJS,0,2815\nrecurring,23015,FixedJS,0,2879\nrecurring,23015,FixedJS,0,2943\nrecurring,23015,FixedJS,0,3007\nrecurring,23015,FixedJS,0,3071\nvariance,23013,KSWIN,0,1209\nvariance,23013,JS,0,767\nvariance,23013,JS,0,2111\nvariance,23013,JS,0,2175\nvariance,23013,JS,0,2239\nvariance,23013,JS,0,2303\nvariance,23013,JS,0,2367\nvariance,23013,JS,0,2431\nvariance,23013,JS,0,2495\nvariance,23013,JS,0,2559\nvariance,23013,JS,0,2623\nvariance,23013,JS,0,2687\nvariance,23013,JS,0,2751\nvariance,23013,JS,0,2815\nvariance,23013,JS,0,2879\nvariance,23013,JS,0,2943\nvariance,23013,JS,0,3007\nvariance,23013,JS,0,3071\nvariance,23013,JS,0,3135\nvariance,23013,JS,0,3199\nvariance,23013,JS,0,3263\nvariance,23013,JS,0,3327\nvariance,23013,JS,0,3391\nvariance,23013,JS,0,3455\nvariance,23013,JS,0,3519\nvariance,23013,JS,0,3583\nvariance,23013,JS,0,3647\nvariance,23013,JS,0,3711\nvariance,23013,JS,0,3775\nvariance,23013,JS,0,3839\nvariance,23013,JS,0,3903\nvariance,23013,JS,0,3967\nvariance,23013,JS,0,4031\nvariance,23013,JS,0,4095\nvariance,23013,PSI,0,767\nvariance,23013,PSI,0,2111\nvariance,23013,PSI,0,2239\nvariance,23013,PSI,0,2367\nvariance,23013,PSI,0,2559\nvariance,23013,PSI,0,2623\nvariance,23013,PSI,0,2687\nvariance,23013,PSI,0,3071\nvariance,23013,PSI,0,3199\nvariance,23013,PSI,0,3263\nvariance,23013,PSI,0,3327\nvariance,23013,PSI,0,3391\nvariance,23013,PSI,0,3519\nvariance,23013,PSI,0,3711\nvariance,23013,PSI,0,3967\nvariance,23013,PSI,0,4031\nvariance,23013,FixedJS,0,2111\nvariance,23013,FixedJS,0,2175\nvariance,23013,FixedJS,0,2239\nvariance,23013,FixedJS,0,2303\nvariance,23013,FixedJS,0,2367\nvariance,23013,FixedJS,0,2431\nvariance,23013,FixedJS,0,2495\nvariance,23013,FixedJS,0,2559\nvariance,23013,FixedJS,0,2623\nvariance,23013,FixedJS,0,2687\nvariance,23013,FixedJS,0,2751\nvariance,23013,FixedJS,0,2815\nvariance,23013,FixedJS,0,2879\nvariance,23013,FixedJS,0,2943\nvariance,23013,FixedJS,0,3007\nvariance,23013,FixedJS,0,3071\nvariance,23013,FixedJS,0,3135\nvariance,23013,FixedJS,0,3199\nvariance,23013,FixedJS,0,3263\nvariance,23013,FixedJS,0,3327\nvariance,23013,FixedJS,0,3391\nvariance,23013,FixedJS,0,3455\nvariance,23013,FixedJS,0,3519\nvariance,23013,FixedJS,0,3583\nvariance,23013,FixedJS,0,3647\nvariance,23013,FixedJS,0,3711\nvariance,23013,FixedJS,0,3775\nvariance,23013,FixedJS,0,3839\nvariance,23013,FixedJS,0,3903\nvariance,23013,FixedJS,0,3967\nvariance,23013,FixedJS,0,4031\nvariance,23013,FixedJS,0,4095\nvariance,23014,KSWIN,0,2195\nvariance,23014,KSWIN,0,2943\nvariance,23014,DDM,0,39\nvariance,23014,DDM,0,178\nvariance,23014,DDM,0,261\nvariance,23014,DDM,1,64\nvariance,23014,DDM,1,192\nvariance,23014,DDM,1,320\nvariance,23014,DDM,3,192\nvariance,23014,DDM,3,320\nvariance,23014,DDM,3,448\nvariance,23014,JS,0,2111\nvariance,23014,JS,0,2175\nvariance,23014,JS,0,2239\nvariance,23014,JS,0,2303\nvariance,23014,JS,0,2367\nvariance,23014,JS,0,2431\nvariance,23014,JS,0,2495\nvariance,23014,JS,0,2559\nvariance,23014,JS,0,2623\nvariance,23014,JS,0,2687\nvariance,23014,JS,0,2751\nvariance,23014,JS,0,2815\nvariance,23014,JS,0,2879\nvariance,23014,JS,0,2943\nvariance,23014,JS,0,3007\nvariance,23014,JS,0,3071\nvariance,23014,JS,0,3135\nvariance,23014,JS,0,3199\nvariance,23014,JS,0,3263\nvariance,23014,JS,0,3327\nvariance,23014,JS,0,3391\nvariance,23014,JS,0,3455\nvariance,23014,JS,0,3519\nvariance,23014,JS,0,3583\nvariance,23014,JS,0,3647\nvariance,23014,JS,0,3711\nvariance,23014,JS,0,3775\nvariance,23014,JS,0,3839\nvariance,23014,JS,0,3903\nvariance,23014,JS,0,3967\nvariance,23014,JS,0,4031\nvariance,23014,JS,0,4095\nvariance,23014,PSI,0,2175\nvariance,23014,PSI,0,2303\nvariance,23014,PSI,0,2367\nvariance,23014,PSI,0,2431\nvariance,23014,PSI,0,2623\nvariance,23014,PSI,0,2687\nvariance,23014,PSI,0,2943\nvariance,23014,PSI,0,3135\nvariance,23014,PSI,0,3263\nvariance,23014,PSI,0,3327\nvariance,23014,PSI,0,3391\nvariance,23014,PSI,0,3519\nvariance,23014,PSI,0,3647\nvariance,23014,PSI,0,3775\nvariance,23014,PSI,0,3839\nvariance,23014,FixedJS,0,2111\nvariance,23014,FixedJS,0,2175\nvariance,23014,FixedJS,0,2239\nvariance,23014,FixedJS,0,2303\nvariance,23014,FixedJS,0,2367\nvariance,23014,FixedJS,0,2431\nvariance,23014,FixedJS,0,2495\nvariance,23014,FixedJS,0,2559\nvariance,23014,FixedJS,0,2623\nvariance,23014,FixedJS,0,2687\nvariance,23014,FixedJS,0,2751\nvariance,23014,FixedJS,0,2815\nvariance,23014,FixedJS,0,2879\nvariance,23014,FixedJS,0,2943\nvariance,23014,FixedJS,0,3007\nvariance,23014,FixedJS,0,3071\nvariance,23014,FixedJS,0,3135\nvariance,23014,FixedJS,0,3199\nvariance,23014,FixedJS,0,3263\nvariance,23014,FixedJS,0,3327\nvariance,23014,FixedJS,0,3391\nvariance,23014,FixedJS,0,3455\nvariance,23014,FixedJS,0,3519\nvariance,23014,FixedJS,0,3583\nvariance,23014,FixedJS,0,3647\nvariance,23014,FixedJS,0,3711\nvariance,23014,FixedJS,0,3775\nvariance,23014,FixedJS,0,3839\nvariance,23014,FixedJS,0,3903\nvariance,23014,FixedJS,0,3967\nvariance,23014,FixedJS,0,4031\nvariance,23014,FixedJS,0,4095\nvariance,23015,KSWIN,0,1305\nvariance,23015,KSWIN,0,1825\nvariance,23015,KSWIN,0,2072\nvariance,23015,KSWIN,0,2356\nvariance,23015,JS,0,2111\nvariance,23015,JS,0,2175\nvariance,23015,JS,0,2239\nvariance,23015,JS,0,2303\nvariance,23015,JS,0,2367\nvariance,23015,JS,0,2431\nvariance,23015,JS,0,2495\nvariance,23015,JS,0,2559\nvariance,23015,JS,0,2623\nvariance,23015,JS,0,2687\nvariance,23015,JS,0,2751\nvariance,23015,JS,0,2815\nvariance,23015,JS,0,2879\nvariance,23015,JS,0,2943\nvariance,23015,JS,0,3007\nvariance,23015,JS,0,3071\nvariance,23015,JS,0,3135\nvariance,23015,JS,0,3199\nvariance,23015,JS,0,3263\nvariance,23015,JS,0,3327\nvariance,23015,JS,0,3391\nvariance,23015,JS,0,3455\nvariance,23015,JS,0,3519\nvariance,23015,JS,0,3583\nvariance,23015,JS,0,3647\nvariance,23015,JS,0,3711\nvariance,23015,JS,0,3775\nvariance,23015,JS,0,3839\nvariance,23015,JS,0,3903\nvariance,23015,JS,0,3967\nvariance,23015,JS,0,4031\nvariance,23015,JS,0,4095\nvariance,23015,PSI,0,2175\nvariance,23015,PSI,0,2239\nvariance,23015,PSI,0,2303\nvariance,23015,PSI,0,2495\nvariance,23015,PSI,0,2623\nvariance,23015,PSI,0,2879\nvariance,23015,PSI,0,2943\nvariance,23015,PSI,0,3199\nvariance,23015,PSI,0,3391\nvariance,23015,PSI,0,3455\nvariance,23015,PSI,0,3519\nvariance,23015,PSI,0,3647\nvariance,23015,PSI,0,3711\nvariance,23015,PSI,0,3839\nvariance,23015,PSI,0,4031\nvariance,23015,FixedJS,0,2111\nvariance,23015,FixedJS,0,2175\nvariance,23015,FixedJS,0,2239\nvariance,23015,FixedJS,0,2303\nvariance,23015,FixedJS,0,2367\nvariance,23015,FixedJS,0,2431\nvariance,23015,FixedJS,0,2495\nvariance,23015,FixedJS,0,2559\nvariance,23015,FixedJS,0,2623\nvariance,23015,FixedJS,0,2687\nvariance,23015,FixedJS,0,2751\nvariance,23015,FixedJS,0,2879\nvariance,23015,FixedJS,0,2943\nvariance,23015,FixedJS,0,3007\nvariance,23015,FixedJS,0,3071\nvariance,23015,FixedJS,0,3135\nvariance,23015,FixedJS,0,3199\nvariance,23015,FixedJS,0,3263\nvariance,23015,FixedJS,0,3327\nvariance,23015,FixedJS,0,3391\nvariance,23015,FixedJS,0,3455\nvariance,23015,FixedJS,0,3519\nvariance,23015,FixedJS,0,3583\nvariance,23015,FixedJS,0,3647\nvariance,23015,FixedJS,0,3711\nvariance,23015,FixedJS,0,3775\nvariance,23015,FixedJS,0,3839\nvariance,23015,FixedJS,0,3903\nvariance,23015,FixedJS,0,3967\nvariance,23015,FixedJS,0,4031\nvariance,23015,FixedJS,0,4095\ncorrelation,23013,ADWIN,0,2111\ncorrelation,23013,PageHinkley,0,2213\ncorrelation,23013,KSWIN,0,183\ncorrelation,23013,KSWIN,0,2067\ncorrelation,23013,KSWIN,0,2192\ncorrelation,23013,JS,0,703\ncorrelation,23013,JS,0,1151\ncorrelation,23013,JS,0,1343\ncorrelation,23013,JS,0,1599\ncorrelation,23013,JS,0,1791\ncorrelation,23013,JS,0,1855\ncorrelation,23013,JS,0,2111\ncorrelation,23013,JS,0,2175\ncorrelation,23013,JS,0,2239\ncorrelation,23013,JS,0,2303\ncorrelation,23013,JS,0,2367\ncorrelation,23013,JS,0,2431\ncorrelation,23013,JS,0,2495\ncorrelation,23013,JS,0,2559\ncorrelation,23013,JS,0,2623\ncorrelation,23013,JS,0,2687\ncorrelation,23013,JS,0,2751\ncorrelation,23013,JS,0,2815\ncorrelation,23013,JS,0,2879\ncorrelation,23013,JS,0,2943\ncorrelation,23013,JS,0,3007\ncorrelation,23013,JS,0,3071\ncorrelation,23013,JS,0,3135\ncorrelation,23013,JS,0,3199\ncorrelation,23013,JS,0,3263\ncorrelation,23013,JS,0,3327\ncorrelation,23013,JS,0,3391\ncorrelation,23013,JS,0,3455\ncorrelation,23013,JS,0,3519\ncorrelation,23013,JS,0,3583\ncorrelation,23013,JS,0,3647\ncorrelation,23013,JS,0,3711\ncorrelation,23013,JS,0,3775\ncorrelation,23013,JS,0,3839\ncorrelation,23013,JS,0,3903\ncorrelation,23013,JS,0,3967\ncorrelation,23013,JS,0,4031\ncorrelation,23013,JS,0,4095\ncorrelation,23013,PSI,0,1151\ncorrelation,23013,PSI,0,1343\ncorrelation,23013,PSI,0,1599\ncorrelation,23013,PSI,0,2111\ncorrelation,23013,PSI,0,2175\ncorrelation,23013,PSI,0,2239\ncorrelation,23013,PSI,0,2303\ncorrelation,23013,PSI,0,2367\ncorrelation,23013,PSI,0,2431\ncorrelation,23013,PSI,0,2495\ncorrelation,23013,PSI,0,2559\ncorrelation,23013,PSI,0,2623\ncorrelation,23013,PSI,0,2687\ncorrelation,23013,PSI,0,2751\ncorrelation,23013,PSI,0,2815\ncorrelation,23013,PSI,0,2879\ncorrelation,23013,PSI,0,2943\ncorrelation,23013,PSI,0,3007\ncorrelation,23013,PSI,0,3071\ncorrelation,23013,PSI,0,3135\ncorrelation,23013,PSI,0,3199\ncorrelation,23013,PSI,0,3263\ncorrelation,23013,PSI,0,3327\ncorrelation,23013,PSI,0,3391\ncorrelation,23013,PSI,0,3455\ncorrelation,23013,PSI,0,3519\ncorrelation,23013,PSI,0,3583\ncorrelation,23013,PSI,0,3647\ncorrelation,23013,PSI,0,3711\ncorrelation,23013,PSI,0,3775\ncorrelation,23013,PSI,0,3839\ncorrelation,23013,PSI,0,3903\ncorrelation,23013,PSI,0,3967\ncorrelation,23013,PSI,0,4031\ncorrelation,23013,PSI,0,4095\ncorrelation,23013,FixedJS,0,1151\ncorrelation,23013,FixedJS,0,2111\ncorrelation,23013,FixedJS,0,2175\ncorrelation,23013,FixedJS,0,2239\ncorrelation,23013,FixedJS,0,2303\ncorrelation,23013,FixedJS,0,2367\ncorrelation,23013,FixedJS,0,2431\ncorrelation,23013,FixedJS,0,2495\ncorrelation,23013,FixedJS,0,2559\ncorrelation,23013,FixedJS,0,2623\ncorrelation,23013,FixedJS,0,2687\ncorrelation,23013,FixedJS,0,2751\ncorrelation,23013,FixedJS,0,2815\ncorrelation,23013,FixedJS,0,2879\ncorrelation,23013,FixedJS,0,2943\ncorrelation,23013,FixedJS,0,3007\ncorrelation,23013,FixedJS,0,3071\ncorrelation,23013,FixedJS,0,3135\ncorrelation,23013,FixedJS,0,3199\ncorrelation,23013,FixedJS,0,3263\ncorrelation,23013,FixedJS,0,3327\ncorrelation,23013,FixedJS,0,3391\ncorrelation,23013,FixedJS,0,3455\ncorrelation,23013,FixedJS,0,3519\ncorrelation,23013,FixedJS,0,3583\ncorrelation,23013,FixedJS,0,3647\ncorrelation,23013,FixedJS,0,3711\ncorrelation,23013,FixedJS,0,3775\ncorrelation,23013,FixedJS,0,3839\ncorrelation,23013,FixedJS,0,3903\ncorrelation,23013,FixedJS,0,3967\ncorrelation,23013,FixedJS,0,4031\ncorrelation,23013,FixedJS,0,4095\ncorrelation,23014,ADWIN,0,2143\ncorrelation,23014,PageHinkley,0,2237\ncorrelation,23014,KSWIN,0,424\ncorrelation,23014,KSWIN,0,1256\ncorrelation,23014,KSWIN,0,2074\ncorrelation,23014,DDM,0,39\ncorrelation,23014,DDM,0,178\ncorrelation,23014,DDM,0,261\ncorrelation,23014,DDM,1,64\ncorrelation,23014,DDM,1,192\ncorrelation,23014,DDM,1,320\ncorrelation,23014,DDM,3,192\ncorrelation,23014,DDM,3,320\ncorrelation,23014,DDM,3,448\ncorrelation,23014,JS,0,831\ncorrelation,23014,JS,0,1343\ncorrelation,23014,JS,0,1663\ncorrelation,23014,JS,0,1919\ncorrelation,23014,JS,0,2047\ncorrelation,23014,JS,0,2111\ncorrelation,23014,JS,0,2175\ncorrelation,23014,JS,0,2239\ncorrelation,23014,JS,0,2303\ncorrelation,23014,JS,0,2367\ncorrelation,23014,JS,0,2431\ncorrelation,23014,JS,0,2495\ncorrelation,23014,JS,0,2559\ncorrelation,23014,JS,0,2623\ncorrelation,23014,JS,0,2687\ncorrelation,23014,JS,0,2751\ncorrelation,23014,JS,0,2815\ncorrelation,23014,JS,0,2879\ncorrelation,23014,JS,0,2943\ncorrelation,23014,JS,0,3007\ncorrelation,23014,JS,0,3071\ncorrelation,23014,JS,0,3135\ncorrelation,23014,JS,0,3199\ncorrelation,23014,JS,0,3263\ncorrelation,23014,JS,0,3327\ncorrelation,23014,JS,0,3391\ncorrelation,23014,JS,0,3455\ncorrelation,23014,JS,0,3519\ncorrelation,23014,JS,0,3583\ncorrelation,23014,JS,0,3647\ncorrelation,23014,JS,0,3711\ncorrelation,23014,JS,0,3775\ncorrelation,23014,JS,0,3839\ncorrelation,23014,JS,0,3903\ncorrelation,23014,JS,0,3967\ncorrelation,23014,JS,0,4031\ncorrelation,23014,JS,0,4095\ncorrelation,23014,PSI,0,383\ncorrelation,23014,PSI,0,1919\ncorrelation,23014,PSI,0,2047\ncorrelation,23014,PSI,0,2111\ncorrelation,23014,PSI,0,2175\ncorrelation,23014,PSI,0,2239\ncorrelation,23014,PSI,0,2303\ncorrelation,23014,PSI,0,2367\ncorrelation,23014,PSI,0,2431\ncorrelation,23014,PSI,0,2495\ncorrelation,23014,PSI,0,2559\ncorrelation,23014,PSI,0,2623\ncorrelation,23014,PSI,0,2687\ncorrelation,23014,PSI,0,2751\ncorrelation,23014,PSI,0,2815\ncorrelation,23014,PSI,0,2879\ncorrelation,23014,PSI,0,2943\ncorrelation,23014,PSI,0,3007\ncorrelation,23014,PSI,0,3071\ncorrelation,23014,PSI,0,3135\ncorrelation,23014,PSI,0,3199\ncorrelation,23014,PSI,0,3263\ncorrelation,23014,PSI,0,3327\ncorrelation,23014,PSI,0,3391\ncorrelation,23014,PSI,0,3455\ncorrelation,23014,PSI,0,3519\ncorrelation,23014,PSI,0,3583\ncorrelation,23014,PSI,0,3647\ncorrelation,23014,PSI,0,3711\ncorrelation,23014,PSI,0,3775\ncorrelation,23014,PSI,0,3839\ncorrelation,23014,PSI,0,3903\ncorrelation,23014,PSI,0,3967\ncorrelation,23014,PSI,0,4031\ncorrelation,23014,PSI,0,4095\ncorrelation,23014,FixedJS,0,2047\ncorrelation,23014,FixedJS,0,2111\ncorrelation,23014,FixedJS,0,2175\ncorrelation,23014,FixedJS,0,2239\ncorrelation,23014,FixedJS,0,2303\ncorrelation,23014,FixedJS,0,2367\ncorrelation,23014,FixedJS,0,2431\ncorrelation,23014,FixedJS,0,2495\ncorrelation,23014,FixedJS,0,2559\ncorrelation,23014,FixedJS,0,2623\ncorrelation,23014,FixedJS,0,2687\ncorrelation,23014,FixedJS,0,2751\ncorrelation,23014,FixedJS,0,2815\ncorrelation,23014,FixedJS,0,2879\ncorrelation,23014,FixedJS,0,2943\ncorrelation,23014,FixedJS,0,3007\ncorrelation,23014,FixedJS,0,3071\ncorrelation,23014,FixedJS,0,3135\ncorrelation,23014,FixedJS,0,3199\ncorrelation,23014,FixedJS,0,3263\ncorrelation,23014,FixedJS,0,3327\ncorrelation,23014,FixedJS,0,3391\ncorrelation,23014,FixedJS,0,3455\ncorrelation,23014,FixedJS,0,3519\ncorrelation,23014,FixedJS,0,3583\ncorrelation,23014,FixedJS,0,3647\ncorrelation,23014,FixedJS,0,3711\ncorrelation,23014,FixedJS,0,3775\ncorrelation,23014,FixedJS,0,3839\ncorrelation,23014,FixedJS,0,3903\ncorrelation,23014,FixedJS,0,3967\ncorrelation,23014,FixedJS,0,4031\ncorrelation,23014,FixedJS,0,4095\ncorrelation,23015,ADWIN,0,2111\ncorrelation,23015,PageHinkley,0,2164\ncorrelation,23015,KSWIN,0,1221\ncorrelation,23015,KSWIN,0,1510\ncorrelation,23015,KSWIN,0,2000\ncorrelation,23015,KSWIN,0,3440\ncorrelation,23015,KSWIN,0,3828\ncorrelation,23015,JS,0,511\ncorrelation,23015,JS,0,1919\ncorrelation,23015,JS,0,2111\ncorrelation,23015,JS,0,2175\ncorrelation,23015,JS,0,2239\ncorrelation,23015,JS,0,2303\ncorrelation,23015,JS,0,2367\ncorrelation,23015,JS,0,2431\ncorrelation,23015,JS,0,2495\ncorrelation,23015,JS,0,2559\ncorrelation,23015,JS,0,2623\ncorrelation,23015,JS,0,2687\ncorrelation,23015,JS,0,2751\ncorrelation,23015,JS,0,2815\ncorrelation,23015,JS,0,2879\ncorrelation,23015,JS,0,2943\ncorrelation,23015,JS,0,3007\ncorrelation,23015,JS,0,3071\ncorrelation,23015,JS,0,3135\ncorrelation,23015,JS,0,3199\ncorrelation,23015,JS,0,3263\ncorrelation,23015,JS,0,3327\ncorrelation,23015,JS,0,3391\ncorrelation,23015,JS,0,3455\ncorrelation,23015,JS,0,3519\ncorrelation,23015,JS,0,3583\ncorrelation,23015,JS,0,3647\ncorrelation,23015,JS,0,3711\ncorrelation,23015,JS,0,3775\ncorrelation,23015,JS,0,3839\ncorrelation,23015,JS,0,3903\ncorrelation,23015,JS,0,3967\ncorrelation,23015,JS,0,4031\ncorrelation,23015,JS,0,4095\ncorrelation,23015,PSI,0,511\ncorrelation,23015,PSI,0,2111\ncorrelation,23015,PSI,0,2175\ncorrelation,23015,PSI,0,2239\ncorrelation,23015,PSI,0,2303\ncorrelation,23015,PSI,0,2367\ncorrelation,23015,PSI,0,2431\ncorrelation,23015,PSI,0,2495\ncorrelation,23015,PSI,0,2559\ncorrelation,23015,PSI,0,2623\ncorrelation,23015,PSI,0,2687\ncorrelation,23015,PSI,0,2751\ncorrelation,23015,PSI,0,2815\ncorrelation,23015,PSI,0,2879\ncorrelation,23015,PSI,0,2943\ncorrelation,23015,PSI,0,3007\ncorrelation,23015,PSI,0,3071\ncorrelation,23015,PSI,0,3135\ncorrelation,23015,PSI,0,3199\ncorrelation,23015,PSI,0,3263\ncorrelation,23015,PSI,0,3327\ncorrelation,23015,PSI,0,3391\ncorrelation,23015,PSI,0,3455\ncorrelation,23015,PSI,0,3519\ncorrelation,23015,PSI,0,3583\ncorrelation,23015,PSI,0,3647\ncorrelation,23015,PSI,0,3711\ncorrelation,23015,PSI,0,3775\ncorrelation,23015,PSI,0,3839\ncorrelation,23015,PSI,0,3903\ncorrelation,23015,PSI,0,3967\ncorrelation,23015,PSI,0,4031\ncorrelation,23015,PSI,0,4095\ncorrelation,23015,FixedJS,0,1919\ncorrelation,23015,FixedJS,0,2111\ncorrelation,23015,FixedJS,0,2175\ncorrelation,23015,FixedJS,0,2239\ncorrelation,23015,FixedJS,0,2303\ncorrelation,23015,FixedJS,0,2367\ncorrelation,23015,FixedJS,0,2431\ncorrelation,23015,FixedJS,0,2495\ncorrelation,23015,FixedJS,0,2559\ncorrelation,23015,FixedJS,0,2623\ncorrelation,23015,FixedJS,0,2687\ncorrelation,23015,FixedJS,0,2751\ncorrelation,23015,FixedJS,0,2815\ncorrelation,23015,FixedJS,0,2879\ncorrelation,23015,FixedJS,0,2943\ncorrelation,23015,FixedJS,0,3007\ncorrelation,23015,FixedJS,0,3071\ncorrelation,23015,FixedJS,0,3135\ncorrelation,23015,FixedJS,0,3199\ncorrelation,23015,FixedJS,0,3263\ncorrelation,23015,FixedJS,0,3327\ncorrelation,23015,FixedJS,0,3391\ncorrelation,23015,FixedJS,0,3455\ncorrelation,23015,FixedJS,0,3519\ncorrelation,23015,FixedJS,0,3583\ncorrelation,23015,FixedJS,0,3647\ncorrelation,23015,FixedJS,0,3711\ncorrelation,23015,FixedJS,0,3775\ncorrelation,23015,FixedJS,0,3839\ncorrelation,23015,FixedJS,0,3903\ncorrelation,23015,FixedJS,0,3967\ncorrelation,23015,FixedJS,0,4031\ncorrelation,23015,FixedJS,0,4095\nprior,23013,ADWIN,0,2079\nprior,23013,PageHinkley,0,2106\nprior,23013,PageHinkley,0,3356\nprior,23013,KSWIN,0,2072\nprior,23013,JS,0,319\nprior,23013,JS,0,383\nprior,23013,JS,0,447\nprior,23013,JS,0,511\nprior,23013,JS,0,575\nprior,23013,JS,0,639\nprior,23013,JS,0,703\nprior,23013,JS,0,767\nprior,23013,JS,0,831\nprior,23013,JS,0,895\nprior,23013,JS,0,959\nprior,23013,JS,0,1023\nprior,23013,JS,0,1087\nprior,23013,JS,0,1151\nprior,23013,JS,0,1215\nprior,23013,JS,0,1279\nprior,23013,JS,0,1343\nprior,23013,JS,0,1407\nprior,23013,JS,0,1471\nprior,23013,JS,0,1535\nprior,23013,JS,0,1599\nprior,23013,JS,0,1663\nprior,23013,JS,0,1727\nprior,23013,JS,0,1791\nprior,23013,JS,0,1855\nprior,23013,JS,0,1919\nprior,23013,JS,0,1983\nprior,23013,JS,0,2047\nprior,23013,JS,0,2111\nprior,23013,JS,0,2175\nprior,23013,JS,0,2239\nprior,23013,JS,0,2303\nprior,23013,JS,0,2367\nprior,23013,JS,0,2431\nprior,23013,JS,0,2495\nprior,23013,JS,0,2559\nprior,23013,JS,0,2623\nprior,23013,JS,0,2687\nprior,23013,JS,0,2751\nprior,23013,JS,0,2815\nprior,23013,JS,0,2879\nprior,23013,JS,0,2943\nprior,23013,JS,0,3007\nprior,23013,JS,0,3071\nprior,23013,JS,0,3135\nprior,23013,JS,0,3199\nprior,23013,JS,0,3263\nprior,23013,JS,0,3327\nprior,23013,JS,0,3391\nprior,23013,JS,0,3455\nprior,23013,JS,0,3519\nprior,23013,JS,0,3583\nprior,23013,JS,0,3647\nprior,23013,JS,0,3711\nprior,23013,JS,0,3775\nprior,23013,JS,0,3839\nprior,23013,JS,0,3903\nprior,23013,JS,0,3967\nprior,23013,JS,0,4031\nprior,23013,JS,0,4095\nprior,23013,PSI,0,319\nprior,23013,PSI,0,383\nprior,23013,PSI,0,447\nprior,23013,PSI,0,511\nprior,23013,PSI,0,575\nprior,23013,PSI,0,639\nprior,23013,PSI,0,703\nprior,23013,PSI,0,767\nprior,23013,PSI,0,831\nprior,23013,PSI,0,895\nprior,23013,PSI,0,959\nprior,23013,PSI,0,1023\nprior,23013,PSI,0,1087\nprior,23013,PSI,0,1151\nprior,23013,PSI,0,1215\nprior,23013,PSI,0,1279\nprior,23013,PSI,0,1343\nprior,23013,PSI,0,1407\nprior,23013,PSI,0,1471\nprior,23013,PSI,0,1535\nprior,23013,PSI,0,1599\nprior,23013,PSI,0,1663\nprior,23013,PSI,0,1727\nprior,23013,PSI,0,1791\nprior,23013,PSI,0,1855\nprior,23013,PSI,0,1919\nprior,23013,PSI,0,1983\nprior,23013,PSI,0,2047\nprior,23013,PSI,0,2111\nprior,23013,PSI,0,2175\nprior,23013,PSI,0,2239\nprior,23013,PSI,0,2303\nprior,23013,PSI,0,2367\nprior,23013,PSI,0,2431\nprior,23013,PSI,0,2495\nprior,23013,PSI,0,2559\nprior,23013,PSI,0,2623\nprior,23013,PSI,0,2687\nprior,23013,PSI,0,2751\nprior,23013,PSI,0,2815\nprior,23013,PSI,0,2879\nprior,23013,PSI,0,2943\nprior,23013,PSI,0,3007\nprior,23013,PSI,0,3071\nprior,23013,PSI,0,3135\nprior,23013,PSI,0,3199\nprior,23013,PSI,0,3263\nprior,23013,PSI,0,3327\nprior,23013,PSI,0,3391\nprior,23013,PSI,0,3455\nprior,23013,PSI,0,3519\nprior,23013,PSI,0,3583\nprior,23013,PSI,0,3647\nprior,23013,PSI,0,3711\nprior,23013,PSI,0,3775\nprior,23013,PSI,0,3839\nprior,23013,PSI,0,3903\nprior,23013,PSI,0,3967\nprior,23013,PSI,0,4031\nprior,23013,PSI,0,4095\nprior,23013,FixedJS,0,319\nprior,23013,FixedJS,0,383\nprior,23013,FixedJS,0,447\nprior,23013,FixedJS,0,511\nprior,23013,FixedJS,0,575\nprior,23013,FixedJS,0,639\nprior,23013,FixedJS,0,703\nprior,23013,FixedJS,0,767\nprior,23013,FixedJS,0,831\nprior,23013,FixedJS,0,895\nprior,23013,FixedJS,0,959\nprior,23013,FixedJS,0,1023\nprior,23013,FixedJS,0,1087\nprior,23013,FixedJS,0,1151\nprior,23013,FixedJS,0,1215\nprior,23013,FixedJS,0,1279\nprior,23013,FixedJS,0,1343\nprior,23013,FixedJS,0,1407\nprior,23013,FixedJS,0,1471\nprior,23013,FixedJS,0,1535\nprior,23013,FixedJS,0,1599\nprior,23013,FixedJS,0,1663\nprior,23013,FixedJS,0,1727\nprior,23013,FixedJS,0,1791\nprior,23013,FixedJS,0,1855\nprior,23013,FixedJS,0,1919\nprior,23013,FixedJS,0,1983\nprior,23013,FixedJS,0,2047\nprior,23013,FixedJS,0,2111\nprior,23013,FixedJS,0,2175\nprior,23013,FixedJS,0,2239\nprior,23013,FixedJS,0,2303\nprior,23013,FixedJS,0,2367\nprior,23013,FixedJS,0,2431\nprior,23013,FixedJS,0,2495\nprior,23013,FixedJS,0,2559\nprior,23013,FixedJS,0,2623\nprior,23013,FixedJS,0,2687\nprior,23013,FixedJS,0,2751\nprior,23013,FixedJS,0,2815\nprior,23013,FixedJS,0,2879\nprior,23013,FixedJS,0,2943\nprior,23013,FixedJS,0,3007\nprior,23013,FixedJS,0,3071\nprior,23013,FixedJS,0,3135\nprior,23013,FixedJS,0,3199\nprior,23013,FixedJS,0,3263\nprior,23013,FixedJS,0,3327\nprior,23013,FixedJS,0,3391\nprior,23013,FixedJS,0,3455\nprior,23013,FixedJS,0,3519\nprior,23013,FixedJS,0,3583\nprior,23013,FixedJS,0,3647\nprior,23013,FixedJS,0,3711\nprior,23013,FixedJS,0,3775\nprior,23013,FixedJS,0,3839\nprior,23013,FixedJS,0,3903\nprior,23013,FixedJS,0,3967\nprior,23013,FixedJS,0,4031\nprior,23013,FixedJS,0,4095\nprior,23014,ADWIN,0,2111\nprior,23014,PageHinkley,0,2106\nprior,23014,KSWIN,0,2075\nprior,23014,DDM,0,39\nprior,23014,DDM,0,178\nprior,23014,DDM,0,261\nprior,23014,DDM,1,64\nprior,23014,DDM,1,192\nprior,23014,DDM,1,320\nprior,23014,DDM,3,192\nprior,23014,DDM,3,320\nprior,23014,DDM,3,448\nprior,23014,JS,0,319\nprior,23014,JS,0,383\nprior,23014,JS,0,447\nprior,23014,JS,0,511\nprior,23014,JS,0,575\nprior,23014,JS,0,639\nprior,23014,JS,0,703\nprior,23014,JS,0,767\nprior,23014,JS,0,831\nprior,23014,JS,0,895\nprior,23014,JS,0,959\nprior,23014,JS,0,1023\nprior,23014,JS,0,1087\nprior,23014,JS,0,1151\nprior,23014,JS,0,1215\nprior,23014,JS,0,1279\nprior,23014,JS,0,1343\nprior,23014,JS,0,1407\nprior,23014,JS,0,1471\nprior,23014,JS,0,1535\nprior,23014,JS,0,1599\nprior,23014,JS,0,1663\nprior,23014,JS,0,1727\nprior,23014,JS,0,1791\nprior,23014,JS,0,1855\nprior,23014,JS,0,1919\nprior,23014,JS,0,1983\nprior,23014,JS,0,2047\nprior,23014,JS,0,2111\nprior,23014,JS,0,2175\nprior,23014,JS,0,2239\nprior,23014,JS,0,2303\nprior,23014,JS,0,2367\nprior,23014,JS,0,2431\nprior,23014,JS,0,2495\nprior,23014,JS,0,2559\nprior,23014,JS,0,2623\nprior,23014,JS,0,2687\nprior,23014,JS,0,2751\nprior,23014,JS,0,2815\nprior,23014,JS,0,2879\nprior,23014,JS,0,2943\nprior,23014,JS,0,3007\nprior,23014,JS,0,3071\nprior,23014,JS,0,3135\nprior,23014,JS,0,3199\nprior,23014,JS,0,3263\nprior,23014,JS,0,3327\nprior,23014,JS,0,3391\nprior,23014,JS,0,3455\nprior,23014,JS,0,3519\nprior,23014,JS,0,3583\nprior,23014,JS,0,3647\nprior,23014,JS,0,3711\nprior,23014,JS,0,3775\nprior,23014,JS,0,3839\nprior,23014,JS,0,3903\nprior,23014,JS,0,3967\nprior,23014,JS,0,4031\nprior,23014,JS,0,4095\nprior,23014,PSI,0,319\nprior,23014,PSI,0,383\nprior,23014,PSI,0,447\nprior,23014,PSI,0,511\nprior,23014,PSI,0,575\nprior,23014,PSI,0,639\nprior,23014,PSI,0,703\nprior,23014,PSI,0,767\nprior,23014,PSI,0,831\nprior,23014,PSI,0,895\nprior,23014,PSI,0,959\nprior,23014,PSI,0,1023\nprior,23014,PSI,0,1087\nprior,23014,PSI,0,1151\nprior,23014,PSI,0,1215\nprior,23014,PSI,0,1279\nprior,23014,PSI,0,1343\nprior,23014,PSI,0,1407\nprior,23014,PSI,0,1471\nprior,23014,PSI,0,1535\nprior,23014,PSI,0,1599\nprior,23014,PSI,0,1663\nprior,23014,PSI,0,1727\nprior,23014,PSI,0,1791\nprior,23014,PSI,0,1855\nprior,23014,PSI,0,1919\nprior,23014,PSI,0,1983\nprior,23014,PSI,0,2047\nprior,23014,PSI,0,2111\nprior,23014,PSI,0,2175\nprior,23014,PSI,0,2239\nprior,23014,PSI,0,2303\nprior,23014,PSI,0,2367\nprior,23014,PSI,0,2431\nprior,23014,PSI,0,2495\nprior,23014,PSI,0,2559\nprior,23014,PSI,0,2623\nprior,23014,PSI,0,2687\nprior,23014,PSI,0,2751\nprior,23014,PSI,0,2815\nprior,23014,PSI,0,2879\nprior,23014,PSI,0,2943\nprior,23014,PSI,0,3007\nprior,23014,PSI,0,3071\nprior,23014,PSI,0,3135\nprior,23014,PSI,0,3199\nprior,23014,PSI,0,3263\nprior,23014,PSI,0,3327\nprior,23014,PSI,0,3391\nprior,23014,PSI,0,3455\nprior,23014,PSI,0,3519\nprior,23014,PSI,0,3583\nprior,23014,PSI,0,3647\nprior,23014,PSI,0,3711\nprior,23014,PSI,0,3775\nprior,23014,PSI,0,3839\nprior,23014,PSI,0,3903\nprior,23014,PSI,0,3967\nprior,23014,PSI,0,4031\nprior,23014,PSI,0,4095\nprior,23014,FixedJS,0,319\nprior,23014,FixedJS,0,383\nprior,23014,FixedJS,0,447\nprior,23014,FixedJS,0,511\nprior,23014,FixedJS,0,575\nprior,23014,FixedJS,0,639\nprior,23014,FixedJS,0,703\nprior,23014,FixedJS,0,767\nprior,23014,FixedJS,0,831\nprior,23014,FixedJS,0,895\nprior,23014,FixedJS,0,959\nprior,23014,FixedJS,0,1023\nprior,23014,FixedJS,0,1087\nprior,23014,FixedJS,0,1151\nprior,23014,FixedJS,0,1215\nprior,23014,FixedJS,0,1279\nprior,23014,FixedJS,0,1343\nprior,23014,FixedJS,0,1407\nprior,23014,FixedJS,0,1471\nprior,23014,FixedJS,0,1535\nprior,23014,FixedJS,0,1599\nprior,23014,FixedJS,0,1663\nprior,23014,FixedJS,0,1727\nprior,23014,FixedJS,0,1791\nprior,23014,FixedJS,0,1855\nprior,23014,FixedJS,0,1919\nprior,23014,FixedJS,0,1983\nprior,23014,FixedJS,0,2047\nprior,23014,FixedJS,0,2111\nprior,23014,FixedJS,0,2175\nprior,23014,FixedJS,0,2239\nprior,23014,FixedJS,0,2303\nprior,23014,FixedJS,0,2367\nprior,23014,FixedJS,0,2431\nprior,23014,FixedJS,0,2495\nprior,23014,FixedJS,0,2559\nprior,23014,FixedJS,0,2623\nprior,23014,FixedJS,0,2687\nprior,23014,FixedJS,0,2751\nprior,23014,FixedJS,0,2815\nprior,23014,FixedJS,0,2879\nprior,23014,FixedJS,0,2943\nprior,23014,FixedJS,0,3007\nprior,23014,FixedJS,0,3071\nprior,23014,FixedJS,0,3135\nprior,23014,FixedJS,0,3199\nprior,23014,FixedJS,0,3263\nprior,23014,FixedJS,0,3327\nprior,23014,FixedJS,0,3391\nprior,23014,FixedJS,0,3455\nprior,23014,FixedJS,0,3519\nprior,23014,FixedJS,0,3583\nprior,23014,FixedJS,0,3647\nprior,23014,FixedJS,0,3711\nprior,23014,FixedJS,0,3775\nprior,23014,FixedJS,0,3839\nprior,23014,FixedJS,0,3903\nprior,23014,FixedJS,0,3967\nprior,23014,FixedJS,0,4031\nprior,23014,FixedJS,0,4095\nprior,23015,ADWIN,0,2111\nprior,23015,PageHinkley,0,2084\nprior,23015,JS,0,319\nprior,23015,JS,0,383\nprior,23015,JS,0,447\nprior,23015,JS,0,511\nprior,23015,JS,0,575\nprior,23015,JS,0,639\nprior,23015,JS,0,703\nprior,23015,JS,0,767\nprior,23015,JS,0,831\nprior,23015,JS,0,895\nprior,23015,JS,0,959\nprior,23015,JS,0,1023\nprior,23015,JS,0,1087\nprior,23015,JS,0,1151\nprior,23015,JS,0,1215\nprior,23015,JS,0,1279\nprior,23015,JS,0,1343\nprior,23015,JS,0,1407\nprior,23015,JS,0,1471\nprior,23015,JS,0,1535\nprior,23015,JS,0,1599\nprior,23015,JS,0,1663\nprior,23015,JS,0,1727\nprior,23015,JS,0,1791\nprior,23015,JS,0,1855\nprior,23015,JS,0,1919\nprior,23015,JS,0,1983\nprior,23015,JS,0,2047\nprior,23015,JS,0,2111\nprior,23015,JS,0,2175\nprior,23015,JS,0,2239\nprior,23015,JS,0,2303\nprior,23015,JS,0,2367\nprior,23015,JS,0,2431\nprior,23015,JS,0,2495\nprior,23015,JS,0,2559\nprior,23015,JS,0,2623\nprior,23015,JS,0,2687\nprior,23015,JS,0,2751\nprior,23015,JS,0,2815\nprior,23015,JS,0,2879\nprior,23015,JS,0,2943\nprior,23015,JS,0,3007\nprior,23015,JS,0,3071\nprior,23015,JS,0,3135\nprior,23015,JS,0,3199\nprior,23015,JS,0,3263\nprior,23015,JS,0,3327\nprior,23015,JS,0,3391\nprior,23015,JS,0,3455\nprior,23015,JS,0,3519\nprior,23015,JS,0,3583\nprior,23015,JS,0,3647\nprior,23015,JS,0,3711\nprior,23015,JS,0,3775\nprior,23015,JS,0,3839\nprior,23015,JS,0,3903\nprior,23015,JS,0,3967\nprior,23015,JS,0,4031\nprior,23015,JS,0,4095\nprior,23015,PSI,0,319\nprior,23015,PSI,0,383\nprior,23015,PSI,0,447\nprior,23015,PSI,0,511\nprior,23015,PSI,0,575\nprior,23015,PSI,0,639\nprior,23015,PSI,0,703\nprior,23015,PSI,0,767\nprior,23015,PSI,0,831\nprior,23015,PSI,0,895\nprior,23015,PSI,0,959\nprior,23015,PSI,0,1023\nprior,23015,PSI,0,1087\nprior,23015,PSI,0,1151\nprior,23015,PSI,0,1215\nprior,23015,PSI,0,1279\nprior,23015,PSI,0,1343\nprior,23015,PSI,0,1407\nprior,23015,PSI,0,1471\nprior,23015,PSI,0,1535\nprior,23015,PSI,0,1599\nprior,23015,PSI,0,1663\nprior,23015,PSI,0,1727\nprior,23015,PSI,0,1791\nprior,23015,PSI,0,1855\nprior,23015,PSI,0,1919\nprior,23015,PSI,0,1983\nprior,23015,PSI,0,2047\nprior,23015,PSI,0,2111\nprior,23015,PSI,0,2175\nprior,23015,PSI,0,2239\nprior,23015,PSI,0,2303\nprior,23015,PSI,0,2367\nprior,23015,PSI,0,2431\nprior,23015,PSI,0,2495\nprior,23015,PSI,0,2559\nprior,23015,PSI,0,2623\nprior,23015,PSI,0,2687\nprior,23015,PSI,0,2751\nprior,23015,PSI,0,2815\nprior,23015,PSI,0,2879\nprior,23015,PSI,0,2943\nprior,23015,PSI,0,3007\nprior,23015,PSI,0,3071\nprior,23015,PSI,0,3135\nprior,23015,PSI,0,3199\nprior,23015,PSI,0,3263\nprior,23015,PSI,0,3327\nprior,23015,PSI,0,3391\nprior,23015,PSI,0,3455\nprior,23015,PSI,0,3519\nprior,23015,PSI,0,3583\nprior,23015,PSI,0,3647\nprior,23015,PSI,0,3711\nprior,23015,PSI,0,3775\nprior,23015,PSI,0,3839\nprior,23015,PSI,0,3903\nprior,23015,PSI,0,3967\nprior,23015,PSI,0,4031\nprior,23015,PSI,0,4095\nprior,23015,FixedJS,0,319\nprior,23015,FixedJS,0,383\nprior,23015,FixedJS,0,447\nprior,23015,FixedJS,0,511\nprior,23015,FixedJS,0,575\nprior,23015,FixedJS,0,639\nprior,23015,FixedJS,0,703\nprior,23015,FixedJS,0,767\nprior,23015,FixedJS,0,831\nprior,23015,FixedJS,0,895\nprior,23015,FixedJS,0,959\nprior,23015,FixedJS,0,1023\nprior,23015,FixedJS,0,1087\nprior,23015,FixedJS,0,1151\nprior,23015,FixedJS,0,1215\nprior,23015,FixedJS,0,1279\nprior,23015,FixedJS,0,1343\nprior,23015,FixedJS,0,1407\nprior,23015,FixedJS,0,1471\nprior,23015,FixedJS,0,1535\nprior,23015,FixedJS,0,1599\nprior,23015,FixedJS,0,1663\nprior,23015,FixedJS,0,1727\nprior,23015,FixedJS,0,1791\nprior,23015,FixedJS,0,1855\nprior,23015,FixedJS,0,1919\nprior,23015,FixedJS,0,1983\nprior,23015,FixedJS,0,2047\nprior,23015,FixedJS,0,2111\nprior,23015,FixedJS,0,2175\nprior,23015,FixedJS,0,2239\nprior,23015,FixedJS,0,2303\nprior,23015,FixedJS,0,2367\nprior,23015,FixedJS,0,2431\nprior,23015,FixedJS,0,2495\nprior,23015,FixedJS,0,2559\nprior,23015,FixedJS,0,2623\nprior,23015,FixedJS,0,2687\nprior,23015,FixedJS,0,2751\nprior,23015,FixedJS,0,2815\nprior,23015,FixedJS,0,2879\nprior,23015,FixedJS,0,2943\nprior,23015,FixedJS,0,3007\nprior,23015,FixedJS,0,3071\nprior,23015,FixedJS,0,3135\nprior,23015,FixedJS,0,3199\nprior,23015,FixedJS,0,3263\nprior,23015,FixedJS,0,3327\nprior,23015,FixedJS,0,3391\nprior,23015,FixedJS,0,3455\nprior,23015,FixedJS,0,3519\nprior,23015,FixedJS,0,3583\nprior,23015,FixedJS,0,3647\nprior,23015,FixedJS,0,3711\nprior,23015,FixedJS,0,3775\nprior,23015,FixedJS,0,3839\nprior,23015,FixedJS,0,3903\nprior,23015,FixedJS,0,3967\nprior,23015,FixedJS,0,4031\nprior,23015,FixedJS,0,4095\nmissingness,23013,ADWIN,0,2079\nmissingness,23013,PageHinkley,0,2161\nmissingness,23013,PageHinkley,0,2715\nmissingness,23013,PageHinkley,0,3371\nmissingness,23013,JS,0,319\nmissingness,23013,JS,0,383\nmissingness,23013,JS,0,447\nmissingness,23013,JS,0,511\nmissingness,23013,JS,0,575\nmissingness,23013,JS,0,639\nmissingness,23013,JS,0,703\nmissingness,23013,JS,0,767\nmissingness,23013,JS,0,831\nmissingness,23013,JS,0,895\nmissingness,23013,JS,0,959\nmissingness,23013,JS,0,1023\nmissingness,23013,JS,0,1087\nmissingness,23013,JS,0,1151\nmissingness,23013,JS,0,1215\nmissingness,23013,JS,0,1279\nmissingness,23013,JS,0,1343\nmissingness,23013,JS,0,1407\nmissingness,23013,JS,0,1471\nmissingness,23013,JS,0,1535\nmissingness,23013,JS,0,1599\nmissingness,23013,JS,0,1663\nmissingness,23013,JS,0,1727\nmissingness,23013,JS,0,1791\nmissingness,23013,JS,0,1855\nmissingness,23013,JS,0,1919\nmissingness,23013,JS,0,1983\nmissingness,23013,JS,0,2047\nmissingness,23013,JS,0,2111\nmissingness,23013,JS,0,2175\nmissingness,23013,JS,0,2239\nmissingness,23013,JS,0,2303\nmissingness,23013,JS,0,2367\nmissingness,23013,JS,0,2431\nmissingness,23013,JS,0,2495\nmissingness,23013,JS,0,2559\nmissingness,23013,JS,0,2623\nmissingness,23013,JS,0,2687\nmissingness,23013,JS,0,2751\nmissingness,23013,JS,0,2815\nmissingness,23013,JS,0,2879\nmissingness,23013,JS,0,2943\nmissingness,23013,JS,0,3007\nmissingness,23013,JS,0,3071\nmissingness,23013,JS,0,3135\nmissingness,23013,JS,0,3199\nmissingness,23013,JS,0,3263\nmissingness,23013,JS,0,3327\nmissingness,23013,JS,0,3391\nmissingness,23013,JS,0,3455\nmissingness,23013,JS,0,3519\nmissingness,23013,JS,0,3583\nmissingness,23013,JS,0,3647\nmissingness,23013,JS,0,3711\nmissingness,23013,JS,0,3775\nmissingness,23013,JS,0,3839\nmissingness,23013,JS,0,3903\nmissingness,23013,JS,0,3967\nmissingness,23013,JS,0,4031\nmissingness,23013,JS,0,4095\nmissingness,23013,PSI,0,319\nmissingness,23013,PSI,0,383\nmissingness,23013,PSI,0,447\nmissingness,23013,PSI,0,511\nmissingness,23013,PSI,0,575\nmissingness,23013,PSI,0,639\nmissingness,23013,PSI,0,703\nmissingness,23013,PSI,0,767\nmissingness,23013,PSI,0,831\nmissingness,23013,PSI,0,895\nmissingness,23013,PSI,0,959\nmissingness,23013,PSI,0,1023\nmissingness,23013,PSI,0,1087\nmissingness,23013,PSI,0,1151\nmissingness,23013,PSI,0,1215\nmissingness,23013,PSI,0,1279\nmissingness,23013,PSI,0,1343\nmissingness,23013,PSI,0,1407\nmissingness,23013,PSI,0,1471\nmissingness,23013,PSI,0,1535\nmissingness,23013,PSI,0,1599\nmissingness,23013,PSI,0,1663\nmissingness,23013,PSI,0,1727\nmissingness,23013,PSI,0,1791\nmissingness,23013,PSI,0,1855\nmissingness,23013,PSI,0,1919\nmissingness,23013,PSI,0,1983\nmissingness,23013,PSI,0,2047\nmissingness,23013,PSI,0,2111\nmissingness,23013,PSI,0,2175\nmissingness,23013,PSI,0,2239\nmissingness,23013,PSI,0,2303\nmissingness,23013,PSI,0,2367\nmissingness,23013,PSI,0,2431\nmissingness,23013,PSI,0,2495\nmissingness,23013,PSI,0,2559\nmissingness,23013,PSI,0,2623\nmissingness,23013,PSI,0,2687\nmissingness,23013,PSI,0,2751\nmissingness,23013,PSI,0,2815\nmissingness,23013,PSI,0,2879\nmissingness,23013,PSI,0,2943\nmissingness,23013,PSI,0,3007\nmissingness,23013,PSI,0,3071\nmissingness,23013,PSI,0,3135\nmissingness,23013,PSI,0,3199\nmissingness,23013,PSI,0,3263\nmissingness,23013,PSI,0,3327\nmissingness,23013,PSI,0,3391\nmissingness,23013,PSI,0,3455\nmissingness,23013,PSI,0,3519\nmissingness,23013,PSI,0,3583\nmissingness,23013,PSI,0,3647\nmissingness,23013,PSI,0,3711\nmissingness,23013,PSI,0,3775\nmissingness,23013,PSI,0,3839\nmissingness,23013,PSI,0,3903\nmissingness,23013,PSI,0,3967\nmissingness,23013,PSI,0,4031\nmissingness,23013,PSI,0,4095\nmissingness,23013,FixedJS,0,319\nmissingness,23013,FixedJS,0,383\nmissingness,23013,FixedJS,0,447\nmissingness,23013,FixedJS,0,511\nmissingness,23013,FixedJS,0,575\nmissingness,23013,FixedJS,0,639\nmissingness,23013,FixedJS,0,703\nmissingness,23013,FixedJS,0,767\nmissingness,23013,FixedJS,0,831\nmissingness,23013,FixedJS,0,895\nmissingness,23013,FixedJS,0,959\nmissingness,23013,FixedJS,0,1023\nmissingness,23013,FixedJS,0,1087\nmissingness,23013,FixedJS,0,1151\nmissingness,23013,FixedJS,0,1215\nmissingness,23013,FixedJS,0,1279\nmissingness,23013,FixedJS,0,1343\nmissingness,23013,FixedJS,0,1407\nmissingness,23013,FixedJS,0,1471\nmissingness,23013,FixedJS,0,1535\nmissingness,23013,FixedJS,0,1599\nmissingness,23013,FixedJS,0,1663\nmissingness,23013,FixedJS,0,1727\nmissingness,23013,FixedJS,0,1791\nmissingness,23013,FixedJS,0,1855\nmissingness,23013,FixedJS,0,1919\nmissingness,23013,FixedJS,0,1983\nmissingness,23013,FixedJS,0,2047\nmissingness,23013,FixedJS,0,2111\nmissingness,23013,FixedJS,0,2175\nmissingness,23013,FixedJS,0,2239\nmissingness,23013,FixedJS,0,2303\nmissingness,23013,FixedJS,0,2367\nmissingness,23013,FixedJS,0,2431\nmissingness,23013,FixedJS,0,2495\nmissingness,23013,FixedJS,0,2559\nmissingness,23013,FixedJS,0,2623\nmissingness,23013,FixedJS,0,2687\nmissingness,23013,FixedJS,0,2751\nmissingness,23013,FixedJS,0,2815\nmissingness,23013,FixedJS,0,2879\nmissingness,23013,FixedJS,0,2943\nmissingness,23013,FixedJS,0,3007\nmissingness,23013,FixedJS,0,3071\nmissingness,23013,FixedJS,0,3135\nmissingness,23013,FixedJS,0,3199\nmissingness,23013,FixedJS,0,3263\nmissingness,23013,FixedJS,0,3327\nmissingness,23013,FixedJS,0,3391\nmissingness,23013,FixedJS,0,3455\nmissingness,23013,FixedJS,0,3519\nmissingness,23013,FixedJS,0,3583\nmissingness,23013,FixedJS,0,3647\nmissingness,23013,FixedJS,0,3711\nmissingness,23013,FixedJS,0,3775\nmissingness,23013,FixedJS,0,3839\nmissingness,23013,FixedJS,0,3903\nmissingness,23013,FixedJS,0,3967\nmissingness,23013,FixedJS,0,4031\nmissingness,23013,FixedJS,0,4095\nmissingness,23014,ADWIN,0,2111\nmissingness,23014,PageHinkley,0,2166\nmissingness,23014,DDM,0,810\nmissingness,23014,DDM,0,860\nmissingness,23014,DDM,1,832\nmissingness,23014,DDM,1,896\nmissingness,23014,DDM,3,960\nmissingness,23014,DDM,3,1024\nmissingness,23014,JS,0,319\nmissingness,23014,JS,0,383\nmissingness,23014,JS,0,447\nmissingness,23014,JS,0,511\nmissingness,23014,JS,0,575\nmissingness,23014,JS,0,639\nmissingness,23014,JS,0,703\nmissingness,23014,JS,0,767\nmissingness,23014,JS,0,831\nmissingness,23014,JS,0,895\nmissingness,23014,JS,0,959\nmissingness,23014,JS,0,1023\nmissingness,23014,JS,0,1087\nmissingness,23014,JS,0,1151\nmissingness,23014,JS,0,1215\nmissingness,23014,JS,0,1279\nmissingness,23014,JS,0,1343\nmissingness,23014,JS,0,1407\nmissingness,23014,JS,0,1471\nmissingness,23014,JS,0,1535\nmissingness,23014,JS,0,1599\nmissingness,23014,JS,0,1663\nmissingness,23014,JS,0,1727\nmissingness,23014,JS,0,1791\nmissingness,23014,JS,0,1855\nmissingness,23014,JS,0,1919\nmissingness,23014,JS,0,1983\nmissingness,23014,JS,0,2047\nmissingness,23014,JS,0,2111\nmissingness,23014,JS,0,2175\nmissingness,23014,JS,0,2239\nmissingness,23014,JS,0,2303\nmissingness,23014,JS,0,2367\nmissingness,23014,JS,0,2431\nmissingness,23014,JS,0,2495\nmissingness,23014,JS,0,2559\nmissingness,23014,JS,0,2623\nmissingness,23014,JS,0,2687\nmissingness,23014,JS,0,2751\nmissingness,23014,JS,0,2815\nmissingness,23014,JS,0,2879\nmissingness,23014,JS,0,2943\nmissingness,23014,JS,0,3007\nmissingness,23014,JS,0,3071\nmissingness,23014,JS,0,3135\nmissingness,23014,JS,0,3199\nmissingness,23014,JS,0,3263\nmissingness,23014,JS,0,3327\nmissingness,23014,JS,0,3391\nmissingness,23014,JS,0,3455\nmissingness,23014,JS,0,3519\nmissingness,23014,JS,0,3583\nmissingness,23014,JS,0,3647\nmissingness,23014,JS,0,3711\nmissingness,23014,JS,0,3775\nmissingness,23014,JS,0,3839\nmissingness,23014,JS,0,3903\nmissingness,23014,JS,0,3967\nmissingness,23014,JS,0,4031\nmissingness,23014,JS,0,4095\nmissingness,23014,PSI,0,319\nmissingness,23014,PSI,0,383\nmissingness,23014,PSI,0,447\nmissingness,23014,PSI,0,511\nmissingness,23014,PSI,0,575\nmissingness,23014,PSI,0,639\nmissingness,23014,PSI,0,703\nmissingness,23014,PSI,0,767\nmissingness,23014,PSI,0,831\nmissingness,23014,PSI,0,895\nmissingness,23014,PSI,0,959\nmissingness,23014,PSI,0,1023\nmissingness,23014,PSI,0,1087\nmissingness,23014,PSI,0,1151\nmissingness,23014,PSI,0,1215\nmissingness,23014,PSI,0,1279\nmissingness,23014,PSI,0,1343\nmissingness,23014,PSI,0,1407\nmissingness,23014,PSI,0,1471\nmissingness,23014,PSI,0,1535\nmissingness,23014,PSI,0,1599\nmissingness,23014,PSI,0,1663\nmissingness,23014,PSI,0,1727\nmissingness,23014,PSI,0,1791\nmissingness,23014,PSI,0,1855\nmissingness,23014,PSI,0,1919\nmissingness,23014,PSI,0,1983\nmissingness,23014,PSI,0,2047\nmissingness,23014,PSI,0,2111\nmissingness,23014,PSI,0,2175\nmissingness,23014,PSI,0,2239\nmissingness,23014,PSI,0,2303\nmissingness,23014,PSI,0,2367\nmissingness,23014,PSI,0,2431\nmissingness,23014,PSI,0,2495\nmissingness,23014,PSI,0,2559\nmissingness,23014,PSI,0,2623\nmissingness,23014,PSI,0,2687\nmissingness,23014,PSI,0,2751\nmissingness,23014,PSI,0,2815\nmissingness,23014,PSI,0,2879\nmissingness,23014,PSI,0,2943\nmissingness,23014,PSI,0,3007\nmissingness,23014,PSI,0,3071\nmissingness,23014,PSI,0,3135\nmissingness,23014,PSI,0,3199\nmissingness,23014,PSI,0,3263\nmissingness,23014,PSI,0,3327\nmissingness,23014,PSI,0,3391\nmissingness,23014,PSI,0,3455\nmissingness,23014,PSI,0,3519\nmissingness,23014,PSI,0,3583\nmissingness,23014,PSI,0,3647\nmissingness,23014,PSI,0,3711\nmissingness,23014,PSI,0,3775\nmissingness,23014,PSI,0,3839\nmissingness,23014,PSI,0,3903\nmissingness,23014,PSI,0,3967\nmissingness,23014,PSI,0,4031\nmissingness,23014,PSI,0,4095\nmissingness,23014,FixedJS,0,319\nmissingness,23014,FixedJS,0,383\nmissingness,23014,FixedJS,0,447\nmissingness,23014,FixedJS,0,511\nmissingness,23014,FixedJS,0,575\nmissingness,23014,FixedJS,0,639\nmissingness,23014,FixedJS,0,703\nmissingness,23014,FixedJS,0,767\nmissingness,23014,FixedJS,0,831\nmissingness,23014,FixedJS,0,895\nmissingness,23014,FixedJS,0,959\nmissingness,23014,FixedJS,0,1023\nmissingness,23014,FixedJS,0,1087\nmissingness,23014,FixedJS,0,1151\nmissingness,23014,FixedJS,0,1215\nmissingness,23014,FixedJS,0,1279\nmissingness,23014,FixedJS,0,1343\nmissingness,23014,FixedJS,0,1407\nmissingness,23014,FixedJS,0,1471\nmissingness,23014,FixedJS,0,1535\nmissingness,23014,FixedJS,0,1599\nmissingness,23014,FixedJS,0,1663\nmissingness,23014,FixedJS,0,1727\nmissingness,23014,FixedJS,0,1791\nmissingness,23014,FixedJS,0,1855\nmissingness,23014,FixedJS,0,1919\nmissingness,23014,FixedJS,0,1983\nmissingness,23014,FixedJS,0,2047\nmissingness,23014,FixedJS,0,2111\nmissingness,23014,FixedJS,0,2175\nmissingness,23014,FixedJS,0,2239\nmissingness,23014,FixedJS,0,2303\nmissingness,23014,FixedJS,0,2367\nmissingness,23014,FixedJS,0,2431\nmissingness,23014,FixedJS,0,2495\nmissingness,23014,FixedJS,0,2559\nmissingness,23014,FixedJS,0,2623\nmissingness,23014,FixedJS,0,2687\nmissingness,23014,FixedJS,0,2751\nmissingness,23014,FixedJS,0,2815\nmissingness,23014,FixedJS,0,2879\nmissingness,23014,FixedJS,0,2943\nmissingness,23014,FixedJS,0,3007\nmissingness,23014,FixedJS,0,3071\nmissingness,23014,FixedJS,0,3135\nmissingness,23014,FixedJS,0,3199\nmissingness,23014,FixedJS,0,3263\nmissingness,23014,FixedJS,0,3327\nmissingness,23014,FixedJS,0,3391\nmissingness,23014,FixedJS,0,3455\nmissingness,23014,FixedJS,0,3519\nmissingness,23014,FixedJS,0,3583\nmissingness,23014,FixedJS,0,3647\nmissingness,23014,FixedJS,0,3711\nmissingness,23014,FixedJS,0,3775\nmissingness,23014,FixedJS,0,3839\nmissingness,23014,FixedJS,0,3903\nmissingness,23014,FixedJS,0,3967\nmissingness,23014,FixedJS,0,4031\nmissingness,23014,FixedJS,0,4095\nmissingness,23015,ADWIN,0,2111\nmissingness,23015,PageHinkley,0,2157\nmissingness,23015,PageHinkley,0,3404\nmissingness,23015,JS,0,319\nmissingness,23015,JS,0,383\nmissingness,23015,JS,0,447\nmissingness,23015,JS,0,511\nmissingness,23015,JS,0,575\nmissingness,23015,JS,0,639\nmissingness,23015,JS,0,703\nmissingness,23015,JS,0,767\nmissingness,23015,JS,0,831\nmissingness,23015,JS,0,895\nmissingness,23015,JS,0,959\nmissingness,23015,JS,0,1023\nmissingness,23015,JS,0,1087\nmissingness,23015,JS,0,1151\nmissingness,23015,JS,0,1215\nmissingness,23015,JS,0,1279\nmissingness,23015,JS,0,1343\nmissingness,23015,JS,0,1407\nmissingness,23015,JS,0,1471\nmissingness,23015,JS,0,1535\nmissingness,23015,JS,0,1599\nmissingness,23015,JS,0,1663\nmissingness,23015,JS,0,1727\nmissingness,23015,JS,0,1791\nmissingness,23015,JS,0,1855\nmissingness,23015,JS,0,1919\nmissingness,23015,JS,0,1983\nmissingness,23015,JS,0,2047\nmissingness,23015,JS,0,2111\nmissingness,23015,JS,0,2175\nmissingness,23015,JS,0,2239\nmissingness,23015,JS,0,2303\nmissingness,23015,JS,0,2367\nmissingness,23015,JS,0,2431\nmissingness,23015,JS,0,2495\nmissingness,23015,JS,0,2559\nmissingness,23015,JS,0,2623\nmissingness,23015,JS,0,2687\nmissingness,23015,JS,0,2751\nmissingness,23015,JS,0,2815\nmissingness,23015,JS,0,2879\nmissingness,23015,JS,0,2943\nmissingness,23015,JS,0,3007\nmissingness,23015,JS,0,3071\nmissingness,23015,JS,0,3135\nmissingness,23015,JS,0,3199\nmissingness,23015,JS,0,3263\nmissingness,23015,JS,0,3327\nmissingness,23015,JS,0,3391\nmissingness,23015,JS,0,3455\nmissingness,23015,JS,0,3519\nmissingness,23015,JS,0,3583\nmissingness,23015,JS,0,3647\nmissingness,23015,JS,0,3711\nmissingness,23015,JS,0,3775\nmissingness,23015,JS,0,3839\nmissingness,23015,JS,0,3903\nmissingness,23015,JS,0,3967\nmissingness,23015,JS,0,4031\nmissingness,23015,JS,0,4095\nmissingness,23015,PSI,0,319\nmissingness,23015,PSI,0,383\nmissingness,23015,PSI,0,447\nmissingness,23015,PSI,0,511\nmissingness,23015,PSI,0,575\nmissingness,23015,PSI,0,639\nmissingness,23015,PSI,0,703\nmissingness,23015,PSI,0,767\nmissingness,23015,PSI,0,831\nmissingness,23015,PSI,0,895\nmissingness,23015,PSI,0,959\nmissingness,23015,PSI,0,1023\nmissingness,23015,PSI,0,1087\nmissingness,23015,PSI,0,1151\nmissingness,23015,PSI,0,1215\nmissingness,23015,PSI,0,1279\nmissingness,23015,PSI,0,1343\nmissingness,23015,PSI,0,1407\nmissingness,23015,PSI,0,1471\nmissingness,23015,PSI,0,1535\nmissingness,23015,PSI,0,1599\nmissingness,23015,PSI,0,1663\nmissingness,23015,PSI,0,1727\nmissingness,23015,PSI,0,1791\nmissingness,23015,PSI,0,1855\nmissingness,23015,PSI,0,1919\nmissingness,23015,PSI,0,1983\nmissingness,23015,PSI,0,2047\nmissingness,23015,PSI,0,2111\nmissingness,23015,PSI,0,2175\nmissingness,23015,PSI,0,2239\nmissingness,23015,PSI,0,2303\nmissingness,23015,PSI,0,2367\nmissingness,23015,PSI,0,2431\nmissingness,23015,PSI,0,2495\nmissingness,23015,PSI,0,2559\nmissingness,23015,PSI,0,2623\nmissingness,23015,PSI,0,2687\nmissingness,23015,PSI,0,2751\nmissingness,23015,PSI,0,2815\nmissingness,23015,PSI,0,2879\nmissingness,23015,PSI,0,2943\nmissingness,23015,PSI,0,3007\nmissingness,23015,PSI,0,3071\nmissingness,23015,PSI,0,3135\nmissingness,23015,PSI,0,3199\nmissingness,23015,PSI,0,3263\nmissingness,23015,PSI,0,3327\nmissingness,23015,PSI,0,3391\nmissingness,23015,PSI,0,3455\nmissingness,23015,PSI,0,3519\nmissingness,23015,PSI,0,3583\nmissingness,23015,PSI,0,3647\nmissingness,23015,PSI,0,3711\nmissingness,23015,PSI,0,3775\nmissingness,23015,PSI,0,3839\nmissingness,23015,PSI,0,3903\nmissingness,23015,PSI,0,3967\nmissingness,23015,PSI,0,4031\nmissingness,23015,PSI,0,4095\nmissingness,23015,FixedJS,0,319\nmissingness,23015,FixedJS,0,383\nmissingness,23015,FixedJS,0,447\nmissingness,23015,FixedJS,0,511\nmissingness,23015,FixedJS,0,575\nmissingness,23015,FixedJS,0,639\nmissingness,23015,FixedJS,0,703\nmissingness,23015,FixedJS,0,767\nmissingness,23015,FixedJS,0,831\nmissingness,23015,FixedJS,0,895\nmissingness,23015,FixedJS,0,959\nmissingness,23015,FixedJS,0,1023\nmissingness,23015,FixedJS,0,1087\nmissingness,23015,FixedJS,0,1151\nmissingness,23015,FixedJS,0,1215\nmissingness,23015,FixedJS,0,1279\nmissingness,23015,FixedJS,0,1343\nmissingness,23015,FixedJS,0,1407\nmissingness,23015,FixedJS,0,1471\nmissingness,23015,FixedJS,0,1535\nmissingness,23015,FixedJS,0,1599\nmissingness,23015,FixedJS,0,1663\nmissingness,23015,FixedJS,0,1727\nmissingness,23015,FixedJS,0,1791\nmissingness,23015,FixedJS,0,1855\nmissingness,23015,FixedJS,0,1919\nmissingness,23015,FixedJS,0,1983\nmissingness,23015,FixedJS,0,2047\nmissingness,23015,FixedJS,0,2111\nmissingness,23015,FixedJS,0,2175\nmissingness,23015,FixedJS,0,2239\nmissingness,23015,FixedJS,0,2303\nmissingness,23015,FixedJS,0,2367\nmissingness,23015,FixedJS,0,2431\nmissingness,23015,FixedJS,0,2495\nmissingness,23015,FixedJS,0,2559\nmissingness,23015,FixedJS,0,2623\nmissingness,23015,FixedJS,0,2687\nmissingness,23015,FixedJS,0,2751\nmissingness,23015,FixedJS,0,2815\nmissingness,23015,FixedJS,0,2879\nmissingness,23015,FixedJS,0,2943\nmissingness,23015,FixedJS,0,3007\nmissingness,23015,FixedJS,0,3071\nmissingness,23015,FixedJS,0,3135\nmissingness,23015,FixedJS,0,3199\nmissingness,23015,FixedJS,0,3263\nmissingness,23015,FixedJS,0,3327\nmissingness,23015,FixedJS,0,3391\nmissingness,23015,FixedJS,0,3455\nmissingness,23015,FixedJS,0,3519\nmissingness,23015,FixedJS,0,3583\nmissingness,23015,FixedJS,0,3647\nmissingness,23015,FixedJS,0,3711\nmissingness,23015,FixedJS,0,3775\nmissingness,23015,FixedJS,0,3839\nmissingness,23015,FixedJS,0,3903\nmissingness,23015,FixedJS,0,3967\nmissingness,23015,FixedJS,0,4031\nmissingness,23015,FixedJS,0,4095\nerror,23013,KSWIN,0,1209\nerror,23013,DDM,0,2095\nerror,23013,DDM,1,2112\nerror,23013,DDM,3,2240\nerror,23013,JS,0,767\nerror,23013,PSI,0,767\nerror,23013,PSI,0,3327\nerror,23013,PSI,0,3711\nerror,23014,KSWIN,0,2195\nerror,23014,KSWIN,0,2943\nerror,23014,DDM,0,39\nerror,23014,DDM,0,178\nerror,23014,DDM,0,261\nerror,23014,DDM,0,2095\nerror,23014,DDM,1,64\nerror,23014,DDM,1,192\nerror,23014,DDM,1,320\nerror,23014,DDM,1,2112\nerror,23014,DDM,3,192\nerror,23014,DDM,3,320\nerror,23014,DDM,3,448\nerror,23014,DDM,3,2240\nerror,23015,KSWIN,0,1305\nerror,23015,KSWIN,0,1825\nerror,23015,KSWIN,0,2789\nerror,23015,KSWIN,0,3640\nerror,23015,DDM,0,2110\nerror,23015,DDM,1,2112\nerror,23015,DDM,3,2240\nconcept,23013,DDM,0,2048\nconcept,23013,DDM,1,2112\nconcept,23013,DDM,3,2240\nconcept,23014,KSWIN,0,499\nconcept,23014,KSWIN,0,2432\nconcept,23014,DDM,0,2048\nconcept,23014,DDM,1,2112\nconcept,23014,DDM,3,2240\nconcept,23015,KSWIN,0,3374\nconcept,23015,DDM,0,2048\nconcept,23015,DDM,1,2112\nconcept,23015,DDM,3,2240\nconcept,23015,JS,0,511\nconcept,23015,JS,0,1087\nconcept,23015,PSI,0,511\nconcept,23015,PSI,0,1087\n', 'reports/phase11_synthetic/frozen.json': '{\n  "ADWIN": "sensitive",\n  "PageHinkley": "conservative",\n  "KSWIN": "conservative",\n  "DDM": "conservative",\n  "JS": 0.07629310991179271,\n  "PSI": 1.682505110410195,\n  "reference_bins": [\n    0.3318130698456872,\n    0.38945933509409947,\n    0.43725219719204383,\n    0.4740768943641612,\n    0.5053532389874869,\n    0.5292868544565196,\n    0.5609761400437144,\n    0.6071473846817884,\n    0.6573034277567646\n  ],\n  "reference_counts": [\n    26,\n    26,\n    25,\n    26,\n    26,\n    25,\n    26,\n    25,\n    26,\n    25\n  ],\n  "window_size": 64\n}\n'}
for name, content in sources.items():
 target=project/name
 target.parent.mkdir(parents=True,exist_ok=True)
 target.write_text(content)


In [ ]:
from src.drift.historical_monitoring import run as history
from src.drift.robustness import run as robustness
history(project, Path('/kaggle/working/mmds_phase11_historical'))
robustness(project/'reports/phase11_synthetic',Path('/kaggle/working/mmds_phase11_robustness'))
